# Exact stopping for residual calibration — optional draft

Frozen logistic-C10; uniform native/color profile, preserved first attempt. Skip omitted-demo folds only when every query has a single candidate. Public development, no hosted or private result.


In [ ]:
from pathlib import Path
import json,subprocess,sys
bundle=Path('arc_portfolio_bundle')
bundle.mkdir(exist_ok=True)
sources={'scripts/solver.py': '"""Frozen, pure unary grid DSL and query-aware breadth-first synthesis.\n\nSPDX-License-Identifier: MIT-0\nInference consumes demonstration outputs and query inputs, never query labels.\nAll grid states must have dimensions 1..30. Undefined operations are absorbing.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom functools import lru_cache\nfrom typing import NamedTuple\nimport time\nimport numpy as np\n\n\ndef pack(a):\n    return bytes(a.shape) + a.astype(np.uint8, copy=False).tobytes()\n\n\ndef unpack(s):\n    return np.frombuffer(s[2:], dtype=np.uint8).reshape(s[0], s[1])\n\n\ndef background(a):\n    return int(np.bincount(a.ravel(), minlength=10).argmax())\n\n\ndef bbox(a, mask):\n    yy, xx = np.nonzero(mask)\n    if not len(yy):\n        return None\n    return a[yy.min():yy.max()+1, xx.min():xx.max()+1]\n\n\n@lru_cache(maxsize=128)\ndef components(s, bg, same_color):\n    a = unpack(s)\n    remaining = set(zip(*np.nonzero(a != bg)))\n    out = []\n    while remaining:\n        start = min(remaining)\n        remaining.remove(start)\n        stack, cells = [start], [start]\n        while stack:\n            y, x = stack.pop()\n            for q in ((y-1,x),(y+1,x),(y,x-1),(y,x+1)):\n                if q in remaining and (not same_color or a[q] == a[start]):\n                    remaining.remove(q)\n                    stack.append(q)\n                    cells.append(q)\n        out.append(tuple(sorted(cells)))\n    return tuple(sorted(out, key=lambda c: (-len(c), c)))\n\n\ndef select_object(a, bg, same, smallest, crop):\n    cs = components(pack(a), bg, same)\n    if not cs:\n        return None\n    c = min(cs, key=lambda c: (len(c), c)) if smallest else cs[0]\n    out = np.full_like(a, bg)\n    for pos in c:\n        out[pos] = a[pos]\n    if crop:\n        yy, xx = zip(*c)\n        out = out[min(yy):max(yy)+1, min(xx):max(xx)+1]\n    return out\n\n\ndef reduce_blocks(a, k):\n    h, w = a.shape\n    if h % k or w % k:\n        return None\n    blocks = a.reshape(h//k, k, w//k, k)\n    out = blocks[:, 0, :, 0]\n    if not np.all(blocks == out[:, None, :, None]):\n        return None\n    return out\n\n\ndef compress(a, axis):\n    if axis == 0:\n        keep = np.r_[True, np.any(a[1:] != a[:-1], axis=1)]\n        return a[keep]\n    return compress(a.T, 0).T\n\n\ndef remove_lines(a):\n    rows = ~np.all(a == a[:, :1], axis=1)\n    cols = ~np.all(a == a[:1, :], axis=0)\n    return a[rows][:, cols]\n\n\ndef half(a, axis, side, gap):\n    n = a.shape[axis]\n    if (n - gap) % 2:\n        return None\n    k = (n-gap)//2\n    if axis == 0:\n        return a[:k] if side == 0 else a[k+gap:]\n    return a[:, :k] if side == 0 else a[:, k+gap:]\n\n\nclass Operation(NamedTuple):\n    name: str\n    fn: object\n\n\ndef operations(train):\n    """Fixed task grammar. Parameters depend only on demonstration inputs/labels."""\n    ops = []\n    def add(name, fn):\n        ops.append(Operation(name, fn))\n    for k in (1,2,3):\n        add(f\'rotate{k*90}\', lambda a,k=k: np.rot90(a,k))\n    add(\'flip_lr\', np.fliplr)\n    add(\'flip_ud\', np.flipud)\n    add(\'transpose\', lambda a:a.T)\n    add(\'anti_transpose\', lambda a:np.rot90(a.T,2))\n    add(\'crop_nonzero\', lambda a:bbox(a,a!=0))\n    add(\'crop_nonbackground\', lambda a:bbox(a,a!=background(a)))\n    add(\'compress_rows\', lambda a:compress(a,0))\n    add(\'compress_columns\', lambda a:compress(a,1))\n    add(\'remove_uniform_lines\', remove_lines)\n    for k in (2,3):\n        add(f\'upscale{k}\',lambda a,k=k:np.repeat(np.repeat(a,k,0),k,1))\n        add(f\'downscale{k}\',lambda a,k=k:reduce_blocks(a,k))\n        add(f\'tile{k}x{k}\',lambda a,k=k:np.tile(a,(k,k)))\n    for axis in (0,1):\n        add(f\'repeat_axis{axis}\',lambda a,axis=axis:np.concatenate((a,a),axis=axis))\n        add(f\'mirror_axis{axis}\',lambda a,axis=axis:np.concatenate((a,np.flip(a,axis)),axis=axis))\n        for side in (0,1):\n            for gap in (0,1):\n                add(f\'half{axis}_{side}_gap{gap}\',lambda a,axis=axis,side=side,gap=gap:half(a,axis,side,gap))\n    colors = sorted({c for t in train for row in t[\'input\'] for c in row})\n    for c in colors:\n        add(f\'crop_color{c}\',lambda a,c=c:bbox(a,a==c))\n        add(f\'keep_color{c}\',lambda a,c=c:np.where(a==c,a,0))\n        add(f\'erase_color{c}\',lambda a,c=c:np.where(a==c,0,a))\n    for bgmode in (\'zero\',\'mode\'):\n        for same in (False,True):\n            for small in (False,True):\n                for crop in (False,True):\n                    add(f\'object_{bgmode}_same{int(same)}_small{int(small)}_crop{int(crop)}\',\n                        lambda a,bgmode=bgmode,same=same,small=small,crop=crop:\n                        select_object(a,0 if bgmode==\'zero\' else background(a),same,small,crop))\n    # Infer a single consistent color map from demonstrations of equal shape.\n    mapping, consistent = {}, True\n    for pair in train:\n        a,b = np.array(pair[\'input\']),np.array(pair[\'output\'])\n        if a.shape != b.shape:\n            consistent = False\n            break\n        for x,y in zip(a.ravel(),b.ravel()):\n            x,y = int(x),int(y)\n            if x in mapping and mapping[x] != y:\n                consistent = False\n                break\n            mapping[x] = y\n    if consistent:\n        lut = np.array([mapping.get(c,c) for c in range(10)],dtype=np.uint8)\n        add(\'demonstration_color_map\',lambda a,lut=lut:lut[a])\n    return ops\n\n\ndef apply(op, values):\n    out = []\n    for s in values:\n        try:\n            a = op.fn(unpack(s))\n        except (ValueError,IndexError):\n            return None\n        if a is None or a.ndim != 2 or min(a.shape)<1 or max(a.shape)>30:\n            return None\n        out.append(pack(a))\n    return tuple(out)\n\n\ndef search(task, mode=\'query\', depth=2, max_states=0, ops_override=None):\n    """Collect goals *before* merging, preserving final-step alternatives in both arms.\n\n    max_states=0 is exhaustive quotient search. A positive cap is approximate.\n    Grammar state is only remaining depth, and BFS stores minimum-depth states.\n    """\n    started = time.perf_counter()\n    train, queries = task[\'train\'], task[\'test\']\n    n = len(train)\n    initial = tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in train+queries)\n    goal = tuple(pack(np.array(p[\'output\'],dtype=np.uint8)) for p in train)\n    ops = operations(train) if ops_override is None else ops_override\n    signature = (lambda v:v[:n]) if mode==\'train\' else (lambda v:v)\n    seen = {signature(initial): (initial, ())}\n    frontier = [(initial,())]\n    solutions, witnesses = {}, []\n    metrics = Counter()\n    def collect(v,path):\n        if v[:n] == goal:\n            predictions = v[n:]\n            if predictions not in solutions:\n                solutions[predictions] = path\n    collect(initial,())\n    layers=[]\n    for d in range(1,depth+1):\n        next_frontier=[]\n        for values,path in frontier:\n            for i,op in enumerate(ops):\n                metrics[\'expansions\'] += 1\n                v = apply(op,values)\n                if v is None:\n                    metrics[\'undefined\'] += 1\n                    continue\n                p = path+(i,)\n                collect(v,p)\n                sig = signature(v)\n                if sig in seen:\n                    metrics[\'merged\'] += 1\n                    old,oldpath=seen[sig]\n                    if old[n:] != v[n:]:\n                        metrics[\'query_divergent_merges\'] += 1\n                        if len(witnesses)<3:\n                            witnesses.append({\'kept\':[ops[j].name for j in oldpath],\n                                              \'discarded\':[ops[j].name for j in p],\n                                              \'at_depth\':d})\n                    continue\n                # Final-depth states need not be stored or deduplicated: all goals\n                # have already been collected. Avoid quadratic memory for evidence.\n                if d < depth:\n                    if max_states and len(next_frontier)>=max_states:\n                        metrics[\'capped\'] += 1\n                        continue\n                    seen[sig]=(v,p)\n                    next_frontier.append((v,p))\n        layers.append({\'depth\':d,\'frontier\':len(frontier),\'next\':len(next_frontier)})\n        frontier=next_frontier\n    # Deterministic shortest-program, then grammar-order tie break. No labels used.\n    candidates=sorted(solutions.items(),key=lambda kv:(len(kv[1]),kv[1],kv[0]))\n    out=[]\n    for pred,path in candidates:\n        out.append({\'predictions\':[unpack(s).tolist() for s in pred],\n                    \'program\':[ops[i].name for i in path]})\n    return {\'candidates\':out,\'metrics\':dict(metrics),\'layers\':layers,\n            \'merge_witnesses\':witnesses,\'num_operations\':len(ops),\n            \'seconds\':time.perf_counter()-started,\'mode\':mode,\'depth\':depth,\n            \'max_states\':max_states}\n\n\ndef solve(task, mode=\'query\', depth=2, max_states=0):\n    """Kaggle-style two attempts per query, deduplicated independently per input."""\n    result=search(task,mode,depth,max_states)\n    answer=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for candidate in result[\'candidates\']:\n            grid=candidate[\'predictions\'][i]\n            if grid not in grids:\n                grids.append(grid)\n            if len(grids)==2:\n                break\n        while len(grids)<2:\n            grids.append(q[\'input\'])\n        answer.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return answer,result\n', 'scripts/solver_extended.py': '"""Structural search with a demonstration-fitted terminal color map.\n\nSPDX-License-Identifier: MIT-0\nThis is a separately versioned grammar extension. Structural-prefix equivalence\nstill preserves query outputs; terminal map fitting is a deterministic function\nof the complete demonstration state vector. Unknown colors map to themselves.\n"""\nfrom collections import Counter\nimport time\nimport numpy as np\nfrom solver import operations,pack,unpack,apply\n\n\ndef fit_map(values,goal):\n    if any(v[:2] != g[:2] for v,g in zip(values,goal)):\n        return None\n    lut=np.arange(10,dtype=np.uint8)\n    known={}\n    for v,g in zip(values,goal):\n        a=np.frombuffer(v[2:],dtype=np.uint8)\n        b=np.frombuffer(g[2:],dtype=np.uint8)\n        pairs=np.unique(a.astype(np.uint16)*10+b)\n        for pair in pairs:\n            x,y=divmod(int(pair),10)\n            if x in known and known[x]!=y:return None\n            known[x]=y;lut[x]=y\n    return lut\n\n\ndef search(task,mode=\'query\',depth=3,max_states=0,ops_override=None):\n    start=time.perf_counter();train=task[\'train\'];n=len(train)\n    initial=tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in train+task[\'test\'])\n    goal=tuple(pack(np.array(p[\'output\'],dtype=np.uint8)) for p in train)\n    ops=operations(train) if ops_override is None else ops_override\n    signature=(lambda v:v[:n]) if mode==\'train\' else (lambda v:v)\n    seen={signature(initial):(initial,())};frontier=[(initial,())]\n    solutions={};fit_cache={};metrics=Counter();witnesses=[];layers=[]\n    def collect(v,path):\n        if v[:n]==goal:\n            predictions=v[n:];names=[ops[i].name for i in path]\n            rank=(len(path),path,0)\n        else:\n            if len(path)>=depth:return\n            key=v[:n]\n            if key not in fit_cache:fit_cache[key]=fit_map(key,goal)\n            lut=fit_cache[key]\n            if lut is None:return\n            predictions=tuple(pack(lut[unpack(s)]) for s in v[n:])\n            names=[ops[i].name for i in path]+[\'fit_color_map:\'+\',\'.join(map(str,lut.tolist()))]\n            rank=(len(path)+1,path,1)\n        if predictions not in solutions or rank<solutions[predictions][0]:\n            solutions[predictions]=(rank,names)\n    collect(initial,())\n    for d in range(1,depth+1):\n        next_frontier=[]\n        for values,path in frontier:\n            for i,op in enumerate(ops):\n                metrics[\'expansions\']+=1\n                v=apply(op,values)\n                if v is None:\n                    metrics[\'undefined\']+=1;continue\n                p=path+(i,);collect(v,p);sig=signature(v)\n                if sig in seen:\n                    metrics[\'merged\']+=1;old,oldpath=seen[sig]\n                    if old[n:]!=v[n:]:\n                        metrics[\'query_divergent_merges\']+=1\n                        if len(witnesses)<3:witnesses.append({\'kept\':[ops[j].name for j in oldpath],\'discarded\':[ops[j].name for j in p],\'at_depth\':d})\n                    continue\n                if d<depth:\n                    if max_states and len(next_frontier)>=max_states:\n                        metrics[\'capped\']+=1;continue\n                    seen[sig]=(v,p);next_frontier.append((v,p))\n        layers.append({\'depth\':d,\'frontier\':len(frontier),\'next\':len(next_frontier)})\n        frontier=next_frontier\n    out=[{\'predictions\':[unpack(s).tolist() for s in pred],\'program\':info[1]}\n         for pred,info in sorted(solutions.items(),key=lambda kv:(kv[1][0],kv[0]))]\n    return {\'candidates\':out,\'metrics\':dict(metrics),\'layers\':layers,\'merge_witnesses\':witnesses,\n            \'num_operations\':len(ops),\'seconds\':time.perf_counter()-start,\'mode\':mode,\n            \'depth\':depth,\'max_states\':max_states,\'grammar\':\'structural_with_terminal_color_map\'}\n\n\ndef solve(task,mode=\'query\',depth=3,max_states=0):\n    result=search(task,mode,depth,max_states)\n    answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for c in result[\'candidates\']:\n            grid=c[\'predictions\'][i]\n            if grid not in grids:grids.append(grid)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return answers,result\n', 'scripts/solver_deferred.py': '"""Layered demonstration DAG with demand-driven query evaluation.\nSPDX-License-Identifier: MIT-0\nPure fixed unary grammars only. No beam, syntax-dependent semantics or cost.\nEach training-equivalent node retains all predecessor edges; no representative\nprefix is discarded. Query vectors are materialized only for goal ancestors.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nimport time\nimport numpy as np\nfrom solver import pack,unpack,operations,apply\nfrom solver_extended import fit_map\n\n\ndef search(task,depth=2,ops_override=None,apply_fn=apply,terminal_map=True):\n    started=time.perf_counter()\n    train=tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in task[\'train\'])\n    queries=tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in task[\'test\'])\n    goal=tuple(pack(np.array(p[\'output\'],dtype=np.uint8)) for p in task[\'train\'])\n    ops=operations(task[\'train\']) if ops_override is None else ops_override\n    layers=[{train:[]}];metrics=Counter();terminals=[];fit_cache={}\n\n    def terminal(d,values):\n        if values==goal:return None,0\n        if not terminal_map or d>=depth:return False,0\n        if values not in fit_cache:fit_cache[values]=fit_map(values,goal)\n        lut=fit_cache[values]\n        return (lut,1) if lut is not None else (False,0)\n\n    for d in range(depth+1):\n        if d:\n            current={}\n            for parent in layers[-1]:\n                for i,op in enumerate(ops):\n                    metrics[\'demonstration_expansions\']+=1\n                    values=apply_fn(op,parent)\n                    if values is None:\n                        metrics[\'demonstration_undefined\']+=1;continue\n                    # Final layer requires only training-fitting terminal nodes.\n                    if d==depth and values!=goal:continue\n                    current.setdefault(values,[]).append((parent,i))\n            layers.append(current)\n        for values in layers[d]:\n            lut,extra=terminal(d,values)\n            if lut is not False:terminals.append((d,values,lut,extra))\n        metrics[\'training_nodes\']+=len(layers[d])\n        metrics[\'provenance_edges\']+=sum(map(len,layers[d].values()))\n\n    @lru_cache(maxsize=None)\n    def materialize(d,values):\n        metrics[\'materialized_nodes\']+=1\n        if d==0:return {queries:()}\n        outputs={}\n        for parent,i in layers[d][values]:\n            for q,path in materialize(d-1,parent).items():\n                metrics[\'query_expansions\']+=1\n                pred=apply_fn(ops[i],q)\n                if pred is None:\n                    metrics[\'query_undefined\']+=1;continue\n                p=path+(i,)\n                if pred not in outputs or p<outputs[pred]:outputs[pred]=p\n        metrics[\'materialized_query_tuples\']+=len(outputs)\n        return outputs\n\n    solutions={}\n    for d,values,lut,extra in terminals:\n        for pred,path in materialize(d,values).items():\n            if lut is not None:\n                pred=tuple(pack(lut[unpack(s)]) for s in pred)\n                metrics[\'terminal_query_maps\']+=len(pred)\n            rank=(d+extra,path,extra)\n            if pred not in solutions or rank<solutions[pred][0]:\n                names=[ops[i].name for i in path]\n                if extra:names+=[\'fit_color_map:\'+\',\'.join(map(str,lut.tolist()))]\n                solutions[pred]=(rank,names)\n    candidates=[{\'predictions\':[unpack(s).tolist() for s in pred],\'program\':info[1]}\n                for pred,info in sorted(solutions.items(),key=lambda kv:(kv[1][0],kv[0]))]\n    return {\'candidates\':candidates,\'metrics\':dict(metrics),\'seconds\':time.perf_counter()-started,\n            \'depth\':depth,\'mode\':\'deferred\',\'max_states\':0,\'num_operations\':len(ops),\n            \'grammar\':\'layered_demonstration_dag_terminal_map\' if terminal_map else \'layered_demonstration_dag\',\n            \'layers\':[{\'depth\':d,\'nodes\':len(nodes)} for d,nodes in enumerate(layers)]}\n\n\ndef solve(task,depth=2,ops_override=None,apply_fn=apply,terminal_map=True):\n    result=search(task,depth,ops_override,apply_fn,terminal_map);answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for candidate in result[\'candidates\']:\n            grid=candidate[\'predictions\'][i]\n            if grid not in grids:grids.append(grid)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return answers,result\n', 'scripts/native_components.py': '"""Optional exact C++ component selector. No search or grammar changes.\nSPDX-License-Identifier: MIT-0\n"""\nimport ctypes\nfrom pathlib import Path\nimport numpy as np\n\n\ndef install():\n    import solver\n    lib=ctypes.CDLL(str(Path(__file__).resolve().parent.parent/\'data/components.so\'))\n    fn=lib.select_component\n    fn.argtypes=[ctypes.POINTER(ctypes.c_uint8),ctypes.c_int,ctypes.c_int,\n                 ctypes.c_int,ctypes.c_int,ctypes.c_int,ctypes.c_int,\n                 ctypes.POINTER(ctypes.c_uint8),ctypes.POINTER(ctypes.c_int)]\n    fn.restype=ctypes.c_int\n    def select(a,bg,same,smallest,crop):\n        a=np.ascontiguousarray(a,dtype=np.uint8)\n        out=np.empty(900,dtype=np.uint8);dims=(ctypes.c_int*2)()\n        success=fn(a.ctypes.data_as(ctypes.POINTER(ctypes.c_uint8)),*a.shape,bg,int(same),\n                   int(smallest),int(crop),out.ctypes.data_as(ctypes.POINTER(ctypes.c_uint8)),dims)\n        if not success:return None\n        return out[:dims[0]*dims[1]].reshape(dims[0],dims[1])\n    original=solver.select_object\n    solver.select_object=select\n    return original,select\n\n\nif __name__==\'__main__\':\n    import itertools,json,random,time\n    rng=random.Random(42)\n    original,native=install();checks=0;started=time.perf_counter()\n    for iteration in range(200):\n        h,w=rng.randrange(1,31),rng.randrange(1,31)\n        a=np.array([[rng.randrange(4) for _ in range(w)] for _ in range(h)],dtype=np.uint8)\n        for bg,same,small,crop in itertools.product((0,1,2,3),(False,True),(False,True),(False,True)):\n            x,y=original(a,bg,same,small,crop),native(a,bg,same,small,crop)\n            assert (x is None and y is None) or (x is not None and y is not None and np.array_equal(x,y))\n            checks+=1\n    dest=Path(\'results/native-checks\');dest.mkdir(parents=True,exist_ok=True)\n    (dest/\'evidence.json\').write_text(json.dumps({\'semantic_equivalence_checks\':checks,\'seconds\':time.perf_counter()-started},indent=2)+\'\\n\')\n    print(f\'Passed {checks} native/Python component comparisons.\')\n', 'scripts/components.cpp': '// SPDX-License-Identifier: MIT-0\n// Exact 4-connected component selection; tie break is first row-major cell.\n#include <cstdint>\n#include <algorithm>\nextern "C" int select_component(const uint8_t* a,int h,int w,int bg,int same,\n                                int smallest,int crop,uint8_t* out,int* dims) {\n    int labels[900]={},queue[900],sizes[901]={},first[901]={};\n    int count=0;\n    for(int start=0;start<h*w;++start) {\n        if(a[start]==bg || labels[start])continue;\n        int label=++count,head=0,tail=1;\n        queue[0]=start;labels[start]=label;first[label]=start;\n        while(head<tail) {\n            int i=queue[head++],y=i/w,x=i%w;\n            int ns[4]={y?i-w:-1,y+1<h?i+w:-1,x?i-1:-1,x+1<w?i+1:-1};\n            for(int q:ns)if(q>=0 && !labels[q] && a[q]!=bg && (!same || a[q]==a[start])) {\n                labels[q]=label;queue[tail++]=q;\n            }\n        }\n        sizes[label]=tail;\n    }\n    if(!count)return 0;\n    int best=1;\n    for(int c=2;c<=count;++c)if((smallest && sizes[c]<sizes[best]) || (!smallest && sizes[c]>sizes[best]))best=c;\n    int ymin=0,ymax=h-1,xmin=0,xmax=w-1;\n    if(crop) {\n        ymin=h;xmin=w;ymax=xmax=-1;\n        for(int i=0;i<h*w;++i)if(labels[i]==best) {\n            ymin=std::min(ymin,i/w);ymax=std::max(ymax,i/w);\n            xmin=std::min(xmin,i%w);xmax=std::max(xmax,i%w);\n        }\n    }\n    dims[0]=ymax-ymin+1;dims[1]=xmax-xmin+1;\n    for(int y=ymin;y<=ymax;++y)for(int x=xmin;x<=xmax;++x) {\n        int i=y*w+x,j=(y-ymin)*dims[1]+x-xmin;\n        out[j]=labels[i]==best?a[i]:bg;\n    }\n    return 1;\n}\n', 'scripts/icecuber_entry.py': '"""Offline two-attempt adapter for the pinned/reset MIT icecuber baseline.\nAuthored adapter SPDX-License-Identifier: MIT-0; embedded solver remains MIT.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport time\n\n\ndef valid(grid):\n    return (isinstance(grid,list) and 1<=len(grid)<=30 and isinstance(grid[0],list)\n            and 1<=len(grid[0])<=30 and all(isinstance(row,list) and len(row)==len(grid[0]) for row in grid)\n            and all(type(c) is int and 0<=c<=9 for row in grid for c in row))\n\n\ndef predict(challenges,root,depth=2,timeout=1500):\n    """Inference has no query outputs. Returns attempts plus execution receipt."""\n    if not challenges:raise ValueError(\'Empty challenge dictionary\')\n    root=Path(root).resolve();target=root/\'dataset/test\'\n    indices=[];source_hashes={}\n    # Serialize only demonstrations and test inputs, including when labels supplied.\n    blind={name:{\'train\':task[\'train\'],\'test\':[{\'input\':q[\'input\']} for q in task[\'test\']]}\n           for name,task in challenges.items()}\n    for name,task in blind.items():\n        if not re.fullmatch(\'[a-z0-9]{8}\',name):raise ValueError(\'Expected official eight-character task ID\')\n        for pair in task[\'train\']:\n            assert valid(pair[\'input\']) and valid(pair[\'output\'])\n        for pair in task[\'test\']:assert valid(pair[\'input\'])\n    if target.exists():shutil.rmtree(target)\n    target.mkdir(parents=True)\n    for name,task in sorted(blind.items()):\n        payload=json.dumps(task,separators=(\',\',\':\'))+\'\\n\'\n        (target/(name+\'.json\')).write_text(payload)\n        source_hashes[name]=hashlib.sha256(payload.encode()).hexdigest()\n        indices.extend((name,i) for i in range(len(task[\'test\'])))\n    output=root/\'output\';output.mkdir(exist_ok=True)\n    for i in range(len(indices)):(output/f\'answer_{i}_{depth}.csv\').unlink(missing_ok=True)\n    env=dict(os.environ)\n    env.pop(\'ICECUBER_PROBE_MAXAREA\',None);env.pop(\'ICECUBER_RESET_NORMALIZATION\',None)\n    started=time.perf_counter()\n    with (output/\'adapter.stdout\').open(\'w\') as stdout,(output/\'adapter.stderr\').open(\'w\') as stderr:\n        process=subprocess.Popen([str(root/\'run\'),\'-1\',str(depth)],cwd=root,env=env,\n                                 stdout=stdout,stderr=stderr,start_new_session=True)\n        try:code=process.wait(timeout=timeout)\n        except subprocess.TimeoutExpired:\n            os.killpg(process.pid,signal.SIGKILL);process.wait();code=\'timeout\'\n    submission={name:[] for name in sorted(blind)};missing=[]\n    for index,(name,qi) in enumerate(indices):\n        csv=output/f\'answer_{index}_{depth}.csv\';grids=[]\n        if csv.exists():\n            lines=csv.read_text().strip().splitlines()\n            assert lines and lines[0]==f\'{name}_{qi}\'\n            for line in lines[1:]:\n                grid=[[int(c) for c in row] for row in line.split()[0].strip(\'|\').split(\'|\')]\n                assert valid(grid)\n                grids.append(grid)\n                if len(grids)==2:break\n        else:missing.append(f\'{name}_{qi}\')\n        while len(grids)<2:grids.append(blind[name][\'test\'][qi][\'input\'])\n        submission[name].append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return submission,{\'binary_sha256\':hashlib.sha256((root/\'run\').read_bytes()).hexdigest(),\n                       \'depth_argument\':depth,\'exit_code\':code,\'missing_queries\':missing,\n                       \'tasks\':len(blind),\'queries\':len(indices),\'wall_seconds\':time.perf_counter()-started,\n                       \'blind_input_hashes\':source_hashes,\'test_outputs_serialized\':False}\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--input\',required=True);p.add_argument(\'--output\',default=\'submission.json\')\n    p.add_argument(\'--root\',default=\'data/icecuber-reset\');p.add_argument(\'--depth\',type=int,default=2)\n    p.add_argument(\'--timeout\',type=int,default=1500);a=p.parse_args()\n    submission,receipt=predict(json.loads(Path(a.input).read_text()),a.root,a.depth,a.timeout)\n    Path(a.output).write_text(json.dumps(submission,separators=(\',\',\':\'))+\'\\n\')\n    Path(a.output+\'.receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\')\n    print(json.dumps({k:v for k,v in receipt.items() if k!=\'blind_input_hashes\'},indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'scripts/portfolio_entry.py': '"""Demonstration-fitting DAG predictions, then distinct pinned-native attempts.\nSPDX-License-Identifier: MIT-0; embedded native solver retains MIT.\n"""\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nfrom icecuber_entry import predict as native_predict, valid\nfrom solver_deferred import search\n\n\ndef select(task, candidates, native):\n    """Use two distinct grids per query, with no labels or task-ID routing."""\n    assert len(native) == len(task[\'test\'])\n    answers = []\n    for qi, query in enumerate(task[\'test\']):\n        grids = []\n        ordered = [c[\'predictions\'][qi] for c in candidates]\n        ordered += [native[qi][\'attempt_1\'], native[qi][\'attempt_2\']]\n        for grid in ordered:\n            assert valid(grid)\n            if grid not in grids:\n                grids.append(grid)\n            if len(grids) == 2:\n                break\n        while len(grids) < 2:\n            grids.append(query[\'input\'])\n        answers.append({\'attempt_1\': grids[0], \'attempt_2\': grids[1]})\n    return answers\n\n\ndef predict(challenges, native_root, depth=3, timeout=1500):\n    blind = {name: {\'train\': task[\'train\'],\n                    \'test\': [{\'input\': q[\'input\']} for q in task[\'test\']]}\n             for name, task in sorted(challenges.items())}\n    started = time.perf_counter()\n    native, native_receipt = native_predict(blind, native_root, depth=2, timeout=timeout)\n    assert native_receipt[\'exit_code\'] == 0 and not native_receipt[\'missing_queries\'], native_receipt\n    submission = {}\n    rows = []\n    for name, task in blind.items():\n        result = search(task, depth=depth, terminal_map=True)\n        answers = select(task, result[\'candidates\'], native[name])\n        submission[name] = answers\n        rows.append({\'task_id\': name, \'candidate_count\': len(result[\'candidates\']),\n                     \'candidates\': result[\'candidates\'], \'dag_seconds\': result[\'seconds\'],\n                     \'dag_metrics\': result[\'metrics\'], \'native_attempts\': native[name],\n                     \'attempts\': answers})\n    return submission, {\'complete\': True, \'depth\': depth, \'tasks\': len(blind),\n                        \'queries\': sum(len(t[\'test\']) for t in blind.values()),\n                        \'native_receipt\': native_receipt, \'rows\': rows,\n                        \'wall_seconds\': time.perf_counter() - started,\n                        \'policy\': \'Ranked DAG candidate grids first; fill remaining distinct slots from reset native depth2. Input fallback only for empty slots. No query-label access.\'}\n\n', 'native/src/brute2.cpp': '#include "precompiled_stl.hpp"\n#include <chrono>\nusing namespace std;\n#include "utils.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n\n#include "timer.hpp"\n\nextern int MAXDEPTH, print_nodes;\n\n//double build_f_time = 0, apply_f_time = 0;\n//double real_f_time = 0;\n\ndouble now() {\n  ll t = chrono::high_resolution_clock::now().time_since_epoch().count();\n  static ll time0 = 0;\n  if (time0 == 0) time0 = t;\n  return (t-time0)*1e-9;\n}\n//double now() { return chrono::steady_clock::now().time_since_epoch().count()*1e-9;}\n\nTimer build_f_time, apply_f_time, real_f_time, add_time, find_child_time, add_child_time, hash_time, map_time, total_time;\nTimer state_time;\n\nvoid Functions3::add(const string& name, int cost_, const function<bool(const State&,State&)>&func, int list) {\n  //if (cost_ != 10) cout << name << endl;\n  //assert(cost_ == 10);\n  if (list) listed.push_back(names.size());\n  names.push_back(name);\n  f_list.push_back(func);\n  cost.push_back(cost_);\n}\n\nvoid Functions3::add(string name, int cost, const function<Image(Image_)>&f, int list) { //list = 1\n  auto func = [f](const State& cur, State& nxt) {\n\n    //if (cur.isvec) return false;\n\n    nxt.vimg.resize(cur.vimg.size());\n    nxt.isvec = cur.isvec;\n\n    int area = 0;\n    for (int i = 0; i < cur.vimg.size(); i++) {\n      real_f_time.start();\n      nxt.vimg[i] = f(cur.vimg[i]);\n      real_f_time.stop();\n\n      area += nxt.vimg[i].w*nxt.vimg[i].h;\n      if (area > MAXPIXELS) return false;\n    }\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(string name, int cost, const function<vImage(Image_)>&f, int list) { //list = 1\n  const int buffer = 5;\n  auto func = [f,cost](const State& cur, State& nxt) {\n    if (cur.isvec || cur.depth+cost+buffer > MAXDEPTH) return false;\n    real_f_time.start();\n    nxt.vimg = f(cur.vimg[0]);\n    real_f_time.stop();\n    nxt.isvec = true;\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(string name, int cost, const function<Image(vImage_)>&f, int list) { //list = 1\n  auto func = [f](const State& cur, State& nxt) {\n    if (!cur.isvec) return false;\n    nxt.vimg.resize(1);\n    real_f_time.start();\n    nxt.vimg[0] = f(cur.vimg);\n    real_f_time.stop();\n    nxt.isvec = false;\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(string name, int cost, const function<vImage(vImage_)>&f, int list) { //list = 1\n  auto func = [f](const State& cur, State& nxt) {\n    if (!cur.isvec) return false;\n    real_f_time.start();\n    nxt.vimg = f(cur.vimg);\n    real_f_time.stop();\n    nxt.isvec = true;\n    return true;\n  };\n  add(name, cost, func, list);\n}\n\nvoid Functions3::add(const vector<point>&sizes, string name, int cost, const function<Image(Image_,Image_)>&f, int list) { //list = 1\n  int szi = 0;\n  for (point sz : sizes) {\n    Image arg2 = core::empty(sz);\n    auto func = [f,arg2](const State& cur, State& nxt) {\n\n      if (cur.isvec) return false;\n\n      nxt.vimg.resize(cur.vimg.size());\n\n      int area = 0;\n      for (int i = 0; i < cur.vimg.size(); i++) {\n\treal_f_time.start();\n\tnxt.vimg[i] = f(cur.vimg[i], arg2);\n\treal_f_time.stop();\n\n\tarea += nxt.vimg[i].w*nxt.vimg[i].h;\n\tif (area > MAXPIXELS) return false;\n      }\n      nxt.isvec = cur.isvec;\n      return true;\n    };\n    add(name+" "+to_string(szi++), cost, func, list);\n  }\n}\n\nstring Functions3::getName(int fi) {\n  assert(fi >= 0 && fi < names.size());\n  return names[fi];\n}\nint Functions3::findfi(string name) {\n  int fi = find(names.begin(), names.end(), name)-names.begin();\n  if (fi == names.size()) {\n    cerr << name << " is not a known function" << endl;\n    assert(0);\n  }\n  return fi;\n}\n\n\nFunctions3 initFuncs3(const vector<point>&sizes) {\n  Functions3 funcs;\n\n  // Unary\n\n  //invert is filterCol(img, 0)\n  for (int c = 0; c < 10; c++)\n    funcs.add("filterCol "+to_string(c), 10, [c](Image_ img) {return filterCol(img, c);});\n  for (int c = 1; c < 10; c++)\n    funcs.add("eraseCol "+to_string(c), 10,\n\t      [c](Image_ img) {return eraseCol(img, c);});\n\n  for (int c = 1; c < 10; c++)\n    funcs.add("colShape "+to_string(c), 10,\n\t      [c](Image_ img) {return colShape(img, c);}, 0);\n\n  funcs.add("compress", 10, [](Image_ img) {return compress(img);});\n  funcs.add("getPos", 10, getPos);\n  funcs.add("getSize0", 10, getSize0);\n  funcs.add("getSize", 10, getSize);\n  funcs.add("hull0", 10, hull0);\n  funcs.add("hull", 10, hull);\n  funcs.add("toOrigin", 10, toOrigin);\n  funcs.add("Fill", 10, Fill);\n  funcs.add("interior", 10, interior);\n  funcs.add("interior2", 10, interior2);\n  funcs.add("border", 10, border);\n  funcs.add("center", 10, center);\n  funcs.add("majCol", 10, majCol);\n\n  //funcs.add("greedyFillBlack", 10, [](Image_ img) {return greedyFillBlack(img);});\n  //funcs.add("greedyFillBlack2", 10, [](Image_ img) {return greedyFillBlack2(img);});\n\n  for (int i = 1; i < 9; i++)\n    funcs.add("rigid "+to_string(i), 10,\n\t      [i](Image_ img) {return rigid(img, i);});\n  for (int a = 0; a < 3; a++)\n    for (int b = 0; b < 3; b++)\n      funcs.add("count "+to_string(a)+" "+to_string(b), 10,\n\t\t[a,b](Image_ img) {return count(img, a, b);});\n  for (int i = 0; i < 15; i++)\n    funcs.add("smear "+to_string(i), 10,\n\t      [i](Image_ img) {return smear(img, i);});\n\n\n  funcs.add("makeBorder", 10,\n\t    [](Image_ img) {return makeBorder(img, 1);});\n\n  for (int id : {0,1})\n    funcs.add("makeBorder2 "+to_string(id), 10,\n\t      [id](Image_ img) {return makeBorder2(img, id);});\n  funcs.add("compress2", 10, compress2);\n  funcs.add("compress3", 10, compress3);\n\n  for (int id = 0; id < 3; id++)\n    funcs.add("connect "+to_string(id), 10,\n\t      [id](Image_ img) {return connect(img,id);});\n\n  for (int id : {0,1})\n    funcs.add("spreadCols "+to_string(id), 10,\n\t      [id](Image_ img) {return spreadCols(img, id);});\n\n  for (int id = 0; id < 4; id++)\n    funcs.add("half "+to_string(id), 10,\n\t      [id](Image_ img) {return half(img, id);});\n\n\n  for (int dy = -2; dy <= 2; dy++) {\n    for (int dx = -2; dx <= 2; dx++) {\n      funcs.add("Move "+to_string(dx)+" "+to_string(dy), 10,\n\t\t[dx,dy](Image_ img) {return Move(img, Pos(dx,dy));}, 0);\n    }\n  }\n\n  // Binary\n  funcs.add(sizes, "embed", 10, embed);\n  funcs.add(sizes, "wrap", 10, wrap);\n  funcs.add(sizes, "broadcast", 10, [](Image_ a, Image_ b) {return broadcast(a,b);});\n  funcs.add(sizes, "repeat 0",  10, [](Image_ a, Image_ b) {return repeat(a,b);});\n  funcs.add(sizes, "repeat 1",  10, [](Image_ a, Image_ b) {return repeat(a,b,1);});\n  funcs.add(sizes, "mirror 0",  10, [](Image_ a, Image_ b) {return mirror(a,b);});\n  funcs.add(sizes, "mirror 1",  10, [](Image_ a, Image_ b) {return mirror(a,b,1);});\n\n\n  //Split\n  funcs.add("cut",       10, [](Image_ img) {return cut(img);});\n  funcs.add("splitCols", 10, [](Image_ img) {return splitCols(img);});\n  funcs.add("splitAll",     10, splitAll);\n  funcs.add("splitColumns", 10, splitColumns);\n  funcs.add("splitRows",    10, splitRows);\n  funcs.add("insideMarked", 10, insideMarked);\n  for (int id = 0; id < 4; id++)\n    funcs.add("gravity "+to_string(id), 10,\n\t      [id](Image_ img) {return gravity(img,id);});\n\n\n  //Join\n  for (int id = 0; id < 14; id++)\n    funcs.add("pickMax "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickMax(v,id);});\n  for (int id = 0; id < 1; id++)\n    funcs.add("pickUnique "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickUnique(v,id);});\n\n  funcs.add("composeGrowing", 10, composeGrowing);\n  funcs.add("stackLine", 10, stackLine);\n  for (int id = 0; id < 2; id++) //consider going to 4\n    funcs.add("myStack "+to_string(id), 10,\n\t      [id](vImage_ v) {return myStack(v,id);}); //\n\n\n  //Vector\n  for (int id = 0; id < 14; id++)\n    funcs.add("pickMaxes "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickMaxes(v,id);});\n  for (int id = 0; id < 14; id++)\n    funcs.add("pickNotMaxes "+to_string(id), 10,\n\t      [id](vImage_ v) {return pickNotMaxes(v,id);});\n\n\n  static int said = 0;\n  if (!said) {\n    cout << "Function count: " << funcs.listed.size() << endl;\n    said = 1;\n  }\n\n\n  //funcs.add("smear",    [](Image_ a, Image_ b) {return smear(a,b,6);});\n\n  //funcs.add(insideMarked); //only do once at depth 0\n\n  // Smear diagonals?\n\n  // outerProducts\n\n  //for (int id = 0; id < 4; id++)\n  //  funcs.add([id](Image_ img) {return gravity(img, id);});\n\n  // Image makeBorder(Image_ img, int bcol = 1);\n  // Image makeBorder2(Image_ img, Image_ bord);\n  // Image greedyFillBlack(Image_ img, int N = 3);\n  // Image extend2(Image_ img, Image_ room);\n  // Image replaceTemplate(Image_ in, Image_ need_, Image_ marked_, int overlapping = 0, int rigids = 0);\n  // Image swapTemplate(Image_ in, Image_ a, Image_ b, int rigids = 0);\n\n  // funcs.add("heuristicCut", heuristicCut);\n\n  return funcs;\n}\n\nImage DAG::getImg(int ni) {\n  return tiny_node.getImg(ni);\n  //assert(tiny_node.getImg(ni) == node[ni].vimg[0]);\n  //return node[ni].state.vimg[0];\n  /*\n  //cout << nodei << endl;\n  assert(nodei >= 0 && nodei < (int)node.size());\n  assert(!node[nodei].isvec);\n  if (node[nodei].pfi == embed1fi) {\n    assert(funcs.f_list[embed1fi](node[nodei], tmp_node));\n    //assert(tmp_node.vimg == node[nodei].vimg);\n    return tmp_node.vimg[0];\n  } else {\n    assert(node[nodei].vimg.size());\n    return node[nodei].vimg[0];\n    }*/\n}\n\n\nint DAG::add(const State&nxt, bool force) { //force = false\n  hash_time.start();\n  ull h = nxt.hash();\n  hash_time.stop();\n  int nodes = tiny_node.size();\n  map_time.start();\n  auto [nodei,inserted] = hashi.insert(h,nodes);\n  //auto [it,inserted] = hashi.insert({h,nodes}); int nodei = it->second;\n  //assert(inserted == tiny_inserted);\n  //assert(nodei == tiny_nodei);\n\n  map_time.stop();\n\n  add_time.start();\n  if (inserted || force) {\n    bool ispiece = !nxt.isvec;\n    if (!nxt.isvec && target_size != point{-1,-1})\n      ispiece &= (nxt.vimg[0].p == point{0,0} && nxt.vimg[0].sz == target_size);\n    /*{\n      Node n;\n      n.state = nxt;\n      n.ispiece = ispiece;\n      n.freed = false;\n      node.push_back(n);\n      }*/\n    tiny_node.append(nxt, ispiece);\n  }\n  add_time.stop();\n  return nodei;\n}\n\n\nvoid DAG::build() {\n  build_f_time.start();\n\n  for (int curi = 0; curi < tiny_node.size(); curi++) {\n    int depth = tiny_node[curi].depth;\n    if (depth+1 > MAXDEPTH) continue;\n\n    //vector<pair<int,int>> child;\n    State nxt;\n    state_time.start();\n    State cur_state = tiny_node.getState(curi);\n    state_time.stop();\n    for (int fi : funcs.listed) {\n      nxt.depth = depth+funcs.cost[fi];\n      if (nxt.depth > MAXDEPTH) continue;\n      if (funcs.f_list[fi](cur_state, nxt)) {\n\tint newi = add(nxt);\n\t//child.emplace_back(fi, newi);\n\ttiny_node.addChild(curi, fi, newi);\n      } else {\n\ttiny_node.addChild(curi, fi, -1);\n\t//child.emplace_back(fi, -1);\n      }\n\n    }\n    //node[curi].child = child;\n  }\n\n  build_f_time.stop();\n}\n\nvoid DAG::initial(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> sizes, int ti) {\n  if (sizes.size() > 1)\n    target_size = sizes[1];\n  else\n    target_size = point{-1,-1};\n\n  Image in = ti < train.size() ? train[ti].first : test_in;\n\n  add(State({in}, false, 0), true);\n\n  //Output sizes\n  for (point sz : sizes)\n    add(State({core::empty(sz)}, false, 10), true);\n\n  // Outputs of other trains\n  for (int tj = 0; tj < train.size(); tj++)\n    add(State({ti != tj ? train[tj].second : core::empty(train[tj].second.sz)}, false, 10), true);\n\n  //add(State({greedyFillBlack2(in)}, false, 10), true);\n\n  //filterCol?\n\n  givens = tiny_node.size();\n}\n\n//time each function\nvoid DAG::benchmark() {\n  vector<pair<double,int>> v;\n  for (int fi : funcs.listed) {\n    double start_time = now();\n    State nxt;\n    for (int i = 0; i < tiny_node.size(); i++) {\n      funcs.f_list[fi](tiny_node.getState(i), nxt);\n    }\n    double elapsed = now()-start_time;\n    v.emplace_back(elapsed, fi);\n  }\n  sort(v.begin(), v.end());\n  for (auto [t,fi] : v)\n    printf("%.1f ms - %s\\n", t*1e3, funcs.getName(fi).c_str());\n}\n\nint DAG::applyFunc(int curi, int fi, const State&state) {\n\n  find_child_time.start();\n  //auto it = lower_bound(node[curi].child.begin(), node[curi].child.end(), make_pair(fi,-1));\n  int it2 = tiny_node.getChild(curi, fi);\n  find_child_time.stop();\n  if (it2 != TinyChildren::None) {//it != node[curi].child.end() && it->first == fi) {\n    //if (it2 != it->second) cout << it2 << \' \' << it->second << endl;\n    //assert(it2 == it->second);\n    return it2;//it->second;\n  }\n  //assert(it2 == -2);\n\n\n  State nxt;\n  nxt.depth = tiny_node[curi].depth+funcs.cost[fi];\n  //nxt.par = curi;\n\n  int newi = -1;\n\n  apply_f_time.start();\n  bool ok = funcs.f_list[fi](state, nxt);\n  apply_f_time.stop();\n\n  if (ok) {\n    //nxt.pfi = fi;\n    newi = add(nxt);\n  }\n\n  add_child_time.start();\n  tiny_node.addChild(curi, fi, newi);\n  //node[curi].child.emplace_back(fi, newi);\n  //sort(node[curi].child.begin(), node[curi].child.end());\n  add_child_time.stop();\n  return newi;\n}\n\nint DAG::applyFunc(int curi, int fi) {\n  state_time.start();\n  State state = tiny_node.getState(curi);\n  state_time.stop();\n  return applyFunc(curi, fi, state);\n}\n\nvoid DAG::applyFunc(string name, bool vec) {\n  int fi = funcs.findfi(name);\n\n  int start_nodes = tiny_node.size();\n  for (int curi = 0; curi < start_nodes; curi++) {\n    if (tiny_node[curi].isvec == vec) applyFunc(curi, fi);\n  }\n}\n\nvoid DAG::applyFuncs(vector<pair<string,int>> names, bool vec) {\n  vector<pair<int,int>> fis;\n  for (auto [name,id] : names) {\n    fis.emplace_back(funcs.findfi(name), id);\n  }\n\n  vector<int> parid(tiny_node.size(),-1);\n  for (int curi = 0; curi < tiny_node.size(); curi++) {\n    if (tiny_node[curi].isvec != vec) continue;\n    state_time.start();\n    State state = tiny_node.getState(curi);\n    state_time.stop();\n\n    for (int i = 0; i < fis.size(); i++) {\n      auto [fi,id] = fis[i];\n      if (id <= parid[curi]) continue;\n\n      if (applyFunc(curi, fi, state) == parid.size()) {\n\tparid.push_back(id);\n\tassert(parid.size() == tiny_node.size());\n      }\n    }\n  }\n}\n\n\n\n\n\nvoid DAG::buildBinary() {\n  int fis = *max_element(funcs.listed.begin(), funcs.listed.end())+1;\n  binary.assign(fis*fis, -1);\n  vector<State> state(fis);\n  vector<int> active(fis), memi(fis);\n  for (int fi : funcs.listed) {\n    int curi = tiny_node.getChild(0, fi);\n    if (curi >= 0) {\n      active[fi] = 1;\n      state[fi] = tiny_node.getState(curi);\n      memi[fi] = curi;\n    }\n  }\n  for (int fa : funcs.listed) {\n    if (!active[fa]) continue;\n    for (int fb : funcs.listed) {\n      if (!active[fb]) continue;\n\n      if (state[fa].isvec || state[fb].isvec) continue;\n      State nxt;\n      nxt.vimg = {align(state[fa].vimg[0], state[fb].vimg[0])};\n      nxt.depth = 2; //TODO\n      nxt.isvec = false;\n      binary[fa*fis+fb] = add(nxt);\n      /*if (binary[fa*fis+fb] != memi[fa]) {\n\tcout << binary[fa*fis+fb] << \' \' << memi[fa] << \' \' << tiny_node.size() << endl;\n      }\n      assert(binary[fa*fis+fb] == memi[fa]);*/\n    }\n  }\n}\n\n\n\nvector<DAG> brutePieces2(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> out_sizes) {\n  int print = 1;\n\n  vector<DAG> dag(train.size()+1);\n\n  int all_train_out_mask = 0, and_train_out_mask = ~0;\n  for (int ti = 0; ti < train.size(); ti++)\n    and_train_out_mask &= core::colMask(train[ti].second);\n\n  for (int ti = 0; ti <= train.size(); ti++) {\n    vector<point> sizes;\n    if (ti < train.size())\n      sizes.push_back(train[ti].first.sz);\n    else\n      sizes.push_back(test_in.sz);\n\n    if (out_sizes.size())\n      sizes.push_back(out_sizes[ti]);\n\n    dag[ti].funcs = initFuncs3(sizes);\n\n    dag[ti].initial(test_in, train, sizes, ti);\n\n    total_time.start();\n\n    double start_time = now();\n    dag[ti].build();\n    if (print) cout << now()-start_time << endl;\n    //dag[ti].buildBinary();\n    //if (print) cout << now()-start_time << endl;\n    dag[ti].applyFunc("composeGrowing", 1);\n    if (print) cout << now()-start_time << endl;\n\n    if (sizes.size() > 1) {\n      vector<pair<string,int>> toapply;\n      toapply.emplace_back("toOrigin",0);\n      for (int c = 1; c <= 5; c++)\n\tif (and_train_out_mask>>c&1)\n\t  toapply.emplace_back("colShape "+to_string(c),1);\n      toapply.emplace_back("embed 1",2);\n      dag[ti].applyFuncs(toapply, 0);\n      /*\n      dag[ti].applyFunc("toOrigin", 0);\n      if (print) cout << now()-start_time << endl;\n\n      int mask;\n      if (ti < train.size()) {\n\tmask = core::colMask(train[ti].second);\n\tall_train_out_mask |= mask;\n      } else {\n\tmask = all_train_out_mask;\n      }\n\n      for (int c = 1; c <= 5; c++)\n\tif (and_train_out_mask>>c&1)\n\tdag[ti].applyFunc("colShape "+to_string(c), 0);\n      if (print) cout << now()-start_time << endl;\n\n      if (ti < train.size())\n\tdeducePositions(dag[ti], train[ti].second);\n      if (print) cout << now()-start_time << endl;\n\n      dag[ti].applyFunc("embed 1", 0);\n      */\n      if (print) cout << now()-start_time << endl;\n\n      /*if (ti < train.size())\n\tdeducePositions(dag[ti], train[ti].second);\n\n\tif (print) cout << now()-start_time << endl;*/\n\n      total_time.stop();\n      total_time.print("Total time");\n      build_f_time.print("Build f time");\n      apply_f_time.print("Apply f time");\n      real_f_time .print("Real f time ");\n\n      add_time.print("Add time");\n      find_child_time.print("Find child");\n      add_child_time.print("Add child");\n      hash_time.print("Hash");\n      map_time.print("Map");\n\n      state_time.print("getState");\n      //exit(0);\n      /*FILE*fp = fopen("images.txt", "w");\n      for (Node&n : dag[ti].node) {\n\tfor (Image_ img : n.vimg) {\n\t  fprintf(fp, "%d %d %d %d\\n", img.x, img.y, img.w, img.h);\n\t  for (char c : img.mask)\n\t    fprintf(fp, "%c", \'0\'+c);\n\t  fprintf(fp, "\\n");\n\t}\n      }\n      exit(0);*/\n      //dag[ti].freeAll();\n    } else total_time.stop();\n    //dag[ti].benchmark();\n    //exit(0);\n\n    /*\n    for (Node&n : dag[ti].node) {\n      if (!n.isvec && n.pfi == dag[ti].embed1fi) {\n\tn.vimg.clear();\n\tn.vimg.shrink_to_fit();\n      }\n    }\n    */\n\n    /*if (ti < train.size()) {\n      for (Node&n : dag[ti].node) {\n\tif (n.par > -1 && (n.isvec || n.img[0].w > 30 || n.img[0].h > 30)) {// || n.img[0].p != point{0,0} || n.img[0].sz != given_sizes[ti][1])) {\n\t  n.img.clear();\n\t  n.img.shrink_to_fit();\n\t}\n      }\n      dag[ti].hashi.clear();\n      }*/\n  }\n\n\n  if (out_sizes.size() && print_nodes) {\n    cout << "Dag sizes: ";\n    for (int i = 0; i < dag.size(); i++)\n      cout << dag[i].tiny_node.size() << " ";\n    cout << endl;\n  }\n\n  return dag;\n}\n', 'native/src/brute_size.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\nextern int MAXDEPTH;\n\nbool operator<(const point a, const point b) {\n  if (a.x != b.x) return a.x < b.x;\n  return a.y < b.y;\n}\n\npair<vector<int>,double> solveSingle(vector<vector<int>>&seeds, const vector<int>& target) {\n  int n = target.size()+1;\n  vector<int> ans(n,1);\n  pair<int,double> best = {-1,1e9};\n\n  auto add = [&](const vector<int>&szs, double loss) {\n    int oks = 0;\n    for (int ti = 0; ti < target.size(); ti++)\n      oks += (szs[ti] == target[ti]);\n    pair<int,double> cand = {oks, -loss-10};\n    int sz = szs.back();\n    if (sz >= 1 && sz <= 30 &&\n\tcand > best) {\n      best = cand;\n      ans = szs;\n    }\n  };\n\n  for (int w = 1; w <= 30; w++) {\n    add(vector<int>(n,w), w);\n  }\n\n  vector<int> szs(n);\n  for (int i = 0; i < seeds.size(); i++) {\n    double a = i+1;\n    for (int w = 1; w < 6; w++) {\n      for (int x = -3; x <= 3; x++) {\n\tfor (int k = 0; k < n; k++)\n\t  szs[k] = seeds[i][k]*w+x;\n\tadd(szs, a*w*(abs(x)+1));\n      }\n    }\n  }\n\n  return {ans,best.second};\n}\n\npoint solveSize(vector<vector<point>>&seeds, const vector<point>& target) {\n  point ans = {1,1};\n  pair<int,double> best = {-1,1e9};\n\n  auto add = [&](const vector<point>&szs, double loss) {\n    int oks = 0;\n    for (int ti = 0; ti < target.size(); ti++)\n      oks += (szs[ti] == target[ti]);\n    pair<int,double> cand = {oks, -loss};\n    point sz = szs.back();\n    if (sz.x >= 1 && sz.x <= 30 &&\n\tsz.y >= 1 && sz.y <= 30 &&\n\tcand > best) {\n      best = cand;\n      ans = sz;\n    }\n  };\n\n  int n = target.size()+1;\n  vector<point> szs(n);\n\n  for (int i = 0; i < seeds.size(); i++) {\n    double a = i+1;\n    for (int h = 1; h < 6; h++) {\n      for (int w = 1; w < 6; w++) {\n\tfor (int y = -3; y <= 3; y++) {\n\t  for (int x = -3; x <= 3; x++) {\n\t    for (int k = 0; k < n; k++)\n\t      szs[k] = {seeds[i][k].x*w+x, seeds[i][k].y*h+y};\n\t    add(szs, a*w*h*(abs(x)+1)*(abs(y)+1));\n\t  }\n\t}\n      }\n    }\n  }\n\n  if (1) {//best.first < target.size()) {\n    for (int i = 0; i < seeds.size(); i++) {\n      double a = i+1;\n      for (int j = 0; j < i; j++) {\n\tdouble b = j+1;\n\tfor (int d = 0; d < 3; d++) {\n\t  for (int k = 0; k < n; k++) {\n\t    szs[k] = seeds[i][k];\n\t    if (d == 0 || d == 2) szs[k].x = szs[k].x+seeds[j][k].x;\n\t    if (d == 1 || d == 2) szs[k].y = szs[k].y+seeds[j][k].y;\n\t  }\n\t  add(szs, a*b);\n\t}\n      }\n    }\n  }\n\n  {\n    vector<vector<int>> seedsx, seedsy;\n    for (vector<point>&seed : seeds) {\n      vector<int> seedx, seedy;\n      for (point p : seed) {\n\tseedx.push_back(p.x);\n\tseedy.push_back(p.y);\n      }\n      seedsx.push_back(seedx);\n      seedsy.push_back(seedy);\n    }\n    vector<int> targetx, targety;\n    for (point p : target) {\n      targetx.push_back(p.x);\n      targety.push_back(p.y);\n    }\n    auto [bestx, scorex] = solveSingle(seedsx, targetx);\n    auto [besty, scorey] = solveSingle(seedsy, targety);\n\n    vector<point> combined;\n    for (int i = 0; i < n; i++) {\n      combined.push_back({bestx[i], besty[i]});\n    }\n    double combloss = scorex*scorey;\n    add(combined, combloss);\n  }\n\n  //cout << best.first << \' \' << target.size() << \' \' << best.second << endl;\n  //assert(best.first == target.size());\n  return ans;\n}\n\nvector<point> bruteSize(Image_ test_in, vector<pair<Image,Image>> train) {\n  vector<point> out_sizes;\n  for (auto [in,out] : train) {\n    out_sizes.push_back(out.sz);\n  }\n  int cp = MAXDEPTH;\n  MAXDEPTH = min(MAXDEPTH, 30);\n  Pieces pieces;\n  {\n    vector<DAG> dags = brutePieces2(test_in, train, {});\n    pieces = makePieces2(dags, train, {});\n  }\n  int dags = pieces.dag.size();\n\n  MAXDEPTH = cp;\n\n  vector<point> target;\n  for (auto [in,out] : train) target.push_back(out.sz);\n\n  //cout << pieces.piece.size() << endl;\n  //cout << pieces.seen.size() << endl;\n  vector<vector<point>> seeds;\n  set<vector<point>> seen;\n  for (Piece3&p : pieces.piece) {\n    vector<point> sz;\n    int ok = 1;\n    int*ind = &pieces.mem[p.memi];\n    for (int ti = 0; ti <= train.size(); ti++) {\n      if (pieces.dag[ti].tiny_node[ind[ti]].isvec) ok = 0;\n      else {\n\tsz.push_back(pieces.dag[ti].getImg(ind[ti]).sz);\n      }\n    }\n    if (ok) {\n      if (seen.insert(sz).second)\n\tseeds.push_back(sz);\n    }\n  }\n\n  point ans = solveSize(seeds, target);\n  out_sizes.push_back(ans);\n  return out_sizes;\n}\n\nvector<point> cheatSize(Image_ test_out, vector<pair<Image,Image>> train) {\n  vector<point> out_sizes;\n  for (auto [in,out] : train) {\n    out_sizes.push_back(out.sz);\n  }\n  out_sizes.push_back(test_out.sz);\n  return out_sizes;\n}\n', 'native/src/compose2.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\n#include "timer.hpp"\n\n/*\nstruct Candidate {\n  vector<Image> imgs;\n  double score;\n};\n*/\n\n\nextern int print_times;\n\n\nstruct mybitset {\n  vector<ull> data;\n  mybitset(int n) {\n    data.resize((n+63)/64);\n  }\n  int operator[](int i) {\n    return data[i>>6]>>(i&63)&1;\n  }\n  void set(int i, ull v) {\n    int bit = i&63;\n    data[i>>6] &= ~(1ull<<bit);\n    data[i>>6] |= (v<<bit);\n  }\n  ull hash() {\n    ull r = 1;\n    for (ull h : data) {\n      r = r*137139+h;\n    }\n    return r;\n  }\n};\n\n\n\nint popcount64c(ull x) {\n  const uint64_t m1  = 0x5555555555555555; //binary: 0101...\n  const uint64_t m2  = 0x3333333333333333; //binary: 00110011..\n  const uint64_t m4  = 0x0f0f0f0f0f0f0f0f; //binary:  4 zeros,  4 ones ...\n  const uint64_t m8  = 0x00ff00ff00ff00ff; //binary:  8 zeros,  8 ones ...\n  const uint64_t m16 = 0x0000ffff0000ffff; //binary: 16 zeros, 16 ones ...\n  const uint64_t m32 = 0x00000000ffffffff; //binary: 32 zeros, 32 ones\n  const uint64_t h01 = 0x0101010101010101; //the sum of 256 to the power of 0,1,2,3...\n  x -= (x >> 1) & m1;             //put count of each 2 bits into those 2 bits\n  x = (x & m2) + ((x >> 2) & m2); //put count of each 4 bits into those 4 bits\n  x = (x + (x >> 4)) & m4;        //put count of each 8 bits into those 8 bits\n  return (x * h01) >> 56;  //returns left 8 bits of x + (x<<8) + (x<<16) + (x<<24) + ...\n}\n\nint popcount64d(ull x) {\n  int pop = 0;\n  while (x) {\n    x &= x-1;\n    pop++;\n  }\n  return pop;\n}\n\n\n\nvector<Candidate> greedyCompose2(Pieces&pieces, vector<Image>&target, vector<point> out_sizes) {\n  if (pieces.piece.empty()) return {};\n\n  Timer greedy_fill_time;\n\n  {\n    int d = -1;\n    for (Piece3&p : pieces.piece) {\n      assert(p.depth >= d);\n      d = p.depth;\n    }\n  }\n\n  vector<Image> init;\n  vector<int> sz;\n  {\n    for (int i = 0; i < pieces.dag.size(); i++) {\n      if (i < target.size()) assert(out_sizes[i] == target[i].sz);\n      init.push_back(core::full(out_sizes[i], 10));\n      sz.push_back(init.back().mask.size());\n    }\n  }\n\n  vector<Candidate> rets;\n\n  int n = pieces.piece.size();\n\n  int M = 0;\n  for (int s : sz) M += s;\n\n  const int M64 = (M+63)/64;\n  vector<ull> bad_mem, active_mem;\n  //vector<mybitset> bad_mem, active_mem;\n  vector<int> img_ind, bad_ind, active_ind;\n\n  {\n    active_mem.reserve(n*M64);\n    bad_mem.reserve(n*M64);\n    TinyHashMap seen;\n    mybitset badi(M), blacki(M);\n    for (int i = 0; i < n; i++) {\n      int x = 0, y = 0;\n      for (int j = 0; j < sz.size(); j++) {\n\tint*ind = &pieces.mem[pieces.piece[i].memi];\n\tImage_ img = pieces.dag[j].getImg(ind[j]);\n\tconst vector<char>&p = img.mask;\n\tconst vector<char>&t = j < target.size() ? target[j].mask : init[j].mask;\n\tassert(p.size() == sz[j]);\n\tassert(t.size() == sz[j]);\n\tfor (int k = 0; k < sz[j]; k++) {\n\t  badi.set  (x++, (p[k] != t[k]));\n\t  blacki.set(y++, (p[k] == 0));\n\t}\n      }\n      img_ind.push_back(i);\n\n      active_ind.push_back(active_mem.size());\n      for (ull v : blacki.data)\n\tactive_mem.push_back(v);\n\n      bad_ind.push_back(bad_mem.size());\n      for (ull v : badi.data)\n\tbad_mem.push_back(v);\n    }\n  }\n\n  //cout << active_mem.size() << endl;\n  //cout << bad_mem.size() << endl;\n  //exit(0);\n\n  int max_piece_depth = 0;\n  for (int i = 0; i < n; i++)\n    max_piece_depth = max(max_piece_depth, pieces.piece[i].depth);\n\n  //mt19937 mrand(0);\n\n\n\n  auto greedyComposeCore = [&](mybitset&cur, const mybitset&careMask, const int piece_depth_thres, vImage&ret) {\n    vector<int> sparsej;\n    for (int j = 0; j < M64; j++) {\n      if (~cur.data[j] & careMask.data[j]) sparsej.push_back(j);\n    }\n\n    vector<ull> best_active(M64), tmp_active(M64);\n    int besti = -1;\n    pair<int,int> bestcnt = {0,0};\n    //for (int i : order) {\n    //  if (skip[i] || pieces.piece[img_ind[i]].depth > piece_depth_thres) continue;\n    for (int i = 0; i < img_ind.size(); i++) {\n      if (pieces.piece[img_ind[i]].depth > piece_depth_thres) continue;\n\n      for (int k : {0,1,2}) {\n\tconst ull*active_data = &active_mem[active_ind[i]];\n\n\tull flip = (k == 0 ? ~0 : 0);\n\tull full = (k == 2 ? ~0 : 0);\n\n\tconst ull*bad_data = &bad_mem[bad_ind[i]];\n\tint cnt = 0, covered = 0, ok = 1;\n\tfor (int j = 0; j < M64; j++) {\n\t  ull active = ((active_data[j]^flip) | full);\n\t  if (~cur.data[j] & bad_data[j] & active) {\n\t    ok = 0;\n\t    break;\n\t  }\n\t}\n\tif (!ok) continue;\n\tfor (int j : sparsej) {\n\t  ull active = ((active_data[j]^flip) | full);\n\t  cnt += popcount64d(active & ~cur.data[j] & careMask.data[j]);\n\t  //covered += popcount64d(bad.data[j] & cur.data[j] & careMask.data[j]);\n\t}\n\tif (ok && make_pair(cnt,-covered) > bestcnt) {\n\t  bestcnt = make_pair(cnt,-covered);\n\t  besti = i;\n\n\t  if (k == 0) {\n\t    for (int j = 0; j < M64; j++)\n\t      tmp_active[j] = ~active_data[j];\n\t  } else if (k == 1) {\n\t    for (int j = 0; j < M64; j++)\n\t      tmp_active[j] = active_data[j];\n\t  } else {\n\t    for (int j = 0; j < M64; j++)\n\t      tmp_active[j] = ~0;\n\t  }\n\t  best_active = tmp_active;\n\t}\n      }\n    }\n\n    if (besti == -1) return -1;\n\n    {\n      int i = img_ind[besti], x = 0;\n\n      int depth = pieces.piece[i].depth;\n\n      for (int l = 0; l < ret.size(); l++) {\n\tint*ind = &pieces.mem[pieces.piece[i].memi];\n\tconst vector<char>&mask = pieces.dag[l].getImg(ind[l]).mask;\n\tfor (int j = 0; j < sz[l]; j++) {\n\t  if ((best_active[x>>6]>>(x&63)&1) && ret[l].mask[j] == 10) {\n\t    //assert(cur[x-1] == 0);\n\t    ret[l].mask[j] = mask[j];\n\t  }\n\n\t  x++;\n\t}\n      }\n      for (int j = 0; j < M; j++) {\n\tif (best_active[j>>6]>>(j&63)&1) cur.set(j, 1);\n      }\n\n      return depth;\n    }\n  };\n\n\n\n\n\n\n\n\n\n  map<ull,Image> greedy_fill_mem;\n\n  int maxiters = 10;\n\n\n  //vector<int> skip(img_ind.size());\n  //vector<int> order;\n  //for (int i = 0; i < img_ind.size(); i++) order.push_back(i);\n\n  for (int pdt = max_piece_depth%10; pdt <= max_piece_depth; pdt += 10) {\n    int piece_depth_thres = pdt;\n\n    for (int it0 = 0; it0 < 10; it0++) {\n      for (int mask = 1; mask < min(1<<target.size(), 1<<5); mask++) {\n\tvector<int> maskv;\n\tfor (int j = 0; j < target.size(); j++)\n\t  if (mask>>j&1) maskv.push_back(j);\n\n\tint caremask;\n\tif (it0 < maskv.size()) {\n\t  caremask = 1<<maskv[it0];\n\t} else {\n\t  continue;\n\t  /*caremask = 1<<maskv[mrand()%maskv.size()];\n\t    for (int i = 0; i < n; i++)\n\t    skip[i] = mrand()%3;\n\t    random_shuffle(order.begin(), order.end());*/\n\t}\n\n\tmybitset cur(M), careMask(M);\n\t{\n\t  int base = 0;\n\t  for (int j = 0; j < sz.size(); j++) {\n\t    if (!(mask>>j&1))\n\t      for (int k = 0; k < sz[j]; k++)\n\t\tcur.set(base+k, 1);\n\t    if ((caremask>>j&1))\n\t      for (int k = 0; k < sz[j]; k++)\n\t\tcareMask.set(base+k, 1);\n\t    base += sz[j];\n\t  }\n\t}\n\n\n\tint cnt_pieces = 0;\n\tvector<int> piece_depths;\n\tint sum_depth = 0, max_depth = 0;\n\n\tvector<Image> ret = init;\n\tfor (int it = 0; it < maxiters; it++) {\n\n\t  int depth = greedyComposeCore(cur, careMask, piece_depth_thres, ret);\n\t  if (depth == -1) break;\n\t  piece_depths.push_back(depth);\n\t  cnt_pieces++;\n\t  sum_depth += depth;\n\t  max_depth = max(max_depth, depth);\n\n\n\t  {\n\t    greedy_fill_time.start();\n\t    vImage cp = ret;\n\t    int carei = 31-__builtin_clz(caremask);\n\t    assert(caremask == 1<<carei);\n\t    int ok = 1;\n\t    {\n\t      Image& img = cp[carei];\n\t      for (char&c : img.mask) if (c == 10) c = 0;\n\t      img = greedyFillBlack(img);\n\t      if (img != target[carei]) ok = 0;\n\t    }\n\t    if (ok) {\n\t      for (int i = 0; i < cp.size(); i++) {\n\t\tif (i == carei) continue;\n\t\tImage& img = cp[i];\n\t\tfor (char&c : img.mask) if (c == 10) c = 0;\n\t\tull h = hashImage(img);\n\t\tif (!greedy_fill_mem.count(h)) {\n\t\t  greedy_fill_mem[h] = greedyFillBlack(img);\n\t\t}\n\t\timg = greedy_fill_mem[h];\n\t\tif (img.w*img.h <= 0) ok = 0;\n\t      }\n\t      if (ok)\n\t\trets.emplace_back(cp, cnt_pieces+1, sum_depth, max_depth);\n\t    }\n\t    greedy_fill_time.stop();\n\t  }\n\t}\n\n\t/*for (Image&img : ret)\n\t  for (char&c : img.mask)\n\t  if (c == 10) c = 0;\n\t*/\n\n\trets.emplace_back(ret, cnt_pieces, sum_depth, max_depth);\n      }\n    }\n  }\n\n  if (print_times)\n    greedy_fill_time.print("Greedy fill time");\n\n  return rets;\n}\n\n\n\nvector<Candidate> composePieces2(Pieces&pieces, vector<pair<Image, Image>> train, vector<point> out_sizes) {\n  vector<Candidate> cands;\n\n  vector<Image> target;\n  for (auto [in,out] : train)\n    target.push_back(out);\n\n  /*\n  for (Piece3&p : pieces.piece) {\n    vector<Image> imgs;\n    //assert(p.ind.size() == pieces.dag.size());\n    int*ind = &pieces.mem[p.memi];\n    for (int i = 0; i < pieces.dag.size(); i++) {\n      //assert(p.ind[i] >= 0 && p.ind[i] < (int)pieces.dag[i].node.size());\n      imgs.push_back(pieces.dag[i].getImg(ind[i]));\n    }\n    cands.emplace_back(imgs, 1, p.depth, p.depth);\n  }\n  */\n\n  for (const Candidate&cand : greedyCompose2(pieces, target, out_sizes)) {\n    cands.push_back(cand);\n  }\n  return cands;\n}\n\n\nvector<Candidate> evaluateCands(const vector<Candidate>&cands, vector<pair<Image,Image>> train) {\n  vector<Candidate> ret;\n  for (const Candidate& cand : cands) {\n    vImage_ imgs = cand.imgs;\n    assert(cand.max_depth >= 0 && cand.max_depth < 100);\n    assert(cand.cnt_pieces >= 0 && cand.cnt_pieces < 100);\n    //cout << cand.max_depth << \' \' << cand.cnt_pieces << endl;\n    double prior = cand.max_depth+cand.cnt_pieces*1e-3;//cnt_pieces;\n\n    int goods = 0;\n    for (int i = 0; i < train.size(); i++) {\n      goods += (imgs[i] == train[i].second);\n    }\n    double score = goods-prior*0.01;\n\n    Image answer = imgs.back();\n    if (answer.w > 30 || answer.h > 30 || answer.w*answer.h == 0) goods = 0;\n    for (int i = 0; i < answer.h; i++)\n      for (int j = 0; j < answer.w; j++)\n\tif (answer(i,j) < 0 || answer(i,j) >= 10) goods = 0;\n\n    if (goods)\n      ret.emplace_back(imgs, score);\n  }\n  sort(ret.begin(), ret.end());\n  //printf("%.20f\\n\\n", ret[0].score);\n  return ret;\n}\n', 'native/src/core_functions.cpp': '/*#include <vector>\n#include <cassert>\n#include <iostream>\n#include <tuple>\n#include <functional>*/\n#include "precompiled_stl.hpp"\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n\nnamespace core {\n  int colMask(Image_ img) {\n    int mask = 0;\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++)\n\tmask |= 1<<img(i,j);\n    return mask;\n  }\n  int countCols(Image_ img, int include0) {//include0 = 0\n    int mask = colMask(img);\n    if (!include0) mask = mask&~1;\n    return __builtin_popcount(mask);\n  }\n  int count(Image_ img) {\n    int ans = 0;\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++)\n\tans += img(i,j) > 0;\n    return ans;\n  }\n\n\n  Image full(point p, point sz, int filling) {//filling = 1\n    Image ret;\n    ret.p = p;\n    ret.sz = sz;\n    ret.mask.assign(ret.h*ret.w, filling);\n    return ret;\n  }\n\n  Image empty(point p, point sz) {\n    return full(p, sz, 0);\n  }\n\n  Image full(point sz, int filling) {//filling = 1\n    Image ret;\n    ret.p = {0,0};\n    ret.sz = sz;\n    ret.mask.assign(ret.h*ret.w, filling);\n    return ret;\n  }\n\n  Image empty(point sz) {\n    return full(point{0,0}, sz, 0);\n  }\n\n\n  bool isRectangle(Image_ a) {\n    return count(a) == a.w*a.h;\n  }\n\n  void countComponents_dfs(Image&img, int r, int c) {\n    img(r,c) = 0;\n    for (int nr = r-1; nr <= r+1; nr++)\n      for (int nc = c-1; nc <= c+1; nc++)\n\tif (nr >= 0 && nr < img.h && nc >= 0 && nc < img.w && img(nr,nc))\n\t  countComponents_dfs(img,nr,nc);\n  }\n\n  int countComponents(Image img) {\n    int ans = 0;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j)) {\n\t  countComponents_dfs(img,i,j);\n\t  /*function<void(int,int)> dfs = [&](int r, int c) {\n\t    if (r < 0 || r >= img.h || c < 0 || c >= img.w || !img(r,c)) return;\n\t    img(r,c) = 0;\n\t    for (int nr : {r-1,r,r+1})\n\t      for (int nc : {c-1,c,c+1})\n\t\tdfs(nr,nc);\n\t  };\n\t  dfs(i,j);*/\n\t  ans++;\n\t}\n      }\n    }\n    return ans;\n  }\n\n\n  char majorityCol(Image_ img, int include0) { //include0 = 0\n    int cnt[10] = {};\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++) {\n\tchar c = img(i,j);\n\tif (c >= 0 && c < 10)\n\t  cnt[c]++;\n      }\n    if (!include0) cnt[0] = 0;\n    int ret = 0;\n    int ma = cnt[ret];\n    for (int c = 1; c < 10; c++) {\n      if (cnt[c] > ma) {\n\tma = cnt[c];\n\tret = c;\n      }\n    }\n    return ret;\n  }\n  Image subImage(Image_ img, point p, point sz) {\n    assert(p.x >= 0 && p.y >= 0 && p.x+sz.x <= img.w && p.y+sz.y <= img.h && sz.x >= 0 && sz.y >= 0);\n    Image ret;\n    ret.p = p+img.p;\n    ret.sz = sz;\n    ret.mask.resize(ret.w*ret.h);\n    for (int i = 0; i < ret.h; i++)\n      for (int j = 0; j < ret.w; j++)\n\tret(i,j) = img(i+p.y, j+p.x);\n    return ret;\n  }\n\n  vector<pair<Image,int>> splitCols(Image_ img, int include0) { //include0 = 0\n    vector<pair<Image,int>> ret;\n    int mask = colMask(img);\n    for (char c = !include0; c < 10; c++) {\n      if (mask>>c&1) {\n\tImage s = img;\n\tfor (int i = 0; i < s.h; i++)\n\t  for (int j = 0; j < s.w; j++)\n\t    s(i,j) = s(i,j) == c;\n\tret.emplace_back(s, c);\n      }\n    }\n    return ret;\n  }\n};\n', 'native/src/count_tasks.cpp': '#include "precompiled_stl.hpp"\n\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "visu.hpp"\n#include "read.hpp"\n\nint main() {\n  vector<Sample> sample = readAll("test", -1);\n  cout << sample.size() << endl;\n}\n', 'native/src/deduce_op.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n\n#include "visu.hpp"\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\n#include "deduce_op.hpp"\n\npair<Image,Image> iOuterProductSI(Image_ img, int w, int h) {\n  if (img.w*img.h <= 0 || img.w%w || img.h%h) return {badImg,badImg};\n  Image big = core::full({img.w/w,img.h/h},-1);\n  Image small = core::full({w,h},-1);\n\n  for (int ii = 0; ii < img.h/h; ii++) {\n    for (int jj = 0; jj < img.w/w; jj++) {\n      int all0 = 1;\n      for (int i = 0; i < h; i++)\n\tfor (int j = 0; j < w; j++)\n\t  if (img(ii*h+i,jj*w+j)) all0 = 0;\n\n      big(ii,jj) = !all0;\n\n      if (!all0) {\n\tfor (int i = 0; i < h; i++) {\n\t  for (int j = 0; j < w; j++) {\n\t    char& a = small(i,j);\n\t    char b = img(ii*h+i,jj*w+j);\n\t    if (a != -1 && a != b) return {badImg,badImg};\n\t    a = b;\n\t  }\n\t}\n      }\n    }\n  }\n  return {big, small};\n}\n\n\npair<Image,Image> iOuterProductIS(Image_ img, int w, int h) {\n  if (img.w*img.h <= 0 || img.w%w || img.h%h) return {badImg,badImg};\n  Image big = core::full({img.w/w,img.h/h},-1);\n  Image small = core::full({w,h},-1);\n\n  for (int ii = 0; ii < img.h/h; ii++) {\n    for (int jj = 0; jj < img.w/w; jj++) {\n      int mask = 0;\n      for (int i = 0; i < h; i++)\n\tfor (int j = 0; j < w; j++)\n\t  mask |= 1<<img(ii*h+i,jj*w+j);\n\n      if (__builtin_popcount(mask&~1) > 1) return {badImg,badImg};\n      big(ii,jj) = 31-__builtin_clz(mask);\n      if (big(ii,jj)) {\n\tfor (int i = 0; i < h; i++) {\n\t  for (int j = 0; j < w; j++) {\n\t    char& a = small(i,j);\n\t    char b = img(ii*h+i,jj*w+j) > 0;\n\t    if (a != -1 && a != b) return {badImg,badImg};\n\t    a = b;\n\t  }\n\t}\n      }\n    }\n  }\n  return {big, small};\n}\n\n\ndeduceOuterProduct::deduceOuterProduct(vector<pair<Image,Image>> train) {\n\n  auto score = [](vector<pair<Image,Image>> pa, int fi) {\n    double ans = 0;\n    for (int k : {0,1}) {\n      vector<Image> imgs;\n      int allequal = 1;\n      for (int ti = 0; ti < pa.size(); ti++) {\n\tImage img = k ? pa[ti].second : pa[ti].first;\n\tif (img.w*img.h <= 0) return 1e9;\n\n\timgs.push_back(img);\n\tif (imgs[0] != imgs.back()) allequal = 0;\n      }\n      if (allequal && imgs.size() > 1) continue;\n\n      for (Image_ img : imgs) {\n\tint cols = __builtin_popcount(core::colMask(img)&~1);\n\tif (cols <= 1 && core::isRectangle(img)) {\n\t  ans += log(img.w+1)+log(img.h+1);\n\t} else if (cols <= 1) {\n\t  ans += log(2)*img.w*img.h;\n\t} else {\n\t  ans += log(10)*img.w*img.h;\n\t}\n      }\n      //ans *= 1-1e-5*1;//(fi*2-1);\n    }\n    return ans;\n  };\n\n\n  int minw = 1e9, minh = 1e9;\n  for (auto [in,out] : train) {\n    minw = min(minw, out.w);\n    minh = min(minh, out.h);\n  }\n\n  /*{\n    Image img = train[0].second;\n    auto [a,b] = iOuterProductSI(img,img.w,img.h);\n    assert(reconstruct(a,b) == img);\n    exit(0);\n    }*/\n\n  rec_funci = -1;\n  double best_score = 1e9;\n  for (int h = 1; h <= minh; h++) {\n    for (int w = 1; w <= minw; w++) {\n      for (int l : {0,1}) {\n\tfor (int k : {0,1}) {\n\t  auto f = k ? iOuterProductSI : iOuterProductIS;\n\t  vector<pair<Image,Image>> is;\n\t  int bad = 0;\n\t  for (auto [in,out] : train) {\n\t    int sw = w, sh = h;\n\t    if (l) {\n\t      if (out.w%w || out.h%h) bad = 1;\n\t      sw = out.w/w;\n\t      sh = out.h/h;\n\t    }\n\t    is.push_back(f(out, sw, sh));\n\t  }\n\t  double entropy = score(is,k);\n\t  if (entropy < best_score) {\n\t    best_score = entropy;\n\t    rec_funci = k;\n\t    train_targets = is;\n\t  }\n\t}\n      }\n    }\n  }\n  assert(rec_funci != -1);\n\n  for (int k : {0,1}) {\n    auto f = k ? iOuterProductSI : iOuterProductIS;\n    vector<double> best_at(train.size(), 1e9);\n    vector<pair<Image,Image>> best_single(train.size(), {badImg,badImg});\n    for (int ti = 0; ti < train.size(); ti++) {\n      Image target = train[ti].second;\n      for (int h = 1; h <= target.h; h++) {\n\tfor (int w = 1; w <= target.w; w++) {\n\t  auto is = f(target, w, h);\n\t  double entropy = score({is},k);\n\t  if (entropy < best_at[ti]) {\n\t    best_at[ti] = entropy;\n\t    best_single[ti] = is;\n\t  }\n\t}\n      }\n    }\n    double entropy = score(best_single,k);\n    if (entropy < best_score) {\n      best_score = entropy;\n      rec_funci = k;\n      train_targets = best_single;\n    }\n  }\n\n  assert(rec_funci != -1);\n  assert(train_targets.size() == train.size());\n  for (int ti = 0; ti < train.size(); ti++) {\n    Image a, b;\n    tie(a,b) = train_targets[ti];\n    //print(a);\n    //print(b);\n    //print(train[ti].second);\n    assert(reconstruct(a,b) == train[ti].second);\n    //cout << "OK" << endl;\n    }\n}\n\n\nImage deduceOuterProduct::reconstruct(Image_ a, Image_ b) {\n  auto f = rec_funci ? outerProductSI : outerProductIS;\n  return f(a,b);\n}\n\nextern int MAXDEPTH;\n\nvoid addDeduceOuterProduct(Pieces&pieces, vector<pair<Image,Image>> train, vector<Candidate>&cands) {\n  deduceOuterProduct deduce_op(train);\n\n  int interestings = 0;\n  for (auto [in,out] : deduce_op.train_targets) {\n    if (core::count(in) > 1 && core::count(out) > 1) interestings++;\n  }\n  if (interestings*2 < train.size()) return;\n\n  vImage a, b;\n  for (int k : {0,1}) {\n    vImage&cand = k ? b : a;\n    int best_match = -1;\n\n    auto add = [&](vImage_ vi) {\n      int matches = 0;\n      for (int i = 0; i < train.size(); i++) {\n\tImage_ target = k ? deduce_op.train_targets[i].second : deduce_op.train_targets[i].first;\n\tmatches += (vi[i] == target);\n      }\n      if (matches > best_match) {\n\tbest_match = matches;\n\tcand = vi;\n      }\n    };\n\n    for (int pi = 0; pi < pieces.mem.size(); pi += pieces.dag.size()) {\n      int*ind = &pieces.mem[pi];\n      //TODO: Use hashes to compare instead of full images\n      vImage imgs;\n      for (int i = 0; i <= train.size(); i++) {\n\tif (pieces.dag[i].tiny_node[ind[i]].isvec) continue;\n\tImage_ img = pieces.dag[i].getImg(ind[i]);\n\timgs.push_back(img);\n\timgs.back().p = point{0,0};\n      }\n      if (imgs.size() == train.size()+1)\n\tadd(imgs);\n    }\n    for (auto [x,y] : deduce_op.train_targets) {\n      add(vImage(train.size()+1, (k ? y : x)));\n    }\n  }\n\n  {\n    assert(a.size() == train.size()+1);\n    assert(b.size() == train.size()+1);\n\n    // TODO: Use correct depths\n    vImage imgs;\n    for (int i = 0; i <= train.size(); i++)\n      imgs.push_back(deduce_op.reconstruct(a[i],b[i]));\n    cands.emplace_back(imgs, 2, MAXDEPTH, MAXDEPTH*2);\n  }\n\n  /*\n  visu.next(to_string(si));\n  for (auto [in,out] : train)\n    visu.add(in,out);\n\n  visu.next(to_string(si)+"!");\n  for (auto [in,out] : deduce_op.train_targets) {\n    visu.add(in,out);\n    }*/\n}\n', 'native/src/efficient.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "brute2.hpp"\n#include "visu.hpp"\n\npair<int,bool> TinyHashMap::insert(ull key, int value) {\n  if (table.size() <= data.size()*sparse_factor) {\n    table.resize(max((int)(table.size() * resize_step), minsz));\n    assert((table.size()&(table.size()-1)) == 0);\n    mask = table.size()-1;\n\n    fill(table.begin(), table.end(), -1);\n    for (int i = 0; i < data.size(); i++) {\n      int&head = table[data[i].key&mask];\n      data[i].next = head;\n      head = i;\n    }\n  }\n  int&head = table[key&mask];\n  int previ = head;\n  while (1) {\n    if (previ == -1) {\n      data.push_back({key, value, head});\n      head = data.size()-1;\n      return {value, true};\n    } else if (data[previ].key == key) {\n      return {data[previ].value, false};\n    }\n    previ = data[previ].next;\n  }\n}\n\n\n\n\nvoid TinyChildren::add(int fi, int to) {\n  assert(fi >= 0);\n  if (sz+1 == dense_thres) {\n    //Convert to dense\n    cap = max(sparse[sz-1].first,fi)+1; //Max fi\n    pair<int,int>*old = sparse;\n    dense = new int[cap];\n    fill_n(dense, cap, None);\n    dense[fi] = to;\n    for (int i = 0; i < sz; i++) {\n      auto [fi, to] = old[i];\n      assert(fi >= 0);\n      assert(fi < cap);\n      dense[fi] = to;\n    }\n    assert(old);\n    delete[]old;\n    sz = dense_thres;\n  }\n\n  if (sz < dense_thres) {\n    //Sparse\n    if (sz+1 > cap) {\n      cap = max((cap+1)*3/2,4);\n      pair<int,int>*old = sparse;\n      sparse = new pair<int,int>[cap];\n      copy_n(old, sz, sparse);\n      if (old) delete[]old;\n    }\n    {\n      int p = sz++;\n      while (p && sparse[p-1].first > fi) {\n\tsparse[p] = sparse[p-1];\n\tp--;\n      }\n      sparse[p] = {fi, to};\n    }\n  } else {\n    //Dense\n    if (cap <= fi) {\n      int oldcap = cap;\n      int*old = dense;\n      cap = (fi+1)*3/2;\n      dense = new int[cap];\n      fill_n(dense+oldcap, cap-oldcap, None);\n      copy_n(old, oldcap, dense);\n      assert(old);\n      delete[]old;\n    }\n    dense[fi] = to;\n  }\n}\nint TinyChildren::get(int fi) {\n  assert(fi >= 0);\n  if (sz < dense_thres) {\n    int low = 0, high = sz-1;\n    while (low <= high) {\n      int mid = (low+high)>>1;\n      int cfi = sparse[mid].first;\n      if (cfi == fi) return sparse[mid].second;\n      else if (cfi > fi) high = mid-1;\n      else low = mid+1;\n    }\n    return None;\n  } else {\n    if (fi >= cap) return None;\n    return dense[fi];\n  }\n}\nvoid TinyChildren::legacy(vector<pair<int,int>>&ret) {\n  if (sz < dense_thres) {\n    //Sparse\n    ret.resize(sz);\n    for (int i = 0; i < sz; i++)\n      ret[i] = sparse[i];\n  } else {\n    //Dense\n    ret.resize(0);\n    for (int i = 0; i < cap; i++) {\n      if (dense[i] != None)\n\tret.emplace_back(i, dense[i]);\n    }\n  }\n}\n\n\n\n\n\n\n\nTinyImage::TinyImage(Image_ img, TinyBank&bank) {\n  for (int c : {img.x, img.y, img.w, img.h}) {\n    assert(c >= -128 && c < 128);\n  }\n  x = img.x, y = img.y, w = img.w, h = img.h;\n\n  int freq[10] = {};\n  for (char c : img.mask) {\n    assert(c >= 0 && c < 10);\n    freq[c]++;\n  }\n\n  priority_queue<pair<int,int>> pq;\n  for (int d = 0; d < 10; d++) {\n    if (freq[d]) {\n      pq.emplace(-freq[d], -d);\n      //cout << d << ": " << freq[d] << endl;\n    }\n  }\n  while (pq.size() < 2) pq.emplace(0,0);\n\n  int nodes = pq.size()-1;\n  int pos = nodes-1;\n  auto convert = [](int a, int p) {\n    if (a <= 0) return -a;\n    else {\n      assert(9+a-p >= 10 && 9+a-p < 16);\n      return 9+a-p;\n    }\n  };\n  while (pq.size() > 1) {\n    auto [mfa,a] = pq.top(); pq.pop();\n    auto [mfb,b] = pq.top(); pq.pop();\n    tree[pos] = convert(a,pos) | convert(b,pos) << 4;\n    pq.emplace(mfa+mfb, pos--);\n  }\n  int code[10] = {}, codelen[10] = {};\n  int path[10] = {}, pathlen[10] = {};\n  for (int p = 0; p < nodes; p++) {\n    for (int k : {0,1}) {\n      int child = tree[p]>>k*4&15;\n      int newpath = path[p] | k<<pathlen[p];\n      if (child < 10) {\n\tcode[child] = newpath;\n\tcodelen[child] = pathlen[p]+1;\n      } else {\n\tchild += p-9;\n\tpath[child] = newpath;\n\tpathlen[child] = pathlen[p]+1;\n      }\n    }\n    //cout << p << ": " << (int)tree[p][0] << \' \' << (int)tree[p][1] << endl;\n  }\n  /*for (int d = 0; d < 10; d++) {\n    cout << d << ": " << bitset<4>(code[d]) << \' \' << codelen[d] << endl;\n    }*/\n\n  assert((bank.curi+align-1)/align < 1ll<<32);\n  memi = (bank.curi+align-1)/align;\n  sz = 0;\n  ll memstart = (ll)memi*align;\n  bank.alloc();\n  for (char c : img.mask) {\n    bank.set(memstart+ sz, code[c], codelen[c]);\n    sz += codelen[c];\n    //for (int i = 0; i < codelen[c]; i++)\n    //bank.set(memstart+ sz++, code[c]>>i&1);\n  }\n  /*for (int it = 0; it < 10; it++)\n    cout << (int)img.mask[it] << \' \';\n    cout << endl;\n    for (int it = 0; it < 20; it++)\n    cout << bank.get(memi+it);\n    cout << endl;*/\n  bank.curi = memstart+sz;\n}\n\n\nImage TinyImage::decompress(TinyBank&bank) {\n  Image ret;\n  ret.x = x, ret.y = y, ret.w = w, ret.h = h;\n  ret.mask.resize(ret.w*ret.h);\n  int treep = 0, maski = 0;\n  ll memstart = (ll)memi*align;\n  for (ll i = memstart; i < memstart+sz; i++) {\n    int bit = bank.get(i);\n    int child = tree[treep]>>bit*4&15;\n    if (child < 10) {\n      ret.mask[maski++] = child;\n      treep = 0;\n    } else\n      treep += child-9;\n  }\n  //cout << maski << \' \' << ret.w*ret.h << endl;\n  assert(maski == ret.w*ret.h);\n  assert(treep == 0);\n  return ret;\n}\n', 'native/src/evals.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "read.hpp"\n#include "visu.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "normalize.hpp"\n#include "spec.hpp"\n#include "image_functions2.hpp"\n\n\n\nint rcDiff(Image_ img) {\n  vector<int> row(img.h,1), col(img.w,1);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      if (img(i,j) == 0) row[i] = col[j] = 0;\n  int ans = 0;\n  for (int i = 0; i < img.h; i++)\n    ans += row[i];\n  for (int j = 0; j < img.w; j++)\n    ans -= col[j];\n  return ans;\n}\n\n\nSimplifier normalizeOrient() {\n  Simplifier sim;\n  sim.in = [](Image_ in) {\n    return rcDiff(in) >= 0 ? in : rigid(in,6);\n  };\n  sim.out = [](Image_ in, Image_ out) {\n    return rcDiff(in) >= 0 ? out : rigid(out,6);\n  };\n  sim.rec = [](Image_ in, Image_ out) {\n    return rcDiff(in) >= 0 ? out : rigid(out,6);\n  };\n  return sim;\n}\n\n\n#define VECIFY(func)\t\t\t\t\\\n  vImage func(vImage_ vimg) {\t\t\t\\\n    vImage ret(vimg.size());\t\t\t\\\n    for (int i = 0; i < vimg.size(); i++) \t\\\n      ret[i] = func(vimg[i]);\t\t\t\\\n    return ret;\t\t\t\t\t\\\n  }\nVECIFY(hull);\n\n#define FOREACH(content)\t\t\t\\\n  vImage ret(vi.size());\t\t\t\\\n  for (int i = 0; i < vi.size(); i++)\t\t\\\n    ret[i] = content;\t\t\t\t\\\n  return ret;\n\nvImage count(vImage_ vi, int a, int b) {\n  FOREACH(count(vi[i],a,b));\n}\n\n\n\n\n\nImage solveEval(Image in, vector<pair<Image,Image>> train, int taski) {\n  //Skipped  23,40,53,58,90\n  //Error in train  7, 27, 44/45, 77\n  auto sim = normalizeCols(train);\n  auto in0 = in;\n\n  if (taski == 0) {\n    vImage out = gravity(in, 0);\n    return embed(composeGrowing(out),in);\n\n  } else if (taski == 1) {\n    in = sim.in(in);\n\n    vImage inside = insideMarked(in);\n    Image out = pickMax(inside,0);\n    return sim.rec(in0,out);\n\n  } else if (taski == 2) {\n    in = eraseCol(in, 2);\n    return greedyFillBlack(in);\n\n  } else if (taski == 3) {\n    Image a = eraseCol(cutIndex(in,0), 2);\n    Image b = cutIndex(in,1);\n    Image c = cutIndex(in,2);\n    Image out = broadcast(b,c);\n    out = replaceTemplate(out,hull(a),a);\n    return compose(c, out);\n\n  } else if (taski == 4) {\n    Image part = rigid(cutPickMax(in, 4), 4);\n    return replaceTemplate(in, invert(part), part);\n\n  } else if (taski == 5) {\n    in = sim.in(in);\n\n    Image dot = compress(filterCol(in, 1));\n    Image out = repeat(dot, in, 1);\n    return sim.rec(in0, compose(out, filterCol(in, 2)));\n\n  } else if (taski == 6) {\n    Simplifier orient = normalizeOrient();\n    in = orient.in(in);\n    Image base = colShape(interior(filterCol(in, 3)),1);\n    base = smear(base, 5);\n    Image cross = compose(filterCol(in,2), base, 2);\n    cross = makeBorder(cross);\n    Image out = compose(compose(compose(base, in), cross), filterCol(in,3));\n    return orient.rec(in0, out);\n\n  } else if (taski == 7) {\n    Image out = repeat(in, core::full({in.w*3,in.h*3}));\n    Image green = colShape(out, 3);\n    Image blue = colShape(smear(out, 4), 1);\n    Image a = Move(green, Pos(1,1));\n    Image b = Move(green, Pos(-1,-1));\n    return compose(blue, compose(out, compose(a, b), 3));\n\n  } else if (taski == 8) {\n    in = sim.in(in);\n    Image squares = interior(filterCol(in,1));\n    Image light = composeGrowing(pickNotMaxes(cut(in), 11));\n    Image border = makeBorder(filterCol(in,1));\n    Image out = composeGrowing({in, colShape(border,2), colShape(light,5), colShape(squares,4)});\n    return sim.rec(in0,out);\n\n  } else if (taski == 9) {\n    Image out = in;\n    for (Image_ img : cut(in)) {\n      Image dot = embed(img, out);\n      out = compose(smear(dot, img.x < in.w-img.x ? 1 : 0), out);\n      out = compose(smear(dot, img.y < in.h-img.y ? 3 : 2), out);\n    }\n    return out;\n\n  } else if (taski == 10) {\n    Image sz = train[0].second;//core::full(in.w*5,in.h);\n    return mirror(in, sz);\n\n  } else if (taski == 11) {\n    return spreadCols(in,1);\n\n  } else if (taski == 12) {\n    in = sim.in(in);\n\n    Image swapcol = compose(colShape(filterCol(in,1),3),\n\t\t\t    colShape(filterCol(in,3),1));\n\n    Image mirror = align(rigid(in, 8), in);\n    Image out = compose(swapcol, mirror, 3);\n    return sim.rec(in0, out);\n\n  } else if (taski == 13) {\n    return greedyFillBlack(in);\n\n  } else if (taski == 14) {\n    Image out = badImg;\n    for (Image img : splitCols(in)) {\n      img = compress(img);\n      Image add = border(count(img,5,0));\n      out = compose(out, align(add, out, 2, 2));\n    }\n    return out;\n\n  } else if (taski == 15) {\n    in = eraseCol(in, 8);\n    return compress3(compress2(in));\n\n  } else if (taski == 16) {\n    Image out = in;\n    Image pat = cutIndex(train[0].second,0);\n    pat = spreadCols(eraseCol(pat,4));\n    for (Image_ img : cut(in)) {\n      Image base = broadcast(pat,img);\n      base = compose(base, colShape(interior2(interior2(img)), 4));\n      out = compose(out, base);\n    }\n    return out;\n\n  } else if (taski == 17) {\n    in = sim.in(in);\n    point ins = compress(interior(filterCol(in, 2))).sz;\n    int borders = ins.x == ins.y ? ins.x : 2;\n    for (int i = 0; i < borders; i++)\n      in = compose(in,makeBorder(in, 1));\n    return sim.rec(in0, in);\n\n  } else if (taski == 18) {\n    return compress(compose(rigid(in,2),in,4));\n\n  } else if (taski == 19 || taski == 20) {\n    in = sim.in(in);\n    Image base = sim.out(train[0].first, train[0].second);\n    Image out = Move(base, outerProductIS(Move(compress(in), Pos(-1,-1)),Square(4)));\n    out = extend2(out, base);\n    return sim.rec(in0,out);\n\n  } else if (taski == 21) {\n    Image parts = smear(connect(in, 0), 3);\n    Image bottom = smear(embed(pickMax(splitRows(in),6),in), 3);\n    parts = compose(parts,bottom,2);\n    return compose(in, compose(bottom, parts));\n\n  } else if (taski == 22) {\n    Image marked = compose(in, hull(compress(filterCol(in,4))), 2);\n    Image need = eraseCol(marked, 4);\n    return replaceTemplate(in, need, marked);\n\n  } else if (taski == 23) { //Wrong\n    in = eraseCol(in, 5);\n    Image ret = invert(hull(in));\n    vImage s = splitAll(in);\n    for (Image_ p : s) {\n      for (Image_ q : s) {\n\tif (p.sz == q.sz && !core::count(p) && core::isRectangle(q)) {\n\t  ret = compose(ret, align(q,p,2,2));\n\t}\n      }\n      if (p.sz != in.sz && core::count(interior(p)))\n\tret = compose(ret, p);\n    }\n    return ret;\n\n  } else if (taski == 24) {\n    Image a = myStack(in,rigid(in,1),0);\n    Image b = myStack(rigid(in,2),rigid(in,3),0);\n    return myStack(a,b,1);\n\n  } else if (taski == 25) {\n    in = sim.in(in);\n    in = eraseCol(in,1);\n    vImage cols = splitCols(in);\n    vImage lens = count(cols, 2,1);\n    Image out = myStack(lens, 1);\n    return sim.rec(in0,out);\n\n  } else if (taski == 26) {\n    in = sim.in(in);\n    Image out = eraseCol(eraseCol(in,1),2);\n    out = connect(out,2);\n    out = compose(out,in);\n    return sim.rec(in0, out);\n\n  } else if (taski == 27) {\n    in = sim.in(in);\n    Image out = compose(in, filterCol(rigid(in,4),1));\n    return sim.rec(in0, out);\n\n  } else if (taski == 28) {\n    return compress3(in);\n\n  } else if (taski == 29) {\n    in = sim.in(in);\n\n    int blue = core::majorityCol(in);\n    int red = 3-blue;\n    Image out = eraseCol(in,blue);\n    vImage to = cut(filterCol(in,3));\n    for (Image img : to) {\n      img = compress(img);\n      out = replaceTemplate(out,colShape(img,red),img,2);\n    }\n    out = compose(out,filterCol(in,blue));\n    return sim.rec(in0,out);\n\n  } else if (taski == 30) {\n    return sim.rec(in0,Col(1));\n\n  } else if (taski == 31) {\n    in = sim.in(in);\n    in = filterCol(in,2);\n    Image out = compose(colShape(smear(in,2),1), in);\n    return sim.rec(in0,out);\n\n  } else if (taski == 32) {\n    Image gray = cutPickMax(in,0);\n    Image cols = eraseCol(gray,5);\n\n    Image shapes = compress(compose(in, gray, 4));\n\n    cols = spreadCols(broadcast(cols,shapes));\n    return embed(compose(replaceCols(shapes,cols), gray), in);\n\n  } else if (taski == 33) {\n    return compose(border(colShape(compose(smear(in,4), smear(in,5), 2), 8)), in);\n\n  } else if (taski == 34) {\n    Image blue = filterCol(in,8);\n    Image squares = Fill(blue);\n\n    Image dots = eraseCol(in,8);\n    Image out = blue;\n    for (Image_ img : splitCols(dots)) {\n      Image part = compress(compose(hull(compress(img)), squares, 2));\n      part = makeBorder2(part, 1);\n      out = compose(part, out);\n    }\n    return out;\n\n  } else if (taski == 35) {\n    return compose(colShape(hull(in),5),Move(in,Pos(-1,0)),3);\n\n  } else if (taski == 36) {\n    in = eraseCol(in,3);\n    in = compose(in,rigid(in,2));\n    in = compose(in,rigid(in,4));\n    return in;\n\n  } else if (taski == 37) {\n    in = eraseCol(in,3);\n\n    Image out = compose(in,rigid(in,2));\n    out = compose(out,rigid(out,4));\n    return compress(compose(out,invert(in),2));\n\n  } else if (taski == 38) {\n    return outerProductIS(in,count(in,1,0));\n\n  } else if (taski == 39) {\n    Image out = in;\n    for (Image_ img : cut(colShape(in,1))) {\n      Image part = compose(in, makeBorder2(img,0), 2);\n      part = eraseCol(part, 5);\n      out = compose(broadcast(majCol(part), img), out);\n    }\n    return out;\n\n  } else if (taski == 40) { //Wrong\n\n    return in;\n\n  } else if (taski == 41) {\n    return outerProductSI(hull(in),in);\n\n  } else if (taski == 42) {\n    Image marked = compress(filterCol(in,8));\n    Image out = replaceTemplate(in, colShape(marked,1), marked);\n    Image marks = filterCol(out,8);\n    Image orange = compose(colShape(connect(marks,2),7), in, 2);\n    return compose(out, compose(orange, marks));\n\n  } else if (taski == 43) {\n    in = sim.in(in);\n    Image bg = compress(filterCol(in,1));\n    Image out = bg;\n    for (Image_ img : cut(in)) {\n      Image need = makeBorder2(colShape(invert(img),1),0);\n      out = replaceTemplate(out, need, makeBorder2(img,0));\n    }\n    out = compose(out,bg);\n    return sim.rec(in0, out);\n\n  } else if (taski == 44 || taski == 45) {\n    Image shape = compress(filterCol(in,8));\n    Image room = invert(Square(3));\n    shape = embed(align(shape,room,1,3), room);\n    return outerProductSI(count(filterCol(in,4),2,1),shape);\n\n  } else if (taski == 46) {\n    Image squares = colShape(repeat(hull(cutPickMax(in,0)), in, 1),2);\n    return compose(squares,in,4);\n\n  } else if (taski == 47 || taski == 48) {\n    Image a = compress(filterCol(in,6));\n    Image b = toOrigin(compress(filterCol(in,5)));\n    return colShape(invert(compose(a,b)),4);\n\n  } else if (taski == 49) {\n    vImage cols = splitCols(in);\n    vImage lens = count(cols, 2,1);\n    Image out = myStack(lens,1);\n    return rigid(out,2);\n\n  } else if (taski == 50) {\n    Image shapes = stackLine(cut(filterCol(in,6)));\n    Image cols = stackLine(cut(eraseCol(in,6)));\n    return compose(broadcast(cols, shapes), shapes, 2);\n\n  } else if (taski == 51) {\n    Image out = replaceTemplate(in, invert(Square(3)), Square(3));\n    out = replaceTemplate(out, invert(Square(2)), Square(2));\n    return out;\n\n  } else if (taski == 52) {\n    Image green = filterCol(in,3);\n    vImage hulls = hull( cut(green) );\n    Image filled = embed(compose(hulls,0),in);\n    Image specials = compose(filterCol(in,2), filled, 2);\n    return compose(in, makeBorder(specials));\n\n  } else if (taski == 53) { //Wrong\n    Image center = interior(in);\n    Image base = train[1].second;\n    base = compose(base, interior(base));\n    base = embed(align(base,center),in);\n    return greedyFillBlack(base);\n\n  } else if (taski == 54) {\n    Image out = toOrigin(cutPickMax(in,0));\n    for (Image_ pa : cut(cutPickMaxes(in,1))) {\n      Image a = half(pa,0);\n      Image b = half(pa,1);\n      out = swapTemplate(out,a,b);\n    }\n    return out;\n\n  } else if (taski == 55 || taski == 56) {\n    Image a = half(in, 2);\n    Image b = toOrigin(half(in, 3));\n    Image or_ = compose(a,b);\n    Image and_= compose(a,b,2);\n    return colShape(compose(or_,and_,4),6);\n\n  } else if (taski == 57) {\n    Image out = toOrigin(cutPickMaxes(in,0));\n    for (Image_ img : pickNotMaxes(cut(in),0)) {\n      out = compose(out,align(img,out));\n    }\n    return out;\n\n  } else if (taski == 58) { //Wrong\n\n\n  } else if (taski == 59) {\n    return compose(smear(eraseCol(in,5),0),in,2);\n\n  } else if (taski == 60) {\n    in = eraseCol(in,6);\n    in = compose(in,rigid(in,6));\n    in = greedyFillBlack(in);\n    return in;\n\n  } else if (taski == 61) {\n    in = sim.in(in);\n    Image a = filterCol(in,2);\n    Image b = filterCol(in,1);\n\n    Image out = Square(3);\n    a = compress(wrap(count(interior(a),0,1),out));\n    b = compress(wrap(count(interior(b),0,1),out));\n    out = embed(toOrigin(myStack(a, b, 1)), out);\n    return sim.rec(in0,out);\n\n  } else if (taski == 62) {\n    Image out = in;\n    int col = 1;\n    for (Image_ img : pickNotMaxes(splitColumns(in),0)) {\n      out = compose(colShape(hull(img), col++), out);\n    }\n    return out;\n\n  } else if (taski == 63) {\n    Image b = half(compress(in), 3);\n    return compose(in, colShape(b,2));\n\n  } else if (taski == 64) {\n    in = sim.in(in);\n\n    int flip = 0;\n    if (core::count(half(in,2)) < core::count(half(in,3))) flip = 1;\n    if (flip) in = rigid(in,5);\n\n    Image both = compose(half(in,2),rigid(in,5),2);\n    Image out = smear(filterCol(both,2),2);\n    out = compose(colShape(out,1), in);\n\n    if (flip) out = rigid(out,5);\n    return sim.rec(in0, out);\n\n  } else if (taski == 65) {\n    Image out = in;\n    for (Image img : cut(in)) {\n      Image piece = img;\n      img = eraseCol(img, 4);\n      piece = makeBorder2(piece, count(img,0,0));\n      out = compose(out, piece);\n    }\n    return out;\n\n  } else if (taski == 66) {\n    Image marked = compress(cutPickMax(in, filterCol(in,5), 1));\n    Image out = replaceTemplate(in, marked, Col(0));\n    return compose(marked, out);\n\n  } else if (taski == 67) {\n    Image marked = compose(pickNotMaxes(splitAll(invert(in)),1),0);\n    return compose(colShape(marked,8),in);\n\n  } else if (taski == 68 || taski == 69) {\n    Image center = interior2(in);\n    return compose(spreadCols(in), smear(center,14), 2);\n\n  } else if (taski == 70) {\n    vImage outs = cut(in);\n    Image out = stackLine(outs);\n    return out;\n\n  } else if (taski == 71) {\n    Image to = colShape(Square(4),4);\n    for (int c = 1; c < 10; c++) {\n      in = replaceTemplate(in,colShape(to,c),to, 1);\n    }\n    return in;\n\n  } else if (taski == 72) {\n    Image a = compress(in);\n    int other_col = 6+3-core::majorityCol(a);\n    Image b = toOrigin(colShape(rigid(a,6),other_col));\n    Image c = myStack(a,b,2);\n    c = myStack(myStack(c,c,2),c,2);\n\n    a = colShape(a,1);\n    c = compose(c,a);\n    c = compose(c,align(rigid(c,4),c));\n    c = compose(c,align(rigid(c,5),c));\n    return embed(compose(c,in),in);\n\n  } else if (taski == 73) {\n    in = rigid(in,2);\n    return mirror(in,outerProductIS(in,Square(2)));\n\n  } else if (taski == 74) {\n    in = sim.in(in);\n    Image shapes = repeat(pickMax(cut(in),0),in,1);\n    Image grid = filterCol(in,1);\n    Image cols = spreadCols(eraseCol(eraseCol(in,1),2));\n    Image out = compose(compose(cols, shapes, 2), grid);\n    return sim.rec(in0,out);\n\n  } else if (taski == 75) {\n    Image out = in;\n    for (Image_ img : cut(in)) {\n      int ccs = core::countComponents(invert(img));\n      int cols[] = {0,1,2,3,7};\n      out = compose(out, colShape(img,cols[ccs]));\n    }\n    return out;\n\n  } else if (taski == 76) {\n    Image inside = cutPickMax(in,filterCol(in,5),1);\n    Image out = in;\n    for (Image img : splitCols(inside)) {\n      img = compress(img);\n      Image green = colShape(img,3);\n      out = replaceTemplate(out,green,img,2, 1);\n    }\n    return out;\n\n  } else if (taski == 77) {\n    Image a = compress(in);\n    for (int i = 0; i < a.h; i++) {\n      for (int j = 0; j < a.w; j++) {\n\tint d = min({i,j,a.h-1-i,a.w-1-j});\n\ta(i,j) = d%2 == 0 ? 5 : d%4 == 1 ? 2 : 0;\n      }\n    }\n    return embed(a,in);\n\n  } else if (taski == 78) {\n    int W = cutIndex(in,0).w;\n\n    Image out = in;\n    for (Image_ img : cut(in)) {\n      map<Image,int> cnt;\n      for (int d = 0; d < 4; d++) {\n\tint nx = img.x+(W+1)*((d==0)-(d==1));\n\tint ny = img.y+(W+1)*((d==2)-(d==3));\n\n\tif (nx >= 0 && ny >= 0 && nx+4 <= in.w && ny+4 <= in.h) {\n\t  Image r = core::subImage(in,{nx,ny},{W,W});\n\t  if (core::count(r) > 0) {\n\t    int rs[] = {4,4,5,5};\n\t    Image here = rigid(r,rs[d]);\n\t    here.p = img.p;\n\t    ++cnt[here];\n\t  }\n\t}\n      }\n      for (auto [here,c] : cnt) {\n\tif (c == 2)\n\t  out = compose(out, here);\n      }\n    }\n    return out;\n\n  } else if (taski == 79) {\n    Image out = in;\n    for (Image_ img : cut(in)) {\n      out = compose(out, repeat(compress(img),img));\n    }\n    return out;\n\n  } else if (taski == 80) {\n    Image pat = compress(compose(pickNotMaxes(splitAll(in),0),0));\n    return extend2(Move(pat,Pos(-1,0)),in);\n\n  } else if (taski == 81) {\n    Image dots = cutPickMaxes(in,1);\n    Image out = in;\n    for (Image_ dot : cut(dots)) {\n      for (Image_ img : cut(in)) {\n\tif (img.w > 1) {\n\t  out = compose(out, align(img, dot), 3);\n\t}\n      }\n    }\n    return out;\n\n  } else if (taski == 82) {\n    return outerProductSI(in,invert(in));\n\n  } else if (taski == 83) {\n    Image out = hull0(in);\n    for (Image_ img : splitCols(in)) {\n      Image top = cutPickMax(img,5);\n      Image side = cutPickMax(img,6);\n      Image add;\n      if (top.p == side.p)\n\tadd = smear(embed(top,in),2);\n      else {\n\tadd = compose(smear(embed(top,in),2),\n\t\t      smear(embed(side,in),4));\n\tadd = compose(add, hull(compress(img)),2);\n      }\n      out = compose(add, out);\n    }\n    return out;\n\n  } else if (taski == 84) {\n    Image out = in;\n    for (Image_ img : cut(in)) {\n      out = compose(out, colShape(majCol(eraseCol(img,1)), interior2(img)));\n    }\n    return out;\n\n  } else if (taski == 85) {\n    Image cols = rigid(cutPickMax(in,1), 3);\n    Image shape = compress(filterCol(in,8));\n    return compose(in,compose(broadcast(cols,shape), shape, 2));\n\n  } else if (taski == 86) {\n    Image out = hull0(in);\n    for (Image_ img : cut(in)) {\n      Image yellow = filterCol(img,4);\n      Image line = cutPickMax(yellow,invert(yellow),0);\n      vImage splitted = cut(compose(img,invert(embed(line,img)),2));\n      Image c = compose(pickUnique(splitted,0), line);\n      out = compose(out, line);\n      out = compose(out,c);\n      out = compose(out,mirror2(c,line));\n    }\n    return out;\n\n  } else if (taski == 87) {\n    return outerProductSI(filterCol(in,5),in);\n\n  } else if (taski == 88) {\n    Image from = Square(2);\n    in = replaceTemplate(in,from,invert(from),1);\n    return greedyFillBlack(in);\n\n  } else if (taski == 89) {\n    Image black = badImg;\n    Image out = badImg;\n    for (Image_ img : pickMaxes(splitAll(in),0)) {\n      Image plus = smear(embed(interior2(img),in),6);\n      Image both = compose(plus,out,2);\n      black = compose(black,both);\n      out = compose(out,plus);\n    }\n    return compose(compose(in,out),black,4);\n\n  } else if (taski == 90) { //Wrong\n\n  } else if (taski == 91 || taski == 92) {\n    Image shape = compress(filterCol(in,1));\n    return outerProductIS(compress2(eraseCol(in,1)),shape);\n\n  } else if (taski == 93 || taski == 94) {\n    int col = core::majorityCol(in);\n    if (col == 1) return train[0].second;\n    else if (col == 2) return train[1].second;\n    else if (col == 3) return train[2].second;\n    else return badImg;\n\n  } else if (taski == 95) {\n    Image cols = eraseCol(in,5);\n    Image gray = filterCol(in,5);\n    cols = compose(cols,rigid(cols,2));\n    cols = spreadCols(cols);\n    return compose(colShape(cols, Fill(in)),gray);\n\n  } else if (taski == 96) {\n    Image side = cutPickMax(in,0);\n    Image out = in;\n    Image red = colShape(Square(5),2);\n    for (Image_ img : cut(side)) {\n      Image a = colShape(majCol(img),red);\n      Image b = compose(red,align(colShape(img,1),red,2,2));\n      out = swapTemplate(out,a,b,1);\n    }\n    return out;\n\n  } else if (taski == 97) {\n    Image a = half(in,2);\n    Image b = toOrigin(half(in,3));\n    return colShape(compose(a,b),4);\n\n  } else if (taski == 98) {\n    Image out = hull0(in);\n    for (Image img : splitCols(in)) {\n      img = compress(img);\n      int col = core::majorityCol(img);\n      if (col == 8)\n\timg.x = 1;\n      else if (col == 2)\n\timg.x = 2;\n      else if (col == 4)\n\timg.x = 3;\n      else if (col == 3)\n\timg.x = 4;\n      out = compose(out,img);\n    }\n    return out;\n\n  } else if (taski == 99) {\n    in = sim.in(in);\n    Image out = filterCol(in,2);\n    for (Image_ img : insideMarked(in)) {\n      Image a = compose(rigid(img,4),img);\n      Image b = compose(rigid(img,5),img);\n      out = compose(out, img.w > img.h ? a : b);\n    }\n    return sim.rec(in0,out);\n  }\n\n  return in;\n}\n\n\nvoid evalEvals(int print = 1) {\n  vector<Sample> sample = readAll("evaluation", -1);\n  int samples = sample.size();\n  sample = vector<Sample>(sample.begin()+samples-100,sample.end());\n\n  Visu visu;\n\n  int corrects = 0;\n  for (int si = 0; si < sample.size(); si++) {\n    Sample&s = sample[si];\n    visu.next(s.id);\n    int ok = 1;\n    for (auto [in,out] : s.train) {\n      Image pred = toOrigin(solveEval(in, s.train, si));\n      ok &= (pred == out);\n      auto sim = normalizeCols(s.train);\n      auto in0 = in;\n      tie(in,out) = sim(in,out);\n      visu.add(in, out);\n      visu.add(in, sim(in0,pred).second);\n    }\n    corrects += ok;\n    if (print)\n      cout << "Task " << si << ": " << (ok ? "OK" : "Failed") << endl;\n  }\n\n  if (print) {\n    cout << corrects << " / " << sample.size() << endl;\n    exit(0);\n  }\n  if (!print)\n    assert(corrects == 95);\n}\n', 'native/src/image_functions.cpp': '#include "precompiled_stl.hpp"\n/*#include <vector>\n#include <cassert>\n#include <queue>\n#include <tuple>\n#include <functional>*/\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n\nImage Col(int id) {\n  assert(id >= 0 && id < 10);\n  return core::full({0,0}, {1,1}, id);\n}\nImage Pos(int dx, int dy) {\n  return core::full({dx,dy},{1,1});\n}\nImage Square(int id) {\n  assert(id >= 1);\n  return core::full({0,0}, {id,id});\n}\nImage Line(int orient, int id) {\n  assert(id >= 1);\n  int w = id, h = 1;\n  if (orient) swap(w,h);\n  return core::full({0,0}, {w,h});\n}\n\nImage getPos(Image_ img) {\n  return core::full(img.p, {1,1}, core::majorityCol(img));\n}\nImage getSize(Image_ img) {\n  return core::full({0,0}, img.sz, core::majorityCol(img));\n}\nImage hull(Image_ img) {\n  return core::full(img.p, img.sz, core::majorityCol(img));\n}\nImage toOrigin(Image img) {\n  img.p = {0,0};\n  return img;\n}\n\nImage getW(Image_ img, int id) {\n  return core::full({0,0}, {img.w,id ? img.w : 1}, core::majorityCol(img));\n}\nImage getH(Image_ img, int id) {\n  return core::full({0,0}, {id ? img.h : 1, img.h}, core::majorityCol(img));\n}\n\nImage hull0(Image_ img) {\n  return core::full(img.p, img.sz, 0);\n}\nImage getSize0(Image_ img) {\n  return core::full({0,0}, img.sz, 0);\n}\n\n\n\nImage Move(Image img, Image_ p) {\n  img.x += p.x;\n  img.y += p.y;\n  return img;\n}\n\nImage filterCol(Image_ img, Image_ palette) {\n  Image ret = img;\n  int palMask = core::colMask(palette);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      if ((palMask>>img(i,j)&1) == 0)\n\tret(i,j) = 0;\n  return ret;\n}\n\nImage filterCol(Image_ img, int id) {\n  assert(id >= 0 && id < 10);\n  if (id == 0) return invert(img);\n  else return filterCol(img, Col(id));\n}\n\n\n\nImage broadcast(Image_ col, Image_ shape, int include0) { //include0 = 1\n  if (col.w*col.h == 0 || shape.w*shape.h == 0) return badImg;\n\n  if (shape.w%col.w == 0 && shape.h%col.h == 0) {\n    Image ret = shape;\n    int dh = shape.h/col.h, dw = shape.w/col.w;\n    for (int ii = 0; ii < col.h; ii++) {\n      for (int jj = 0; jj < col.w; jj++) {\n\tint c = col(ii,jj);\n\tfor (int i = ii*dh; i < ii*dh+dh; i++)\n\t  for (int j = jj*dw; j < jj*dw+dw; j++)\n\t    ret(i,j) = c;\n      }\n    }\n    return ret;\n  }\n\n  Image ret = shape;\n  double fh = col.h*1./shape.h, fw = col.w*1./shape.w;\n\n  const double eps = 1e-9;\n  double w0[10] = {};\n  for (int c : col.mask) w0[c] += 1e-6;\n\n  double tot = fh*fw;\n  double weight[10];\n  for (int i = 0; i < shape.h; i++) {\n    for (int j = 0; j < shape.w; j++) {\n      copy_n(w0, 10, weight);\n\n      double r0 = i*fh+eps, r1 = (i+1)*fh-eps;\n      double c0 = j*fw+eps, c1 = (j+1)*fw-eps;\n\n      int guess = !include0;\n      for (int y = r0; y < r1; y++) {\n\tdouble wy = min((double)y+1,r1)-max((double)y,r0);\n\tfor (int x = c0; x < c1; x++) {\n\t  double wx = min((double)x+1,c1)-max((double)x,c0);\n\t  char c = col(y,x);\n\t  weight[c] += wx*wy;\n\t  guess = c;\n\t}\n      }\n\n      if (weight[guess]*2 > tot) {\n\tret(i,j) = guess;\n\tcontinue;\n      }\n\n      int maj = !include0;\n      double w = weight[maj];\n      for (int c = 1; c < 10; c++) {\n\tif (weight[c] > w) maj = c, w = weight[c];\n      }\n      ret(i,j) = maj;\n      //point sz = {max(c1-c0, 1), max(r1-r0, 1)};\n      //ret(i,j) = core::majorityCol(core::subImage(col, {c0,r0}, sz), include0);\n    }\n  }\n  return ret;\n}\n\nImage colShape(Image_ col, Image_ shape) {\n  if (shape.w*shape.h == 0 || col.w*col.h == 0) return badImg;\n  Image ret = broadcast(col, getSize(shape));\n  ret.p = shape.p;\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      if (!shape(i,j)) ret(i,j) = 0;\n  return ret;\n}\nImage colShape(Image_ shape, int id) {\n  assert(id >= 0 && id < 10);\n  Image ret = shape;\n  for (char&c : ret.mask)\n    c = c ? id : 0;\n  return ret;\n}\n\n\n\nImage compress(Image_ img, Image_ bg) { // bg = Col(0)\n  int bgmask = core::colMask(bg);\n\n  int xmi = 1e9, xma = 0, ymi = 1e9, yma = 0;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if ((bgmask>>img(i,j)&1) == 0) {\n\txmi = min(xmi, j);\n\txma = max(xma, j);\n\tymi = min(ymi, i);\n\tyma = max(yma, i);\n      }\n    }\n  }\n  Image ret;\n  if (xmi == 1e9) {\n    ret.p = {0,0};\n    ret.sz = {0,0};\n    return ret;\n  }\n  ret.p = img.p + point{xmi, ymi};\n  ret.sz = {xma-xmi+1, yma-ymi+1};\n  ret.mask.resize(ret.h*ret.w);\n  for (int i = ymi; i <= yma; i++) {\n    for (int j = xmi; j <= xma; j++) {\n      ret(i-ymi,j-xmi) = img(i,j);\n    }\n  }\n  return ret;\n}\n\n\n\nImage embedSlow(Image_ img, Image_ shape) {\n  Image ret = core::empty(shape.p, shape.sz);\n  point d = shape.p-img.p;\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      ret(i,j) = img.safe(i+d.y, j+d.x);\n  return ret;\n}\n\nImage embed(Image_ img, Image_ shape) {\n  Image ret = core::empty(shape.p, shape.sz);\n  point d = shape.p-img.p;\n  int sx = max(0,-d.x);\n  int sy = max(0,-d.y);\n  int ex = min(ret.w,img.w-d.x);\n  int ey = min(ret.h,img.h-d.y);\n\n  int retw = ret.w, imgw = img.w, off = d.y*img.w+d.x;\n  for (int i = sy; i < ey; i++)\n    for (int j = sx; j < ex; j++)\n      ret.mask[i*retw+j] = img.mask[i*imgw+j+off];\n  //assert(ret == embedSlow(img,shape));\n  return ret;\n}\n\n\nImage compose(Image_ a, Image_ b, const function<int(int,int)>&f, int overlap_only) {\n  Image ret;\n  if (overlap_only == 1) {\n    ret.p = {max(a.p.x,b.p.x),\n\t     max(a.p.y,b.p.y)};\n    point ra = a.p+a.sz, rb = b.p+b.sz;\n    ret.sz = {min(ra.x,rb.x),min(ra.y,rb.y)};\n    ret.sz = ret.sz-ret.p;\n    if (ret.w <= 0 || ret.h <= 0) return badImg;\n  } else if (overlap_only == 0) {\n    ret.p = {min(a.p.x,b.p.x),\n\t     min(a.p.y,b.p.y)};\n    point ra = a.p+a.sz, rb = b.p+b.sz;\n    ret.sz = {max(ra.x,rb.x),max(ra.y,rb.y)};\n    ret.sz = ret.sz-ret.p;\n  } else if (overlap_only == 2) {\n    ret.p = a.p;\n    ret.sz = a.sz;\n  } else assert(0);\n  if (ret.w > MAXSIDE || ret.h > MAXSIDE || ret.w*ret.h > MAXAREA) return badImg;\n  ret.mask.assign(ret.w*ret.h, 0);\n  point da = ret.p-a.p;\n  point db = ret.p-b.p;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      int ca = a.safe(i+da.y, j+da.x);\n      int cb = b.safe(i+db.y, j+db.x);\n      ret(i,j) = f(ca,cb);\n    }\n  }\n  return ret;\n}\n\n\nImage compose(Image_ a, Image_ b, int id) { //id = 0\n  if (id == 0) {\n    return compose(a, b, [](int a, int b) {return b ? b : a;}, 0); //a then b, inside either\n  } else if (id == 1) {\n    return compose(a, b, [](int a, int b) {return b ? b : a;}, 1); //a then b, inside both\n  } else if (id == 2) {\n    return compose(a, b, [](int a, int b) {return b ? a : 0;}, 1); //a masked by b\n  } else if (id == 3) {\n    return compose(a, b, [](int a, int b) {return b ? b : a;}, 2); //a then b, inside of a\n  } else if (id == 4) {\n    return compose(a, b, [](int a, int b) {return b ? 0 : a;}, 2); //a masked by inverse of b, inside of a\n  } else assert(id >= 0 && id < 5);\n  return badImg;\n}\n\nImage outerProductIS(Image_ a, Image_ b) {\n  if (a.w*b.w > MAXSIDE || a.h*b.h > MAXSIDE || a.w*b.w*a.h*b.h > MAXAREA) return badImg;\n  point rpos = {a.p.x*b.w+b.p.x,\n\t\ta.p.y*b.h+b.p.y};\n  Image ret = core::empty(rpos, {a.w*b.w, a.h*b.h});\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      for (int k = 0; k < b.h; k++)\n\tfor (int l = 0; l < b.w; l++)\n\t  ret(i*b.h+k, j*b.w+l) = a(i,j) * !!b(k,l);\n  return ret;\n}\nImage outerProductSI(Image_ a, Image_ b) {\n  if (a.w*b.w > MAXSIDE || a.h*b.h > MAXSIDE || a.w*b.w*a.h*b.h > MAXAREA) return badImg;\n  point rpos = {a.p.x*b.w+b.p.x,\n\t\ta.p.y*b.h+b.p.y};\n  Image ret = core::empty(rpos, {a.w*b.w, a.h*b.h});\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      for (int k = 0; k < b.h; k++)\n\tfor (int l = 0; l < b.w; l++)\n\t  ret(i*b.h+k, j*b.w+l) = (a(i,j)>0) * b(k,l);\n  return ret;\n}\n\n\n\nImage Fill(Image_ a) {\n  Image ret = core::full(a.p, a.sz, core::majorityCol(a));\n  vector<pair<int,int>> q;\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      if ((i == 0 || j == 0 || i == a.h-1 || j == a.w-1) && !a(i,j)) {\n\tq.emplace_back(i,j);\n\tret(i,j) = 0;\n      }\n  while (q.size()) {\n    auto [r,c] = q.back();\n    q.pop_back();\n    for (int d = 0; d < 4; d++) {\n      int nr = r+(d==2)-(d==3);\n      int nc = c+(d==0)-(d==1);\n      if (nr >= 0 && nr < a.h && nc >= 0 && nc < a.w && !a(nr,nc) && ret(nr,nc)) {\n\tq.emplace_back(nr,nc);\n\tret(nr,nc) = 0;\n      }\n    }\n  }\n  return ret;\n}\n\nImage interior(Image_ a) {\n  return compose(Fill(a), a, [](int x, int y) {return y ? 0 : x;}, 0);\n}\n\nImage border(Image_ a) {\n  Image ret = core::empty(a.p, a.sz);\n  vector<pair<int,int>> q;\n  for (int i = 0; i < a.h; i++)\n    for (int j = 0; j < a.w; j++)\n      if (i == 0 || j == 0 || i == a.h-1 || j == a.w-1) {\n\tif (!a(i,j))\n\t  q.emplace_back(i,j);\n\tret(i,j) = 1;\n      }\n  while (q.size()) {\n    auto [r,c] = q.back();\n    q.pop_back();\n#define DO(nr,nc) {if (!ret(nr,nc)) { ret(nr,nc) = 1; if (!a(nr,nc)) q.emplace_back(nr,nc); }}\n    if (r > 0) {\n      if (c > 0) DO(r-1,c-1);\n      DO(r-1,c);\n      if (c+1 < a.w) DO(r-1,c+1);\n    }\n    if (r+1 < a.h) {\n      if (c > 0) DO(r+1,c-1);\n      DO(r+1,c);\n      if (c+1 < a.w) DO(r+1,c+1);\n    }\n\n    if (c > 0) DO(r,c-1);\n    if (c+1 < a.w) DO(r,c+1);\n#undef DO\n    /*for (int nr : {r-1,r,r+1}) {\n      for (int nc : {c-1,c,c+1}) {\n\tif (nr >= 0 && nr < a.h && nc >= 0 && nc < a.w && !ret(nr,nc)) {\n    if (!a(nr,nc))\n\t  q.emplace_back(nr,nc);\n\t  ret(nr,nc) = 1;\n\t}\n      }\n      }*/\n  }\n    for (int i = 0; i < a.mask.size(); i++)\n      ret.mask[i] = ret.mask[i]*a.mask[i];\n  return ret;\n}\n\n\nImage alignx(Image_ a, Image_ b, int id) {\n  assert(id >= 0 && id < 5);\n  Image ret = a;\n  if (id == 0)      ret.x = b.x-a.w;\n  else if (id == 1) ret.x = b.x;\n  else if (id == 2) ret.x = b.x+(b.w-a.w)/2;\n  else if (id == 3) ret.x = b.x+b.w-a.w;\n  else if (id == 4) ret.x = b.x+b.w;\n  return ret;\n}\nImage aligny(Image_ a, Image_ b, int id) {\n  assert(id >= 0 && id < 5);\n  Image ret = a;\n  if (id == 0)      ret.y = b.y-a.h;\n  else if (id == 1) ret.y = b.y;\n  else if (id == 2) ret.y = b.y+(b.h-a.h)/2;\n  else if (id == 3) ret.y = b.y+b.h-a.h;\n  else if (id == 4) ret.y = b.y+b.h;\n  return ret;\n}\nImage align(Image_ a, Image_ b, int idx, int idy) {\n  assert(idx >= 0 && idx < 6);\n  assert(idy >= 0 && idy < 6);\n  Image ret = a;\n  if      (idx == 0) ret.x = b.x-a.w;\n  else if (idx == 1) ret.x = b.x;\n  else if (idx == 2) ret.x = b.x+(b.w-a.w)/2;\n  else if (idx == 3) ret.x = b.x+b.w-a.w;\n  else if (idx == 4) ret.x = b.x+b.w;\n\n  if      (idy == 0) ret.y = b.y-a.h;\n  else if (idy == 1) ret.y = b.y;\n  else if (idy == 2) ret.y = b.y+(b.h-a.h)/2;\n  else if (idy == 3) ret.y = b.y+b.h-a.h;\n  else if (idy == 4) ret.y = b.y+b.h;\n  return ret;\n}\n\n\n\nImage align(Image_ a, Image_ b) {\n  //Find most matching color and align a to b using it\n  Image ret = a;\n  int match_size = 0;\n  for (int c = 1; c < 10; c++) {\n    Image ca = compress(filterCol(a, c));\n    Image cb = compress(filterCol(b, c));\n    if (ca.mask == cb.mask) {\n      int cnt = core::count(ca);\n      if (cnt > match_size) {\n\tmatch_size = cnt;\n\tret.p = a.p+cb.p-ca.p;\n      }\n    }\n  }\n  if (match_size == 0) return badImg;\n  return ret;\n}\n\nImage replaceCols(Image_ base, Image_ cols) {\n  Image ret = base;\n  Image done = core::empty(base.p,base.sz);\n  point d = base.p-cols.p;\n  for (int i = 0; i < base.h; i++) {\n    for (int j = 0; j < base.w; j++) {\n      if (!done(i,j) && base(i,j)) {\n\tint acol = base(i,j);\n\tint cnt[10] = {};\n\tvector<pair<int,int>> path;\n\tfunction<void(int,int)> dfs = [&](int r, int c) {\n\t  if (r < 0 || r >= base.h || c < 0 || c >= base.w || base(r,c) != acol || done(r,c)) return;\n\t  cnt[cols.safe(r+d.y,c+d.x)]++;\n\t  path.emplace_back(r,c);\n\t  done(r,c) = 1;\n\t  for (int nr : {r-1,r,r+1})\n\t    for (int nc : {c-1,c,c+1})\n\t      dfs(nr,nc);\n\t};\n\tdfs(i,j);\n\tpair<int,int> maj = {0,0};\n\tfor (int c = 1; c < 10; c++) {\n\t  maj = max(maj, make_pair(cnt[c], -c));\n\t}\n\tfor (auto [r,c] : path)\n\t  ret(r,c) = -maj.second;\n      }\n    }\n  }\n  return ret;\n}\n\n//Exploit symmetries?\nImage center(Image_ img) {\n  point sz = {(img.w+1)%2+1,\n\t      (img.h+1)%2+1};\n  return core::full(img.p+(img.sz-sz)/2, sz);\n}\n\nImage transform(Image_ img, int A00, int A01, int A10, int A11) {\n  if (img.w*img.h == 0) return img;\n  Image c = center(img);\n  point off = point{1-c.w,1-c.h}+(img.p-c.p)*2;\n  auto t = [&](point p) {\n    p = p*2+off;\n    p = {A00*p.x+A01*p.y,\n\t A10*p.x+A11*p.y};\n    p = p-off;\n    p.x >>= 1;\n    p.y >>= 1;\n    return p;\n  };\n  point corner[4] = {t({0,0}),t({img.w-1,0}),t({0,img.h-1}),t({img.w-1,img.h-1})};\n  point a = corner[0], b = corner[0];\n  for (int i = 1; i < 4; i++) {\n    a.x = min(a.x, corner[i].x);\n    a.y = min(a.y, corner[i].y);\n    b.x = max(b.x, corner[i].x);\n    b.y = max(b.y, corner[i].y);\n  }\n  Image ret = core::empty(img.p, b-a+point{1,1});\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      point go = t({j,i})-a;\n      ret(go.y,go.x) = img(i,j);\n    }\n  }\n  return ret;\n}\n\nint mirrorHeuristic(Image_ img) {\n  //Meant to be used for mirroring, flip either x or y, depending on center of gravity\n  int cnt = 0, sumx = 0, sumy = 0;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (img(i,j)) {\n\tcnt++;\n\tsumx += j;\n\tsumy += i;\n      }\n    }\n  }\n  return abs(sumx*2-(img.w-1)*cnt) < abs(sumy*2-(img.h-1)*cnt);\n}\n\nImage rigid(Image_ img, int id) {\n  if (id == 0) return img;\n  else if (id == 1) return transform(img, 0, 1,-1, 0); //CCW\n  else if (id == 2) return transform(img,-1, 0, 0,-1); //180\n  else if (id == 3) return transform(img, 0,-1, 1, 0); //CW\n  else if (id == 4) return transform(img,-1, 0 ,0, 1); //flip x\n  else if (id == 5) return transform(img, 1, 0, 0,-1); //flip y\n  else if (id == 6) return transform(img, 0, 1, 1, 0); //swap xy\n  else if (id == 7) return transform(img, 0,-1,-1, 0); //swap other diagonal\n  else if (id == 8) return rigid(img, 4+mirrorHeuristic(img));\n  else assert(id >= 0 && id < 9);\n  return badImg;\n}\n\nImage invert(Image img) {\n  if (img.w*img.h == 0) return img;\n  int mask = core::colMask(img);\n  int col = 1;\n  while (col < 10 && (mask>>col&1) == 0) col++;\n  if (col == 10) col = 1;\n\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      img(i,j) = img(i,j) ? 0 : col;\n  return img;\n}\n\n\nImage interior2(Image_ a) {\n  return compose(a, invert(border(a)), 2);\n}\n\n\n\nImage count(Image_ img, int id, int outType) {\n  assert(id >= 0 && id < 7);\n  assert(outType >= 0 && outType < 3);\n  int num;\n  if (id == 0)      num = core::count(img);\n  else if (id == 1) num = core::countCols(img);\n  else if (id == 2) num = core::countComponents(img);\n  else if (id == 3) num = img.w;\n  else if (id == 4) num = img.h;\n  else if (id == 5) num = max(img.w,img.h);\n  else if (id == 6) num = min(img.w,img.h);\n  else assert(0);\n\n  point sz;\n  if (outType == 0) sz = {num,num};\n  else if (outType == 1) sz = {num,1};\n  else if (outType == 2) sz = {1,num};\n  else assert(0);\n\n  if (max(sz.x,sz.y) > MAXSIDE || sz.x*sz.y > MAXAREA) return badImg;\n  return core::full(sz, core::majorityCol(img));\n}\n\n\nImage myStack(Image_ a, Image b, int orient) {\n  assert(orient >= 0 && orient <= 3);\n  b.p = a.p;\n  if (orient == 0) { //Horizontal\n    b.x += a.w;\n  } else if (orient == 1) { //Vertical\n    b.y += a.h;\n  } else if (orient == 2) { //Diagonal\n    b.x += a.w;\n    b.y += a.h;\n  } else { //Other diagonal, bottom-left / top-right\n    Image c = a;\n    c.y += b.h;\n    b.x += a.w;\n    return compose(c,b);\n  }\n  return compose(a,b);\n}\n\n\nImage wrap(Image_ line, Image_ area) {\n  if (line.w*line.h == 0 || area.w*area.h == 0) return badImg;\n  Image ans = core::empty(area.sz);\n  for (int i = 0; i < line.h; i++) {\n    for (int j = 0; j < line.w; j++) {\n      int x = j, y = i;\n      x += y/area.h * line.w;\n      y %= area.h;\n\n      y += x/area.w * line.h;\n      x %= area.w;\n\n      if (x >= 0 && y >= 0 && x < ans.w && y < ans.h)\n\tans(y,x) = line(i,j);\n    }\n  }\n  return ans;\n}\n\nImage smear(Image_ base, Image_ room, int id) {\n  assert(id >= 0 && id < 7);\n  const int arr[] = {1,2,4,8,3,12,15};\n  int mask = arr[id];\n\n  point d = room.p-base.p;\n\n  Image ret = embed(base, hull(room));\n  if (mask&1) {\n    for (int i = 0; i < ret.h; i++) {\n      char c = 0;\n      for (int j = 0; j < ret.w; j++) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  if (mask>>1&1) {\n    for (int i = 0; i < ret.h; i++) {\n      char c = 0;\n      for (int j = ret.w-1; j >= 0; j--) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  if (mask>>2&1) {\n    for (int j = 0; j < ret.w; j++) {\n      char c = 0;\n      for (int i = 0; i < ret.h; i++) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  if (mask>>3&1) {\n    for (int j = 0; j < ret.w; j++) {\n      char c = 0;\n      for (int i = ret.h-1; i >= 0; i--) {\n\tif (!room(i,j)) c = 0;\n\telse if (base.safe(i+d.y,j+d.x)) c = base(i+d.y,j+d.x);\n\tif (c) ret(i,j) = c;\n      }\n    }\n  }\n\n  return ret;\n}\n\n/*\nImage smear(Image_ base, int id) {\n  return smear(base, hull(base), id);\n}\n*/\n\nImage extend(Image_ img, Image_ room) {\n  if (img.w*img.h == 0) return badImg;\n  Image ret = room;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      point p = point{j,i}+room.p-img.p;\n      p.x = clamp(p.x, 0, img.w-1);\n      p.y = clamp(p.y, 0, img.h-1);\n      ret(i,j) = img(p.y,p.x);\n    }\n  }\n  return ret;\n}\n\n\n\n\n\n\n\n\nImage pickMax(vImage_ v, const function<int(Image_)>& f) {\n  if (v.empty()) return badImg;\n  int ma = f(v[0]), maxi = 0;\n  for (int i = 1; i < v.size(); i++) {\n    int score = f(v[i]);\n    if (score > ma) {\n      ma = score;\n      maxi = i;\n    }\n  }\n  return v[maxi];\n}\n\nint maxCriterion(Image_ img, int id) {\n  assert(id >= 0 && id < 14);\n  switch (id) {\n  case 0: return  core::count(img);\n  case 1: return -core::count(img);\n  case 2: return  img.w*img.h;\n  case 3: return -img.w*img.h;\n  case 4: return core::countCols(img);\n  case 5: return -img.p.y;\n  case 6: return  img.p.y;\n  case 7: return  core::countComponents(img);\n  case 8:\n    {\n      Image comp = compress(img);\n      return comp.w*comp.h-core::count(comp);\n    }\n  case 9:\n    {\n      Image comp = compress(img);\n      return -(comp.w*comp.h-core::count(comp));\n    }\n  case 10: return  core::count(interior(img));\n  case 11: return -core::count(interior(img));\n  case 12: return -img.p.x;\n  case 13: return  img.p.x;\n  }\n  return -1;\n}\n\n\nImage pickMax(vImage_ v, int id) {\n  return pickMax(v, [id](Image_ img) {return maxCriterion(img,id);});\n}\n\nvImage cut(Image_ img, Image_ a) {\n  vector<Image> ret;\n  Image done = core::empty(img.p,img.sz);\n  point d = img.p-a.p;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (!done(i,j) && !a.safe(i+d.y,j+d.x)) {\n\tImage toadd = core::empty(img.p,img.sz);\n\tfunction<void(int,int)> dfs = [&](int r, int c) {\n\t  if (r < 0 || r >= img.h || c < 0 || c >= img.w || a.safe(r+d.y,c+d.x) || done(r,c)) return;\n\t  toadd(r,c) = img(r,c)+1;\n\t  done(r,c) = 1;\n\t  for (int nr : {r-1,r,r+1})\n\t    for (int nc : {c-1,c,c+1})\n\t      dfs(nr,nc);\n\t};\n\tdfs(i,j);\n\ttoadd = compress(toadd);\n\tfor (int i = 0; i < toadd.h; i++) {\n\t  for (int j = 0; j < toadd.w; j++) {\n\t    toadd(i,j) = max(0, toadd(i,j)-1);\n\t  }\n\t}\n\tret.push_back(toadd);\n      }\n    }\n  }\n  return ret;\n}\n\n\nvImage splitCols(Image_ img, int include0) { //include0 = 0\n  vector<Image> ret;\n  int mask = core::colMask(img);\n  for (int c = !include0; c < 10; c++) {\n    if (mask>>c&1) {\n      Image s = img;\n      for (int i = 0; i < s.h; i++)\n\tfor (int j = 0; j < s.w; j++)\n\t  s(i,j) = (s(i,j) == c ? c : 0);\n      ret.push_back(s);\n    }\n  }\n  return ret;\n}\n\nImage compose(vImage_ imgs, int id) {\n  if (imgs.empty()) return badImg;\n  Image ret = imgs[0];\n  for (int i = 1; i < imgs.size(); i++)\n    ret = compose(ret, imgs[i], id);\n  return ret;\n}\n\nvoid getRegular(vector<int>&col) {\n  int colw = col.size();\n\n  for (int w = 1; w < colw; w++) {\n    int s = -1;\n    if (colw%(w+1) == w) { //No outer border\n      s = w;\n    } else if (colw%(w+1) == 1) { //Outer border\n      s = 0;\n    }\n    if (s != -1) {\n      int ok = 1;\n      for (int i = 0; i < colw; i++) {\n\tif (col[i] != (i%(w+1) == s)) {\n\t  ok = 0;\n\t  break;\n\t}\n      }\n      if (ok) {\n\treturn;\n      }\n    }\n  }\n  fill(col.begin(), col.end(), 0);\n}\n\nImage getRegular(Image_ img) {\n  //Look for regular grid division in single color\n  Image ret = img;\n  vector<int> col(img.w,1), row(img.h,1);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (img(i,j) != img(i,0)) row[i] = 0;\n      if (img(i,j) != img(0,j)) col[j] = 0;\n    }\n  }\n  getRegular(col);\n  getRegular(row);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      ret(i,j) = row[i] || col[j];\n    }\n  }\n  return ret;\n}\n\n\nImage cutPickMax(Image_ a, Image_ b, int id) {\n  return pickMax(cut(a,b), id);\n}\nImage regularCutPickMax(Image_ a, int id) {\n  Image b = getRegular(a);\n  return pickMax(cut(a,b), id);\n}\nImage splitPickMax(Image_ a, int id, int include0) { //include0 = 0\n  return pickMax(splitCols(a, include0), id);\n}\n\nImage cutCompose(Image_ a, Image_ b, int id) {\n  auto v = cut(a,b);\n  for (Image& img : v) img = toOrigin(img);\n  return compose(v, id);\n}\nImage regularCutCompose(Image_ a, int id) {\n  Image b = getRegular(a);\n  auto v = cut(a,b);\n  for (Image& img : v) img = toOrigin(img);\n  return compose(v, id);\n}\nImage splitCompose(Image_ a, int id, int include0) { //include0 = 0\n  auto v = splitCols(a, include0);\n  for (Image& img : v) img = toOrigin(compress(img));\n  return compose(v, id);\n}\n\nImage cutIndex(Image_ a, Image_ b, int ind) {\n  auto v = cut(a,b);\n  if (ind < 0 || ind >= (int)v.size()) return badImg;\n  return v[ind];\n}\n\n\nvImage pickMaxes(vImage_ v, function<int(Image_)> f, int invert = 0) {\n  int n = v.size();\n  if (!n) return {};\n  vector<int> score(n);\n  int ma = -1e9;\n  for (int i = 0; i < n; i++) {\n    score[i] = f(v[i]);\n    if (!i || score[i] > ma)\n      ma = score[i];\n  }\n  vector<Image> ret_imgs;\n  for (int i = 0; i < n; i++)\n    if ((score[i] == ma) ^ invert)\n      ret_imgs.push_back(v[i]);\n  return ret_imgs;\n}\n\nvImage pickMaxes(vImage_ v, int id) {\n  return pickMaxes(v, [id](Image_ img){return maxCriterion(img, id);}, 0);\n}\nvImage pickNotMaxes(vImage_ v, int id) {\n  return pickMaxes(v, [id](Image_ img){return maxCriterion(img, id);}, 1);\n}\n\nImage cutPickMaxes(Image_ a, Image_ b, int id) {\n  return compose(pickMaxes(cut(a,b), id), 0);\n}\nImage splitPickMaxes(Image_ a, int id) {\n  return compose(pickMaxes(splitCols(a), id), 0);\n}\n\n//Return a single color\n//Cut into at least 2 pieces\n//No nested pieces\n//Must touch at least 2 opposite sides\n//Smallest piece should be as big as possible\n\nImage heuristicCut(Image_ img) {\n  int ret = core::majorityCol(img, 1);\n  int ret_score = -1;\n\n  int mask = core::colMask(img);\n  Image done = core::empty(img.p,img.sz);\n  for (int col = 0; col < 10; col++) {\n    if ((mask>>col&1) == 0) continue;\n    fill(done.mask.begin(), done.mask.end(), 0);\n    function<void(int,int)> edgy = [&](int r, int c) {\n      if (r < 0 || r >= img.h || c < 0 || c >= img.w || img(r,c) != col || done(r,c)) return;\n      done(r,c) = 1;\n      for (int nr : {r-1,r,r+1})\n\tfor (int nc : {c-1,c,c+1})\n\t  edgy(nr,nc);\n    };\n    int top = 0, bot = 0, left = 0, right = 0;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j) == col) {\n\t  if (i == 0) top = 1;\n\t  if (j == 0) left = 1;\n\t  if (i == img.h-1) bot = 1;\n\t  if (j == img.w-1) right = 1;\n\t}\n\tif ((i == 0 || j == 0 || i == img.h-1 || j == img.w-1) && img(i,j) == col && !done(i,j)) {\n\t  edgy(i,j);\n\t}\n      }\n    }\n\n    if (!(top && bot || left && right)) continue;\n\n    int score = 1e9, components = 0, nocontained = 1;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint cnt = 0, contained = 1;\n\tif (!done(i,j) && img(i,j) != col) {\n\t  function<void(int,int)> dfs = [&](int r, int c) {\n\t    if (r < 0 || r >= img.h || c < 0 || c >= img.w) return;\n\t    if (img(r,c) == col) {\n\t      if (done(r,c)) contained = 0;\n\t      return;\n\t    }\n\t    if (done(r,c)) return;\n\t    cnt++;\n\t    done(r,c) = 1;\n\t    for (int nr : {r-1,r,r+1})\n\t      for (int nc : {c-1,c,c+1})\n\t\tdfs(nr,nc);\n\t  };\n\t  dfs(i,j);\n\t  components++;\n\t  score = min(score, cnt);\n\t  if (contained) nocontained = 0;\n\t}\n      }\n    }\n    if (components >= 2 && nocontained && score > ret_score) {\n      ret_score = score;\n      ret = col;\n    }\n  }\n  return filterCol(img,ret);\n}\n\nvImage cut(Image_ img) {\n  return cut(img, heuristicCut(img));\n}\n\nImage cutPickMax(Image_ a, int id) {\n  return cutPickMax(a, heuristicCut(a), id);\n}\nImage cutIndex(Image_ a, int ind) {\n  return cutIndex(a, heuristicCut(a), ind);\n}\nImage cutPickMaxes(Image_ a, int id) {\n  return cutPickMaxes(a, heuristicCut(a), id);\n}\n\n\n\nImage repeat(Image_ a, Image_ b, int pad) { //pad = 0\n  if (a.w*a.h <= 0 || b.w*b.h <= 0) return badImg;\n  Image ret;\n  ret.p = b.p;\n  ret.sz = b.sz;\n  ret.mask.resize(ret.w*ret.h,0);\n\n  const int W = a.w+pad, H = a.h+pad;\n  int ai  = ((b.y-a.y)%H+H)%H;\n  int aj0 = ((b.x-a.x)%W+W)%W;\n  for (int i = 0; i < ret.h; i++) {\n    int aj = aj0;\n    for (int j = 0; j < ret.w; j++) {\n      if (ai < a.h && aj < a.w)\n\tret(i,j) = a(ai,aj);\n      if (++aj == W) aj = 0;\n    }\n    if (++ai == H) ai = 0;\n  }\n  return ret;\n}\n\nImage mirror(Image_ a, Image_ b, int pad) { //pad = 0\n  if (a.w*a.h <= 0 || b.w*b.h <= 0) return badImg;\n  Image ret;\n  ret.p = b.p;\n  ret.sz = b.sz;\n  ret.mask.resize(ret.w*ret.h);\n  const int W = a.w+pad, H = a.h+pad;\n  const int W2 = W*2, H2 = H*2;\n  int ai  = ((b.y-a.y)%H2+H2)%H2;\n  int aj0 = ((b.x-a.x)%W2+W2)%W2;\n  for (int i = 0; i < ret.h; i++) {\n    int aj = aj0;\n    for (int j = 0; j < ret.w; j++) {\n      int x = -1, y = -1;\n      if (aj < a.w) x = aj;\n      else if (aj >= W && aj < W+a.w) x = W+a.w-1-aj;\n      if (ai < a.h) y = ai;\n      else if (ai >= H && ai < H+a.h) y = H+a.h-1-ai;\n      if (x != -1 && y != -1)\n\tret(i,j) = a(y,x);\n      if (++aj == W2) aj = 0;\n    }\n    if (++ai == H2) ai = 0;\n  }\n  return ret;\n}\n\nImage majCol(Image_ img) {\n  return Col(core::majorityCol(img));\n}\n', 'native/src/image_functions2.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "spec.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n\n\nvImage splitAll(Image_ img) {\n  vector<Image> ret;\n  Image done = core::empty(img.p,img.sz);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (!done(i,j)) {\n\tImage toadd = core::empty(img.p,img.sz);\n\tfunction<void(int,int,int)> dfs = [&](int r, int c, int col) {\n\t  if (r < 0 || r >= img.h || c < 0 || c >= img.w || img(r,c) != col || done(r,c)) return;\n\t  toadd(r,c) = img(r,c)+1;\n\t  done(r,c) = 1;\n\t  for (int d = 0; d < 4; d++)\n\t    dfs(r+(d==0)-(d==1),c+(d==2)-(d==3),col);\n\t};\n\tdfs(i,j,img(i,j));\n\ttoadd = compress(toadd);\n\tfor (int i = 0; i < toadd.h; i++) {\n\t  for (int j = 0; j < toadd.w; j++) {\n\t    toadd(i,j) = max(0, toadd(i,j)-1);\n\t  }\n\t}\n\t//Image i = interior(toadd);\n\t//if (!core::count(compose(img,i,2)))\n\tif (core::count(toadd))\n\t  ret.push_back(toadd);\n      }\n    }\n  }\n  return ret;\n}\n\n\nImage eraseCol(Image img, int col) {\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      if (img(i,j) == col) img(i,j) = 0;\n  return img;\n}\n\n\n// Looks for 4 corners\nvImage insideMarked(Image_ in) {\n  vector<Image> ret;\n  for (int i = 0; i+1 < in.h; i++) {\n    for (int j = 0; j+1 < in.w; j++) {\n      for (int h = 1; i+h+1 < in.h; h++) {\n\tfor (int w = 1; j+w+1 < in.w; w++) {\n\t  char col = in(i,j);\n\t  if (!col) continue;\n\t  int ok = 1;\n\t  for (int k = 0; k < 4; k++) {\n\t    int x = j+k%2*w, y = i+k/2*h;\n\t    for (int d = 0; d < 4; d++) {\n\t      if ((d != 3-k) == (in(y+d/2,x+d%2) != col)) {\n\t\tok = 0;\n\t\tgoto fail;\n\t      }\n\t    }\n\t  }\n\tfail:\n\t  if (ok) {\n\t    Image inside = invert(core::full(point{j+1,i+1}, point{w,h}));\n\t    ret.push_back(compose(inside,in,3));\n\t  }\n\t}\n      }\n    }\n  }\n  return ret;\n}\n\nImage makeBorder(Image_ img, int bcol = 1) {\n  Image ret = hull0(img);\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      if (img(i,j) == 0) {\n\tint ok = 0;\n\tfor (int ni : {i-1,i,i+1}) {\n\t  for (int nj : {j-1,j,j+1}) {\n\t    if (img.safe(ni,nj)) {\n\t      ok = 1;\n\t      break;\n\t    }\n\t  }\n\t}\n\tif (ok) {\n\t  ret(i,j) = bcol;\n\t}\n      }\n    }\n  }\n  return ret;\n}\n\n\nImage makeBorder2(Image_ img, int usemaj = 1) {\n  int bcol = 1;\n  if (usemaj) bcol = core::majorityCol(img);\n\n  point rsz = img.sz+point{2,2};\n  if (max(rsz.x,rsz.y) > MAXSIDE || rsz.x*rsz.y > MAXAREA) return badImg;\n  Image ret = core::full(img.p-point{1,1}, rsz, bcol);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      ret(i+1,j+1) = img(i,j);\n  return ret;\n}\n\nImage makeBorder2(Image_ img, Image_ bord) {\n  int bcol = core::majorityCol(bord);\n  point rsz = img.sz+bord.sz+bord.sz;\n  if (max(rsz.x,rsz.y) > MAXSIDE || rsz.x*rsz.y > MAXAREA) return badImg;\n  Image ret = core::full(img.p-bord.sz, rsz, bcol);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      ret(i+bord.h,j+bord.w) = img(i,j);\n  return ret;\n}\n\n\n//Delete black rows / cols\nImage compress2(Image_ img) {\n  vector<int> row(img.h), col(img.w);\n  for (int i = 0; i < img.h; i++)\n    for (int j = 0; j < img.w; j++)\n      if (img(i,j)) row[i] = col[j] = 1;\n  vector<int> rows, cols;\n  for (int i = 0; i < img.h; i++) if (row[i]) rows.push_back(i);\n  for (int j = 0; j < img.w; j++) if (col[j]) cols.push_back(j);\n  Image ret = core::empty(point{(int)cols.size(), (int)rows.size()});\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      ret(i,j) = img(rows[i],cols[j]);\n  return ret;\n}\n\n\n//Group single color rectangles\nImage compress3(Image_ img) {\n  if (img.w*img.h <= 0) return badImg;\n  vector<int> row(img.h), col(img.w);\n  row[0] = col[0] = 1;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (i && img(i,j) != img(i-1,j)) row[i] = 1;\n      if (j && img(i,j) != img(i,j-1)) col[j] = 1;\n    }\n  }\n  vector<int> rows, cols;\n  for (int i = 0; i < img.h; i++) if (row[i]) rows.push_back(i);\n  for (int j = 0; j < img.w; j++) if (col[j]) cols.push_back(j);\n  Image ret = core::empty(point{(int)cols.size(), (int)rows.size()});\n  for (int i = 0; i < ret.h; i++)\n    for (int j = 0; j < ret.w; j++)\n      ret(i,j) = img(rows[i],cols[j]);\n  return ret;\n}\n\n\n//TODO: return badImg if fail\n\nImage greedyFill(Image& ret, vector<pair<int,vector<int>>>&piece, Spec&done, int bw, int bh, int&donew) {\n  sort(piece.rbegin(), piece.rend());\n\n  const int dw = ret.w-bw+1, dh = ret.h-bh+1;\n  if (dw < 1 || dh < 1) return badImg;\n\n  vector<int> dones(dw*dh, -1);\n  priority_queue<tuple<int,int,int>> pq;\n  auto recalc = [&](int i, int j) {\n    int cnt = 0;\n    for (int y = 0; y < bh; y++)\n      for (int x = 0; x < bw; x++)\n\tcnt += done(i+y,j+x);\n    if (cnt != dones[i*dw+j]) {\n      dones[i*dw+j] = cnt;\n      pq.emplace(cnt,j,i);\n    }\n  };\n  for (int i = 0; i+bh <= ret.h; i++)\n    for (int j = 0; j+bw <= ret.w; j++)\n      recalc(i,j);\n\n  while (pq.size()) {\n    auto [ds,j,i] = pq.top();\n    pq.pop();\n    if (ds != dones[i*dw+j]) continue;\n    int found = 0;\n    for (auto [cnt,mask] : piece) {\n      int ok = 1;\n      for (int y = 0; y < bh; y++)\n\tfor (int x = 0; x < bw; x++)\n\t  if (done(i+y,j+x) && ret(i+y,j+x) != mask[y*bw+x])\n\t    ok = 0;\n      if (ok) {\n\tfor (int y = 0; y < bh; y++) {\n\t  for (int x = 0; x < bw; x++) {\n\t    if (!done(i+y,j+x)) {\n\t      done(i+y,j+x) = donew;\n\t      if (donew > 1) donew--;\n\t      ret(i+y,j+x) = mask[y*bw+x];\n\t    }\n\t  }\n\t}\n\tfor (int y = max(i-bh+1,0); y < min(i+bh, dh); y++)\n\t  for (int x = max(j-bw+1,0); x < min(j+bw, dw); x++)\n\t    recalc(y,x);\n\tfound = 1;\n\tbreak;\n      }\n    }\n    if (!found)\n      return badImg;//ret;\n  }\n  return ret;\n}\n\n\n\n\nImage greedyFillBlack(Image_ img, int N = 3) {\n  Image ret = core::empty(img.p, img.sz);\n  Spec done;\n  done.sz = img.sz;\n  done.mask.assign(done.w*done.h,0);\n\n  int donew = 1e6;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      if (img(i,j)) {\n\tret(i,j) = img(i,j);\n\tdone(i,j) = donew;\n      }\n    }\n  }\n\n  map<vector<int>,int> piece_cnt;\n  vector<int> mask;\n  const int bw = N, bh = N;\n  for (int r = 0; r < 8; r++) {\n    Image rot = rigid(img,r);\n    for (int i = 0; i+bh <= rot.h; i++) {\n      for (int j = 0; j+bw <= rot.w; j++) {\n\tmask.reserve(bw*bh);\n\tmask.resize(0);\n\tint ok = 1;\n\tfor (int y = 0; y < bh; y++)\n\t  for (int x = 0; x < bw; x++) {\n\t    char c = rot(i+y,j+x);\n\t    mask.push_back(c);\n\t    if (!c) ok = 0;\n\t  }\n\tif (ok)\n\t  ++piece_cnt[mask];\n      }\n    }\n  }\n  vector<pair<int,vector<int>>> piece;\n  for (auto&[p,c] : piece_cnt)\n    piece.emplace_back(c,p);\n\n  return greedyFill(ret, piece, done, bw, bh, donew);\n}\n\nImage greedyFillBlack2(Image_ img, int N = 3) {\n  Image filled = greedyFillBlack(img, N);\n  return compose(filled, img, 4);\n}\n\nImage extend2(Image_ img, Image_ room) {\n  Image ret = core::empty(room.p, room.sz);\n  Spec done;\n  done.sz = room.sz;\n  done.mask.assign(done.w*done.h,0);\n\n  point d = room.p-img.p;\n  int donew = 1e6;\n  for (int i = 0; i < ret.h; i++) {\n    for (int j = 0; j < ret.w; j++) {\n      int x = j+d.x, y = i+d.y;\n      if (x >= 0 && y >= 0 && x < img.w && y < img.h) {\n\tret(i,j) = img(y,x);\n\tdone(i,j) = donew;\n      }\n    }\n  }\n\n  map<vector<int>,int> piece_cnt;\n  vector<int> mask;\n  const int bw = 3, bh = 3;\n  for (int r = 0; r < 8; r++) {\n    Image rot = rigid(img,r);\n    for (int i = 0; i+bh <= rot.h; i++) {\n      for (int j = 0; j+bw <= rot.w; j++) {\n\tmask.reserve(bw*bh);\n\tmask.resize(0);\n\tfor (int y = 0; y < bh; y++)\n\t  for (int x = 0; x < bw; x++)\n\t    mask.push_back(rot(i+y,j+x));\n\t++piece_cnt[mask];\n      }\n    }\n  }\n  vector<pair<int,vector<int>>> piece;\n  for (auto&[p,c] : piece_cnt)\n    piece.emplace_back(c,p);\n\n  return greedyFill(ret, piece, done, bw, bh, donew);\n}\n\n\n\nImage connect(Image_ img, int id) {\n  assert(id >= 0 && id < 3);\n  Image ret = core::empty(img.p, img.sz);\n\n  //Horizontal\n  if (id == 0 || id == 2) {\n    for (int i = 0; i < img.h; i++) {\n      int last = -1, lastc = -1;\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j)) {\n\t  if (img(i,j) == lastc) {\n\t    for (int k = last+1; k < j; k++)\n\t      ret(i,k) = lastc;\n\t  }\n\t  lastc = img(i,j);\n\t  last = j;\n\t}\n      }\n    }\n  }\n\n  //Vertical\n  if (id == 1 || id == 2) {\n    for (int j = 0; j < img.w; j++) {\n      int last = -1, lastc = -1;\n      for (int i = 0; i < img.h; i++) {\n\tif (img(i,j)) {\n\t  if (img(i,j) == lastc) {\n\t    for (int k = last+1; k < i; k++)\n\t      ret(k,j) = lastc;\n\t  }\n\t  lastc = img(i,j);\n\t  last = i;\n\t}\n      }\n    }\n  }\n\n  return ret;\n}\n\nImage replaceTemplate(Image_ in, Image_ need_, Image_ marked_, int overlapping = 0, int rigids = 0) {\n  if (marked_.sz != need_.sz) return badImg;\n  if (need_.w*need_.h <= 0) return in;\n\n  const int rots = rigids ? 8 : 1;\n  vector<Image> needr(rots), markedr(rots);\n  for (int r = 0; r < rots; r++) {\n    needr[r] = rigid(need_,r);\n    markedr[r] = rigid(marked_,r);\n  }\n\n  Image ret = in;\n  for (int r = 0; r < rots; r++) {\n    Image_ need = needr[r];\n    Image_ marked = markedr[r];\n\n    for (int i = 0; i+need.h <= ret.h; i++) {\n      for (int j = 0; j+need.w <= ret.w; j++) {\n\tint ok = 1;\n\tfor (int y = 0; y < need.h; y++)\n\t  for (int x = 0; x < need.w; x++)\n\t    if ((overlapping ? in : ret)(i+y,j+x) != need(y,x)) ok = 0;\n\n\tif (overlapping == 2) {\n\t  for (int y = -1; y <= need.h; y++) {\n\t    for (int x = -1; x <= need.w; x++) {\n\t      if (x >= 0 && y >= 0 && x < need.w && y < need.h) continue;\n\n\t      char nn = need(clamp(y,0,need.h-1),\n\t\t\t     clamp(x,0,need.w-1));\n\t      if (nn && nn == in.safe(i+y,j+x)) ok = 0;\n\t    }\n\t  }\n\t}\n\n\tif (ok) {\n\t  for (int y = 0; y < need.h; y++)\n\t    for (int x = 0; x < need.w; x++)\n\t      ret(i+y,j+x) = marked(y,x);\n\t}\n      }\n    }\n  }\n  return ret;\n}\n\n\n\nImage swapTemplate(Image_ in, Image_ a, Image_ b, int rigids = 0) {\n  if (a.sz != b.sz) return badImg;\n  if (a.w*a.h <= 0) return in;\n\n  const int rots = rigids ? 8 : 1;\n  vector<Image> ar(rots), br(rots);\n  for (int r = 0; r < rots; r++) {\n    ar[r] = rigid(a,r);\n    br[r] = rigid(b,r);\n  }\n  Image done = hull0(in), ret = in;\n  for (int k : {0,1}) {\n    for (int r = 0; r < rots; r++) {\n      Image_ need = k ? ar[r] : br[r];\n      Image_ to   = k ? br[r] : ar[r];\n\n      for (int i = 0; i+need.h <= ret.h; i++) {\n\tfor (int j = 0; j+need.w <= ret.w; j++) {\n\n\t  int ok = 1;\n\t  for (int y = 0; y < need.h; y++)\n\t    for (int x = 0; x < need.w; x++)\n\t      if (done(i+y,j+x) || ret(i+y,j+x) != need(y,x)) ok = 0;\n\t  if (ok) {\n\t    for (int y = 0; y < need.h; y++) {\n\t      for (int x = 0; x < need.w; x++) {\n\t\tret(i+y,j+x) = to(y,x);\n\t\tdone(i+y,j+x) = 1;\n\t      }\n\t    }\n\t  }\n\t}\n\n      }\n    }\n  }\n  return ret;\n}\n\n\nImage spreadCols(Image img, int skipmaj = 0) {\n  int skipcol = -1;\n  if (skipmaj)\n    skipcol = core::majorityCol(img);\n\n  Image done = hull0(img);\n  queue<tuple<int,int,int>> q;\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      if (img(i,j)) {\n\tif (img(i,j) != skipcol)\n\t  q.emplace(j,i,img(i,j));\n\tdone(i,j) = 1;\n      }\n    }\n  }\n  while (q.size()) {\n    auto [j,i,c] = q.front();\n    q.pop();\n    for (int d = 0; d < 4; d++) {\n      int ni = i+(d==0)-(d==1);\n      int nj = j+(d==2)-(d==3);\n      if (ni >= 0 && nj >= 0 && ni < img.h && nj < img.w && !done(ni,nj)) {\n\timg(ni,nj) = c;\n\tdone(ni,nj) = 1;\n\tq.emplace(nj,ni,c);\n      }\n    }\n  }\n  return img;\n}\n\n\nvImage splitColumns(Image_ img) {\n  if (img.w*img.h <= 0) return {};\n  vector<Image> ret(img.w);\n  for (int j = 0; j < img.w; j++) {\n    ret[j].p = {j,0};\n    ret[j].sz = {1,img.h};\n    ret[j].mask.resize(img.h);\n    for (int i = 0; i < img.h; i++)\n      ret[j].mask[i] = img(i,j);\n  }\n  return ret;\n}\nvImage splitRows(Image_ img) {\n  if (img.w*img.h <= 0) return {};\n  vector<Image> ret(img.h);\n  for (int i = 0; i < img.h; i++) {\n    ret[i].p = {0,i};\n    ret[i].sz = {img.w,1};\n    ret[i].mask.resize(img.w);\n    for (int j = 0; j < img.w; j++)\n      ret[i].mask[j] = img(i,j);\n  }\n  return ret;\n}\n\n\nImage half(Image_ img, int id) {\n  assert(id >= 0 && id < 4);\n  if (id == 0) {\n    return core::subImage(img, {0,0}, {img.w/2, img.h});\n  } else if (id == 1) {\n    return core::subImage(img, {img.w-img.w/2,0}, {img.w/2, img.h});\n  } else if (id == 2) {\n    return core::subImage(img, {0,0}, {img.w, img.h/2});\n  } else if (id == 3) {\n    return core::subImage(img, {0,img.h-img.h/2}, {img.w, img.h/2});\n  }\n  return badImg;\n}\n\n\nImage smear(Image_ img, int id) {\n  assert(id >= 0 && id < 15);\n  const pair<int,int> R = {1,0}, L = {-1,0}, D = {0,1}, U = {0,-1};\n  const pair<int,int> X = {1,1}, Y = {-1,-1}, Z = {1,-1}, W = {-1,1};\n  const vector<pair<int,int>> d[15] = {{R},{L},{D},{U},\n\t\t\t\t       {R,L},{D,U},\n\t\t\t\t       {R,L,D,U},\n\t\t\t\t       {X},{Y},{Z},{W},\n\t\t\t\t       {X,Y},{Z,W},\n\t\t\t\t       {X,Y,Z,W},\n\t\t\t\t       {R,L,D,U,X,Y,Z,W}};\n  const int w = img.w;\n  Image ret = img;\n\n\n  for (auto [dx,dy] : d[id]) {\n    int di = dy*w+dx;\n\n    for (int i = 0; i < ret.h; i++) {\n      int step = i == 0 || i == ret.h-1 ? 1 : max(ret.w-1,1);\n      for (int j = 0; j < ret.w; j += step) {\n\tif (i-dy < 0 || j-dx < 0 || i-dy >= img.h || j-dx >= img.w) {\n\t  int steps = MAXSIDE;\n\t  if (dx ==-1) steps = min(steps, j+1);\n\t  if (dx == 1) steps = min(steps, img.w-j);\n\t  if (dy ==-1) steps = min(steps, i+1);\n\t  if (dy == 1) steps = min(steps, img.h-i);\n\n\t  int ind = i*w+j;\n\t  int end_ind = ind+steps*di;\n\t  int c = 0;\n\t  for (; ind != end_ind; ind += di) {\n\t    if (img.mask[ind]) c = img.mask[ind];\n\t    if (c) ret.mask[ind] = c;\n\t  }\n\t}\n      }\n    }\n  }\n\n  return ret;\n}\n\n\nImage mirror2(Image_ a, Image_ line) {\n  Image ret;\n  if (line.w > line.h) {\n    ret = rigid(a,5);\n    ret.x = a.x;\n    ret.y = line.y*2+line.h-a.y-a.h;\n  } else {\n    ret = rigid(a,4);\n    ret.y = a.y;\n    ret.x = line.x*2+line.w-a.x-a.w;\n  }\n  return ret;\n}\n\nvImage gravity(Image_ in, int d) {\n  vImage pieces = splitAll(in);\n  Image room = hull0(in);\n  int dx = (d==0)-(d==1);\n  int dy = (d==2)-(d==3);\n\n  vImage ret;\n  Image out = room;\n  sort(pieces.begin(), pieces.end(), [dx,dy](Image_ a, Image_ b) {\n      return (a.x-b.x)*dx+(a.y-b.y)*dy > 0;});\n  for (Image p : pieces) {\n    while (1) {\n      p.x += dx;\n      p.y += dy;\n      for (int i = 0; i < p.h; i++) {\n\tfor (int j = 0; j < p.w; j++) {\n\t  if (p(i,j) == 0) continue;\n\t  int x = j+p.x-out.x;\n\t  int y = i+p.y-out.y;\n\t  if (x < 0 || y < 0 || x >= out.w || y >= out.h || out(y,x)) {\n\t    p.x -= dx;\n\t    p.y -= dy;\n\t    goto done;\n\t  }\n\t}\n      }\n    }\n  done:\n    ret.push_back(p);\n    out = compose(out, p, 3);\n  }\n  return ret;\n}\n\n\n\nImage myStack(vImage_ lens, int id) {\n  int n = lens.size();\n  if (!n) return badImg;\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < n; i++) {\n    order[i] = {lens[i].w*lens[i].h,i};\n  }\n  sort(order.begin(), order.end());\n\n  Image out = lens[order[0].second];\n  for (int i = 1; i < n; i++)\n    out = myStack(out,lens[order[i].second],id);\n  return out;\n}\n\nImage stackLine(vImage_ shapes) {\n  int n = shapes.size();\n  if (!n) return badImg;\n  else if (n == 1) return shapes[0];\n  vector<int> xs(n), ys(n);\n  for (int i = 0; i < n; i++) {\n    xs[i] = shapes[i].x;\n    ys[i] = shapes[i].y;\n  }\n  sort(xs.begin(), xs.end());\n  sort(ys.begin(), ys.end());\n  int xmin = 1e9, ymin = 1e9;\n  for (int i = 1; i < n; i++) {\n    xmin = min(xmin, xs[i]-xs[i-1]);\n    ymin = min(ymin, ys[i]-ys[i-1]);\n  }\n  int dx = 1, dy = 0;\n  if (xmin < ymin) dx = 0, dy = 1;\n\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < shapes.size(); i++) {\n    order[i] = {shapes[i].x*dx+shapes[i].y*dy,i};\n  }\n  sort(order.begin(), order.end());\n\n  Image out = shapes[order[0].second];\n  for (int i = 1; i < n; i++)\n    out = myStack(out,shapes[order[i].second],dy);\n  return out;\n}\n\n\nImage composeGrowingSlow(vImage_ imgs) {\n  int n = imgs.size();\n  if (!n) return badImg;\n\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < n; i++) {\n    order[i] = {core::count(imgs[i]), i};\n  }\n  sort(order.rbegin(), order.rend());\n\n  Image ret = imgs[order[0].second];\n  for (int i = 1; i < n; i++)\n    ret = compose(ret, imgs[order[i].second], 0);\n  return ret;\n}\n\n\nImage composeGrowing(vImage_ imgs) {\n  int n = imgs.size();\n  if (!n) return badImg;\n  if (n == 1) return imgs[0];\n\n  int minx = 1e9, miny = 1e9, maxx = -1e9, maxy = -1e9;\n  for (Image_ img : imgs) {\n    minx = min(minx, img.x);\n    miny = min(miny, img.y);\n    maxx = max(maxx, img.x+img.w);\n    maxy = max(maxy, img.y+img.h);\n  }\n\n  point rsz = {maxx-minx, maxy-miny};\n  if (max(rsz.x, rsz.y) > MAXSIDE || rsz.x*rsz.y > MAXAREA || rsz.x <= 0 || rsz.y <= 0)\n    return badImg;\n\n  vector<pair<int,int>> order(n);\n  for (int i = 0; i < n; i++) {\n    order[i] = {core::count(imgs[i]), i};\n  }\n  sort(order.rbegin(), order.rend());\n\n  Image ret = core::empty(point{minx, miny}, rsz);\n  for (auto [cnt,imgi] : order) {\n    Image_ img = imgs[imgi];\n    int dx = img.x-ret.x, dy = img.y-ret.y;\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tif (img(i,j))\n\t  ret(i+dy,j+dx) = img(i,j);\n      }\n    }\n  }\n  //assert(ret == composeGrowingSlow(imgs));\n  return ret;\n}\n\n\nImage pickUnique(vImage_ imgs, int id) {\n  assert(id == 0);\n\n  int n = imgs.size();\n  if (!n) return badImg;\n\n  //Pick the one with the unique color\n  vector<int> mask(n);\n  vector<int> cnt(10);\n  for (int i = 0; i < n; i++) {\n    mask[i] = core::colMask(imgs[i]);\n    for (int c = 0; c < 10; c++) {\n      if (mask[i]>>c&1) cnt[c]++;\n    }\n  }\n  int reti = -1;\n  for (int i = 0; i < n; i++) {\n    for (int c = 0; c < 10; c++) {\n      if (mask[i]>>c&1) {\n\tif (cnt[c] == 1) {\n\t  if (reti == -1) reti = i;\n\t  else return badImg;\n\t}\n      }\n    }\n  }\n  if (reti == -1) return badImg;\n  return imgs[reti];\n}\n', 'native/src/load.cpp': '#include <stdio.h>\n#include <string>\nusing std::string;\n#include "load.hpp"\n\nconst int loading_len = 40;\n\nvoid Loader::operator()() {\n  long long a = counter++, b = n;\n  const char*title = text.c_str();\n\n  long long pb = prev*b*2, x = a*2000+b;\n  if (pb > x-b*2 && pb <= x) return;\n\n#pragma omp critical\n  {\n    long long p = prev = x/(2*b);\n\n    if (title) printf("%s ", title);\n    printf("|");\n    int dist = (a*2LL*loading_len+(b>>1))/b;\n    for (int i = 0; i < dist>>1; i++)\n      printf("=");\n    if (dist&1)\n      printf("-");\n    for (int i = (dist+1)>>1; i < loading_len; i++)\n      printf(" ");\n    printf("| %lld / %lld (%lld.%lld%%)\\r", a, b, p/10, p%10);\n    fflush(stdout);\n  }\n}\n\nLoader::~Loader() {\n#pragma omp critical\n  {\n    const char*title = text.c_str();\n    if (title) {\n      for (int i = 0; title[i]; i++)\n\tprintf(" ");\n    }\n    for (int i = 0; i < loading_len+35+5; i++)\n      printf(" ");\n    printf("\\r");\n    fflush(stdout);\n    if (title && keep_title) {\n      printf("%s \\n", title);\n    }\n  }\n}\n', 'native/src/main.cpp': '#include "runner.hpp"\n#include <stdio.h>\n#include <stdlib.h>\n\n#include <unistd.h>\n\nint main(int argc, char**argv) {\n  //rankFeatures();\n  //evalNormalizeRigid();\n  //evalTasks();\n  //bruteSubmission();\n  //bruteSolve();\n  int only_sid = -1;\n  if (argc >= 2) {\n    only_sid = atoi(argv[1]);\n    printf("Running only task # %d\\n", only_sid);\n  }\n  int maxdepth = -1;\n  if (argc >= 3) {\n    maxdepth = atoi(argv[2]);\n    printf("Using max depth %d\\n", maxdepth);\n  }\n  run(only_sid, maxdepth);\n}\n', 'native/src/normalize.cpp': '#include "precompiled_stl.hpp"\n/*#include <cassert>\n#include <functional>\n#include <vector>\n#include <iostream>\n#include <map>\n#include <cmath>*/\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "visu.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n\nvector<double> shapeFeatures(Image_ img, int col) {\n  double fill_cnt = core::count(img);\n  double comp_cnt = core::countComponents(img);\n  double int_cnt = core::count(interior(img));\n  vector<double> r =  {fill_cnt, 1./(fill_cnt+1e-3),\n\t\t       comp_cnt, 1./(comp_cnt+1e-3),\n\t\t       int_cnt, 1./(int_cnt+1e-3)};\n  for (int c = 0; c < 10; c++)\n    r.push_back((c == col)+2);\n\n  point center2 = img.p*2+img.sz-point{1,1};\n  for (int a = 0; a < 2; a++) {\n    for (int b = 0; b < 2; b++) {\n      for (int c = 0; c < 2; c++) {\n\tpoint dir;\n\tif (c) dir = point{a*2-1, b*2-1}*2;\n\telse dir = point{(a*2-1)*b, (a*2-1)*!b}*3;\n\tint ma = -50;\n\tfor (int i = 0; i < img.h; i++)\n\t  for (int j = 0; j < img.w; j++)\n\t    if (img(i,j)) ma = max(ma, (point{i*2,j*2}-center2)*dir);\n\tr.push_back(ma+1000);\n      }\n    }\n  }\n  return r;\n}\n\nstruct UniquePicker {\n  vector<int> feature_dim;\n  int save;\n  UniquePicker(const vector<Image>&ins, int save_) {\n    save = save_ | 1;\n\n    int nfeats = -1;\n    vector<vector<vector<double>>> feat; // inp, col, feat\n    for (Image_ in : ins) {\n      vector<vector<double>> features;\n      vector<pair<Image, int>> split = core::splitCols(in);\n      for (auto&[sh,col] : split) {\n\tfeatures.push_back(shapeFeatures(sh,col));\n\tnfeats = features.back().size();\n      }\n      feat.push_back(features);\n    }\n\n    int nins = ins.size();\n    vector<vector<int>> done(nins); // inp, cole\n    vector<int> cols_left(nins);\n    for (int i = 0; i < nins; i++) {\n      done[i].assign(feat[i].size(), 0);\n      cols_left[i] = feat[i].size();\n    }\n\n    if (nfeats == -1) return;\n\n    while (1) {\n      vector<double> score(nfeats, 1e3); //feat\n      vector<vector<int>> picki(nins); //inp, feat -> col\n\n      int found = 0;\n      for (int inpi = 0; inpi < nins; inpi++) {\n\tif (!cols_left[inpi]) continue;\n\tfound = 1;\n\n\tauto&f = feat[inpi];\n\tint cols = f.size();\n\tfor (int fi = 0; fi < nfeats; fi++) {\n\t  pair<double,int> best = {-1,-1};\n\t  for (int i = 0; i < cols; i++) {\n\t    if (done[inpi][i]) continue;\n\t    double worst = 1e3;\n\t    for (int j = 0; j < cols; j++) {\n\t      if (done[inpi][j] || i == j) continue;\n\t      worst = min(worst, diff(f[i][fi], f[j][fi]));\n\t    }\n\t    best = max(best, make_pair(worst, i));\n\t  }\n\t  assert(best.second != -1);\n\t  score[fi] = min(score[fi], best.first);\n\t  picki[inpi].push_back(best.second);\n\t}\n      }\n      if (!found) break;\n\n      pair<double,int> best = {-1,-1};\n      for (int fi = 0; fi < nfeats; fi++) {\n\tbest = max(best, make_pair(score[fi], fi));\n\t//cout << score[fi] << \'-\' << fi << "  ";\n      }\n      //cout << endl;\n      //cout << best.first << \'-\' << best.second << endl;\n      int pickf = best.second;\n      assert(pickf != -1);\n      feature_dim.push_back(pickf);\n      for (int inpi = 0; inpi < nins; inpi++) {\n\tif (!cols_left[inpi]) continue;\n\tauto&col = picki[inpi];\n\tdone[inpi][col[pickf]] = 1;\n\tcols_left[inpi]--;\n      }\n    }\n\n    //for (int i : feature_dim) cout << i << endl;\n    //cout << endl;\n    //exit(0);\n  }\n\n  double diff(double a, double b) const {\n    return a/(b+1e-5);\n  }\n\n  int getUnique(const vector<vector<double>>&features, const vector<int>&done, int fi) const {\n    pair<double,int> best = {-1,-1};\n    int n = features.size();\n    for (int i = 0; i < n; i++) {\n      if (done[i]) continue;\n      double score = 0;\n      for (int j = 0; j < features[i].size(); j++) {\n\tif (fi != -1) j = fi;\n\tdouble fscore = 1e3;\n\tfor (int k = 0; k < n; k++) {\n\t  if (k == i || done[k]) continue;\n\t  fscore = min(fscore, diff(features[i][j], features[k][j]));\n\t}\n\tscore = max(score, fscore);\n\tif (fi != -1) break;\n      }\n      best = max(best, make_pair(score, i));\n      //cout << score << \' \';\n    }\n    //cout << endl;\n    //exit(0);\n    assert(best.first != -1);\n    return best.second;\n  }\n\n  void getMap(Image_ in, int cols[10]) const {\n    int j = 0, done = 0;\n    for (int i = 0; i < 10; i++)\n      if (save>>i&1) {\n\tdone |= 1<<i;\n\tcols[i] = j++;\n      }\n\n    vector<vector<double>> features;\n    vector<pair<Image, int>> split = core::splitCols(in);\n    vector<int> splitcol;\n    for (auto&[sh,col] : split) {\n      if (save>>col&1) continue;\n      features.push_back(shapeFeatures(sh, col));\n      splitcol.push_back(col);\n    }\n\n    vector<int> order, fdone(features.size());\n    for (int it = 0; it < features.size(); it++) {\n      int fi = it < feature_dim.size() ? feature_dim[it] : -1;\n      int i = getUnique(features, fdone, fi);\n      fdone[i] = 1;\n      order.push_back(i);\n    }\n    for (int&i : order) i = splitcol[i];\n\n    for (int i : order) {\n      if (done>>i&1) continue;\n      cols[i] = j++;\n      done |= 1<<i;\n    }\n    for (int i = 0; i < 10; i++) {\n      if ((done>>i&1) == 0) {\n\tcols[i] = j++;\n\tdone |= 1<<i;\n      }\n    }\n    assert(done == (1<<10)-1);\n  }\n};\n\n\n\n\n\n\nImage remapCols(Image_ img, int cols[10]) {\n  Image r = img;\n  for (int i = 0; i < r.h; i++)\n    for (int j = 0; j < r.w; j++)\n      r(i,j) = cols[img(i,j)];\n  return r;\n}\n\n\n\nvoid remapCols(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims) {\n  int orin = 0, orout = 0, andin = ~0, andout = ~0;\n  for (auto [in,out] : train) {\n    int maskin = core::colMask(in);\n    int maskout = core::colMask(out);\n    orin |= maskin;\n    andin &= maskin;\n    orout |= maskout;\n    andout &= maskout;\n  }\n\n  int save = andout & ~orin;\n  if (orout == andout) save = andout;\n  save |= andin&~orout;\n\n  vector<Image> train_ins;\n  for (auto&[in,out] : train) train_ins.push_back(in);\n  UniquePicker up(train_ins, save | (orout&~orin));\n\n  Simplifier ret;\n\n  ret.in = [up](Image_ in) {\n    int cols[10];\n    up.getMap(in, cols);\n    return remapCols(in, cols);\n  };\n  ret.out = [up](Image_ in, Image_ out) {\n    int cols[10];\n    up.getMap(in, cols);\n    return remapCols(out, cols);\n  };\n  ret.rec = [up](Image_ in, Image_ out) {\n    int cols[10];\n    up.getMap(in, cols);\n    int icols[10];\n    for (int i = 0; i < 10; i++)\n      icols[cols[i]] = i;\n    return remapCols(out, icols);\n  };\n\n  sims.push_back(ret);\n}\n\n\nImage listCols(Image_ img, int extra) {\n  int mask = core::colMask(img);\n  int w = __builtin_popcount(mask);\n  Image ret = core::full({w,2}, 9);\n  int j = 0;\n  for (int i = 0; i < 10; i++)\n    if ((mask>>i&1) && !(extra>>i&1))\n      ret(0,j++) = i;\n  j = 0;\n  for (int i = 0; i < 10; i++)\n    if (extra>>i&1)\n      ret(1,j++) = i;\n  return ret;\n}\n\n\n\nvoid normalizeCols(vector<Sample>&sample) {\n  Visu visu;\n\n  int count = 0;\n  for (Sample&s : sample) {\n    auto train = s.train;\n    vector<Simplifier> sims;\n    remapCols(train, sims);\n    if (sims.size()) {\n      Simplifier&sim = sims[0];\n      for (auto&[in,out] : train) {\n\tImage cp = out;\n\tout = sim.out(in, out);\n\tassert(cp == sim.rec(in, out));\n\tin = sim.in(in);\n      }\n\n      count++;\n      visu.next(s.id);\n      for (auto [in,out] : s.train)\n\tvisu.add(in,out);\n      visu.next(s.id);\n      for (auto [in,out] : train) {\n\tvisu.add(in,out);\n      }\n    }\n    continue;\n    vector<pair<int,int>> masks;\n    int orin = 0, orout = 0, andin = ~0, andout = ~0;\n    for (auto [in,out] : s.train) {\n      masks.emplace_back(core::colMask(in), core::colMask(out));\n      orin |= masks.back().first;\n      andin &= masks.back().first;\n      orout |= masks.back().second;\n      andout &= masks.back().second;\n    }\n    int eq = core::countCols(s.train[0].first);\n\n    int meaningCols = checkAll(s.train, [&](pair<Image,Image> p) {\n\treturn core::countCols(p.first) == eq;});\n    //if (__builtin_popcount(eq&~1) != 1) continue;\n\n    if (meaningCols) {\n      count++;\n      visu.next(s.id);\n      for (auto [in,out] : s.train)\n\tvisu.add(in,out);\n      visu.next(s.id);\n      for (auto [in,out] : s.train) {\n\t//visu.add(in,core::embed(core::filterCol(out,10), out.sz));\n\tvisu.add(listCols(in, andin&~orout),listCols(out, andout&~orin));\n      }\n    }\n  }\n  cout << count << " tasks" << endl;\n}\n\n\n\n\n\n\n\n\n\n\n\n\n//Needs rigid:\n//0, 2, 12, 24, 52, (63), 66, 76, (91), 96, 130, (134), 138, 176, (182)\n\nstruct OrientationPicker {\n  int feature_dim;\n  OrientationPicker(const vector<Image>&ins) {\n    feature_dim = 0;\n    double best_score = 1.5;\n\n    {\n      double score = 1e3;\n      for (Image_ in : ins) {\n\tscore = min(score, max(in.w*1./in.h, in.h*1./in.w));\n      }\n      if (score > best_score) {\n\tfeature_dim = 1;\n\tbest_score = score;\n      }\n    }\n\n    for (int c = 0; c < 10; c++) {\n      double score = 1e3;\n      for (Image_ in : ins) {\n\tauto [p,q] = inertia(in, c);\n\tdouble s = max(p*1./(q+1.), q*1./(p+1.));\n\tscore = min(score, sqrt(s));\n      }\n      if (score > best_score) {\n\tfeature_dim = 2+c;\n\tbest_score = score;\n      }\n    }\n\n    //TODO: improve heuristics for when to use\n    {\n      double scard = 1e3, sdiag = 1e3;\n      for (Image_ in : ins) {\n\tdouble x[10] = {}, y[10] = {}, cnt[10] = {};\n\tcolorMeans(in, x, y, cnt);\n\n\tdouble sx = 1e3, sy = 1e3, sxy = 1e3, syx = 1e3;\n\tint found = 0;\n\tfor (int a = 1; a < 10; a++) {\n\t  for (int b = 1; b < a; b++) {\n\t    if (cnt[a] && cnt[b]) {\n\t      double dx = x[a]-x[b];\n\t      double dy = y[a]-y[b];\n\t      sx = min(sx, abs(dx)/(abs(dy)+1e-1));\n\t      sy = min(sy, abs(dy)/(abs(dx)+1e-1));\n\t      sxy = min(sxy, abs(dx+dy)/(abs(dx-dy)+1e-1));\n\t      syx = min(syx, abs(dx-dy)/(abs(dx+dy)+1e-1));\n\t      found = 1;\n\t      goto outside;\n\t    }\n\t  }\n\t}\n      outside:\n\tif (!found) scard = sdiag = -1;\n\tscard = min(scard, max(sx, sy));\n\tsdiag = min(sdiag, max(sxy, syx));\n      }\n      scard /= 5., sdiag /= 5.;\n      if (scard > best_score) {\n\tbest_score = scard;\n\tfeature_dim = 12;\n      }\n      if (sdiag > best_score) {\n\tbest_score = sdiag;\n\tfeature_dim = 13;\n      }\n    }\n  }\n\n\n\n  pair<double,double> inertia(Image_ img, char c) const {\n    double x = 0, y = 0, xx = 0, yy = 0, cnt = 0;\n    for (int i = 0; i < img.h; i++)\n      for (int j = 0; j < img.w; j++)\n\tif (img(i,j) == c) {\n\t  x += j, xx += j*j;\n\t  y += i, yy += i*i;\n\t  cnt++;\n\t}\n    if (cnt < 4) return {0.,0.};\n    return {(xx*cnt-x*x), (yy*cnt-y*y)};\n  }\n\n  void colorMeans(Image_ in, double*x, double*y, double*cnt) const {\n    for (int i = 0; i < in.h; i++)\n      for (int j = 0; j < in.w; j++) {\n\tchar c = in(i,j);\n\tx[c] += j;\n\ty[c] += i;\n\tcnt[c]++;\n      }\n\n    for (int c = 0; c < 10; c++) {\n      if (cnt[c]) {\n\tx[c] /= cnt[c];\n\ty[c] /= cnt[c];\n      }\n    }\n  }\n\n  int getRigid(Image_ in) const {\n    if (feature_dim == 0) return 0;\n    else if (feature_dim == 1) return in.h > in.w ? 6 : 0;\n    else if (feature_dim >= 2 && feature_dim < 12) {\n      auto [p,q] = inertia(in, feature_dim-2);\n      return p > q ? 6 : 0;\n    } else if (feature_dim == 12 || feature_dim == 13) {\n\n      double x[10] = {}, y[10] = {}, cnt[10] = {};\n      colorMeans(in, x, y, cnt);\n\n      for (int a = 1; a < 10; a++)\n\tfor (int b = 1; b < a; b++) {\n\t  if (cnt[a] && cnt[b]) {\n\t    double dx = x[a]-x[b];\n\t    double dy = y[a]-y[b];\n\t    if (feature_dim == 12) {\n\t      if (abs(dx) > abs(dy))\n\t\treturn dx < 0 ? 4 : 0;\n\t      else\n\t\treturn dy < 0 ? 7 : 6;\n\t    } else {\n\t      if (abs(dx+dy) > abs(dx-dy))\n\t\treturn dx+dy < 0 ? 7 : 0;\n\t      else\n\t\treturn dx-dy < 0 ? 4 : 5;\n\t    }\n\t  }\n\t}\n      return 0;\n    }\n    else assert(0);\n\n\n    //Criteria:\n    //Size should be horizontal\n    //Colored lines should be vertical\n    //2 colors should be top-left and bottom-right diagonally\n    //2 colors should be left and right horizontally\n\n    //Preserve:\n    //Things touching one side / corner\n    //Colored lines in output\n\n  }\n};\n\nvoid normalizeRigid(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims) {\n  Simplifier ret;\n\n  vector<Image> train_ins;\n  for (auto&[in,out] : train) train_ins.push_back(in);\n  OrientationPicker up(train_ins);\n\n  ret.in = [up](Image_ in) {\n    int rid = up.getRigid(in);\n    return toOrigin(rigid(in, rid));\n  };\n  ret.out = [up](Image_ in, Image_ out) {\n    int rid = up.getRigid(in);\n    return toOrigin(rigid(out, rid));\n  };\n  ret.rec = [up](Image_ in, Image_ out) {\n    int rid = up.getRigid(in);\n    int inv[] = {0,3,2,1,4,5,6,7};\n    return toOrigin(rigid(out, inv[rid]));\n  };\n\n  sims.push_back(ret);\n}\n\n\n\nvoid evalNormalizeRigid() {\n  vector<Sample> sample = readAll("training", 100);//evaluation\n  Visu visu;\n\n  int place_count[11] = {};\n  //#pragma omp parallel for\n  for (int si = 0; si < sample.size(); si++) {\n    Sample&s = sample[si];\n\n    {\n      vector<Simplifier> sims;\n      remapCols(s.train, sims);\n      for (auto&[in,out] : s.train)\n\ttie(in,out) = make_pair(sims[0].in(in), sims[0].out(in,out));\n      for (auto&[in,out] : s.test)\n\ttie(in,out) = make_pair(sims[0].in(in), sims[0].out(in,out));\n    }\n\n    if (1) {\n      visu.next(s.id);\n      for (auto [in,out] : s.train) {\n\tvisu.add(in,out);\n      }\n      vector<Simplifier> sims;\n      normalizeRigid(s.train, sims);\n      for (auto&[in,out] : s.train) {\n\ttie(in,out) = make_pair(sims[0].in(in), sims[0].out(in,out));\n      }\n      for (auto&[in,out] : s.test) {\n\ttie(in,out) = make_pair(sims[0].in(in), sims[0].out(in,out));\n      }\n\n      visu.next(s.id);\n      for (auto [in,out] : s.train) {\n\tvisu.add(in,out);\n      }\n      continue;\n    }\n  }\n}\n\n\n\npair<Image,Image> Simplifier::operator()(Image_ a, Image_ b) {\n  return {in(a), out(a,b)};\n}\n\nSimplifier normalizeCols(const vector<pair<Image,Image>>&train) {\n  vector<Simplifier> sims;\n  remapCols(train, sims);\n  return sims[0];\n}\n\nSimplifier normalizeDummy(const vector<pair<Image,Image>>&train) {\n  Simplifier ret;\n  ret.in = [](Image_ in) { return in; };\n  ret.out = [](Image_ in, Image_ out) { return out; };\n  ret.rec = [](Image_ in, Image_ out) { return out; };\n  return ret;\n}\n', 'native/src/pieces.cpp': '#include "precompiled_stl.hpp"\n#include <chrono>\nusing namespace std;\n#include "utils.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n\n#include "visu.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "timer.hpp"\n/*\nstruct Piece2 {\n  vector<int> ind;\n  int depth;\n};\n\nstruct Pieces {\n  vector<DAG> dag;\n  vector<Piece2> piece;\n  set<vector<int>> seen;\n};\n*/\n\nextern int print_nodes;\n\null hashVec(const vector<int>&vec) {\n  ull r = 1;\n  for (int v : vec) {\n    r = r*1069388789821391921+v;\n  }\n  return r;\n}\n\nPieces makePieces2(vector<DAG>&dag, vector<pair<Image,Image>> train, vector<point> out_sizes) {\n  Timer set_time, piece_time, child_time;\n\n  Pieces pieces;\n\n  vector<int>&mem = pieces.mem;\n  vector<int> depth_mem;\n\n  int dags = dag.size();\n\n  TinyHashMap seen;\n  //set<vector<int>> seen;\n  vector<queue<int>> q;\n\n  auto add = [&](int d, const vector<int>&v) {\n    assert(v.size() == dags);\n\n    set_time.start();\n    auto [memi, inserted] = seen.insert(hashVec(v), (int)mem.size());\n    set_time.stop();\n\n    if (inserted) {\n      for (int i : v) {\n\tmem.push_back(i);\n      }\n      depth_mem.push_back(d);\n    }\n\n    if (inserted || depth_mem[memi/dags] > d) {\n      depth_mem[memi/dags] = d;\n      while (q.size() <= d) q.push_back({});\n      q[d].push(memi);\n    }\n  };\n  for (int i = 0; i < dag[0].givens; i++) {\n    vector<int> v(dags,i);\n    add(dag[0].tiny_node[i].depth, v);\n  }\n\n  /*\n  for (int fi2 = 0; fi2 < dag[0].binary.size(); fi2++) {\n    vector<int> v;\n    v.reserve(dags);\n    int ok = 1, depth = -1;\n    for (DAG&d : dag) {\n      int ni = d.binary[fi2];\n      if (ni == -1) {\n\tok = 0;\n\tbreak;\n      }\n      assert(ni >= 0 && ni < d.tiny_node.size());\n      depth = max(depth, (int)d.tiny_node[ni].depth);\n      v.push_back(ni);\n    }\n    if (ok) {\n      pushQueue(depth, mem.size());\n      add(v, false);\n    }\n  }\n  */\n\n  vector<vector<pair<int,int>>> slow_child(train.size()+1);\n  vector<pair<int,vector<int>>> newi_list;\n\n  piece_time.start();\n  for (int depth = 0; depth < q.size(); depth++) {\n    while (q[depth].size()) {\n      int memi = q[depth].front();\n      q[depth].pop();\n      if (depth > depth_mem[memi/dags]) continue;\n      assert(depth == depth_mem[memi/dags]);\n\n      vector<int> ind(mem.begin()+memi, mem.begin()+memi+dags);\n      {\n\tint ok = 1, maxdepth = -1;\n\tfor (int i = 0; i < dags; i++) {\n\t  maxdepth = max(maxdepth, (int)dag[i].tiny_node[ind[i]].depth);\n\t  ok &= dag[i].tiny_node[ind[i]].ispiece;\n\t}\n\tif (ok && maxdepth >= depth) {\n\t  Piece3 p;\n\t  p.memi = memi;\n\t  p.depth = depth;\n\t  pieces.piece.push_back(p);\n\t}\n\tif (maxdepth < depth) continue;\n      }\n\n      newi_list.clear();\n\n      child_time.start();\n      {\n\tfor (int i = 0; i <= train.size(); i++)\n\t  dag[i].tiny_node[ind[i]].child.legacy(slow_child[i]);\n\tvector<int> newi(dags), ci(dags); //current index into child[]\n\tint fi = 0;\n\twhile (1) {\n\tnext_iteration:\n\t  for (int i = 0; i <= train.size(); i++) {\n\t    auto&child = slow_child[i];//dag[i].node[ind[i]].child;\n\t    while (ci[i] < child.size() &&\n\t\t   child[ci[i]].first < fi) ci[i]++;\n\t    if (ci[i] == child.size()) goto finish;\n\n\t    int next_available_fi = child[ci[i]].first;\n\t    if (next_available_fi > fi) {\n\t      fi = next_available_fi;\n\t      goto next_iteration;\n\n\t    } else {\n\n\t      newi[i] = child[ci[i]].second;\n\n\t      if (newi[i] == -1) {\n\t\tfi++;\n\t\tgoto next_iteration;\n\t      }\n\t    }\n\t  }\n\t  newi_list.emplace_back(fi, newi);\n\t  fi++;\n\t}\n      finish:;\n      }\n      child_time.stop();\n\n      for (auto&[fi, newi] : newi_list) {\n\tif (0) {\n\t  int i = train.size();\n\t  //auto&child = dag[i].node[ind[i]].child;\n\t  //auto it = lower_bound(child.begin(), child.end(), make_pair(fi, -1));\n\t  //if (it == child.end() || it->first != fi) {\n\t  int to = dag[i].tiny_node.getChild(ind[i], fi);\n\t  if (to == TinyChildren::None) {\n\t    string name = dag[i].funcs.getName(fi);\n\t    if (name.substr(0,4) == "Move") {\n\t      newi[i] = dag[i].applyFunc(ind[i], fi);\n\t      if (newi[i] != -1 && out_sizes.size())\n\t\tdag[i].applyFunc(newi[i], dag[i].funcs.findfi("embed 1"));\n\t    } else continue;\n\t  } else {\n\t    newi[i] = to; //it->second\n\t  }\n\t  if (newi[i] == -1) continue;\n\t}\n\n\tint new_depth = -1;\n\tfor (int i = 0; i < dags; i++) {\n\t  new_depth = max(new_depth, (int)dag[i].tiny_node[newi[i]].depth);\n\t}\n\n\tint cost = dag[0].funcs.cost[fi];\n\n\tif (new_depth >= depth+cost) {\n\t  add(depth+cost, newi);\n\t}\n      }\n    }\n  }\n  piece_time.stop();\n  piece_time.print("Piece loop time");\n  set_time.print("Set time");\n  child_time.print("Child looking time");\n\n\n  auto lookFor = [&](vector<string> name_list) {\n    vector<Image> look_imgs;\n    vector<int> look_v;\n\n    int di = 0;\n    for (DAG&d : dag) {\n      int p = 0;\n      for (string name : name_list) {\n\tint fi = d.funcs.findfi(name);\n\n\t//auto&child = d.node[p].child;\n\t//auto it = lower_bound(child.begin(), child.end(), make_pair(fi,-1));\n\t//assert(it != child.end());\n\tint ret = d.tiny_node.getChild(p, fi);\n\tassert(ret >= 0);\n\t  //auto [fi_, ret] = *it;\n\tif (0) {//fi != fi_) {\n\t  cout << p << \' \' << di << " / " << train.size() << endl;\n\t  /*for (auto [fi,nxti] : child) {\n\t    string name = d.funcs.getName(fi);\n\t    if (name.substr(0,4) == "Move") {\n\t      cout << name << \' \';\n\t    }\n\t  }\n\t  cout << endl;*/\n\t}\n\t//assert(fi == fi_);\n\tp = ret;\n\tassert(p != -1);\n      }\n      look_v.push_back(p);\n      look_imgs.push_back(d.getImg(p));\n      /*if (n.isvec || n.img[0].sz != given_sizes[di][1]) {\n\tcout << "Bad" << endl;\n\t}*/\n      di++;\n    }\n    if (!seen.insert(hashVec(look_v),0).second) cout << "Found indices" << endl;\n    //exit(0);\n  };\n\n  /*if (out_sizes.size()) {\n    lookFor({"compress", "toOrigin"});\n    }*/\n    /*\n    lookFor({"repeat 0 1", "colShape 2", "Move -1 -1", "embed 1"});\n    lookFor({"repeat 0 1", "colShape 2", "Move 1 1", "embed 1"});\n    lookFor({"repeat 0 1", "smear 4", "colShape 1"});\n    }*/\n  /*lookFor({"filterCol 1", "interior", "colShape 2"});\n    lookFor({"cut", "pickNotMaxes 11", "composeGrowing", "colShape 3", "embed 1"});\n    lookFor({"filterCol 5", "makeBorder", "colShape 1", "embed 1"});\n    lookFor({});\n  */\n\n  if (out_sizes.size() && print_nodes) {\n    int nodes = 0;\n    for (DAG&d : dag) nodes += d.tiny_node.size();\n    cout << "Nodes:  " << nodes << endl;\n    cout << "Pieces: " << pieces.piece.size() << endl;\n  }\n  pieces.dag = move(dag);\n  //pieces.seen = move(seen);\n\n  for (Piece3&p : pieces.piece) {\n    for (int i = 0; i < pieces.dag.size(); i++) {\n      int*ind = &mem[p.memi];\n      assert(ind[i] >= 0 && ind[i] < pieces.dag[i].tiny_node.size());\n    }\n  }\n\n  return pieces;\n}\n', 'native/src/read.cpp': '/*#include <vector>\n#include <regex>\n#include <cassert>\n#include <iostream>*/\n#include "precompiled_stl.hpp"\n#include <experimental/filesystem>\n\nusing namespace std;\n#include "utils.hpp"\n#include "read.hpp"\n\nSample::Sample(string filename) {\n  vector<Image> train_input, train_output;\n  vector<Image> test_input, test_output;\n\n  regex re(".*/([a-z0-9]{8}).json");\n  smatch match;\n  assert(std::regex_search(filename, match, re) && match.size() == 2);\n  id = match.str(1);\n\n  fp = fopen(filename.c_str(), "r");\n  assert(fp);\n  expect(\'{\');\n  while (1) {\n    string train_test = getQuote();\n    expect(\'[\');\n    while (1) {\n      expect(\'{\');\n      while (1) {\n\tstring input_output = getQuote();\n\tImage img = readImage();\n\tif (train_test == "train" && input_output == "input") {\n\t  train_input.push_back(img);\n\t} else if (train_test == "train" && input_output == "output") {\n\t  train_output.push_back(img);\n\t} else if (train_test == "test" && input_output == "input") {\n\t  test_input.push_back(img);\n\t} else if (train_test == "test" && input_output == "output") {\n\t  test_output.push_back(img);\n\t} else {\n\t  cerr << train_test << \' \' << input_output << endl;\n\t  assert(!"Unexpected tag");\n\t}\n\tif (end(\'}\')) break;\n      }\n      if (end(\']\')) break;\n    }\n    if (end(\'}\')) break;\n  }\n  assert(mygetc() == -1);\n  assert(feof(fp));\n  fclose(fp);\n\n  assert(train_input.size() == train_output.size());\n  for (int i = 0; i < (int)train_input.size(); i++) {\n    train.emplace_back(train_input[i], train_output[i]);\n  }\n  if (test_input.size() == test_output.size()) {\n    for (int i = 0; i < (int)test_input.size(); i++) {\n      test.emplace_back(test_input[i], test_output[i]);\n    }\n  } else {\n    assert(test_output.empty());\n    for (int i = 0; i < (int)test_input.size(); i++) {\n      test.emplace_back(test_input[i], badImg);\n    }\n  }\n}\n\nvector<Sample> Sample::split() {\n  vector<Sample> ret;\n  for (int i = 0; i < test.size(); i++) {\n    ret.push_back(*this);\n    Sample&s = ret.back();\n    tie(s.test_in, s.test_out) = test[i];\n    s.test = {{s.test_in, s.test_out}};\n    s.id_ind = i;\n  }\n  return ret;\n}\n\nchar Sample::mygetc() {\n  char c = fgetc(fp);\n  while (c == \' \' || c == \'\\n\') c = fgetc(fp);\n  return c;\n}\nint Sample::end(char endc) {\n  char c = mygetc();\n  if (c == \',\') {\n    return 0;\n  } else {\n    if (c != endc)\n      cerr << "|"<<c<<"|" << " != " << "|"<<endc<<"|" << endl;\n    assert(c == endc);\n    return 1;\n  }\n}\nvoid Sample::expect(char c) {\n  char got = mygetc();\n  if (got != c)\n    cerr << "|"<<got<<"|" << " != " << "|"<<c<<"|" << endl;\n  assert(got == c);\n}\nstring Sample::getQuote() {\n  expect(\'\\"\');\n  char str[200];\n  assert(fscanf(fp, "%[^\\"]", str));\n  expect(\'\\"\');\n  expect(\':\');\n  return string(str);\n}\nvector<int> Sample::readRow() {\n  vector<int> ret;\n  expect(\'[\');\n  while (1) {\n    int v;\n    assert(fscanf(fp, "%d", &v) == 1);\n    ret.push_back(v);\n    if (end(\']\')) break;\n  }\n  return ret;\n}\nImage Sample::readImage() {\n  Image ret;\n  ret.p = {0,0};\n  expect(\'[\');\n  vector<int> widths;\n  while (1) {\n    vector<int> row = readRow();\n    widths.push_back(row.size());\n    for (int col : row)\n      ret.mask.push_back(col);\n    if (end(\']\')) break;\n  }\n  ret.h = widths.size();\n  assert(ret.h >= 1 && ret.h <= 30);\n  ret.w = widths[0];\n  for (int i = 0; i < ret.h; i++)\n    assert(widths[i] == ret.w);\n  assert(ret.w >= 1 && ret.w <= 30);\n  return ret;\n}\n\n\nvector<Sample> readAll(string path, int maxn) { //maxn = -1\n  const string base_path[2] = {"/kaggle/input/abstraction-and-reasoning-challenge/", "./dataset/"};\n\n  int base_pathi = 0;\n  while (!experimental::filesystem::exists(base_path[base_pathi]+path)) {\n    base_pathi++;\n    assert(base_pathi < 2);\n  }\n\n  vector<string> files;\n  for (auto magic_file_type : experimental::filesystem::directory_iterator(base_path[base_pathi]+path)) {\n    string name = magic_file_type.path().u8string();\n    if (name.size() >= 5 && name.substr(name.size()-5,5) == ".json")\n      files.push_back(name);\n  }\n  sort(files.begin(), files.end()); // Lane patch: reproducible indices.\n  if (maxn >= 0) files.resize(maxn);\n  vector<Sample> sample;\n  for (string sid : files) {\n    //sample.push_back(Sample(sid));\n    for (Sample s : Sample(sid).split()) {\n      //if (maxn < 0 || sample.size() < maxn)\n      sample.push_back(s);\n    }\n  }\n  return sample;\n}\n\n\nWriter::Writer(string filename) { //filename = "submission_part.csv"\n  filename = filename;\n  fp = fopen(filename.c_str(), "w");\n  assert(fp);\n  fprintf(fp, "output_id,output\\n");\n}\n\nvoid Writer::operator()(const Sample& s, vector<Image> imgs) {\n  //int cnt = seen[s.id]++;\n  //assert(cnt == s.id_ind);\n  fprintf(fp, "%s_%d,", s.id.c_str(), s.id_ind);\n  //TODO: is empty output allowed? Looks like no!\n  if (imgs.empty()) imgs = {dummyImg};\n  assert(imgs.size() <= 3);\n  int notfirst = 0;\n  for (Image_ img : imgs) {\n    assert(img.p == point({0,0}));\n    assert(img.w >= 1 && img.w <= 30 && img.h >= 1 && img.h <= 30);\n    if (notfirst++) fprintf(fp, " ");\n    fprintf(fp, "|");\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint c = img(i,j);\n\tassert(c >= 0 && c <= 9);\n\tfprintf(fp, "%d", c);\n      }\n      fprintf(fp, "|");\n    }\n  }\n  fprintf(fp, "\\n");\n}\n\nWriter::~Writer() {\n  fclose(fp);\n}\n\n\nvoid writeAnswersWithScores(const Sample&s, string fn, vector<Image> imgs, vector<double> scores) {\n  FILE*fp = fopen(fn.c_str(), "w");\n  assert(fp);\n  fprintf(fp, "%s_%d\\n", s.id.c_str(), s.id_ind);\n  assert(imgs.size() == scores.size());\n  if (imgs.empty()) imgs = {dummyImg}, scores = {-1};\n  assert(imgs.size() <= 3);\n\n  for (int i = 0; i < imgs.size(); i++) {\n    Image_ img = imgs[i];\n    double score = scores[i];\n    assert(img.p == point({0,0}));\n    assert(img.w >= 1 && img.w <= 30 && img.h >= 1 && img.h <= 30);\n    fprintf(fp, "|");\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint c = img(i,j);\n\tassert(c >= 0 && c <= 9);\n\tfprintf(fp, "%d", c);\n      }\n      fprintf(fp, "|");\n    }\n    fprintf(fp, " %.20f", score);\n    fprintf(fp, "\\n");\n  }\n  fclose(fp);\n}\n', 'native/src/runner.cpp': '#include "precompiled_stl.hpp"\n\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "visu.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n#include "tasks.hpp"\n#include "evals.hpp"\n\n#include "brute2.hpp"\n\n#include "score.hpp"\n#include "load.hpp"\n\n#include "deduce_op.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\n#include "brute_size.hpp"\n\n#include <thread>\n\nstring green(string s) {\n  return ("\\033[1;32m"+s+"\\033[0m");\n}\nstring blue(string s) {\n  return ("\\033[1;34m"+s+"\\033[0m");\n}\nstring yellow(string s) {\n  return ("\\033[1;33m"+s+"\\033[0m");\n}\nstring red(string s) {\n  return ("\\033[1;31m"+s+"\\033[0m");\n}\n\n\nvoid writeVerdict(int si, string sid, int verdict) {\n  printf("Task #%2d (%s): ", si, sid.c_str());\n  switch (verdict) {\n  case 3: cout << green("Correct") << endl; break;\n  case 2: cout << yellow("Candidate") << endl; break;\n  case 1: cout << blue("Dimensions") << endl; break;\n  case 0: cout << red("Nothing") << endl; break;\n  default: assert(0);\n  }\n}\n\n\nint MAXDEPTH = -1; //Argument\n\nint MAXSIDE = 100, MAXAREA = 40*40, MAXPIXELS = 40*40*5; //Just default values\n\nint print_times = 1, print_mem = 1, print_nodes = 1;\n\nvoid run(int only_sid = -1, int arg = -1) {\n  //rankFeatures();\n  //evalNormalizeRigid();\n  //evalTasks();\n  //bruteSubmission();\n  //bruteSolve();\n  //evalEvals(1);\n  //deduceEvals();\n\n  int no_norm   = (arg >= 10 && arg < 20);\n  int add_flips = (arg >= 20 && arg < 40);\n  int add_flip_id = (arg >= 30 && arg < 40 ? 7 : 6);\n\n  if (arg == -1) arg = 2;\n  MAXDEPTH = arg % 10 * 10;\n\n  int eval = 1; // Lane patch: never load query labels.\n\n  int skips = 0;\n\n  string sample_dir = "evaluation";\n  int samples = -1;\n  if (eval) {\n    sample_dir = "test";\n    samples = -1;\n  }\n\n  /*vector<Sample> sample = readAll("evaluation", -1);\n  samples = sample.size();\n  sample = vector<Sample>(sample.begin()+samples-100,sample.end());*/\n  vector<Sample> sample = readAll(sample_dir, samples);\n  //sample = vector<Sample>(sample.begin()+200, sample.begin()+300);\n\n  int scores[4] = {};\n\n  Visu visu;\n\n  vector<int> verdict(sample.size());\n\n  int dones = 0;\n  Loader load(sample.size());\n\n\n  assert(only_sid == -1 || only_sid < (int)sample.size()); // Lane patch: signed batch sentinel.\n  //Remember to fix Timers before running parallel\n\n  for (int si = 0; si < sample.size(); si++) {\n    if (only_sid != -1 && si != only_sid) continue;\n\n    //if (si == 30) assert(0);\n\n    if (eval) load();\n    else if (++dones % 10 == 0) {\n      cout << dones << " / " << sample.size() << endl;\n    }\n\n    const Sample&s = sample[si];\n\n    // Reset process-global bounds before color-normalization features.\n    MAXSIDE = 100; MAXAREA = 40*40; MAXPIXELS = 40*40*5;\n    //Normalize sample\n    Simplifier sim = normalizeCols(s.train);\n    if (no_norm) sim = normalizeDummy(s.train);\n\n    vector<pair<Image,Image>> train;\n    for (auto&[in,out] : s.train) {\n      train.push_back(sim(in,out));\n    }\n    //auto base_train = train;\n    if (add_flips) {\n      for (auto&[in,out] : s.train) {\n\tauto [rin,rout] = sim(in,out);\n\ttrain.push_back({rigid(rin,add_flip_id),rigid(rout,add_flip_id)});\n      }\n    }\n    auto [test_in,test_out] = sim(s.test_in, s.test_out);\n\n    {\n      int insumsz = 0, outsumsz = 0, macols = 0;\n      int maxside = 0, maxarea = 0;\n      for (auto&[in,out] : s.train) {\n\tmaxside = max({maxside, in.w, in.h, out.w, out.h});\n\tmaxarea = max({maxarea, in.w*in.h, out.w*out.h});\n\tinsumsz += in.w*in.h;\n\toutsumsz += out.w*out.h;\n\tmacols = max(macols, __builtin_popcount(core::colMask(in)));\n      }\n      int sumsz = max(insumsz, outsumsz);\n      cerr << "Features: " << insumsz << \' \' << outsumsz << \' \' << macols << endl;\n\n      double w[4] = {1.2772523019346949, 0.00655104, 0.70820414, 0.00194519};\n      double expect_time3 = w[0]+w[1]*sumsz+w[2]*macols+w[1]*w[2]*sumsz*macols;\n      //MAXDEPTH = 2;//(expect_time3 < 30 ? 4 : 3);//sumsz < 20*20*3 ? 3 : 2;\n      cerr << "MAXDEPTH: " << MAXDEPTH << endl;\n\n\n      MAXSIDE = 100;\n      MAXAREA = maxarea*2;\n      MAXPIXELS = MAXAREA*5;\n    }\n    //#warning Only 1 training example\n    //train.resize(1);\n\n    vector<point> out_sizes = bruteSize(test_in, train);\n    /*if (add_flips) {\n      point predsz = out_sizes.back();\n      out_sizes.clear();\n      for (auto [in,out] : train)\n\tout_sizes.push_back(out.sz);\n      out_sizes.push_back(predsz);\n      assert(out_sizes.size() == train.size()+1);\n      }*/\n    //vector<point> out_sizes = cheatSize(test_out, train); assert(!eval);\n\n    /*verdict[si] = (out_sizes.back() == test_out.sz ? 3 : 0);\n    scores[verdict[si]]++;\n    writeVerdict(si, s.id, verdict[si]);\n    continue;*/\n    //Generate candidate pieces\n    Pieces pieces;\n    {\n      double start_time = now();\n      vector<DAG> dags = brutePieces2(test_in, train, out_sizes);\n\n      if (print_times) cout << "brutePieces time: " << now()-start_time << endl;\n      start_time = now();\n      pieces = makePieces2(dags, train, out_sizes);\n      if (print_times) cout << "makePieces time: " << now()-start_time << endl;\n    }\n\n    if (print_mem) {\n      double size = 0, child = 0, other = 0, inds = 0, maps = 0;\n      for (DAG&d : pieces.dag) {\n\t/*for (Node&n : d.node) {\n\t  for (Image_ img : n.state.vimg) {\n\t    size += img.mask.size();\n\t  }\n\t  child += n.child.size()*8;\n\t  other += sizeof(Node);\n\t  }*/\n\tother += sizeof(TinyNode)*d.tiny_node.size();\n\tsize += 4*d.tiny_node.bank.mem.size();\n\tfor (TinyNode&n : d.tiny_node.node) {\n\t  if (n.child.sz < TinyChildren::dense_thres)\n\t    child += n.child.cap*8;\n\t  else\n\t    child += n.child.cap*4;\n\t}\n\tmaps += 16*d.hashi.data.size()+4*d.hashi.table.size();\n\t//maps += (d.hashi.bucket_count()*32+d.hashi.size()*16);\n      }\n      for (Piece3&p : pieces.piece) {\n\tinds += sizeof(p);\n      }\n      inds += sizeof(pieces.mem[0])*pieces.mem.size();\n      printf("Memory: %.1f + %.1f + %.1f + %.1f + %.1f MB\\n", size/1e6, child/1e6, other/1e6, maps/1e6, inds/1e6);\n      //break;\n    }\n\n    for (DAG&d : pieces.dag) {\n      d.hashi.clear();\n      for (TinyNode&n : d.tiny_node.node) {\n\tn.child.clear();\n      }\n    }\n\n    int s1 = 0;\n    if (!eval) s1 = (out_sizes.back() == test_out.sz);\n\n    //Assemble pieces into candidates\n    vector<Candidate> cands;\n    {\n      double start_time = now();\n      cands = composePieces2(pieces, train, out_sizes);\n      if (print_times) cout << "composePieces time: " << now()-start_time << endl;\n    }\n    addDeduceOuterProduct(pieces, train, cands);\n\n    cands = evaluateCands(cands, train);\n\n    int s2 = 0;\n    if (!eval) s2 = scoreCands(cands, test_in, test_out);\n\n\n    //Pick best candidates\n    vector<Candidate> answers = cands;\n\n    {\n      sort(cands.begin(), cands.end());\n      vector<Candidate> filtered;\n      set<ull> seen;\n      for (const Candidate&cand : cands) {\n\n\t//printf("%.20f\\n", cand.score);\n\n\tull h = hashImage(cand.imgs.back());\n\tif (seen.insert(h).second) {\n\t  filtered.push_back(cand);\n\t  if (filtered.size() == 3+skips*3) break;\n\t}\n      }\n      for (int i = 0; i < skips*3 && filtered.size(); i++)\n\tfiltered.erase(filtered.begin());\n      answers = move(filtered);\n    }\n\n    //Reconstruct answers\n    vector<Image> rec_answers;\n    vector<double> answer_scores;\n    for (Candidate&cand : answers) {\n      rec_answers.push_back(sim.rec(s.test_in, cand.imgs.back()));\n      double score = cand.score;\n      if (add_flips) {\n\tscore /= 2-1e-5;\n      }\n      answer_scores.push_back(score);\n    }\n\n    int s3 = 0;\n    if (!eval) s3 = scoreAnswers(rec_answers, s.test_in, s.test_out);\n\n    if (!eval) {//!eval && s1 && !s2) {\n      {\n\tvisu.next(to_string(si) + " - test");\n\tfor (auto&[in,out] : train) visu.add(in,out);\n\tvisu.next(to_string(si) + " - test");\n\tvisu.add(test_in,test_out);\n\tvisu.next(to_string(si) + " - cands");\n\tfor (int i = 0; i < min((int)answers.size(), 5); i++) {\n\t  visu.add(test_in, answers[i].imgs.back());\n\t}\n      }\n    }\n\n    /*if (!eval && (s2 || s3) && !s1) {\n      cout << si << endl;\n      }*/\n\n    if (!eval) {\n      if (s3) verdict[si] = 3;\n      else if (s2) verdict[si] = 2;\n      else if (s1) verdict[si] = 1;\n      else verdict[si] = 0;\n      scores[verdict[si]]++;\n\n      writeVerdict(si, s.id, verdict[si]);\n    }\n    {\n      string fn = "output/answer_"+to_string(si)+"_"+to_string(arg)+".csv";\n      //Writer writer(fn);\n      //writer(s, rec_answers);\n      writeAnswersWithScores(s, fn, rec_answers, answer_scores);\n    }\n  }\n\n\n  //auto now = []() {return chrono::steady_clock::now().time_since_epoch().count()/1e9;};\n  //for (double start = now(); now() < start+10;);\n\n  if (!eval && only_sid == -1) {\n    for (int si = 0; si < sample.size(); si++) {\n      Sample&s = sample[si];\n      writeVerdict(si, s.id, verdict[si]);\n    }\n    for (int si = 0; si < sample.size(); si++) cout << verdict[si];\n    cout << endl;\n\n    for (int i = 3; i; i--) scores[i-1] += scores[i];\n    printf("\\n");\n    printf("Total: % 4d\\n", scores[0]);\n    printf("Pieces:% 4d\\n", scores[1]);\n    printf("Cands: % 4d\\n", scores[2]);\n    printf("Correct:% 3d\\n", scores[3]);\n  }\n}\n', 'native/src/score.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n\n#include "brute2.hpp"\n#include "pieces.hpp"\n#include "compose2.hpp"\n\nint scoreCands(const vector<Candidate>&cands, Image_ test_in, Image_ test_out) {\n  for (const Candidate&cand : cands)\n    if (cand.imgs.back() == test_out) return 1;\n  return 0;\n}\n\nint scoreAnswers(vImage_ answers, Image_ test_in, Image_ test_out) {\n  assert(answers.size() <= 3);\n  for (Image_ answer : answers)\n    if (answer.sz == test_out.sz && answer.mask == test_out.mask) return 1;\n  return 0;\n}\n', 'native/src/tasks.cpp': '#include "precompiled_stl.hpp"\nusing namespace std;\n\n#include "utils.hpp"\n#include "core_functions.hpp"\n#include "image_functions.hpp"\n#include "image_functions2.hpp"\n#include "visu.hpp"\n#include "read.hpp"\n#include "normalize.hpp"\n#include "spec.hpp"\n\nImage solveTask(const Image& img, const vector<pair<Image,Image>>&train, int taski) {\n  if (taski == 0) {\n    return toOrigin(compress(filterCol(img, 1)));\n  } else if (taski == 1) {\n    Image cols = cutPickMax(img, 4);\n    Image shape = cutPickMax(img, 0);\n    return toOrigin(colShape(cols, shape));\n  } else if (taski == 2 || taski == 3) {\n    return interior(filterCol(img, 2));\n  } else if (taski == 4) {\n    return outerProductIS(img, count(img, 1, 0));\n  } else if (taski == 5) {\n    return train[0].second;\n  } else if (taski == 6) {\n    Image cutter = filterCol(train[0].second, 1);\n    Image yellow = compose(cutter, filterCol(img, 4));\n    Image part = cutPickMax(yellow, 0);\n    Image mask = hull(part);\n    Image chosen = toOrigin(compose(img, mask, 2));\n    Image positioned = outerProductSI(toOrigin(part), myStack(chosen, Square(1), 2));\n    return compose(positioned, cutter, 1);\n  } else if (taski == 7) {\n    Image blue = filterCol(img, 1);\n    Image red = compress(filterCol(img, 2));\n    return compose(blue, Move(red, Pos(2,0)), 0);\n  } else if (taski == 8) {\n    return rigid(img, 2);\n  } else if (taski == 9 || taski == 10) {\n    Image a = compress(filterCol(img, 2));\n    Image b = toOrigin(compress(filterCol(img, 3)));\n    return colShape(compose(a, b), 1);\n  } else if (taski == 11) {\n    return train[0].second;\n  } else if (taski == 12) {\n    return compose(Square(3), splitPickMax(img, 0));\n  } else if (taski == 13) {\n    return train[0].second;\n\n  } else if (taski == 15) {\n    return toOrigin(compress(cutIndex(img, 0)));\n  } else if (taski == 16) {\n    Image good = filterCol(img, train[0].second);\n    Image a = compose(good, rigid(good, 6));\n    Image b = compose(a, Move(rigid(a, 2), Pos(2,2)));\n    return embed(b, img);\n  } else if (taski == 17) {\n    Image a = filterCol(img, 2);\n    Image b = filterCol(img, 3);\n    return compose(a, colShape(b, 1));\n  } else if (taski == 18) {\n    Image cross = filterCol(rigid(img,4), 1);\n    Image sq = count(filterCol(img, 2), 0, 0);\n    return rigid(extend(compress(myStack(invert(sq), cross, 2)), img), 4);\n  } else if (taski == 19) {\n    Image a = cutIndex(img, 0);\n    Image b = cutIndex(img, 1);\n    return compose(colShape(compose(hull(compress(a)), hull(compress(b))), 1), img);\n\n  } else if (taski == 21) {\n    return rigid(img, 6);\n  } else if (taski == 22) {\n    Image shape = compress(cutPickMax(img, 8));\n    Image points = filterCol(img, 1);\n    Image a = cutIndex(points, 0);\n    Image b = cutIndex(points, 1);\n    Image c = cutIndex(points, 2);\n    Image d = cutIndex(points, 3);\n    Image ans = img;\n    ans = compose(ans, replaceCols(align(outerProductIS(shape, a), a), img));\n    ans = compose(ans, replaceCols(align(outerProductIS(shape, b), b), img));\n    ans = compose(ans, replaceCols(align(outerProductIS(shape, c), c), img));\n    ans = compose(ans, replaceCols(align(outerProductIS(shape, d), d), img));\n    return ans;\n  } else if (taski == 23) {\n    Image a = smear(filterCol(img,2), 6);\n    Image b = smear(filterCol(img,3), 6);\n    return compose(compose(a, b), colShape(compose(a, b, 2), 1));\n\n  } else if (taski == 25) {\n    return embed(Move(toOrigin(compress(train[0].second)), compress(filterCol(img,1))), img);\n  } else if (taski == 26) {\n    Image dup = colShape(toOrigin(cutPickMax(img, 0)), 1);\n    return compose(outerProductSI(Square(3), myStack(dup, Col(1), 2)), img, 1);\n  } else if (taski == 27) {\n    return broadcast(img, Square(3));\n\n  } else if (taski == 29) {\n    return compose(img, invert(interior2(img)), 2);\n\n  } else if (taski == 31) {\n    Image a = cutIndex(img, 0);\n    Image b = cutIndex(img, 1);\n    Image c = cutIndex(img, 2);\n    Image d = cutIndex(img, 3);\n    Image ans = compose(invert(Square(3)), Pos(1,1));\n    ans = compose(ans, align(a, ans));\n    ans = compose(ans, align(b, ans));\n    ans = compose(ans, align(c, ans));\n    ans = compose(ans, align(d, ans));\n    return ans;\n  } else if (taski == 32) {\n    return toOrigin(compress(filterCol(cutPickMax(img, 4), 1)));\n  } else if (taski == 33) {\n    Image a = toOrigin(compress(filterCol(img, 3)));\n    Image b = toOrigin(compress(filterCol(img, 4)));\n    Image both = compose(a, b, 2);\n    Image one = compose(a, b);\n    return colShape(compose(one, invert(both), 2), 1);\n  } else if (taski == 34) {\n    return count(img, 0, 1);\n  } else if (taski == 35) {\n    Image a = smear(filterCol(img, 2), 5);\n    Image b = smear(filterCol(img, 3), 4);\n    return compose(compose(a, b), colShape(compose(a,b,2), 1));\n  } else if (taski == 36) {\n    Image a = toOrigin(compress(cutPickMax(img, 0)));\n    Image b = toOrigin(compress(cutPickMax(img, 1)));\n    return colShape(invert(compose(a, b)), 1);\n\n  } else if (taski == 38) {\n    Image gray = colShape(filterCol(img, 5), 1);\n    return compose(img, gray);\n  } else if (taski == 39) {\n    return broadcast(rigid(img,4), Square(4));\n  } else if (taski == 40) {\n    Image a = smear(filterCol(img, 1), 4);\n    Image b = filterCol(img, 2);\n    return compose(b, a);\n  } else if (taski == 41) {\n    Image cutter = filterCol(img, 3);\n    Image ret = colShape(getSize(img), 2);\n    ret = compose(ret, colShape(cutPickMaxes(invert(img), 3), 1));\n    ret = compose(ret, cutter);\n    return ret;\n  } else if (taski == 42) {\n    return compose(train[0].second, compose(img, rigid(img, 6)));\n  } else if (taski == 43) {\n    Image a = smear(filterCol(img, 1), 4);\n    Image b = smear(filterCol(img, 3), 4);\n    Image c = smear(filterCol(img, 2), 5);\n    return compose(c, compose(a, b));\n\n  } else if (taski == 47) {\n    Image fill = colShape(invert(compress(filterCol(img, 2))), 1);\n    return compose(smear(fill, invert(img), 6), img);\n\n  } else if (taski == 49) {\n    Image base = colShape(cutPickMax(img, 4), 1);\n    base = myStack(base, rigid(base, 4), 0);\n    base = myStack(base, rigid(base, 5), 1);\n    return base;\n\n  } else if (taski == 51) {\n    return rigid(compose(rigid(train[0].second, 5), getSize(img), 2), 5);\n\n  } else if (taski == 52) {\n    return embed(toOrigin(compress(img)), Square(3));\n  } else if (taski == 53) {\n    Image takes = splitPickMax(img, 0);\n    return toOrigin(cutIndex(takes, 0));\n  } else if (taski == 54) {\n    Image a = compress(cutIndex(img,0));\n    Image b = compress(cutIndex(img,1));\n    Image ca = align(Col(1), a, 2, 2);\n    Image cb = align(Col(1), b, 2, 2);\n    ca = smear(ca, getSize(img), 6);\n    cb = smear(cb, getSize(img), 6);\n    return compose(compose(compose(compose(img, ca), cb), a), b);\n  } else if (taski == 55) {\n    Image twos = cutPickMaxes(img, 0);\n    return colShape(broadcast(twos, Square(3), 0), 1);\n  } else if (taski == 56) {\n    Image base = compress(train[0].second);\n    Image pos = compress(img);\n    return embed(align(base, pos, 2, 2), img);\n\n  } else if (taski == 58) {\n    Image blues = count(img, 0, 1);\n    Image blacks = count(invert(img), 0, 0);\n    blues = outerProductSI(blues, img);\n    blacks = outerProductSI(blacks, img);\n    return wrap(blues, blacks);\n  } else if (taski == 59) {\n    return toOrigin(compress(filterCol(img, 1)));\n  } else if (taski == 60) {\n    return compose(img, rigid(img, 5));\n\n  } else if (taski == 62) {\n    Image reds = cutPickMaxes(img, 11);\n    return compose(colShape(img, 1), reds);\n  } else if (taski == 63) {\n    Image blues = colShape(cutPickMaxes(img, 0), 1);\n    Image greens = colShape(cutPickMaxes(img, 1), 3);\n    return compose(compose(colShape(img, 2), blues), greens);\n  } else if (taski == 64) {\n    return colShape(filterCol(img, 1), 2);\n\n  } else if (taski == 66) {\n    return broadcast(compress(img), Square(3)); //Cheating\n  } else if (taski == 67) {\n    return compose(smear(filterCol(img, 2), 3), filterCol(img, 1)); //Needs good normalize_rigid\n\n  } else if (taski == 69) {\n    Image a = invert(toOrigin(compress(filterCol(img, 2))));\n    Image b = invert(toOrigin(compress(filterCol(img, 3))));\n    return colShape(compose(a,b,2), 1);\n  } else if (taski == 70) {\n    Image shape = compress(filterCol(train[0].second, 1));\n    Image a = cutIndex(img, 0);\n    Image b = cutIndex(img, 1);\n    Image c = cutIndex(img, 2);\n    a = replaceCols(align(shape, a, 2, 5), img);\n    b = replaceCols(align(shape, b, 2, 5), img);\n    c = replaceCols(align(shape, c, 2, 5), img);\n    return embed(compose(compose(a,b), c), img);\n  } else if (taski == 71) {\n    Image shape = hull(compress(img));\n    shape = compose(shape, colShape(interior2(shape), 2));\n    return embed(shape, img);\n  } else if (taski == 72) {\n    return myStack(img, rigid(img,5), 1);\n  } else if (taski == 73) {\n    Image base = broadcast(compress(img), Square(3));\n    return outerProductIS(base, base);\n\n  } else if (taski == 77) {\n    return myStack(img, rigid(img,5), 1);\n  } else if (taski == 78) {\n    Image a = compose(img, rigid(img, 6));\n    Image b = compose(a, Move(rigid(a, 2), Pos(2,2)));\n    return embed(b, img);\n  } else if (taski == 79) {\n    Image ship = filterCol(img, 2);\n    Image cover = smear(ship, 2);\n    return compose(compose(smear(filterCol(img, 1), 5), cover, 2), img);\n  } else if (taski == 82) {\n    Image mask = compress(interior(smear(filterCol(img, 2), 6)));\n    return toOrigin(colShape(embed(img,mask), 2));\n  } else if (taski == 83) {\n    Image ret = myStack(img,rigid(img,3), 0);\n    return myStack(ret,rigid(ret,2),1);\n  } else if (taski == 84) {\n    return embed(count(img, 0, 1), Square(3));\n  } else if (taski == 85) {\n    return train.back().second;\n  } else if (taski == 86) {\n    Image a = cutIndex(img,0);\n    Image b = cutIndex(img,1);\n    Image c = cutIndex(img,2);\n    a = align(a,a,5,0);\n    b = align(b,b,5,0);\n    c = align(c,c,5,0);\n    return embed(compose(compose(a,b),c), img);\n  } else if (taski == 88) {\n    Image cutter = filterCol(img, 3);\n    Image mi = colShape(cutPickMaxes(invert(img),1), 2);\n    Image ma = colShape(cutPickMaxes(invert(img),0), 1);\n    return compose(compose(mi, ma), cutter);\n  } else if (taski == 89) {\n    Image dots = filterCol(img, 3);\n    Image lines = smear(dots, 1);\n    Image red = filterCol(img, 2);\n    Image top = cutIndex(red,0);\n    Image bot = cutIndex(red,1);\n\n    Image overlap = compose(lines,top,2);\n    overlap = align(overlap, bot, 1, 1);\n    Image lines2 = smear(overlap,getSize(img),1);\n    dots = colShape(dots, 1);\n    return compose(compose(compose(lines, lines2), red), dots);\n  } else if (taski == 90) {\n    Image a = cutIndex(img,0);\n    Image b = cutIndex(img,1);\n    Image c = cutIndex(img,2);\n    Image d = cutIndex(img,3);\n    Image whole = getSize(img);\n    Image blue = img;\n    blue = compose(smear(a,whole,2), blue);\n    blue = compose(smear(b,whole,2), blue);\n    blue = compose(smear(c,whole,2), blue);\n    blue = compose(smear(d,whole,2), blue);\n    blue = colShape(blue, 1);\n    Image red = img;\n    red = compose(align(outerProductIS(a,Square(2)),a,2,2), red);\n    red = compose(align(outerProductIS(b,Square(2)),b,2,2), red);\n    red = compose(align(outerProductIS(c,Square(2)),c,2,2), red);\n    red = compose(align(outerProductIS(d,Square(2)),d,2,2), red);\n    red = colShape(red, 2);\n    return compose(compose(blue,red),img);\n    /*} else if (taski == 88) {\n    //Needs better align\n    Image cutter = filterCol(img,Col(1));\n    Image ret = cutPickMax(img,cutter,0);\n    Image a = cutIndex(img,cutter,0);\n    Image b = cutIndex(img,cutter,1);\n    Image c = cutIndex(img,cutter,2);\n    Image d = cutIndex(img,cutter,3);\n    ret = compose(ret, align(a,ret), 3);\n    ret = compose(ret, align(b,ret), 3);\n    ret = compose(ret, align(c,ret), 3);\n    ret = compose(ret, align(d,ret), 3);\n    return toOrigin(ret);*/\n  } else if (taski == 93) {\n    Image blue = compress(filterCol(img,1));\n    return embed(compose(align(colShape(Square(3), 3), blue, 2, 2), blue), img);\n  } else if (taski == 95) {\n    Image cols = smear(splitPickMaxes(img, 1), 2);\n    return replaceCols(img, cols);\n  } else if (taski == 96) {\n    Image base = compress(img);\n    return toOrigin(outerProductIS(base,base));\n  } else if (taski == 97) {\n    return compose(img, colShape(interior2(img), 1));\n  } else if (taski == 101) {\n    return colShape(broadcast(compress(filterCol(img,2)), Square(3)), 1);\n  }\n  return img;\n}\n\n\n\n\n\nvoid evalTasks() {\n  vector<Sample> sample = readAll("training", 100);//evaluation\n  Visu visu;\n\n  int corrects = 0;\n  int place_count[11] = {};\n  //#pragma omp parallel for\n  for (int si = 0; si < sample.size(); si++) {\n    Sample&s = sample[si];\n    {\n      vector<Simplifier> sims;\n      remapCols(s.train, sims);\n      for (auto&[in,out] : s.train)\n\ttie(in,out) = make_pair(sims[0].in(in), sims[0].out(in,out));\n      tie(s.test_in,s.test_out) = make_pair(sims[0].in(s.test_in), sims[0].out(s.test_in,s.test_out));\n      s.test = {{s.test_in, s.test_out}};\n    }\n    /*{\n      vector<Simplifier> sims;\n      normalizeRigid(s.train, sims);\n      for (auto&[in,out] : s.train)\n\ttie(in,out) = make_pair(sims[0].in(in), sims[0].out(in,out));\n      for (auto&[in,out] : s.test)\n\ttie(in,out) = make_pair(sims[0].in(in), sims[0].out(in,out));\n\t}*/\n\n    /*visu.next(s.id);\n    for (auto [in,out] : s.train) {\n      visu.add(in,out);\n      }*/\n\n    Image pred = solveTask(s.test[0].first, s.train, si);\n    cout << "Task " << si << ": " << (pred == s.test[0].second ? "OK" : "Failed") << endl;\n    corrects += (pred == s.test[0].second);\n    if (pred != s.test_out) {// && pred != s.test_in) {\n      visu.next(to_string(si));//s.id);\n      visu.add(s.test_in, pred);\n      visu.add(s.test_in, s.test_out);\n      for (auto [in,out] : s.train)\n\tvisu.add(in, out);\n    }\n  }\n  cout << corrects << " / " << sample.size() << endl;\n  exit(0);\n}\n', 'native/src/timer.cpp': '', 'native/src/visu.cpp': '/*#include "stdio.h"\n#include <vector>\n#include <string>\n#include <tuple>\n#include <cassert>\n#include <functional>*/\n#include "precompiled_stl.hpp"\nusing namespace std;\n#include "utils.hpp"\n#include "visu.hpp"\n\nVisu::Visu() {\n  fp = fopen("visu.txt", "w");\n}\nVisu::~Visu() {\n  fclose(fp);\n}\nvoid Visu::next(string s) {\n  fprintf(fp, "Task %s\\n", s.c_str());\n}\nvoid Visu::add(Image in, Image out) {\n  fprintf(fp, "Pair\\n");\n  for (Image_ img : {in,out}) {\n    fprintf(fp, "Image %d %d\\n", img.w, img.h);\n    for (int i = 0; i < img.h; i++) {\n      for (int j = 0; j < img.w; j++) {\n\tint col = img(i,j);\n\tfprintf(fp, "%d", col);\n      }\n      fprintf(fp, "\\n");\n    }\n  }\n}\n\n\nvoid plot(const vector<vector<int>>&inp, const char*filename) { //filename = out.ppm\n  int h = inp.size();\n  int w = inp[0].size();\n\n  int tw = 512/max(w,h);\n  int bw = max(1, 10/max(w,h));\n\n  int W = (tw+bw)*w+bw, H = (tw+bw)*h+bw;\n  FILE*fp = fopen(filename, "w");\n  fprintf(fp, "P6\\n%d %d\\n255\\n", W, H);\n  vector<int> cols = {0x000000, 0x0074D9, 0xFF4136, 0x2ECC40, 0xFFDC00, 0xAAAAAA, 0xF012BE, 0xFF851B, 0x7FDBFF, 0x870C25};\n\n  vector<unsigned char> output(W*H*3, 0x60);\n  for (int i = 0; i < h; i++) {\n    for (int j = 0; j < w; j++) {\n      for (int k = 0; k < tw; k++) {\n\tfor (int l = 0; l < tw; l++) {\n\t  for (int c = 0; c < 3; c++) {\n\t    output[((i*(tw+bw)+bw+k)*W+\n\t\t    (j*(tw+bw)+bw+l))*3 + c] = cols[inp[i][j]] >> (2-c)*8 & 255;\n\t  }\n\t}\n      }\n    }\n  }\n  fwrite(output.data(), 1, W*H*3, fp);\n  fclose(fp);\n}\n\nvoid print(Image img) {\n  printf("[%d %d %d %d]\\n", img.p.x, img.p.y, img.w, img.h);\n  for (int i = 0; i < img.h; i++) {\n    for (int j = 0; j < img.w; j++) {\n      int col = img(i,j);\n      if (col)\n\tprintf("%d", col);\n      else printf(".");\n    }\n    printf("\\n");\n  }\n}\n', 'native/headers/brute2.hpp': '//#include <ext/pb_ds/assoc_container.hpp>\n//using namespace __gnu_pbds;\n\ndouble now();\n\nstruct State {\n  vImage vimg;\n  int depth;\n  bool isvec;\n  State() {}\n  State(vImage_ vimg_, bool isvec_, int depth_) : vimg(vimg_), isvec(isvec_), depth(depth_) {}\n  ull hash() const {\n    ull r = isvec;\n    for (Image_ img : vimg) {\n      r += hashImage(img)*123413491;\n    }\n    return r;\n  }\n};\n\n\n#include "efficient.hpp"\n\n\n\nstruct Node {\n  State state;\n  //vImage vimg;\n  //bool isvec;\n  vector<pair<int,int>> child;\n  //int depth;\n  int par, pfi;\n  bool freed, ispiece;\n  Node() {\n    freed = ispiece = false;\n  }\n  Node(vImage_ vimg_, bool isvec_, int depth_, int par_ = -1, int pfi_ = -1) : par(par_), pfi(pfi_) {\n    freed = ispiece = false;\n    state.vimg = vimg_;\n    state.isvec = isvec_;\n    state.depth = depth_;\n  }\n};\n\n\nstruct Functions3 {\n  vector<int> listed, cost;\n  vector<string> names;\n  vector<function<bool(const State&,State&)>> f_list;\n\n  void add(const string& name, int cost, const function<bool(const State&,State&)>&func, int list);\n  void add(string name, int cost, const function<Image(Image_)>&f, int list = 1);\n  void add(string name, int cost, const function<vImage(Image_)>&f, int list = 1);\n  void add(string name, int cost, const function<Image(vImage_)>&f, int list = 1);\n  void add(string name, int cost, const function<vImage(vImage_)>&f, int list = 1);\n  void add(const vector<point>&sizes, string name, int cost, const function<Image(Image_,Image_)>&f, int list = 1);\n  string getName(int fi);\n  int findfi(string name);\n};\n\n\n\nstruct DAG {\n  Functions3 funcs;\n  //vector<Node> node;\n  TinyNodeBank tiny_node;\n  int givens;\n  point target_size;\n  //gp_hash_table<ull, int> hashi;\n  TinyHashMap hashi;\n  vector<int> binary;\n  int add(const State&nxt, bool force = false);\n  Image getImg(int nodei);\n  void build();\n  void buildBinary();\n  void initial(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> sizes, int ti);\n  void benchmark();\n  int applyFunc(int curi, int fi, const State&state);\n  int applyFunc(int curi, int fi);\n  void applyFunc(string name, bool vec);\n  void applyFuncs(vector<pair<string,int>> names, bool vec);\n};\n\n\nstruct Pieces;\nvector<DAG> brutePieces2(Image_ test_in, const vector<pair<Image,Image>>&train, vector<point> out_sizes);\n', 'native/headers/brute_size.hpp': 'vector<point> bruteSize(Image_ test_in, vector<pair<Image,Image>> train);\nvector<point> cheatSize(Image_ test_out, vector<pair<Image,Image>> train);\n', 'native/headers/compose2.hpp': 'struct Candidate {\n  vImage imgs;\n  double score;\n  int cnt_pieces, sum_depth, max_depth;\n  Candidate(vImage_ imgs_, double score_) : imgs(imgs_), score(score_) {\n    cnt_pieces = sum_depth = max_depth = -1;\n  }\n  Candidate(vImage_ imgs_, int cnt_pieces_, int sum_depth_, int max_depth_) :\n    imgs(imgs_), cnt_pieces(cnt_pieces_), sum_depth(sum_depth_), max_depth(max_depth_) {\n    score = -1;\n  }\n};\n\ninline bool operator<(const Candidate& a, const Candidate& b) {\n  return a.score > b.score;\n}\n\nvector<Candidate> composePieces2(Pieces&pieces, vector<pair<Image, Image>> train, vector<point> out_sizes);\nvector<Candidate> evaluateCands(const vector<Candidate>&cand, vector<pair<Image,Image>> train);\n', 'native/headers/core_functions.hpp': 'namespace core {\n  int colMask(Image_ img);\n  int countCols(Image_ img, int include0 = 0);\n  int count(Image_ img);\n  Image full(point p, point sz, int filling = 1);\n  Image empty(point p, point sz);\n  Image full(point sz, int filling = 1);\n  Image empty(point sz);\n  bool isRectangle(Image_ a);\n  int countComponents(Image img);\n  char majorityCol(Image_ img, int include0 = 0);\n  Image subImage(Image_ img, point p, point sz);\n  vector<pair<Image,int>> splitCols(Image_ img, int include0 = 0);\n};\n', 'native/headers/deduce_op.hpp': '\nstruct deduceOuterProduct {\n  vector<pair<Image,Image>> train_targets;\n  int rec_funci;\n  deduceOuterProduct(vector<pair<Image,Image>> train);\n  Image reconstruct(Image_ a, Image_ b);\n};\nvoid addDeduceOuterProduct(Pieces&pieces, vector<pair<Image,Image>> train, vector<Candidate>&cands);\n', 'native/headers/efficient.hpp': 'typedef unsigned long long ull;\nstruct hashEntry {\n  ull key;\n  int value, next;\n};\nstruct TinyHashMap {\n  static constexpr double sparse_factor = 1.5, resize_step = 2;\n  static constexpr int minsz = 1<<20;\n  vector<hashEntry> data;\n  vector<int> table;\n  ull mask;\n  pair<int,bool> insert(ull key, int value);\n  unsigned int size() {\n    return data.size();\n  }\n  void clear() {\n    data.clear();\n    table.clear();\n  }\n};\n\n\n\n\n\nstruct TinyChildren {\n  static constexpr int dense_thres = 10;\n  static constexpr int None = -2;\n  union {\n    int*dense;\n    pair<int,int>*sparse;\n  };\n  short sz = 0, cap = 0;\n  TinyChildren() {\n    dense = NULL;\n  }\n  void add(int fi, int to);\n  int get(int fi);\n  ~TinyChildren() {\n    delete[]dense;\n    dense = NULL;\n  }\n  TinyChildren(const TinyChildren&) = delete;\n  TinyChildren(TinyChildren&&o) {\n    dense = o.dense;\n    sz = o.sz, cap = o.cap;\n    o.dense = NULL;\n  }\n\n  void legacy(vector<pair<int,int>>&child);\n  void clear() {\n    delete[]dense;\n    dense = NULL;\n    sz = cap = 0;\n  }\n};\n\n\n\n\nstruct TinyBank {\n  vector<unsigned int> mem;\n  ll curi = 0;\n  void alloc() {\n    if (curi/32+2000 >= mem.size())\n      mem.resize(curi/32+2000);\n  }\n  inline void set(ll bi, unsigned int v) {\n    int i = bi>>5, j = bi&31;\n    if (i >= mem.size()) mem.resize(max(i+1,1024));\n    mem[i] |= v<<j;\n    //if (v) mem[i] |= 1u<<j;\n    //else mem[i] &= ~(1u<<j);\n  }\n  inline void set(ll bi, unsigned int v, int len) {\n    int i = bi>>5, j = bi&31;\n\n    mem[i] |= v << j;\n    if (j+len > 32)\n      mem[i+1] |= v >> 32-j;\n  }\n  inline int get(ll bi) {\n    return mem[bi>>5]>>(bi&31)&1;\n  }\n};\n\nstruct TinyImage {\n  static constexpr int align = 16;\n  uint32_t memi;\n  short sz;\n  char x, y, w, h;\n  uint8_t tree[9];\n  TinyImage(Image_ img, TinyBank&bank);\n  Image decompress(TinyBank&bank);\n};\n\nstruct TinyNode {\n  int*vimg = NULL;\n  bool isvec, ispiece;\n  short imgs; //TODO change to unsigned char\n  char depth;\n  TinyChildren child;\n  TinyNode() {}\n  TinyNode(TinyNode&&o) = default;\n};\n\nstruct TinyNodeBank {\n  TinyBank bank;\n  vector<TinyImage> imgs;\n  vector<TinyNode> node;\n  Image read(int i) {\n    assert(i >= 0 && i < imgs.size());\n    return imgs[i].decompress(bank);\n  }\n  Image getImg(int ni) {\n    return imgs[node[ni].vimg[0]].decompress(bank);\n  }\n  State getState(int ni) {\n    State ret;\n    ret.vimg.resize(node[ni].imgs);\n    for (int i = 0; i < node[ni].imgs; i++)\n      ret.vimg[i] = imgs[node[ni].vimg[i]].decompress(bank);\n    ret.depth = node[ni].depth;\n    ret.isvec = node[ni].isvec;\n    return ret;\n  }\n  void append(const State&state, bool ispiece) {\n    assert(state.depth >= 0 && state.depth < 128);\n    TinyNode v;\n    //assert(state.vimg.size() < 1<<16); //TODO: change to unsigned char\n\n    v.imgs = min((int)state.vimg.size(),100000);\n    v.vimg = new int[v.imgs];\n    for (int i = 0; i < v.imgs; i++) {\n      v.vimg[i] = imgs.size();\n      imgs.emplace_back(state.vimg[i], bank);\n    }\n    v.isvec = state.isvec;\n    v.ispiece = ispiece;\n    v.depth = state.depth;\n    node.push_back(move(v));\n  }\n  void addChild(int ni, int fi, int to) {\n    node[ni].child.add(fi, to);\n  }\n  int getChild(int ni, int fi) {\n    return node[ni].child.get(fi);\n  }\n  ~TinyNodeBank() {\n    for (TinyNode&n : node)\n      delete[]n.vimg;\n  }\n  TinyNode& operator[](int i) {\n    return node[i];\n  }\n  int size() { return node.size(); }\n};\n', 'native/headers/evals.hpp': 'void evalEvals(int print = 1);\n', 'native/headers/image_functions.hpp': 'Image Col(int id);\nImage Pos(int dx, int dy);\nImage Square(int id);\nImage Line(int orient, int id);\nImage getPos(Image_ img);\nImage getSize(Image_ img);\nImage hull(Image_ img);\nImage toOrigin(Image img);\nImage getW(Image_ img, int id);\nImage getH(Image_ img, int id);\nImage filterCol(Image_ img, int col);\nImage colShape(Image_ shape, int col);\nImage compress(Image_ img, Image_ bg = Col(0));\nImage Fill(Image_ a);\nImage interior(Image_ a);\nImage border(Image_ a);\nImage center(Image_ img);\n//Image transform(Image_ img, int A00, int A01, int A10, int A11);\nImage rigid(Image_ img, int id);\nImage invert(Image img);\nImage interior2(Image_ a);\nImage count(Image_ img, int id, int outType);\n//Image smear(Image_ base, int id);\n\nImage hull0(Image_ img);\nImage getSize0(Image_ img);\nImage getRegular(Image_ img);\nvImage cut(Image_ img);\n\n//One and a bit of the other\nImage Move(Image img, Image_ p);\nImage embed(Image_ img, Image_ shape);\nImage extend(Image_ img, Image_ room);\nImage wrap(Image_ line, Image_ area);\n\n//Two inputs\nImage filterCol(Image_ img, Image_ palette);\nImage colShape(Image_ col, Image_ shape);\nImage broadcast(Image_ col, Image_ shape, int include0 = 1);\nImage replaceCols(Image_ base, Image_ cols);\nImage smear(Image_ base, Image_ room, int id);\n//Image alignx(Image_ a, Image_ b, int id);\n//Image aligny(Image_ a, Image_ b, int id);\nImage align(Image_ a, Image_ b, int idx, int idy);\nImage align(Image_ a, Image_ b);\n//Image compose(Image_ a, Image_ b, const function<int(int,int)>&f, int overlap_only);\nImage compose(Image_ a, Image_ b, int id = 0);\nImage outerProductIS(Image_ a, Image_ b);\nImage outerProductSI(Image_ a, Image_ b);\nImage myStack(Image_ a, Image b, int orient);\n\n\n//Image pickMax(const vector<Image>& v, function<int(Image_)> f);\nImage pickMax(vImage_ v, int id);\n//vector<Image> cut(Image_ img, Image_ a);\n//vector<Image> splitCols(Image_ img, int include0 = 0);\n//Image compose(const vector<Image>&imgs, int id);\n//void getRegular(vector<int>&col);\n//Image getRegular(Image_ img);\nImage cutPickMax(Image_ a, int id);\n//Image regularCutPickMax(Image_ a, int id);\nImage splitPickMax(Image_ a, int id, int include0 = 0);\n//Image cutCompose(Image_ a, Image_ b, int id);\n//Image regularCutCompose(Image_ a, int id);\n//Image splitCompose(Image_ a, int id, int include0 = 0);\n//Image cutIndex(Image_ a, Image_ b, int ind);\nImage splitPickMaxes(Image_ a, int id);\nImage cutIndex(Image_ a, int ind);\nImage cutPickMaxes(Image_ a, int id);\nImage heuristicCut(Image_ a);\n\nvImage pickNotMaxes(vImage_ v, int id);\n\nImage cutPickMax(Image_ a, Image_ b, int id);\nImage cutPickMaxes(Image_ a, Image_ b, int id);\nvImage pickMaxes(vImage_ v, int id);\nImage compose(vImage_ imgs, int id);\nvImage cut(Image_ img, Image_ a);\nvImage splitCols(Image_ img, int include0 = 0);\n\nImage repeat(Image_ a, Image_ b, int pad = 0);\nImage mirror(Image_ a, Image_ b, int pad = 0);\nImage majCol(Image_ img);\n', 'native/headers/image_functions2.hpp': 'vImage splitAll(Image_ img);\nImage eraseCol(Image img, int col);\nvImage insideMarked(Image_ in);\nImage makeBorder(Image_ img, int bcol = 1);\nImage makeBorder2(Image_ img, int usemaj = 1);\nImage makeBorder2(Image_ img, Image_ bord);\nImage compress2(Image_ img);\nImage compress3(Image_ img);\nImage greedyFillBlack(Image_ img, int N = 3);\nImage greedyFillBlack2(Image_ img, int N = 3);\nImage extend2(Image_ img, Image_ room);\nImage connect(Image_ img, int id);\nImage replaceTemplate(Image_ in, Image_ need_, Image_ marked_, int overlapping = 0, int rigids = 0);\nImage swapTemplate(Image_ in, Image_ a, Image_ b, int rigids = 0);\nImage spreadCols(Image img, int skipmaj = 0);\nvImage splitColumns(Image_ img);\nvImage splitRows(Image_ img);\nImage half(Image_ img, int id);\nImage smear(Image_ img, int id);\nImage mirror2(Image_ a, Image_ line);\nvImage gravity(Image_ in, int d);\n\nImage myStack(vImage_ lens, int id);\nImage stackLine(vImage_ shapes);\nImage composeGrowing(vImage_ imgs);\n\nImage pickUnique(vImage_ imgs, int id);\n', 'native/headers/load.hpp': 'struct Loader {\n  long long n;\n  string text;\n  long long counter;\n  long long prev;\n  int keep_title;\n  Loader(long long n_, string text_ = "") {\n    n = n_;\n    text = text_;\n    counter = 0;\n    prev = -1;\n    keep_title = 0;\n  }\n  void operator()();\n  ~Loader();\n};\n', 'native/headers/normalize.hpp': 'struct Simplifier {\n  function<Image(Image_)> in;\n  function<Image(Image_,Image_)> out, rec;\n  pair<Image,Image> operator()(Image_ a, Image_ b);\n};\n\nSimplifier normalizeCols(const vector<pair<Image,Image>>&train);\nSimplifier normalizeDummy(const vector<pair<Image,Image>>&train);\n\n//inline vector<double> shapeFeatures(Image_ img, int col);\n//inline Image remapCols(Image_ img, int cols[10]);\nvoid remapCols(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims);\n//inline Image listCols(Image_ img, int extra);\nvoid normalizeCols(vector<Sample>&sample);\n\nvoid normalizeRigid(const vector<pair<Image,Image>>&train, vector<Simplifier>&sims);\nvoid evalNormalizeRigid();\n', 'native/headers/pieces.hpp': '/*struct Piece2 {\n  vector<int> ind;\n  int depth;\n  };*/\nstruct Piece3 {\n  int memi, depth;\n};\n\nstruct Pieces {\n  vector<DAG> dag;\n  vector<Piece3> piece;\n  //vector<vector<int>> seen;\n  vector<int> mem;\n};\n\nPieces makePieces2(vector<DAG>&dag, vector<pair<Image,Image>> train, vector<point> out_sizes);\n', 'native/headers/precompiled_stl.hpp': '#include <vector>\n#include <unordered_map>\n#include <cassert>\n#include <iostream>\n#include <set>\n#include <random>\n#include <algorithm>\n#include <tuple>\n#include <functional>\n#include <regex>\n#include <queue>\n\n//#include <bits/stdc++.h>\n', 'native/headers/read.hpp': '\nstruct Sample {\n  vector<pair<Image,Image>> train, test;\n  Image test_in, test_out;\n  string id;\n  int id_ind;\n  FILE*fp;\n  Sample(string filename);\n  char mygetc();\n  int end(char endc);\n  void expect(char c);\n  string getQuote();\n  vector<int> readRow();\n  Image readImage();\n  vector<Sample> split();\n};\n\nvector<Sample> readAll(string path, int maxn);\n\nstruct Writer {\n  FILE*fp;\n  map<string,int> seen;\n  Writer(string filename = "submission_part.csv");\n  void operator()(const Sample& s, vector<Image> imgs);\n  ~Writer();\n};\n\nvoid writeAnswersWithScores(const Sample&s, string fn, vector<Image> imgs, vector<double> scores);\n', 'native/headers/runner.hpp': 'void run(int only_sid = -1, int maxdepth = -1);\n', 'native/headers/score.hpp': 'struct Candidate;\nint scorePieces(const vector<Piece>&piece, Image_ test_in, Image_ test_out, const vector<pair<Image,Image>>&train);\nint scoreCands(const vector<Candidate>&cands, Image_ test_in, Image_ test_out);\nint scoreAnswers(vImage_ answers, Image_ test_in, Image_ test_out);\n', 'native/headers/spec.hpp': '\nstruct Spec {\n  bool bad = 0, anypos = 0;\n  union {\n    point p;\n    struct {\n      int x, y;\n    };\n  };\n  union {\n    point sz;\n    struct {\n      int w, h;\n    };\n  };\n  vector<int> mask;\n  inline int& operator()(int i, int j) {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  inline int operator()(int i, int j) const {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  bool check(Image_ img) const {\n    if (bad) return 0;\n    if (img.sz != sz || (!anypos && img.p != p)) return 0;\n    for (int i = 0; i < mask.size(); i++)\n      if ((mask[i]>>img.mask[i]&1) == 0) return 0;\n    return 1;\n  }\n};\n\ntypedef const Spec& Spec_;\nconst int allCols = (1<<10)-1;\n', 'native/headers/store.hpp': 'void storePieces(vector<vector<Piece>>&pieces);\nvector<vector<Piece>> loadPieces();\nvoid storeCands(vector<vector<Image>>&cands);\nvector<vector<Image>> loadCands();\n\nvoid storePieces(vector<Piece>&vp, string id, int id_ind);\nvector<Piece> loadPieces(string id, int id_ind);\n', 'native/headers/tasks.hpp': 'void evalTasks();\n', 'native/headers/timer.hpp': '#include <chrono>\n\nstruct Timer {\n  const int skips = 100, brute_thres = 1000;\n  bool running = false;\n  double cnt = 0, brute_sum = 0;\n  int sample_counter = 0, target = 0;\n  double samples = 0, sample_sum = 0, sample_var = 0;\n  mt19937 mrand;\n  double now() {\n    return chrono::high_resolution_clock::now().time_since_epoch().count()*1e-9;\n  }\n  double start_time;\n  void start() {\n    assert(!running);\n    running = true;\n    if (cnt < brute_thres || sample_counter == target)\n      start_time = now();\n  }\n  void stop() {\n    assert(running);\n    running = false;\n\n    double dt;\n    if (cnt < brute_thres || sample_counter == target)\n      dt = now()-start_time;\n\n    if (cnt < brute_thres) {\n      brute_sum += dt;\n    }\n    if (sample_counter == target) {\n      samples++;\n      sample_sum += dt;\n      sample_var += dt*dt;\n      target = mrand()%skips;\n    }\n\n    if (++sample_counter == skips)\n      sample_counter = 0;\n    cnt++;\n  }\n  tuple<double,double,double> read() {\n    if (cnt <= brute_thres)\n      return {cnt, brute_sum, 0};\n    double mean = sample_sum/samples;\n    double var = (sample_var/samples-mean*mean)/(samples-1);\n    return {cnt, mean*cnt, sqrt(var)*cnt};\n  }\n  void print(const string& label) {\n    auto [cnt,sum,std] = read();\n    printf("%5.1f ± %4.1f ms - %s\\n", sum*1e3, std*1e3, label.c_str());\n  }\n};\n', 'native/headers/utils.hpp': '//const int MAXSIDE = 100, MAXAREA = 40*40, MAXPIXELS = 40*40*5;\nextern int MAXSIDE, MAXAREA, MAXPIXELS;\n\n//#define CHECK_BOUNDS\n\nstruct point {\n  int x, y;\n};\ninline point operator+(const point& a, const point& b) {\n  return {a.x+b.x, a.y+b.y};\n}\ninline point operator-(const point& a, const point& b) {\n  return {a.x-b.x, a.y-b.y};\n}\ninline point operator*(const point& a, int f) {\n  return {a.x*f, a.y*f};\n}\ninline point operator/(const point& a, int f) {\n  assert(a.x%f == 0 && a.y%f == 0);\n  return {a.x/f, a.y/f};\n}\ninline int operator*(const point& a, const point& b) {\n  return a.x*b.x+a.y*b.y;\n}\ninline int operator%(const point& a, const point& b) {\n  return a.x*b.y-a.y*b.x;\n}\ninline bool operator==(const point& a, const point& b) {\n  return a.x == b.x && a.y == b.y;\n}\ninline bool operator!=(const point& a, const point& b) {\n  return a.x != b.x || a.y != b.y;\n}\n\n\n\nstruct Image {\n  union {\n    point p;\n    struct {\n      int x, y;\n    };\n  };\n  union {\n    point sz;\n    struct {\n      int w, h;\n    };\n  };\n  vector<char> mask;\n  inline char& operator()(int i, int j) {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  inline char operator()(int i, int j) const {\n#if defined CHECK_BOUNDS\n    assert(i >= 0 && i < h && j >= 0 && j < w);\n#endif\n    return mask[i*w+j];\n  }\n  inline char safe(int i, int j) const {\n    return (i<0 || j<0 || i>=h || j>=w ? (char)0 : mask[i*w+j]);\n  }\n};\n\nusing Image_ = const Image&;\nusing vImage = vector<Image>;\nusing vImage_ = const vector<Image>&;\n\ninline bool operator==(Image_ a, Image_ b) {\n  return tie(a.p,a.sz,a.mask) == tie(b.p,b.sz,b.mask);\n}\ninline bool operator!=(Image_ a, Image_ b) {\n  return tie(a.p,a.sz,a.mask) != tie(b.p,b.sz,b.mask);\n}\ninline bool operator<(Image_ a, Image_ b) {\n  if (a.sz != b.sz) return tie(a.w,a.h) < tie(b.w,b.h);\n  return a.mask < b.mask;\n}\n\nusing ll = long long;\nusing ull = unsigned long long;\ninline ull hashImage(Image_ img) {\n  const ull base = 137;\n  ull r = 1543;\n  r = r*base+img.w;\n  r = r*base+img.h;\n  r = r*base+img.x;\n  r = r*base+img.y;\n  for(char c : img.mask) {\n    //assert(c >= 0 && c < 10);\n    r = r*base+c;\n  }\n  return r;\n}\n\nstruct Piece {\n  vector<Image> imgs;\n  double node_prob;\n  int keepi, knowi;\n};\n\nconst Image badImg = {{0,0},{0,0},{}};\nconst Image dummyImg = {{0,0},{1,1},{0}};\n\nusing Piece_ = const Piece&;\n\ntemplate<class T, class F>\nint checkAll(const vector<T>&v, F f) {\n  int all = 1;\n  for (const T& it : v)\n    all &= f(it);\n  return all;\n}\ntemplate<class T, class F>\nint allEqual(const vector<T>&v, F f) {\n  int need = f(v[0]);\n  for (const T& it : v) {\n    if (f(it) != need) return 0;\n  }\n  return 1;\n}\n', 'native/headers/visu.hpp': 'struct Visu {\n  FILE*fp;\n  Visu();\n  ~Visu();\n  void next(string s);\n  void add(Image in, Image out);\n};\n\nvoid plot(const vector<vector<int>>&inp, const char*filename = "out.ppm");\nvoid print(Image img);\n', 'native/LICENSE': 'MIT License\n\nCopyright (c) 2020 top-quarks\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'native/Makefile': '\nDEBUG = -Wshadow -Wall -fsanitize=address -fsanitize=undefined -D_GLIBCXX_DEBUG -g -Wno-sign-compare -Wno-shadow -Wno-char-subscripts -Wno-unused-variable\n\nFLAGS = -std=c++17 -g -O2 #-fopenmp # $(DEBUG)\nLIBS = -lstdc++ -lstdc++fs\n\n.DEFAULT_GOAL := run\n\nPREC = headers/precompiled_stl.hpp\nHEAD = utils.hpp read.hpp visu.hpp image_functions.hpp normalize.hpp core_functions.hpp brute2.hpp timer.hpp efficient.hpp\nHEAD_PATH = $(addprefix headers/,$(HEAD)) $(PREC).gch | obj\n\n\nOBJ = read.o core_functions.o image_functions.o image_functions2.o visu.o normalize.o tasks.o runner.o score.o load.o evals.o brute2.o deduce_op.o pieces.o compose2.o brute_size.o efficient.o\n\nOBJ_PATH = $(addprefix obj/,$(OBJ))\n\nobj:\n\tmkdir -p obj\n\tmkdir -p output\n\n$(PREC).gch: $(PREC)\n\tg++ -c $< $(FLAGS)\n\nobj/%.o: src/%.cpp $(HEAD_PATH)\n\tg++ -c $< $(FLAGS) -o $@ -I headers\nall: $(OBJ)\n\nrun: src/main.cpp $(OBJ_PATH) $(HEAD_PATH) headers/tasks.hpp\n\tg++ src/main.cpp $(OBJ_PATH) $(FLAGS) $(LIBS) -o run -I headers\n\ncount_tasks: src/count_tasks.cpp obj/read.o headers/utils.hpp\n\tg++ src/count_tasks.cpp obj/read.o $(FLAGS) $(LIBS) -o count_tasks -I headers\n', 'LICENSE': 'MIT No Attribution\n\nCopyright 2026 Touko\'s research campaign contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of this\nsoftware and associated documentation files (the "Software"), to deal in the\nSoftware without restriction, including without limitation the rights to use,\ncopy, modify, merge, publish, distribute, sublicense, and/or sell copies of the\nSoftware, and to permit persons to whom the Software is furnished to do so.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'scripts/solver_panels.py': '"""Separate general 2D grammar extension: Boolean composition of equal panels.\nSPDX-License-Identifier: MIT-0\nAll parameters are fixed; demonstration-fitted terminal color maps are separate.\nNo task IDs, category or query labels enter operations.\n"""\nimport numpy as np\nfrom solver import Operation, operations as base_operations, background, half\n\n\ndef combine(a, axis, gap, bgmode, kind):\n    left = half(a, axis, 0, gap)\n    right = half(a, axis, 1, gap)\n    if left is None or right is None or left.shape != right.shape or min(left.shape) < 1:\n        return None\n    bg = 0 if bgmode == \'zero\' else background(a)\n    x, y = left != bg, right != bg\n    if kind == \'union\':\n        mask = x | y\n    elif kind == \'intersection\':\n        mask = x & y\n    elif kind == \'xor\':\n        mask = x ^ y\n    elif kind == \'left_only\':\n        mask = x & ~y\n    elif kind == \'right_only\':\n        mask = y & ~x\n    else:\n        raise ValueError(kind)\n    # For overlap the first panel\'s foreground wins. A terminal color map can\n    # consistently recolor the structural result using all demonstrations.\n    colors = np.where(x, left, right)\n    return np.where(mask, colors, bg).astype(np.uint8)\n\n\ndef operations(train):\n    ops = base_operations(train)\n    for axis in (0, 1):\n        for gap in (0, 1):\n            for bgmode in (\'zero\', \'mode\'):\n                for kind in (\'union\', \'intersection\', \'xor\', \'left_only\', \'right_only\'):\n                    ops.append(Operation(f\'panel_axis{axis}_gap{gap}_{bgmode}_{kind}\',\n                        lambda a, axis=axis, gap=gap, bgmode=bgmode, kind=kind:\n                        combine(a, axis, gap, bgmode, kind)))\n    return ops\n', 'scripts/panel_portfolio_entry.py': '"""Original depth3 plus panel-depth2 fitting programs, then native depth2.\nSPDX-License-Identifier: MIT-0; native solver retains MIT.\n"""\nfrom pathlib import Path\nfrom concurrent.futures import ProcessPoolExecutor\nimport json\nimport time\nimport numpy as np\nfrom solver import pack\nfrom solver_panels import operations\nfrom solver_deferred import search\nfrom portfolio_entry import select\nfrom icecuber_entry import predict as native_predict\n\n\ndef merge(task, original, panels):\n    indices = {op.name:i for i,op in enumerate(operations(task[\'train\']))}\n    candidates = {}\n    for c in original + panels:\n        terminal = bool(c[\'program\'] and c[\'program\'][-1].startswith(\'fit_color_map:\'))\n        names = c[\'program\'][:-1] if terminal else c[\'program\']\n        path = tuple(indices[n] for n in names)\n        rank = (len(c[\'program\']),path,int(terminal))\n        grids = tuple(pack(np.array(g,dtype=np.uint8)) for g in c[\'predictions\'])\n        if grids not in candidates or rank < candidates[grids][0]: candidates[grids] = (rank,c)\n    return [v[1] for k,v in sorted(candidates.items(),key=lambda kv:(kv[1][0],kv[0]))]\n\n\ndef dag_one(item):\n    name,task = item\n    original = search(task,3,terminal_map=True)\n    panels = search(task,2,ops_override=operations(task[\'train\']),terminal_map=True)\n    return name,original,panels,merge(task,original[\'candidates\'],panels[\'candidates\'])\n\n\ndef predict(challenges, native_root, timeout=1500, workers=2):\n    blind = {name:{\'train\':t[\'train\'],\'test\':[{\'input\':q[\'input\']} for q in t[\'test\']]}\n             for name,t in sorted(challenges.items())}\n    started = time.perf_counter()\n    native,native_receipt = native_predict(blind,native_root,depth=2,timeout=timeout)\n    assert native_receipt[\'exit_code\'] == 0 and not native_receipt[\'missing_queries\']\n    rows = [];submission = {}\n    with ProcessPoolExecutor(max_workers=workers) as pool:\n        for name,original,panels,candidates in pool.map(dag_one,blind.items(),chunksize=1):\n            submission[name] = select(blind[name],candidates,native[name])\n            rows.append({\'task_id\':name,\'original_candidates\':original[\'candidates\'],\n                         \'panel_candidates\':panels[\'candidates\'],\'candidates\':candidates,\n                         \'native_attempts\':native[name],\'attempts\':submission[name]})\n    return submission,{\'complete\':True,\'tasks\':len(blind),\'queries\':sum(len(t[\'test\']) for t in blind.values()),\n                       \'native_receipt\':native_receipt,\'rows\':rows,\'wall_seconds\':time.perf_counter()-started,\n                       \'workers\':workers,\'policy\':\'Original-depth3 and panel-depth2 candidate union, ranked by the combined grammar shortest program/path order, distinct per-query predictions first then reset-native depth2. No task-ID/query-label routing.\'}\n', 'scripts/solver_relational_v3.py': '"""Demonstration-fitted relational cell rules; no query labels or task IDs.\nSPDX-License-Identifier: MIT-0\n"""\nfrom __future__ import annotations\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport time\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.tree import DecisionTreeClassifier\n\nVERSION = \'relational-rules-v3\'\nDEPTHS = (2, 4, 6, 10)\nGROUPS = (\'local\', \'objects\', \'rays\', \'all\')\n\n\ndef clean_task(task):\n    return {\'train\': [{\'input\': p[\'input\'], \'output\': p[\'output\']} for p in task[\'train\']],\n            \'test\': [{\'input\': p[\'input\']} for p in task[\'test\']]}\n\n\ndef mode(a):\n    return int(np.bincount(a.ravel(), minlength=10).argmax())\n\n\ndef transform(a, name):\n    if name == \'identity\': return a\n    bg = 0 if name == \'crop_zero\' else mode(a)\n    y, x = np.nonzero(a != bg)\n    if len(y) == 0: return a\n    return a[y.min():y.max()+1, x.min():x.max()+1]\n\n\ndef features(a):\n    """Fixed input-only features. Each feature is an integer per grid cell."""\n    a = np.asarray(a, dtype=np.int16)\n    h, w = a.shape\n    yy, xx = np.indices((h, w))\n    cols, names, tags = [], [], []\n    bg = mode(a)\n    def add(name, v, tag):\n        if np.isscalar(v): v = np.full_like(a, v)\n        cols.append(np.array(v, dtype=np.int16, copy=True).ravel()); names.append(name); tags.append(tag)\n    add(\'color\', a, \'base\')\n    add(\'nonzero\', a != 0, \'base\'); add(\'nonbackground\', a != bg, \'base\')\n    add(\'background\', bg, \'base\')\n    # Absolute geometry is separate: simpler feature subsets can generalize away\n    # from accidental coordinate correlations.\n    for name, v in [(\'row\',yy),(\'column\',xx),(\'bottom\',h-1-yy),(\'right\',w-1-xx),\n                    (\'row_parity\',yy%2),(\'col_parity\',xx%2),\n                    (\'border\',np.minimum.reduce([yy,xx,h-1-yy,w-1-xx])),\n                    (\'diagonal\',yy-xx),(\'antidiagonal\',yy+xx),\n                    (\'h\',h),(\'w\',w)]: add(name,v,\'position\')\n    pad = np.pad(a,2,constant_values=10)\n    for dy in range(-2,3):\n        for dx in range(-2,3):\n            if dy == dx == 0: continue\n            add(f\'neighbor_{dy}_{dx}\',pad[2+dy:2+dy+h,2+dx:2+dx+w], \'local\')\n    add(\'mirror_ud\',a[::-1], \'local\'); add(\'mirror_lr\',a[:,::-1], \'local\')\n    add(\'mirror_both\',a[::-1,::-1], \'local\')\n    if h == w:\n        add(\'transpose\',a.T, \'local\'); add(\'antitranspose\',a[::-1,::-1].T,\'local\')\n    else:\n        add(\'transpose\',10,\'local\'); add(\'antitranspose\',10,\'local\')\n    for c in range(10):\n        mask = a == c\n        rows = mask.sum(1)[:,None] + np.zeros_like(a)\n        columns = mask.sum(0)[None,:] + np.zeros_like(a)\n        add(f\'color{c}_count\',mask.sum(),\'color\')\n        add(f\'color{c}_rowcount\',rows,\'rays\'); add(f\'color{c}_colcount\',columns,\'rays\')\n        y,x=np.nonzero(mask)\n        if len(y):\n            y0,y1,x0,x1=y.min(),y.max(),x.min(),x.max()\n            add(f\'color{c}_inbbox\',(yy>=y0)&(yy<=y1)&(xx>=x0)&(xx<=x1),\'color\')\n            add(f\'color{c}_bboxdy\',np.maximum(y0-yy,0)+np.maximum(yy-y1,0),\'color\')\n            add(f\'color{c}_bboxdx\',np.maximum(x0-xx,0)+np.maximum(xx-x1,0),\'color\')\n            dist = ndi.distance_transform_cdt(~mask,metric=\'taxicab\')\n            add(f\'color{c}_distance\',dist,\'color\')\n        else:\n            for suffix in (\'inbbox\',\'bboxdy\',\'bboxdx\',\'distance\'): add(f\'color{c}_{suffix}\',-1,\'color\')\n        # Counts on each side support intervals, parity, rays, and crossings.\n        for axis, label in [(0,\'vertical\'),(1,\'horizontal\')]:\n            forward=np.cumsum(mask,axis=axis)-mask\n            backward=np.flip(np.cumsum(np.flip(mask,axis=axis),axis=axis),axis=axis)-mask\n            add(f\'color{c}_{label}_before\',forward,\'rays\')\n            add(f\'color{c}_{label}_after\',backward,\'rays\')\n    # Component geometry for both monochrome regions and foreground objects.\n    for conn in (1,2):\n        structure=ndi.generate_binary_structure(2,conn)\n        maps={name:np.zeros_like(a) for name in (\n            \'area\',\'height\',\'width\',\'top\',\'left\',\'bottom\',\'right\',\'row\',\'col\',\n            \'edge\',\'holes\',\'colors\',\'color\',\'fill\',\'perimeter\',\'rankarea\',\n            \'nobjects\',\'rankrow\',\'rankcol\',\'sym_ud\',\'sym_lr\')}\n        groups=[]\n        for c in range(10):\n            labels,num=ndi.label(a==c,structure)\n            for label,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; sublabels=labels[sl]\n                mask=sublabels==label\n                groups.append((sy,sx,mask,c))\n        def component_maps(groups, prefix):\n            areas=[int(m.sum()) for _,_,m,_ in groups]\n            rows=sorted(set(sy.start for sy,_,_,_ in groups))\n            xs=sorted(set(sx.start for _,sx,_,_ in groups))\n            for sy,sx,mask,c in groups:\n                ah,aw=mask.shape; area=int(mask.sum())\n                gy,gx=np.indices(mask.shape); filled=ndi.binary_fill_holes(mask)\n                holes=int((filled&~mask).sum())\n                erosion=ndi.binary_erosion(mask)\n                properties={\'area\':area,\'height\':ah,\'width\':aw,\n                    \'top\':sy.start,\'left\':sx.start,\'bottom\':h-sy.stop,\'right\':w-sx.stop,\n                    \'row\':gy,\'col\':gx,\'edge\':int(sy.start==0 or sx.start==0 or sy.stop==h or sx.stop==w),\n                    \'holes\':holes,\'colors\':len(np.unique(a[sy,sx][mask])),\n                    \'color\':c,\'fill\':area==ah*aw,\'perimeter\':int((mask&~erosion).sum()),\n                    \'rankarea\':sum(z<area for z in areas),\'nobjects\':len(groups),\n                    \'rankrow\':rows.index(sy.start),\'rankcol\':xs.index(sx.start),\n                    \'sym_ud\':int(np.array_equal(mask,mask[::-1])),\n                    \'sym_lr\':int(np.array_equal(mask,mask[:,::-1]))}\n                for k,v in properties.items():\n                    view=maps[k][sy,sx]\n                    view[mask]=v[mask] if isinstance(v,np.ndarray) else v\n            for name,v in maps.items(): add(prefix+name,v,\'objects\')\n        component_maps(groups,f\'same{conn}_\')\n        for bgval,bgname in [(0,\'zero\'),(bg,\'mode\')]:\n            for v in maps.values(): v.fill(0)\n            labels,num=ndi.label(a!=bgval,structure)\n            groups=[]\n            for i,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; mask=labels[sl]==i\n                vals=a[sl][mask]\n                c=int(np.bincount(vals,minlength=10).argmax())\n                groups.append((sy,sx,mask,c))\n            component_maps(groups,f\'fg_{bgname}{conn}_\')\n    # Nearest non-background and nearest differing cell in eight directions.\n    for dy,dx in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:\n        raycolor=np.full_like(a,10); raydistance=np.full_like(a,31)\n        diffcolor=np.full_like(a,10); diffdistance=np.full_like(a,31)\n        for y in range(h):\n            for x in range(w):\n                for dist in range(1,31):\n                    ny,nx=y+dy*dist,x+dx*dist\n                    if not (0<=ny<h and 0<=nx<w): break\n                    if raycolor[y,x]==10 and a[ny,nx]!=bg:\n                        raycolor[y,x]=a[ny,nx];raydistance[y,x]=dist\n                    if diffcolor[y,x]==10 and a[ny,nx]!=a[y,x]:\n                        diffcolor[y,x]=a[ny,nx];diffdistance[y,x]=dist\n                    if raycolor[y,x]!=10 and diffcolor[y,x]!=10: break\n        for name,v in [(\'color\',raycolor),(\'distance\',raydistance),(\'diffcolor\',diffcolor),(\'diffdistance\',diffdistance)]:\n            add(f\'ray{dy}_{dx}_{name}\',v,\'rays\')\n    return np.stack(cols,axis=1).astype(np.float32),names,tags\n\n\ndef feature_indices(tags, group):\n    allowed={\'local\':{\'base\',\'local\'},\'objects\':{\'base\',\'objects\',\'color\'},\n             \'rays\':{\'base\',\'rays\',\'color\'},\'all\':set(tags)}[group]\n    return np.array([i for i,t in enumerate(tags) if t in allowed])\n\n\ndef tree_spec(tree,names,indices):\n    t=tree.tree_\n    def node(i):\n        if t.children_left[i] < 0:\n            return int(tree.classes_[np.argmax(t.value[i][0])])\n        return [names[indices[t.feature[i]]],float(t.threshold[i]),node(t.children_left[i]),node(t.children_right[i])]\n    return node(0)\n\n\ndef solve(task, execution=\'deferred\'):\n    startcpu,startwall=time.process_time(),time.perf_counter()\n    task=clean_task(task)\n    stats=Counter();candidates=[];all_proposals=[]\n    cache={}\n    def feat(a):\n        key=(a.shape,a.tobytes())\n        if key not in cache:\n            stats[\'feature_grid_calls\']+=1\n            stats[\'feature_cells\']+=a.size\n            cache[key]=features(a)\n        return cache[key]\n    train=[(np.array(p[\'input\'],dtype=np.int16),np.array(p[\'output\'],dtype=np.int16)) for p in task[\'train\']]\n    query=[np.array(p[\'input\'],dtype=np.int16) for p in task[\'test\']]\n    for transform_name in (\'identity\',\'crop_zero\',\'crop_mode\'):\n        transformed=[transform(a,transform_name) for a,b in train]\n        if any(a.shape!=b.shape for a,(_,b) in zip(transformed,train)):continue\n        # Identical transform behavior may differ on a query: preserve all names.\n        ftrain=[feat(a) for a in transformed]\n        names,tags=ftrain[0][1:]\n        y=np.concatenate([b.ravel() for a,b in train])\n        boundaries=np.cumsum([0]+[a.size for a in transformed])\n        for group in GROUPS:\n            ix=feature_indices(tags,group)\n            X=np.concatenate([f[0][:,ix] for f in ftrain])\n            for depth in DEPTHS:\n                model=DecisionTreeClassifier(max_depth=depth,random_state=0,criterion=\'entropy\')\n                model.fit(X,y); stats[\'models_fit\']+=1\n                predtrain=model.predict(X);stats[\'model_training_cells\']+=len(y)\n                fit=bool(np.array_equal(predtrain,y))\n                preds=None\n                # Eager explicitly materializes every proposal. Deferred delays\n                # query features and predictions until the demonstration fit.\n                if execution==\'eager\' or fit:\n                    preds=[]\n                    for q in query:\n                        tq=transform(q,transform_name)\n                        fq=feat(tq)[0][:,ix]\n                        pq=model.predict(fq).astype(np.int16).reshape(tq.shape).tolist()\n                        stats[\'query_model_grid_calls\']+=1;stats[\'query_model_cells\']+=tq.size\n                        preds.append(pq)\n                proposal={\'transform\':transform_name,\'group\':group,\'depth\':depth,\'fits\':fit,\n                          \'nodes\':int(model.tree_.node_count)}\n                all_proposals.append(proposal)\n                if not fit:continue\n                loo_exact=loo_cells=loo_total=0\n                if len(train)>1:\n                    for i in range(len(train)):\n                        keep=np.ones(len(y),dtype=bool);keep[boundaries[i]:boundaries[i+1]]=False\n                        cv=DecisionTreeClassifier(max_depth=depth,random_state=0,criterion=\'entropy\')\n                        cv.fit(X[keep],y[keep]);stats[\'models_fit\']+=1\n                        got=cv.predict(X[~keep]);truth=y[~keep]\n                        loo_exact+=int(np.array_equal(got,truth));loo_cells+=int((got==truth).sum());loo_total+=len(truth)\n                rank=[-loo_exact,-(loo_cells/max(1,loo_total)),int(model.tree_.node_count),\n                      (\'identity\',\'crop_zero\',\'crop_mode\').index(transform_name),GROUPS.index(group),depth]\n                candidates.append({\'predictions\':preds,\'program\':{\'transform\':transform_name,\'tree\':tree_spec(model,names,ix)},\n                    \'rank\':rank,\'loo_exact\':loo_exact,\'loo_cells\':loo_cells,\'loo_total\':loo_total,\'proposal\':proposal})\n    candidates.sort(key=lambda c:(c[\'rank\'],json.dumps(c[\'predictions\'],separators=(\',\',\':\'))))\n    distinct=[];seen=set()\n    for c in candidates:\n        key=json.dumps(c[\'predictions\'],separators=(\',\',\':\'))\n        if key not in seen: seen.add(key);distinct.append(c)\n    answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for c in distinct:\n            g=c[\'predictions\'][i]\n            if g not in grids:grids.append(g)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return {\'attempts\':answers,\'candidates\':distinct,\'consistent_programs\':candidates,\n            \'proposals\':all_proposals,\'metrics\':dict(stats),\'cpu_seconds\':time.process_time()-startcpu,\n            \'wall_seconds\':time.perf_counter()-startwall,\'execution\':execution}\n\n\ndef replay(program, grid):\n    a=transform(np.asarray(grid,dtype=np.int16),program[\'transform\'])\n    X,names,tags=features(a);index={s:i for i,s in enumerate(names)}\n    def node(tree, selected):\n        if isinstance(tree,int):return np.full(len(selected),tree,dtype=np.int16)\n        name,threshold,left,right=tree\n        mask=X[selected,index[name]]<=threshold\n        out=np.empty(len(selected),dtype=np.int16)\n        out[mask]=node(left,selected[mask]);out[~mask]=node(right,selected[~mask])\n        return out\n    return node(program[\'tree\'],np.arange(a.size)).reshape(a.shape).tolist()\n', 'scripts/solver_periodic_v2.py': '"""Minimum-description periodic denoising inside delimiter-separated panels.\nSPDX-License-Identifier: MIT-0\n"""\nimport numpy as np\nfrom scipy import ndimage as ndi\n\n\ndef denoise_line(line,min_repeats,penalty,max_fraction):\n    n=len(line);best=None\n    if n<2*min_repeats:return line\n    for p in range(1,n//min_repeats+1):\n        motif=np.array([np.bincount(line[i::p],minlength=10).argmax() for i in range(p)],dtype=np.uint8)\n        pred=motif[np.arange(n)%p]\n        errors=int(np.count_nonzero(pred!=line))\n        if errors>max_fraction*n:continue\n        # Literal motif symbols plus penalized correction locations.\n        rank=(p+penalty*errors,errors,p,tuple(motif))\n        if best is None or rank<best[0]:best=(rank,pred)\n    return line if best is None else best[1]\n\n\ndef periodic_repair(a,separator,axis,min_repeats,penalty,max_fraction,scope):\n    b=a.copy();h,w=a.shape\n    if scope==\'whole\':regions=[(slice(0,h),slice(0,w))]\n    else:\n        labels,n=ndi.label(a!=separator)\n        regions=[]\n        for i,sl in enumerate(ndi.find_objects(labels),1):\n            if sl is None:continue\n            sy,sx=sl\n            # Solid rectangular panels avoid cutting through arbitrary objects.\n            if not np.all(labels[sl]==i):continue\n            if scope==\'interior\' and (sy.start==0 or sx.start==0 or sy.stop==h or sx.stop==w):continue\n            regions.append(sl)\n    for sy,sx in regions:\n        region=a[sy,sx];target=b[sy,sx]\n        if axis==0:region=region.T;target=target.T\n        for i,line in enumerate(region):\n            target[i]=denoise_line(line,min_repeats,penalty,max_fraction)\n    return b if np.any(b!=a) else None\n\n\ndef role_repair(a,selector,axis,min_repeats,penalty,max_fraction,scope):\n    h,w=a.shape\n    counts=np.bincount(a.ravel(),minlength=10)\n    present=[int(c) for c in np.flatnonzero(counts)]\n    if selector==\'all\':separators=present\n    elif selector==\'second\':\n        ranked=sorted(present,key=lambda c:(-int(counts[c]),c))\n        if len(ranked)<2:return None\n        separators=[ranked[1]]\n    else:raise ValueError(selector)\n    regions=set()\n    if scope==\'whole\':regions.add((0,h,0,w))\n    else:\n        for sep in separators:\n            labels,n=ndi.label(a!=sep)\n            for i,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None:continue\n                sy,sx=sl\n                if scope==\'interior\' and (sy.start==0 or sx.start==0 or sy.stop==h or sx.stop==w):continue\n                if not np.all(labels[sl]==i):continue\n                regions.add((sy.start,sy.stop,sx.start,sx.stop))\n    chosen=a.copy();edits=np.zeros(a.shape,dtype=bool)\n    for y0,y1,x0,x1 in sorted(regions):\n        source=a[y0:y1,x0:x1];target=source.copy()\n        current_axis=(1 if source.shape[1]>=source.shape[0] else 0) if axis==\'long\' else axis\n        if current_axis==0:source=source.T;target=target.T\n        for i,line in enumerate(source):target[i]=denoise_line(line,min_repeats,penalty,max_fraction)\n        if current_axis==0:target=target.T\n        before=a[y0:y1,x0:x1];mask=target!=before\n        old=chosen[y0:y1,x0:x1];already=edits[y0:y1,x0:x1]\n        if np.any(mask&already&(old!=target)):return None\n        old[mask]=target[mask];already|=mask\n    return chosen if edits.any() else None\n\n\ndef operations(train):\n    from solver import Operation\n    out=[]\n    for scope in (\'interior\',\'panels\',\'whole\'):\n     for selector in ((\'all\',\'second\') if scope!=\'whole\' else (\'all\',)):\n      for axis in (\'long\',1,0):\n       for min_repeats in (3,2,4):\n        for penalty in (2,3,4):\n         for max_fraction in (0.2,0.1,0.3):\n          name=f\'role_periodic_{scope}_{selector}_axis{axis}_reps{min_repeats}_penalty{penalty}_fraction{max_fraction}\'\n          out.append(Operation(name,lambda a,selector=selector,axis=axis,min_repeats=min_repeats,penalty=penalty,max_fraction=max_fraction,scope=scope:\n            role_repair(a,selector,axis,min_repeats,penalty,max_fraction,scope)))\n    return out\n', 'scripts/diversity_selection.py': '"""Uniform selection; no task IDs, categories or labels. MIT-0."""\ndef select(baseline,periodic,relational,query_count):\n    out=[]\n    for i in range(query_count):\n        if periodic:\n            proposed=[c[\'predictions\'][i] for c in periodic]+list(baseline[i].values())\n        else:\n            proposed=[baseline[i][\'attempt_1\']]\n            if relational:proposed.append(relational[0][\'predictions\'][i])\n            proposed.append(baseline[i][\'attempt_2\'])\n        grids=[]\n        for g in proposed:\n            if g not in grids:grids.append(g)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(baseline[i][\'attempt_1\'])\n        out.append(dict(zip([\'attempt_1\',\'attempt_2\'],grids)))\n    return out\n', 'scripts/worker_portfolio_entry.py': '"""Fresh native + worker solvers. Two disclosed, uniform development policies.\nSPDX-License-Identifier: MIT-0\n"""\nfrom concurrent.futures import ProcessPoolExecutor\nimport resource\nimport time\nfrom solver_relational_v3 import clean_task, solve as relational_solve\nfrom solver_periodic_v2 import operations as periodic_operations\nfrom solver_deferred import search\nfrom diversity_selection import select as diversity_select\nfrom panel_portfolio_entry import predict as panel_predict\nfrom icecuber_entry import predict as native_predict\n\n\ndef extra_one(item):\n    name, task, baseline, profile = item\n    started = time.process_time()\n    relational = relational_solve(task, \'deferred\')\n    periodic = None\n    if profile == \'conservative\':\n        candidates = relational[\'candidates\']\n        use = bool(candidates and candidates[0][\'loo_exact\'] == len(task[\'train\']))\n        answers = []\n        for i, previous in enumerate(baseline):\n            grids = [candidates[0][\'predictions\'][i]] if use else []\n            for key in (\'attempt_1\', \'attempt_2\'):\n                if previous[key] not in grids:\n                    grids.append(previous[key])\n            while len(grids) < 2:\n                grids.append(previous[\'attempt_1\'])\n            answers.append(dict(zip((\'attempt_1\', \'attempt_2\'), grids[:2])))\n    else:\n        periodic = search(task, depth=1, ops_override=periodic_operations(task[\'train\']), terminal_map=False)\n        answers = diversity_select(baseline, periodic[\'candidates\'], relational[\'candidates\'], len(task[\'test\']))\n    return name, answers, {\'task_id\': name, \'relational\': relational, \'periodic\': periodic,\n                           \'baseline_attempts\': baseline, \'attempts\': answers,\n                           \'extra_cpu_seconds\': time.process_time() - started}\n\n\ndef predict(challenges, native_root, profile=\'diversity\', timeout=1500, workers=2):\n    if profile not in (\'conservative\', \'diversity\'):\n        raise ValueError(profile)\n    blind = {name: clean_task(task) for name, task in sorted(challenges.items())}\n    start = time.perf_counter()\n    cpu_start = time.process_time()\n    child_start = resource.getrusage(resource.RUSAGE_CHILDREN)\n    if profile == \'conservative\':\n        baseline, baseline_receipt = panel_predict(blind, native_root, timeout, workers)\n    else:\n        baseline, native_receipt = native_predict(blind, native_root, depth=2, timeout=timeout)\n        assert native_receipt[\'exit_code\'] == 0 and not native_receipt[\'missing_queries\']\n        baseline_receipt = {\'complete\': True, \'native_receipt\': native_receipt}\n    baseline_wall = time.perf_counter() - start\n    items = [(name, task, baseline[name], profile) for name, task in blind.items()]\n    output, rows = {}, []\n    with ProcessPoolExecutor(max_workers=workers) as pool:\n        for name, attempts, row in pool.map(extra_one, items, chunksize=1):\n            output[name] = attempts\n            rows.append(row)\n    children = resource.getrusage(resource.RUSAGE_CHILDREN)\n    return output, {\'complete\': True, \'profile\': profile, \'tasks\': len(blind),\n                    \'queries\': sum(len(t[\'test\']) for t in blind.values()), \'rows\': rows,\n                    \'baseline_receipt\': baseline_receipt, \'baseline_wall_seconds\': baseline_wall,\n                    \'wall_seconds\': time.perf_counter() - start,\n                    \'cpu_seconds\': time.process_time() - cpu_start + children.ru_utime + children.ru_stime\n                                   - child_start.ru_utime - child_start.ru_stime,\n                    \'workers\': workers, \'baseline_recomputed\': True,\n                    \'policy\': \'Conservative: panel/native then exact-LOO relational insertion. Diversity: native then periodic-first; otherwise native attempt1 plus top relational prediction. Profile is explicit, independent of task IDs and labels.\'}\n', 'scripts/solver_copy_v4.py': '"""Demonstration-fitted relational cell rules; no query labels or task IDs.\nSPDX-License-Identifier: MIT-0\n"""\nfrom __future__ import annotations\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport time\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.tree import DecisionTreeClassifier\n\nVERSION = \'copy-rules-v4\'\nDEPTHS = (2, 4, 6, 10)\nGROUPS = (\'local\', \'objects\', \'rays\', \'all\')\n\n\ndef clean_task(task):\n    return {\'train\': [{\'input\': p[\'input\'], \'output\': p[\'output\']} for p in task[\'train\']],\n            \'test\': [{\'input\': p[\'input\']} for p in task[\'test\']]}\n\n\ndef mode(a):\n    return int(np.bincount(a.ravel(), minlength=10).argmax())\n\n\ndef transform(a, name):\n    if name == \'identity\': return a\n    bg = 0 if name == \'crop_zero\' else mode(a)\n    y, x = np.nonzero(a != bg)\n    if len(y) == 0: return a\n    return a[y.min():y.max()+1, x.min():x.max()+1]\n\n\ndef features(a):\n    """Fixed input-only features. Each feature is an integer per grid cell."""\n    a = np.asarray(a, dtype=np.int16)\n    h, w = a.shape\n    yy, xx = np.indices((h, w))\n    cols, names, tags = [], [], []\n    bg = mode(a)\n    def add(name, v, tag):\n        if np.isscalar(v): v = np.full_like(a, v)\n        cols.append(np.array(v, dtype=np.int16, copy=True).ravel()); names.append(name); tags.append(tag)\n    add(\'color\', a, \'base\')\n    add(\'nonzero\', a != 0, \'base\'); add(\'nonbackground\', a != bg, \'base\')\n    add(\'background\', bg, \'base\')\n    # Absolute geometry is separate: simpler feature subsets can generalize away\n    # from accidental coordinate correlations.\n    for name, v in [(\'row\',yy),(\'column\',xx),(\'bottom\',h-1-yy),(\'right\',w-1-xx),\n                    (\'row_parity\',yy%2),(\'col_parity\',xx%2),\n                    (\'border\',np.minimum.reduce([yy,xx,h-1-yy,w-1-xx])),\n                    (\'diagonal\',yy-xx),(\'antidiagonal\',yy+xx),\n                    (\'h\',h),(\'w\',w)]: add(name,v,\'position\')\n    pad = np.pad(a,2,constant_values=10)\n    for dy in range(-2,3):\n        for dx in range(-2,3):\n            if dy == dx == 0: continue\n            add(f\'neighbor_{dy}_{dx}\',pad[2+dy:2+dy+h,2+dx:2+dx+w], \'local\')\n    add(\'mirror_ud\',a[::-1], \'local\'); add(\'mirror_lr\',a[:,::-1], \'local\')\n    add(\'mirror_both\',a[::-1,::-1], \'local\')\n    if h == w:\n        add(\'transpose\',a.T, \'local\'); add(\'antitranspose\',a[::-1,::-1].T,\'local\')\n    else:\n        add(\'transpose\',10,\'local\'); add(\'antitranspose\',10,\'local\')\n    for c in range(10):\n        mask = a == c\n        rows = mask.sum(1)[:,None] + np.zeros_like(a)\n        columns = mask.sum(0)[None,:] + np.zeros_like(a)\n        add(f\'color{c}_count\',mask.sum(),\'color\')\n        add(f\'color{c}_rowcount\',rows,\'rays\'); add(f\'color{c}_colcount\',columns,\'rays\')\n        y,x=np.nonzero(mask)\n        if len(y):\n            y0,y1,x0,x1=y.min(),y.max(),x.min(),x.max()\n            add(f\'color{c}_inbbox\',(yy>=y0)&(yy<=y1)&(xx>=x0)&(xx<=x1),\'color\')\n            add(f\'color{c}_bboxdy\',np.maximum(y0-yy,0)+np.maximum(yy-y1,0),\'color\')\n            add(f\'color{c}_bboxdx\',np.maximum(x0-xx,0)+np.maximum(xx-x1,0),\'color\')\n            dist = ndi.distance_transform_cdt(~mask,metric=\'taxicab\')\n            add(f\'color{c}_distance\',dist,\'color\')\n        else:\n            for suffix in (\'inbbox\',\'bboxdy\',\'bboxdx\',\'distance\'): add(f\'color{c}_{suffix}\',-1,\'color\')\n        # Counts on each side support intervals, parity, rays, and crossings.\n        for axis, label in [(0,\'vertical\'),(1,\'horizontal\')]:\n            forward=np.cumsum(mask,axis=axis)-mask\n            backward=np.flip(np.cumsum(np.flip(mask,axis=axis),axis=axis),axis=axis)-mask\n            add(f\'color{c}_{label}_before\',forward,\'rays\')\n            add(f\'color{c}_{label}_after\',backward,\'rays\')\n    # Component geometry for both monochrome regions and foreground objects.\n    for conn in (1,2):\n        structure=ndi.generate_binary_structure(2,conn)\n        maps={name:np.zeros_like(a) for name in (\n            \'area\',\'height\',\'width\',\'top\',\'left\',\'bottom\',\'right\',\'row\',\'col\',\n            \'edge\',\'holes\',\'colors\',\'color\',\'fill\',\'perimeter\',\'rankarea\',\n            \'nobjects\',\'rankrow\',\'rankcol\',\'sym_ud\',\'sym_lr\')}\n        groups=[]\n        for c in range(10):\n            labels,num=ndi.label(a==c,structure)\n            for label,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; sublabels=labels[sl]\n                mask=sublabels==label\n                groups.append((sy,sx,mask,c))\n        def component_maps(groups, prefix):\n            areas=[int(m.sum()) for _,_,m,_ in groups]\n            rows=sorted(set(sy.start for sy,_,_,_ in groups))\n            xs=sorted(set(sx.start for _,sx,_,_ in groups))\n            for sy,sx,mask,c in groups:\n                ah,aw=mask.shape; area=int(mask.sum())\n                gy,gx=np.indices(mask.shape); filled=ndi.binary_fill_holes(mask)\n                holes=int((filled&~mask).sum())\n                erosion=ndi.binary_erosion(mask)\n                properties={\'area\':area,\'height\':ah,\'width\':aw,\n                    \'top\':sy.start,\'left\':sx.start,\'bottom\':h-sy.stop,\'right\':w-sx.stop,\n                    \'row\':gy,\'col\':gx,\'edge\':int(sy.start==0 or sx.start==0 or sy.stop==h or sx.stop==w),\n                    \'holes\':holes,\'colors\':len(np.unique(a[sy,sx][mask])),\n                    \'color\':c,\'fill\':area==ah*aw,\'perimeter\':int((mask&~erosion).sum()),\n                    \'rankarea\':sum(z<area for z in areas),\'nobjects\':len(groups),\n                    \'rankrow\':rows.index(sy.start),\'rankcol\':xs.index(sx.start),\n                    \'sym_ud\':int(np.array_equal(mask,mask[::-1])),\n                    \'sym_lr\':int(np.array_equal(mask,mask[:,::-1]))}\n                for k,v in properties.items():\n                    view=maps[k][sy,sx]\n                    view[mask]=v[mask] if isinstance(v,np.ndarray) else v\n            for name,v in maps.items(): add(prefix+name,v,\'objects\')\n        component_maps(groups,f\'same{conn}_\')\n        for bgval,bgname in [(0,\'zero\'),(bg,\'mode\')]:\n            for v in maps.values(): v.fill(0)\n            labels,num=ndi.label(a!=bgval,structure)\n            groups=[]\n            for i,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; mask=labels[sl]==i\n                vals=a[sl][mask]\n                c=int(np.bincount(vals,minlength=10).argmax())\n                groups.append((sy,sx,mask,c))\n            component_maps(groups,f\'fg_{bgname}{conn}_\')\n    # Nearest non-background and nearest differing cell in eight directions.\n    for dy,dx in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:\n        raycolor=np.full_like(a,10); raydistance=np.full_like(a,31)\n        diffcolor=np.full_like(a,10); diffdistance=np.full_like(a,31)\n        for y in range(h):\n            for x in range(w):\n                for dist in range(1,31):\n                    ny,nx=y+dy*dist,x+dx*dist\n                    if not (0<=ny<h and 0<=nx<w): break\n                    if raycolor[y,x]==10 and a[ny,nx]!=bg:\n                        raycolor[y,x]=a[ny,nx];raydistance[y,x]=dist\n                    if diffcolor[y,x]==10 and a[ny,nx]!=a[y,x]:\n                        diffcolor[y,x]=a[ny,nx];diffdistance[y,x]=dist\n                    if raycolor[y,x]!=10 and diffcolor[y,x]!=10: break\n        for name,v in [(\'color\',raycolor),(\'distance\',raydistance),(\'diffcolor\',diffcolor),(\'diffdistance\',diffdistance)]:\n            add(f\'ray{dy}_{dx}_{name}\',v,\'rays\')\n    return np.stack(cols,axis=1).astype(np.float32),names,tags\n\n\ndef feature_indices(tags, group):\n    allowed={\'local\':{\'base\',\'local\'},\'objects\':{\'base\',\'objects\',\'color\'},\n             \'rays\':{\'base\',\'rays\',\'color\'},\'all\':set(tags)}[group]\n    return np.array([i for i,t in enumerate(tags) if t in allowed])\n\n\ndef tree_spec(tree,names,indices):\n    t=tree.tree_\n    def node(i):\n        if t.children_left[i] < 0:\n            return int(tree.classes_[np.argmax(t.value[i][0])])\n        return [names[indices[t.feature[i]]],float(t.threshold[i]),node(t.children_left[i]),node(t.children_right[i])]\n    return node(0)\n\n\ndef output_sources(X,names):\n    """Input-dependent color expressions plus literal colors."""\n    source_names=[\'constant:\'+str(c) for c in range(10)]\n    values=[np.full(len(X),c,dtype=np.int16) for c in range(10)]\n    for i,name in enumerate(names):\n        if name==\'color\' or name.startswith(\'neighbor_\') or name.startswith(\'mirror_\') or name in (\'transpose\',\'antitranspose\') or (name.startswith(\'ray\') and name.endswith(\'color\')) or (name.startswith(\'fg_\') and name.endswith(\'_color\')):\n            source_names.append(name)\n            v=X[:,i].astype(np.int16)\n            # An out-of-grid lookup keeps the original cell rather than emitting\n            # the feature vocabulary\'s sentinel value 10.\n            values.append(np.where((v>=0)&(v<=9),v,X[:,0]))\n    return np.stack(values,axis=1),source_names\n\n\ndef target_encodings(X,y,names):\n    yield \'literal\',y,None\n    S,source_names=output_sources(X,names)\n    identity=source_names.index(\'color\');changed=X[:,0]!=y\n    if not changed.any():return\n    eligible=np.flatnonzero(np.all(S[changed]==y[changed,None],axis=0))\n    for j in eligible:\n        if j==identity:continue\n        yield \'copy:\'+source_names[j],changed.astype(np.int16),[\'color\',source_names[j]]\n    # Greedy action assignment is a proposal constructor, not exhaustive search.\n    # It covers all target cells with at most the ten literal-color actions.\n    labels=np.full(len(y),-1,dtype=np.int16);labels[~changed]=0\n    actions=[\'color\']\n    while (labels<0).any():\n        unresolved=labels<0\n        coverage=(S[unresolved]==y[unresolved,None]).sum(0)\n        # Prefer input expressions to fixed colors when support ties.\n        best=max(range(S.shape[1]),key=lambda j:(int(coverage[j]),j>=10,-j))\n        assert coverage[best]>0\n        labels[unresolved&(S[:,best]==y)]=len(actions);actions.append(source_names[best])\n    yield \'greedy_copy\',labels,actions\n\n\ndef decode_actions(values,expressions,X,names):\n    if expressions is None:return values.astype(np.int16)\n    S,sources=output_sources(X,names)\n    indices=[sources.index(s) for s in expressions]\n    return S[np.arange(len(X)),np.array(indices)[values.astype(int)]].astype(np.int16)\n\n\ndef solve(task, execution=\'deferred\'):\n    startcpu,startwall=time.process_time(),time.perf_counter()\n    task=clean_task(task)\n    stats=Counter();candidates=[];all_proposals=[]\n    cache={}\n    def feat(a):\n        key=(a.shape,a.tobytes())\n        if key not in cache:\n            stats[\'feature_grid_calls\']+=1\n            stats[\'feature_cells\']+=a.size\n            cache[key]=features(a)\n        return cache[key]\n    train=[(np.array(p[\'input\'],dtype=np.int16),np.array(p[\'output\'],dtype=np.int16)) for p in task[\'train\']]\n    query=[np.array(p[\'input\'],dtype=np.int16) for p in task[\'test\']]\n    for transform_name in (\'identity\',\'crop_zero\',\'crop_mode\'):\n        transformed=[transform(a,transform_name) for a,b in train]\n        if any(a.shape!=b.shape for a,(_,b) in zip(transformed,train)):continue\n        # Identical transform behavior may differ on a query: preserve all names.\n        ftrain=[feat(a) for a in transformed]\n        names,tags=ftrain[0][1:]\n        y=np.concatenate([b.ravel() for a,b in train])\n        boundaries=np.cumsum([0]+[a.size for a in transformed])\n        Xall=np.concatenate([f[0] for f in ftrain])\n        model_cache={};fold_cache={}\n        def fit_once(X,targets,group,depth,fold):\n            # Within a transform and group, X is fixed; a fold fixes its row\n            # subset. Distinct action decoders can share identical fitted trees.\n            key=(group,depth,fold,targets.dtype.str,targets.tobytes())\n            stats[\'model_fit_requests\']+=1\n            if key not in model_cache:\n                fitted=DecisionTreeClassifier(max_depth=depth,random_state=0,criterion=\'entropy\')\n                fitted.fit(X,targets);model_cache[key]=fitted;stats[\'models_fit\']+=1\n            else:stats[\'model_fit_cache_hits\']+=1\n            return model_cache[key]\n        for encoding,targets,expressions in target_encodings(Xall,y,names):\n          for group in GROUPS:\n            ix=feature_indices(tags,group)\n            X=Xall[:,ix]\n            for depth in DEPTHS:\n                model=fit_once(X,targets,group,depth,-1)\n                rawtrain=model.predict(X)\n                predtrain=decode_actions(rawtrain,expressions,Xall,names)\n                stats[\'model_training_cells\']+=len(y)\n                fit=bool(np.array_equal(predtrain,y))\n                preds=None\n                if execution==\'eager\' or fit:\n                    preds=[]\n                    for q in query:\n                        tq=transform(q,transform_name)\n                        fq=feat(tq)[0]\n                        raw=model.predict(fq[:,ix])\n                        pq=decode_actions(raw,expressions,fq,names).reshape(tq.shape).tolist()\n                        stats[\'query_model_grid_calls\']+=1;stats[\'query_model_cells\']+=tq.size\n                        preds.append(pq)\n                proposal={\'transform\':transform_name,\'group\':group,\'depth\':depth,\'fits\':fit,\n                          \'nodes\':int(model.tree_.node_count),\'encoding\':encoding}\n                all_proposals.append(proposal)\n                if not fit:continue\n                loo_exact=loo_cells=loo_total=0\n                if len(train)>1:\n                    for i in range(len(train)):\n                        keep=np.ones(len(y),dtype=bool);keep[boundaries[i]:boundaries[i+1]]=False\n                        # Reconstruct action vocabulary and assignments without\n                        # the omitted demonstration\'s outputs. The old v2 folds\n                        # reused full-demonstration greedy target assignments.\n                        fold_key=(encoding,i)\n                        if fold_key not in fold_cache:\n                            fold_cache[fold_key]=next(((yy,ex) for en,yy,ex in target_encodings(Xall[keep],y[keep],names)\n                                                      if en==encoding),None)\n                        fold=fold_cache[fold_key]\n                        truth=y[~keep];loo_total+=len(truth)\n                        if fold is None:continue\n                        fold_targets,fold_expressions=fold\n                        cv=fit_once(X[keep],fold_targets,group,depth,i)\n                        raw=cv.predict(X[~keep])\n                        got=decode_actions(raw,fold_expressions,Xall[~keep],names)\n                        loo_exact+=int(np.array_equal(got,truth));loo_cells+=int((got==truth).sum())\n                rank=[-loo_exact,-(loo_cells/max(1,loo_total)),int(model.tree_.node_count)+(len(expressions) if expressions else 0),\n                      (\'identity\',\'crop_zero\',\'crop_mode\').index(transform_name),GROUPS.index(group),depth,encoding]\n                candidates.append({\'predictions\':preds,\'program\':{\'transform\':transform_name,\'tree\':tree_spec(model,names,ix),\'expressions\':expressions},\n                    \'rank\':rank,\'loo_exact\':loo_exact,\'loo_cells\':loo_cells,\'loo_total\':loo_total,\'proposal\':proposal})\n    candidates.sort(key=lambda c:(c[\'rank\'],json.dumps(c[\'predictions\'],separators=(\',\',\':\'))))\n    distinct=[];seen=set()\n    for c in candidates:\n        key=json.dumps(c[\'predictions\'],separators=(\',\',\':\'))\n        if key not in seen: seen.add(key);distinct.append(c)\n    answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for c in distinct:\n            g=c[\'predictions\'][i]\n            if g not in grids:grids.append(g)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return {\'attempts\':answers,\'candidates\':distinct,\'consistent_programs\':candidates,\n            \'proposals\':all_proposals,\'metrics\':dict(stats),\'cpu_seconds\':time.process_time()-startcpu,\n            \'wall_seconds\':time.perf_counter()-startwall,\'execution\':execution}\n\n\ndef replay(program, grid):\n    a=transform(np.asarray(grid,dtype=np.int16),program[\'transform\'])\n    X,names,tags=features(a);index={s:i for i,s in enumerate(names)}\n    def node(tree, selected):\n        if isinstance(tree,int):return np.full(len(selected),tree,dtype=np.int16)\n        name,threshold,left,right=tree\n        mask=X[selected,index[name]]<=threshold\n        out=np.empty(len(selected),dtype=np.int16)\n        out[mask]=node(left,selected[mask]);out[~mask]=node(right,selected[~mask])\n        return out\n    raw=node(program[\'tree\'],np.arange(a.size))\n    return decode_actions(raw,program.get(\'expressions\'),X,names).reshape(a.shape).tolist()\n', 'scripts/copy_portfolio_entry.py': '"""Fresh native + worker solvers. Two disclosed, uniform development policies.\nSPDX-License-Identifier: MIT-0\n"""\nfrom concurrent.futures import ProcessPoolExecutor\nimport resource\nimport time\nfrom solver_copy_v4 import clean_task, solve as relational_solve\nfrom solver_periodic_v2 import operations as periodic_operations\nfrom solver_deferred import search\nfrom diversity_selection import select as diversity_select\nfrom panel_portfolio_entry import predict as panel_predict\nfrom icecuber_entry import predict as native_predict\n\n\ndef extra_one(item):\n    name, task, baseline, profile = item\n    started = time.process_time()\n    relational = relational_solve(task, \'deferred\')\n    periodic = None\n    if profile == \'copy-conservative\':\n        candidates = relational[\'candidates\']\n        use = bool(candidates and candidates[0][\'loo_exact\'] == len(task[\'train\']))\n        answers = []\n        for i, previous in enumerate(baseline):\n            grids = [candidates[0][\'predictions\'][i]] if use else []\n            for key in (\'attempt_1\', \'attempt_2\'):\n                if previous[key] not in grids:\n                    grids.append(previous[key])\n            while len(grids) < 2:\n                grids.append(previous[\'attempt_1\'])\n            answers.append(dict(zip((\'attempt_1\', \'attempt_2\'), grids[:2])))\n    else:\n        periodic = search(task, depth=1, ops_override=periodic_operations(task[\'train\']), terminal_map=False)\n        answers = diversity_select(baseline, periodic[\'candidates\'], relational[\'candidates\'], len(task[\'test\']))\n    return name, answers, {\'task_id\': name, \'relational\': relational, \'periodic\': periodic,\n                           \'baseline_attempts\': baseline, \'attempts\': answers,\n                           \'extra_cpu_seconds\': time.process_time() - started}\n\n\ndef predict(challenges, native_root, profile=\'copy-conservative\', timeout=1500, workers=2):\n    if profile not in (\'copy-conservative\',):\n        raise ValueError(profile)\n    blind = {name: clean_task(task) for name, task in sorted(challenges.items())}\n    start = time.perf_counter()\n    cpu_start = time.process_time()\n    child_start = resource.getrusage(resource.RUSAGE_CHILDREN)\n    if profile == \'copy-conservative\':\n        baseline, baseline_receipt = panel_predict(blind, native_root, timeout, workers)\n    else:\n        baseline, native_receipt = native_predict(blind, native_root, depth=2, timeout=timeout)\n        assert native_receipt[\'exit_code\'] == 0 and not native_receipt[\'missing_queries\']\n        baseline_receipt = {\'complete\': True, \'native_receipt\': native_receipt}\n    baseline_wall = time.perf_counter() - start\n    items = [(name, task, baseline[name], profile) for name, task in blind.items()]\n    output, rows = {}, []\n    with ProcessPoolExecutor(max_workers=workers) as pool:\n        for name, attempts, row in pool.map(extra_one, items, chunksize=1):\n            output[name] = attempts\n            rows.append(row)\n    children = resource.getrusage(resource.RUSAGE_CHILDREN)\n    return output, {\'complete\': True, \'profile\': profile, \'tasks\': len(blind),\n                    \'queries\': sum(len(t[\'test\']) for t in blind.values()), \'rows\': rows,\n                    \'baseline_receipt\': baseline_receipt, \'baseline_wall_seconds\': baseline_wall,\n                    \'wall_seconds\': time.perf_counter() - start,\n                    \'cpu_seconds\': time.process_time() - cpu_start + children.ru_utime + children.ru_stime\n                                   - child_start.ru_utime - child_start.ru_stime,\n                    \'workers\': workers, \'baseline_recomputed\': True,\n                    \'policy\': \'Panel/native then copy-action v4 insertion only when all omitted-demonstration folds predict exactly. Uniform policy, independent of IDs and query labels.\'}\n', 'scripts/solver_color_base.py': '"""Demonstration-fitted relational cell rules; no query labels or task IDs.\nSPDX-License-Identifier: MIT-0\n"""\nfrom __future__ import annotations\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport time\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.tree import DecisionTreeClassifier\n\nVERSION = \'copy-rules-v4\'\nDEPTHS = (2, 4, 6, 10)\nGROUPS = (\'local\', \'objects\', \'rays\', \'all\')\n\n\ndef clean_task(task):\n    return {\'train\': [{\'input\': p[\'input\'], \'output\': p[\'output\']} for p in task[\'train\']],\n            \'test\': [{\'input\': p[\'input\']} for p in task[\'test\']]}\n\n\ndef mode(a):\n    return int(np.bincount(a.ravel(), minlength=10).argmax())\n\n\ndef transform(a, name):\n    if name == \'identity\': return a\n    bg = 0 if name == \'crop_zero\' else mode(a)\n    y, x = np.nonzero(a != bg)\n    if len(y) == 0: return a\n    return a[y.min():y.max()+1, x.min():x.max()+1]\n\n\ndef features(a):\n    """Fixed input-only features. Each feature is an integer per grid cell."""\n    a = np.asarray(a, dtype=np.int16)\n    h, w = a.shape\n    yy, xx = np.indices((h, w))\n    cols, names, tags = [], [], []\n    bg = mode(a)\n    def add(name, v, tag):\n        if np.isscalar(v): v = np.full_like(a, v)\n        cols.append(np.array(v, dtype=np.int16, copy=True).ravel()); names.append(name); tags.append(tag)\n    add(\'color\', a, \'base\')\n    add(\'nonzero\', a != 0, \'base\'); add(\'nonbackground\', a != bg, \'base\')\n    add(\'background\', bg, \'base\')\n    # Absolute geometry is separate: simpler feature subsets can generalize away\n    # from accidental coordinate correlations.\n    for name, v in [(\'row\',yy),(\'column\',xx),(\'bottom\',h-1-yy),(\'right\',w-1-xx),\n                    (\'row_parity\',yy%2),(\'col_parity\',xx%2),\n                    (\'border\',np.minimum.reduce([yy,xx,h-1-yy,w-1-xx])),\n                    (\'diagonal\',yy-xx),(\'antidiagonal\',yy+xx),\n                    (\'h\',h),(\'w\',w)]: add(name,v,\'position\')\n    pad = np.pad(a,2,constant_values=10)\n    for dy in range(-2,3):\n        for dx in range(-2,3):\n            if dy == dx == 0: continue\n            add(f\'neighbor_{dy}_{dx}\',pad[2+dy:2+dy+h,2+dx:2+dx+w], \'local\')\n    add(\'mirror_ud\',a[::-1], \'local\'); add(\'mirror_lr\',a[:,::-1], \'local\')\n    add(\'mirror_both\',a[::-1,::-1], \'local\')\n    if h == w:\n        add(\'transpose\',a.T, \'local\'); add(\'antitranspose\',a[::-1,::-1].T,\'local\')\n    else:\n        add(\'transpose\',10,\'local\'); add(\'antitranspose\',10,\'local\')\n    for c in range(10):\n        mask = a == c\n        rows = mask.sum(1)[:,None] + np.zeros_like(a)\n        columns = mask.sum(0)[None,:] + np.zeros_like(a)\n        add(f\'color{c}_count\',mask.sum(),\'color\')\n        add(f\'color{c}_rowcount\',rows,\'rays\'); add(f\'color{c}_colcount\',columns,\'rays\')\n        y,x=np.nonzero(mask)\n        if len(y):\n            y0,y1,x0,x1=y.min(),y.max(),x.min(),x.max()\n            add(f\'color{c}_inbbox\',(yy>=y0)&(yy<=y1)&(xx>=x0)&(xx<=x1),\'color\')\n            add(f\'color{c}_bboxdy\',np.maximum(y0-yy,0)+np.maximum(yy-y1,0),\'color\')\n            add(f\'color{c}_bboxdx\',np.maximum(x0-xx,0)+np.maximum(xx-x1,0),\'color\')\n            dist = ndi.distance_transform_cdt(~mask,metric=\'taxicab\')\n            add(f\'color{c}_distance\',dist,\'color\')\n        else:\n            for suffix in (\'inbbox\',\'bboxdy\',\'bboxdx\',\'distance\'): add(f\'color{c}_{suffix}\',-1,\'color\')\n        # Counts on each side support intervals, parity, rays, and crossings.\n        for axis, label in [(0,\'vertical\'),(1,\'horizontal\')]:\n            forward=np.cumsum(mask,axis=axis)-mask\n            backward=np.flip(np.cumsum(np.flip(mask,axis=axis),axis=axis),axis=axis)-mask\n            add(f\'color{c}_{label}_before\',forward,\'rays\')\n            add(f\'color{c}_{label}_after\',backward,\'rays\')\n    # Component geometry for both monochrome regions and foreground objects.\n    for conn in (1,2):\n        structure=ndi.generate_binary_structure(2,conn)\n        maps={name:np.zeros_like(a) for name in (\n            \'area\',\'height\',\'width\',\'top\',\'left\',\'bottom\',\'right\',\'row\',\'col\',\n            \'edge\',\'holes\',\'colors\',\'color\',\'fill\',\'perimeter\',\'rankarea\',\n            \'nobjects\',\'rankrow\',\'rankcol\',\'sym_ud\',\'sym_lr\')}\n        groups=[]\n        for c in range(10):\n            labels,num=ndi.label(a==c,structure)\n            for label,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; sublabels=labels[sl]\n                mask=sublabels==label\n                groups.append((sy,sx,mask,c))\n        def component_maps(groups, prefix):\n            areas=[int(m.sum()) for _,_,m,_ in groups]\n            rows=sorted(set(sy.start for sy,_,_,_ in groups))\n            xs=sorted(set(sx.start for _,sx,_,_ in groups))\n            for sy,sx,mask,c in groups:\n                ah,aw=mask.shape; area=int(mask.sum())\n                gy,gx=np.indices(mask.shape); filled=ndi.binary_fill_holes(mask)\n                holes=int((filled&~mask).sum())\n                erosion=ndi.binary_erosion(mask)\n                properties={\'area\':area,\'height\':ah,\'width\':aw,\n                    \'top\':sy.start,\'left\':sx.start,\'bottom\':h-sy.stop,\'right\':w-sx.stop,\n                    \'row\':gy,\'col\':gx,\'edge\':int(sy.start==0 or sx.start==0 or sy.stop==h or sx.stop==w),\n                    \'holes\':holes,\'colors\':len(np.unique(a[sy,sx][mask])),\n                    \'color\':c,\'fill\':area==ah*aw,\'perimeter\':int((mask&~erosion).sum()),\n                    \'rankarea\':sum(z<area for z in areas),\'nobjects\':len(groups),\n                    \'rankrow\':rows.index(sy.start),\'rankcol\':xs.index(sx.start),\n                    \'sym_ud\':int(np.array_equal(mask,mask[::-1])),\n                    \'sym_lr\':int(np.array_equal(mask,mask[:,::-1]))}\n                for k,v in properties.items():\n                    view=maps[k][sy,sx]\n                    view[mask]=v[mask] if isinstance(v,np.ndarray) else v\n            for name,v in maps.items(): add(prefix+name,v,\'objects\')\n        component_maps(groups,f\'same{conn}_\')\n        for bgval,bgname in [(0,\'zero\'),(bg,\'mode\')]:\n            for v in maps.values(): v.fill(0)\n            labels,num=ndi.label(a!=bgval,structure)\n            groups=[]\n            for i,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; mask=labels[sl]==i\n                vals=a[sl][mask]\n                c=int(np.bincount(vals,minlength=10).argmax())\n                groups.append((sy,sx,mask,c))\n            component_maps(groups,f\'fg_{bgname}{conn}_\')\n    # Nearest non-background and nearest differing cell in eight directions.\n    for dy,dx in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:\n        raycolor=np.full_like(a,10); raydistance=np.full_like(a,31)\n        diffcolor=np.full_like(a,10); diffdistance=np.full_like(a,31)\n        for y in range(h):\n            for x in range(w):\n                for dist in range(1,31):\n                    ny,nx=y+dy*dist,x+dx*dist\n                    if not (0<=ny<h and 0<=nx<w): break\n                    if raycolor[y,x]==10 and a[ny,nx]!=bg:\n                        raycolor[y,x]=a[ny,nx];raydistance[y,x]=dist\n                    if diffcolor[y,x]==10 and a[ny,nx]!=a[y,x]:\n                        diffcolor[y,x]=a[ny,nx];diffdistance[y,x]=dist\n                    if raycolor[y,x]!=10 and diffcolor[y,x]!=10: break\n        for name,v in [(\'color\',raycolor),(\'distance\',raydistance),(\'diffcolor\',diffcolor),(\'diffdistance\',diffdistance)]:\n            add(f\'ray{dy}_{dx}_{name}\',v,\'rays\')\n    return np.stack(cols,axis=1).astype(np.float32),names,tags\n\n\ndef feature_indices(tags, group):\n    allowed={\'local\':{\'base\',\'local\'},\'objects\':{\'base\',\'objects\',\'color\'},\n             \'rays\':{\'base\',\'rays\',\'color\'},\'all\':set(tags)}[group]\n    return np.array([i for i,t in enumerate(tags) if t in allowed])\n\n\ndef tree_spec(tree,names,indices):\n    t=tree.tree_\n    def node(i):\n        if t.children_left[i] < 0:\n            return int(tree.classes_[np.argmax(t.value[i][0])])\n        return [names[indices[t.feature[i]]],float(t.threshold[i]),node(t.children_left[i]),node(t.children_right[i])]\n    return node(0)\n\n\ndef output_sources(X,names):\n    """Input-dependent color expressions plus literal colors."""\n    source_names=[\'constant:\'+str(c) for c in range(10)]\n    values=[np.full(len(X),c,dtype=np.int16) for c in range(10)]\n    for i,name in enumerate(names):\n        if name==\'color\' or name.startswith(\'neighbor_\') or name.startswith(\'mirror_\') or name in (\'transpose\',\'antitranspose\') or (name.startswith(\'ray\') and name.endswith(\'color\')) or (name.startswith(\'fg_\') and name.endswith(\'_color\')):\n            source_names.append(name)\n            v=X[:,i].astype(np.int16)\n            # An out-of-grid lookup keeps the original cell rather than emitting\n            # the feature vocabulary\'s sentinel value 10.\n            values.append(np.where((v>=0)&(v<=9),v,X[:,0]))\n    return np.stack(values,axis=1),source_names\n\n\ndef target_encodings(X,y,names):\n    yield \'literal\',y,None\n    S,source_names=output_sources(X,names)\n    identity=source_names.index(\'color\');changed=X[:,0]!=y\n    if not changed.any():return\n    eligible=np.flatnonzero(np.all(S[changed]==y[changed,None],axis=0))\n    for j in eligible:\n        if j==identity:continue\n        yield \'copy:\'+source_names[j],changed.astype(np.int16),[\'color\',source_names[j]]\n    # Greedy action assignment is a proposal constructor, not exhaustive search.\n    # It covers all target cells with at most the ten literal-color actions.\n    labels=np.full(len(y),-1,dtype=np.int16);labels[~changed]=0\n    actions=[\'color\']\n    while (labels<0).any():\n        unresolved=labels<0\n        coverage=(S[unresolved]==y[unresolved,None]).sum(0)\n        # Prefer input expressions to fixed colors when support ties.\n        best=max(range(S.shape[1]),key=lambda j:(int(coverage[j]),j>=10,-j))\n        assert coverage[best]>0\n        labels[unresolved&(S[:,best]==y)]=len(actions);actions.append(source_names[best])\n    yield \'greedy_copy\',labels,actions\n\n\ndef decode_actions(values,expressions,X,names):\n    if expressions is None:return values.astype(np.int16)\n    S,sources=output_sources(X,names)\n    indices=[sources.index(s) for s in expressions]\n    return S[np.arange(len(X)),np.array(indices)[values.astype(int)]].astype(np.int16)\n\n\ndef solve(task, execution=\'deferred\'):\n    startcpu,startwall=time.process_time(),time.perf_counter()\n    task=clean_task(task)\n    stats=Counter();candidates=[];all_proposals=[]\n    cache={}\n    def feat(a):\n        key=(a.shape,a.tobytes())\n        if key not in cache:\n            stats[\'feature_grid_calls\']+=1\n            stats[\'feature_cells\']+=a.size\n            cache[key]=features(a)\n        return cache[key]\n    train=[(np.array(p[\'input\'],dtype=np.int16),np.array(p[\'output\'],dtype=np.int16)) for p in task[\'train\']]\n    query=[np.array(p[\'input\'],dtype=np.int16) for p in task[\'test\']]\n    for transform_name in (\'identity\',\'crop_zero\',\'crop_mode\'):\n        transformed=[transform(a,transform_name) for a,b in train]\n        if any(a.shape!=b.shape for a,(_,b) in zip(transformed,train)):continue\n        # Identical transform behavior may differ on a query: preserve all names.\n        ftrain=[feat(a) for a in transformed]\n        names,tags=ftrain[0][1:]\n        y=np.concatenate([b.ravel() for a,b in train])\n        boundaries=np.cumsum([0]+[a.size for a in transformed])\n        Xall=np.concatenate([f[0] for f in ftrain])\n        model_cache={};fold_cache={}\n        def fit_once(X,targets,group,depth,fold):\n            # Within a transform and group, X is fixed; a fold fixes its row\n            # subset. Distinct action decoders can share identical fitted trees.\n            key=(group,depth,fold,targets.dtype.str,targets.tobytes())\n            stats[\'model_fit_requests\']+=1\n            if key not in model_cache:\n                fitted=DecisionTreeClassifier(max_depth=depth,random_state=0,criterion=\'entropy\')\n                fitted.fit(X,targets);model_cache[key]=fitted;stats[\'models_fit\']+=1\n            else:stats[\'model_fit_cache_hits\']+=1\n            return model_cache[key]\n        for encoding,targets,expressions in target_encodings(Xall,y,names):\n          for group in GROUPS:\n            ix=feature_indices(tags,group)\n            X=Xall[:,ix]\n            for depth in DEPTHS:\n                model=fit_once(X,targets,group,depth,-1)\n                rawtrain=model.predict(X)\n                predtrain=decode_actions(rawtrain,expressions,Xall,names)\n                stats[\'model_training_cells\']+=len(y)\n                fit=bool(np.array_equal(predtrain,y))\n                preds=None\n                if execution==\'eager\' or fit:\n                    preds=[]\n                    for q in query:\n                        tq=transform(q,transform_name)\n                        fq=feat(tq)[0]\n                        raw=model.predict(fq[:,ix])\n                        pq=decode_actions(raw,expressions,fq,names).reshape(tq.shape).tolist()\n                        stats[\'query_model_grid_calls\']+=1;stats[\'query_model_cells\']+=tq.size\n                        preds.append(pq)\n                proposal={\'transform\':transform_name,\'group\':group,\'depth\':depth,\'fits\':fit,\n                          \'nodes\':int(model.tree_.node_count),\'encoding\':encoding}\n                all_proposals.append(proposal)\n                if not fit:continue\n                loo_exact=loo_cells=loo_total=0\n                if len(train)>1:\n                    for i in range(len(train)):\n                        keep=np.ones(len(y),dtype=bool);keep[boundaries[i]:boundaries[i+1]]=False\n                        # Reconstruct action vocabulary and assignments without\n                        # the omitted demonstration\'s outputs. The old v2 folds\n                        # reused full-demonstration greedy target assignments.\n                        fold_key=(encoding,i)\n                        if fold_key not in fold_cache:\n                            fold_cache[fold_key]=next(((yy,ex) for en,yy,ex in target_encodings(Xall[keep],y[keep],names)\n                                                      if en==encoding),None)\n                        fold=fold_cache[fold_key]\n                        truth=y[~keep];loo_total+=len(truth)\n                        if fold is None:continue\n                        fold_targets,fold_expressions=fold\n                        cv=fit_once(X[keep],fold_targets,group,depth,i)\n                        raw=cv.predict(X[~keep])\n                        got=decode_actions(raw,fold_expressions,Xall[~keep],names)\n                        loo_exact+=int(np.array_equal(got,truth));loo_cells+=int((got==truth).sum())\n                rank=[-loo_exact,-(loo_cells/max(1,loo_total)),int(model.tree_.node_count)+(len(expressions) if expressions else 0),\n                      (\'identity\',\'crop_zero\',\'crop_mode\').index(transform_name),GROUPS.index(group),depth,encoding]\n                candidates.append({\'predictions\':preds,\'program\':{\'transform\':transform_name,\'tree\':tree_spec(model,names,ix),\'expressions\':expressions},\n                    \'rank\':rank,\'loo_exact\':loo_exact,\'loo_cells\':loo_cells,\'loo_total\':loo_total,\'proposal\':proposal})\n    candidates.sort(key=lambda c:(c[\'rank\'],json.dumps(c[\'predictions\'],separators=(\',\',\':\'))))\n    distinct=[];seen=set()\n    for c in candidates:\n        key=json.dumps(c[\'predictions\'],separators=(\',\',\':\'))\n        if key not in seen: seen.add(key);distinct.append(c)\n    answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for c in distinct:\n            g=c[\'predictions\'][i]\n            if g not in grids:grids.append(g)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return {\'attempts\':answers,\'candidates\':distinct,\'consistent_programs\':candidates,\n            \'proposals\':all_proposals,\'metrics\':dict(stats),\'cpu_seconds\':time.process_time()-startcpu,\n            \'wall_seconds\':time.perf_counter()-startwall,\'execution\':execution}\n\n\ndef replay(program, grid):\n    a=transform(np.asarray(grid,dtype=np.int16),program[\'transform\'])\n    X,names,tags=features(a);index={s:i for i,s in enumerate(names)}\n    def node(tree, selected):\n        if isinstance(tree,int):return np.full(len(selected),tree,dtype=np.int16)\n        name,threshold,left,right=tree\n        mask=X[selected,index[name]]<=threshold\n        out=np.empty(len(selected),dtype=np.int16)\n        out[mask]=node(left,selected[mask]);out[~mask]=node(right,selected[~mask])\n        return out\n    raw=node(program[\'tree\'],np.arange(a.size))\n    return decode_actions(raw,program.get(\'expressions\'),X,names).reshape(a.shape).tolist()\n', 'scripts/solver_color_relations.py': '"""Input-only color equality/role features for finite learned copy proposals.\nSPDX-License-Identifier: MIT-0. Constructor and fitter: adjacent frozen v4.\n"""\nfrom collections import Counter\nfrom itertools import combinations\nimport numpy as np\nfrom scipy import ndimage as ndi\nimport solver_color_base as base\n\nVERSION = \'color-relations-v1\'\nORIGINAL_FEATURES = base.features\nORIGINAL_SOURCES = base.output_sources\n\n\ndef features(a):\n    a = np.asarray(a, dtype=np.int16)\n    X, names, tags = ORIGINAL_FEATURES(a)\n    values = [X]\n    names, tags = list(names), list(tags)\n    byname = {n: X[:, i] for i, n in enumerate(names)}\n    h, w = a.shape\n    counts = np.bincount(a.ravel(), minlength=10)\n    bg = int(counts.argmax())\n    present = list(map(int, np.flatnonzero(counts)))\n    appearance = list(dict.fromkeys(map(int, a.ravel())))\n    orderings = {\n        \'frequency\': sorted(present, key=lambda c: (-counts[c], appearance.index(c))),\n        \'rare\': sorted(present, key=lambda c: (counts[c], appearance.index(c))),\n        \'appearance\': appearance,\n        \'foreground\': sorted([c for c in present if c != bg],\n                             key=lambda c: (-counts[c], appearance.index(c))),\n    }\n    def add(name, v, tag):\n        if np.isscalar(v): v = np.full(a.size, v)\n        v = np.asarray(v, dtype=np.float32).reshape(-1)\n        assert len(v) == a.size\n        values.append(v[:, None]); names.append(name); tags.append(tag)\n        byname[name] = v\n\n    for role, colors in orderings.items():\n        rank = np.full(10, 10)\n        for i, c in enumerate(colors): rank[c] = i\n        add(\'role_rank_\' + role, rank[a], \'color\')\n        for i in range(3):\n            c = colors[i] if len(colors) > i else bg\n            add(f\'role_color_{role}_{i}\', c, \'base\')\n            add(f\'role_is_{role}_{i}\', a == c, \'color\')\n    add(\'own_color_count\', counts[a], \'color\')\n    add(\'own_color_unique\', counts[a] == 1, \'color\')\n    add(\'palette_size\', len(present), \'color\')\n\n    # Equality edges form short color-independent conditions instead of numeric\n    # color-ID thresholds. The finite fixed list is independent of task/labels.\n    local = [\'color\'] + [f\'neighbor_{dy}_{dx}\' for dy, dx in\n        [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]]\n    mirrors = [\'color\', \'mirror_ud\', \'mirror_lr\', \'mirror_both\',\n               \'transpose\', \'antitranspose\']\n    rays = [f\'ray{dy}_{dx}_color\' for dy, dx in\n        [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]]\n    differing = [r[:-5] + \'diffcolor\' for r in rays]\n    for group, sources, tag in [(\'local\', local, \'local\'),\n                                (\'mirrors\', mirrors, \'local\'),\n                                (\'rays\', [\'color\'] + rays + differing, \'rays\')]:\n        for left, right in combinations(sources, 2):\n            x, y = byname[left], byname[right]\n            add(\'equal_\' + group + \'_\' + left + \'_\' + right,\n                (x == y) & (x < 10) & (y < 10), tag)\n        for name in sources:\n            color = byname[name].astype(int)\n            valid = color < 10\n            safe = np.clip(color, 0, 9)\n            add(\'source_is_bg_\' + group + \'_\' + name,\n                valid & (color == bg), tag)\n            add(\'source_count_\' + group + \'_\' + name,\n                np.where(valid, counts[safe], -1), tag)\n\n    # Candidate decoders can select the nearest nonbackground/different color\n    # without learning its direction or literal color identifier.\n    for kind, sources in [(\'ray\', rays), (\'different\', differing)]:\n        distances = np.stack([byname[s[:-5] + \'distance\'] for s in sources], axis=1)\n        colors = np.stack([byname[s] for s in sources], axis=1).astype(int)\n        minimum = distances.min(axis=1)\n        chosen = distances.argmin(axis=1)\n        copied = colors[np.arange(a.size), chosen]\n        add(\'closest_color_\' + kind, np.where(copied < 10, copied, a.ravel()), \'rays\')\n        add(\'closest_distance_\' + kind, minimum, \'rays\')\n        add(\'closest_unique_\' + kind, (distances == minimum[:, None]).sum(axis=1) == 1, \'rays\')\n\n    # Global nearest foreground color also works away from the eight rays.\n    for bgval, label in [(0, \'zero\'), (bg, \'mode\')]:\n        mask = a != bgval\n        if mask.any():\n            idx = ndi.distance_transform_edt(~mask, return_distances=False, return_indices=True)\n            copied = a[tuple(idx)]\n        else: copied = a\n        add(\'closest_color_euclidean_\' + label, copied, \'color\')\n\n    return np.concatenate(values, axis=1), names, tags\n\n\ndef output_sources(X, names):\n    S, sources = ORIGINAL_SOURCES(X, names)\n    ix = [i for i, name in enumerate(names)\n          if name.startswith((\'role_color_\', \'closest_color_\'))]\n    if ix:\n        S = np.concatenate((S, X[:, ix].astype(np.int16)), axis=1)\n        sources += [names[i] for i in ix]\n    return S, sources\n\n\nbase.features = features\nbase.output_sources = output_sources\nclean_task = base.clean_task\nsolve = base.solve\nreplay = base.replay\ntransform = base.transform\n', 'scripts/color_portfolio_entry.py': '"""Native/panel/copy portfolio followed by isolated color-relation proposals.\nSPDX-License-Identifier: MIT-0\n"""\nfrom concurrent.futures import ProcessPoolExecutor\nimport resource\nimport time\nfrom solver_color_relations import clean_task, solve\nfrom copy_portfolio_entry import predict as copy_predict\n\n\ndef merge(result, baseline, ntrain):\n    candidates = result[\'candidates\']\n    use = bool(candidates and candidates[0][\'loo_exact\'] == ntrain)\n    if not use:\n        return baseline\n    output = []\n    for i, previous in enumerate(baseline):\n        grids = [candidates[0][\'predictions\'][i]]\n        for key in (\'attempt_1\', \'attempt_2\'):\n            if previous[key] not in grids:\n                grids.append(previous[key])\n        while len(grids) < 2:\n            grids.append(previous[\'attempt_1\'])\n        output.append(dict(zip((\'attempt_1\', \'attempt_2\'), grids[:2])))\n    return output\n\n\ndef color_one(item):\n    name, task, baseline = item\n    blind = clean_task(task)\n    result = solve(blind, \'deferred\')\n    return name, merge(result, baseline, len(blind[\'train\'])), result\n\n\ndef predict(challenges, native_root, profile=\'color-conservative\', timeout=1500, workers=2):\n    if profile != \'color-conservative\':\n        raise ValueError(profile)\n    blind = {name: clean_task(task) for name, task in sorted(challenges.items())}\n    started = time.perf_counter()\n    cpu = time.process_time()\n    children = resource.getrusage(resource.RUSAGE_CHILDREN)\n    baseline, baseline_receipt = copy_predict(blind, native_root, \'copy-conservative\', timeout, workers)\n    output, rows = {}, []\n    jobs = [(name, task, baseline[name]) for name, task in blind.items()]\n    with ProcessPoolExecutor(max_workers=workers) as pool:\n        for name, attempts, result in pool.map(color_one, jobs, chunksize=1):\n            output[name] = attempts\n            rows.append({\'task_id\': name, \'relational\': result, \'attempts\': attempts})\n    usage = resource.getrusage(resource.RUSAGE_CHILDREN)\n    return output, {\'complete\': True, \'profile\': profile, \'tasks\': len(blind),\n                    \'queries\': sum(len(t[\'test\']) for t in blind.values()),\n                    \'baseline_receipt\': baseline_receipt, \'rows\': rows,\n                    \'wall_seconds\': time.perf_counter() - started,\n                    \'cpu_seconds\': time.process_time() - cpu + usage.ru_utime + usage.ru_stime\n                                   - children.ru_utime - children.ru_stime,\n                    \'policy\': \'Uniform perfect leave-one-demonstration insertion after native/panel/copy. No query labels or task metadata.\'}\n'}
for name,source in sources.items():
    target=bundle/name
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists() or target.read_text()!=source:
        target.write_text(source)
(bundle/'data').mkdir(exist_ok=True)
(bundle/'native'/'obj').mkdir(exist_ok=True)
(bundle/'native'/'output').mkdir(exist_ok=True)
subprocess.run(['make','-j','2'],cwd=bundle/'native',check=True)
subprocess.run(['g++','-O3','-std=c++17','-shared','-fPIC',str(bundle/'scripts'/'components.cpp'),'-o',str(bundle/'data'/'components.so')],check=True)
sys.path.insert(0,str((bundle/'scripts').resolve()))
from native_components import install
install()
from color_portfolio_entry import predict as color_predict
from worker_portfolio_entry import predict as diversity_predict

learner_sources={'learner/singleton_residual.py': '"""Skip validation only when each residual pool has exactly one grid. MIT-0."""\nimport time\nfrom selector import certify as original,learned_solve,candidates\nfrom residual import features,predict_from_features\ndef certify(task,baseline,model):\n    start=time.process_time()\n    blind=dict(train=[dict(input=p[\'input\'],output=p[\'output\']) for p in task[\'train\']],\n               test=[dict(input=q[\'input\']) for q in task[\'test\']])\n    full=learned_solve(blind)\n    empty=dict(full=full,folds=[])\n    entries=features(blind,empty,baseline)\n    n=len(blind[\'train\']);count=n if n>=2 and candidates(full) else 0\n    singleton=all(len(es)==1 for es in entries)\n    if singleton:\n        row=dict(full=full,folds=[],fold_cpu_seconds=0.0,fold_primitive_calls=0)\n        answer=[dict(attempt_1=b[\'attempt_1\'],attempt_2=es[0][\'grid\']) for es,b in zip(entries,baseline)]\n    else:\n        row=original(blind,full=full)\n        entries=features(blind,row,baseline)\n        answer=predict_from_features(baseline,row,entries,model)\n    row.update(entries=entries,selected_attempts=answer,singleton_pool=singleton,\n               original_fold_count=count,completed_folds=len(row[\'folds\']),\n               skipped_folds=count-len(row[\'folds\']),certificate_cpu_seconds=time.process_time()-start)\n    return row\n', 'learner/residual_runner.py': '"""Standalone learned search and residual selector, native input is explicit. MIT-0."""\nimport argparse,gzip,json,time\nfrom pathlib import Path\nfrom concurrent.futures import ProcessPoolExecutor\nimport numpy as np\nfrom sklearn.tree import DecisionTreeClassifier\nfrom selector import certify\nfrom residual import predict\nfrom inference_guard import blind_view,inference_access\ndef one(item):\n    name,task,baseline,model=item\n    DecisionTreeClassifier(max_depth=1,random_state=0).fit(np.array([[0.],[1.]]),[0,1])\n    start=time.process_time()\n    with inference_access():\n        blind=blind_view(task);row=certify(blind);answer=predict(blind,row,baseline,model)\n    row[\'selector_cpu_seconds\']=time.process_time()-start\n    row[\'task_id\']=name;row[\'selected_attempts\']=answer\n    return name,answer,row\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--challenges\',required=True);p.add_argument(\'--baseline\',required=True);p.add_argument(\'--gate\',required=True);p.add_argument(\'--out\',required=True);p.add_argument(\'--workers\',type=int,default=2);a=p.parse_args()\n    tasks=json.loads(Path(a.challenges).read_text());base=json.loads(Path(a.baseline).read_text());model=json.loads(Path(a.gate).read_text());assert set(tasks)==set(base)\n    items=[(n,{\'train\':[dict(input=p[\'input\'],output=p[\'output\']) for p in t[\'train\']],\'test\':[{\'input\':q[\'input\']} for q in t[\'test\']]},base[n],model) for n,t in sorted(tasks.items())]\n    out=Path(a.out);out.mkdir(exist_ok=False);submission={};cpu=fold_cpu=primitive=0;start=time.perf_counter()\n    with ProcessPoolExecutor(max_workers=a.workers) as pool,gzip.open(out/\'certificates.jsonl.gz\',\'wt\') as f:\n        for n,ans,row in pool.map(one,items,chunksize=1):\n            submission[n]=ans;cpu+=row[\'selector_cpu_seconds\'];fold_cpu+=row[\'fold_cpu_seconds\'];primitive+=row[\'fold_primitive_calls\']+row[\'full\'][\'work_counts\'][\'primitive_grid_evaluations\']\n            f.write(json.dumps(row,separators=(\',\',\':\'))+\'\\n\');f.flush()\n            if len(submission)%20==0:print(\'residual selector\',len(submission),len(tasks),flush=True)\n    (out/\'submission.json\').write_text(json.dumps(submission,separators=(\',\',\':\'))+\'\\n\')\n    receipt=dict(complete=True,tasks=len(tasks),stage_cpu_seconds=cpu,fold_cpu_seconds=fold_cpu,primitive_calls=primitive,wall_seconds=time.perf_counter()-start,model_name=model[\'name\'],workers=a.workers,baseline_recomputed=False)\n    (out/\'receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\');print(json.dumps(receipt),flush=True)\nif __name__==\'__main__\':main()\n', 'learner/component_features.py': '"""Exact component feature lookup with per-property demand. MIT-0.\n\nComponent labels are grid-local. Tables include a zero-valued background row;\nno label numbering is exposed to the grammar. Rectangular component shortcuts\nare identities for the reference\'s fill, erosion and symmetry definitions.\n"""\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom fast_features import FeatureGrid as BaseFeatureGrid, NAMES, TAGS\n\n\nclass ComponentTable:\n    def __init__(self, a, conn, bg):\n        self.a = a\n        self.h, self.w = a.shape\n        labels = np.zeros(a.shape, dtype=np.int32)\n        n = 0\n        structure = ndi.generate_binary_structure(2, conn)\n        for color in (range(10) if bg is None else (None,)):\n            part, count = ndi.label(a == color if bg is None else a != bg, structure)\n            selected = part != 0\n            labels[selected] = part[selected] + n\n            n += count\n        self.labels, self.n = labels, n\n        flat = labels.ravel()\n        self.area = np.bincount(flat, minlength=n + 1)\n        y, x = np.indices(a.shape)\n        self.y, self.x = y, x\n        self.y0, self.x0 = np.full(n+1, self.h), np.full(n+1, self.w)\n        self.y1, self.x1 = np.zeros(n+1, dtype=int), np.zeros(n+1, dtype=int)\n        np.minimum.at(self.y0, flat, y.ravel())\n        np.minimum.at(self.x0, flat, x.ravel())\n        np.maximum.at(self.y1, flat, y.ravel())\n        np.maximum.at(self.x1, flat, x.ravel())\n        self.height, self.width = self.y1-self.y0+1, self.x1-self.x0+1\n        self.full = self.area == self.height*self.width\n        self.cache = {}\n\n    def __getitem__(self, name):\n        if name not in self.cache:\n            values = self.property(name)\n            # The reference initializes all background feature cells to zero.\n            if np.ndim(values) == 2:\n                result = np.where(self.labels, values, 0)\n            else:\n                values = np.array(values, dtype=np.int16, copy=True)\n                values[0] = 0\n                result = values[self.labels]\n            self.cache[name] = result\n        return self.cache[name]\n\n    def property(self, name):\n        n = self.n\n        if name == \'area\': return self.area\n        if name == \'height\': return self.height\n        if name == \'width\': return self.width\n        if name == \'top\': return self.y0\n        if name == \'left\': return self.x0\n        if name == \'bottom\': return self.h-1-self.y1\n        if name == \'right\': return self.w-1-self.x1\n        if name == \'row\': return self.y-self.y0[self.labels]\n        if name == \'col\': return self.x-self.x0[self.labels]\n        if name == \'edge\': return (self.y0 == 0) | (self.x0 == 0) | (self.y1 == self.h-1) | (self.x1 == self.w-1)\n        if name == \'fill\': return self.full\n        if name == \'rankarea\': return np.searchsorted(np.sort(self.area[1:]), self.area)\n        if name == \'nobjects\': return np.full(n+1, n)\n        if name == \'rankrow\': return np.searchsorted(np.unique(self.y0[1:]), self.y0)\n        if name == \'rankcol\': return np.searchsorted(np.unique(self.x0[1:]), self.x0)\n        if name in (\'colors\', \'color\'):\n            if \'color_counts\' not in self.__dict__:\n                joint = self.labels.ravel()*10 + self.a.ravel()\n                self.color_counts = np.bincount(joint, minlength=(n+1)*10).reshape(n+1, 10)\n            return (self.color_counts > 0).sum(1) if name == \'colors\' else self.color_counts.argmax(1)\n        values = np.zeros(n+1, dtype=np.int16)\n        if name == \'perimeter\':\n            values[1:] = self.area[1:] - np.maximum(self.height[1:]-2, 0)*np.maximum(self.width[1:]-2, 0)\n        elif name in (\'sym_ud\', \'sym_lr\'):\n            values[1:] = 1\n        elif name != \'holes\':\n            raise KeyError(name)\n        for label in np.flatnonzero(~self.full[1:])+1:\n            sl = (slice(self.y0[label], self.y1[label]+1), slice(self.x0[label], self.x1[label]+1))\n            mask = self.labels[sl] == label\n            if name == \'holes\': value = (ndi.binary_fill_holes(mask) & ~mask).sum()\n            elif name == \'perimeter\': value = (mask & ~ndi.binary_erosion(mask)).sum()\n            elif name == \'sym_ud\': value = np.array_equal(mask, mask[::-1])\n            else: value = np.array_equal(mask, mask[:, ::-1])\n            values[label] = value\n        return values\n\n\nclass FeatureGrid(BaseFeatureGrid):\n    def component_maps(self, conn, bg):\n        return ComponentTable(self.a, conn, bg)\n\n\ndef features(a):\n    grid = FeatureGrid(a)\n    return np.stack([grid.column(name) for name in NAMES], axis=1).astype(np.float32), list(NAMES), list(TAGS)\n', 'learner/adaptive.py': '"""Monotone attempt extension using bounded, exact deeper searches. MIT-0."""\nfrom collections import Counter\nimport hashlib\nimport json\nimport time\nfrom grid_cache import GridEvaluator\nfrom cached_grammar import CachedGrammar\nfrom packed_2d import PackedGrammar\nfrom staged_solver import search as staged\nfrom solver_deferred import search as deferred\nfrom entry import attempts\n\nEXTRA_BUDGET = 50000\nMAX_DEPTH = 4\n\nclass BudgetExpired(Exception):\n    pass\n\nclass BudgetEvaluator(GridEvaluator):\n    def __init__(self, operations, apply_fn):\n        super().__init__(operations, apply_fn, True, 200000)\n        self.ceiling = None\n\n    def __call__(self, op, values):\n        result = []\n        index = self.operation_ids[id(op)]\n        for packed in values:\n            # Check before evaluation, including undefined outcomes. Cached\n            # lookups remain free but cannot add another primitive execution.\n            if (self.ceiling is not None and (index, packed) not in self.cache\n                and self.counts[\'primitive_grid_evaluations\'] >= self.ceiling):\n                raise BudgetExpired()\n            output = super().__call__(op, (packed,))\n            if output is None:\n                return None\n            result.append(output[0])\n        return tuple(result)\n\n\ndef fill_slots(prior, candidates):\n    """Keep all existing grids; canonical candidates fill duplicate slots only."""\n    result = []\n    witnesses = []\n    for qi, answer in enumerate(prior):\n        first, second = answer[\'attempt_1\'], answer[\'attempt_2\']\n        witness = None\n        if first == second:\n            for candidate in candidates:\n                proposed = candidate[\'predictions\'][qi]\n                if proposed != first:\n                    second = proposed\n                    witness = candidate[\'program\']\n                    break\n        result.append({\'attempt_1\': first, \'attempt_2\': second})\n        witnesses.append(witness)\n    return result, witnesses\n\n\ndef has_vacancy(answers):\n    return any(a[\'attempt_1\'] == a[\'attempt_2\'] for a in answers)\n\n\ndef solve(task, engine=\'staged\', grammar=\'objects\', depth=2,\n          extra_budget=EXTRA_BUDGET, max_depth=MAX_DEPTH):\n    cpu, wall = time.process_time(), time.perf_counter()\n    blind = {\'train\': [{\'input\': p[\'input\'], \'output\': p[\'output\']} for p in task[\'train\']],\n             \'test\': [{\'input\': p[\'input\']} for p in task[\'test\']]}\n    factory = (CachedGrammar if grammar == \'objects\' else PackedGrammar)(blind[\'train\'])\n    factory.clear_cache()\n    constructor_cpu = time.process_time() - cpu\n    evaluator = BudgetEvaluator(factory.operations, factory.apply)\n    search = {\'staged\': staged, \'deferred\': deferred}[engine]\n    base = search(blind, depth, factory.operations, evaluator, True)\n    base_answers = attempts(blind, base[\'candidates\'])\n    baseline = dict(candidates=base[\'candidates\'], attempts=base_answers,\n        metrics=base[\'metrics\'], stages=base.get(\'stages\'), work_counts=evaluator.metrics(),\n        constructor_metrics=getattr(factory,\'constructor_metrics\',{}),\n        fit_counts=getattr(factory,\'fit_counts\',{}),\n        grammar_sha256=hashlib.sha256(json.dumps([op.name for op in factory.operations],separators=(\',\',\':\')).encode()).hexdigest(),\n        constructor_cpu_seconds=constructor_cpu, cpu_seconds=time.process_time()-cpu)\n    start_count = evaluator.counts[\'primitive_grid_evaluations\']\n    evaluator.ceiling = start_count + extra_budget\n    selected = base_answers\n    levels = []\n    for next_depth in range(depth+1, max_depth+1):\n        if not extra_budget or not has_vacancy(selected):\n            break\n        level_cpu = time.process_time()\n        before = dict(evaluator.counts)\n        try:\n            result = search(blind, next_depth, factory.operations, evaluator, True)\n        except BudgetExpired:\n            levels.append(dict(depth=next_depth, complete=False, budget_expired=True,\n                cpu_seconds=time.process_time()-level_cpu,\n                work_counts=dict(evaluator.counts-Counter(before)), attempts=selected))\n            break\n        selected, witnesses = fill_slots(selected, result[\'candidates\'])\n        levels.append(dict(depth=next_depth, complete=True, budget_expired=False,\n            cpu_seconds=time.process_time()-level_cpu,\n            work_counts=dict(evaluator.counts-Counter(before)), attempts=selected,\n            witnesses=witnesses, candidates=result[\'candidates\'], metrics=result[\'metrics\'],\n            stages=result.get(\'stages\')))\n    calls = evaluator.counts[\'primitive_grid_evaluations\']-start_count\n    assert calls <= extra_budget\n    return dict(attempts=selected, baseline=baseline, levels=levels,\n        fallback_requested=bool(levels), budget_expired=any(x[\'budget_expired\'] for x in levels),\n        extra_primitive_calls=calls, extra_budget=extra_budget, max_depth=max_depth,\n        work_counts=evaluator.metrics(), cpu_seconds=time.process_time()-cpu,\n        wall_seconds=time.perf_counter()-wall, engine=engine, grammar=grammar)\n\n\ndef run(challenges, engine=\'staged\', grammar=\'objects\', depth=2):\n    import gzip\n    from pathlib import Path\n    submission, totals = {}, Counter()\n    with gzip.open(\'tasks.jsonl.gz\', \'wt\') as stream:\n        for task_id, task in challenges.items():\n            row = solve(task, engine, grammar, depth)\n            submission[task_id] = row[\'attempts\']\n            row[\'task_id\'] = task_id\n            stream.write(json.dumps(row,separators=(\',\',\':\'))+\'\\n\'); stream.flush()\n            totals[\'tasks\'] += 1\n            totals[\'primitive_calls\'] += row[\'work_counts\'][\'primitive_grid_evaluations\']\n            totals[\'extra_primitive_calls\'] += row[\'extra_primitive_calls\']\n            totals[\'cpu_seconds\'] += row[\'cpu_seconds\']\n    Path(\'submission.json\').write_text(json.dumps(submission)+\'\\n\')\n    receipt=dict(complete=True,totals=dict(totals),engine=engine,grammar=grammar,\n                 depth=depth,extra_budget=EXTRA_BUDGET,max_depth=MAX_DEPTH,verified=False)\n    Path(\'receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\')\n    return receipt\n', 'learner/inference_guard.py': '"""Python-level scientific inference controls; not an OS sandbox. MIT-0."""\nfrom collections.abc import Mapping\nfrom contextlib import contextmanager\nimport builtins\nimport io\nimport os\nimport socket\nimport sys\n\n\nclass Guard(Mapping):\n    """Keep only allowed payloads so mapping copies cannot reveal labels."""\n    def __init__(self, value, allowed):\n        self._allowed = frozenset(allowed)\n        self._data = {k: value[k] for k in self._allowed if k in value}\n\n    def __getitem__(self, key):\n        if key not in self._allowed:\n            raise AssertionError(\'Forbidden inference field: \' + str(key))\n        return self._data[key]\n\n    def __iter__(self):\n        return iter(self._data)\n\n    def __len__(self):\n        return len(self._data)\n\n\n_ACTIVE = False\n\n\ndef _audit_event(event, args):\n    if _ACTIVE and (event == \'open\' or\n                   event.startswith((\'socket.\', \'subprocess.\', \'ctypes.dlopen\', \'ctypes.dlsym\')) or\n                   event in {\'os.system\', \'os.posix_spawn\', \'os.exec\', \'os.fork\', \'os.forkpty\',\n                             \'os.startfile\', \'os.listdir\', \'os.scandir\'}):\n        raise AssertionError(\'Forbidden inference audit event: \' + event)\n\n\nsys.addaudithook(_audit_event)\n\n\n@contextmanager\ndef inference_access():\n    global _ACTIVE\n    assert not _ACTIVE, \'Nested inference guard\'\n    targets = [(builtins, \'open\'), (io, \'open\'), (os, \'open\'), (socket, \'socket\'),\n               (socket, \'getaddrinfo\'), (socket, \'create_connection\')]\n    originals = [(module, name, getattr(module, name)) for module, name in targets]\n    def denied(*args, **kwargs):\n        raise AssertionError(\'Forbidden inference file/network access\')\n    try:\n        _ACTIVE = True\n        for module, name in targets:\n            setattr(module, name, denied)\n        yield\n    finally:\n        _ACTIVE = False\n        for module, name, original in originals:\n            setattr(module, name, original)\n\n\ndef blind_view(task):\n    return Guard({\'train\': [Guard(p, {\'input\', \'output\'}) for p in task[\'train\']],\n                  \'test\': [Guard(p, {\'input\'}) for p in task[\'test\']]}, {\'train\', \'test\'})\n', 'learner/solver.py': '"""Frozen, pure unary grid DSL and query-aware breadth-first synthesis.\n\nSPDX-License-Identifier: MIT-0\nInference consumes demonstration outputs and query inputs, never query labels.\nAll grid states must have dimensions 1..30. Undefined operations are absorbing.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom functools import lru_cache\nfrom typing import NamedTuple\nimport time\nimport numpy as np\n\n\ndef pack(a):\n    return bytes(a.shape) + a.astype(np.uint8, copy=False).tobytes()\n\n\ndef unpack(s):\n    return np.frombuffer(s[2:], dtype=np.uint8).reshape(s[0], s[1])\n\n\ndef background(a):\n    return int(np.bincount(a.ravel(), minlength=10).argmax())\n\n\ndef bbox(a, mask):\n    yy, xx = np.nonzero(mask)\n    if not len(yy):\n        return None\n    return a[yy.min():yy.max()+1, xx.min():xx.max()+1]\n\n\n@lru_cache(maxsize=128)\ndef components(s, bg, same_color):\n    a = unpack(s)\n    remaining = set(zip(*np.nonzero(a != bg)))\n    out = []\n    while remaining:\n        start = min(remaining)\n        remaining.remove(start)\n        stack, cells = [start], [start]\n        while stack:\n            y, x = stack.pop()\n            for q in ((y-1,x),(y+1,x),(y,x-1),(y,x+1)):\n                if q in remaining and (not same_color or a[q] == a[start]):\n                    remaining.remove(q)\n                    stack.append(q)\n                    cells.append(q)\n        out.append(tuple(sorted(cells)))\n    return tuple(sorted(out, key=lambda c: (-len(c), c)))\n\n\ndef select_object(a, bg, same, smallest, crop):\n    cs = components(pack(a), bg, same)\n    if not cs:\n        return None\n    c = min(cs, key=lambda c: (len(c), c)) if smallest else cs[0]\n    out = np.full_like(a, bg)\n    for pos in c:\n        out[pos] = a[pos]\n    if crop:\n        yy, xx = zip(*c)\n        out = out[min(yy):max(yy)+1, min(xx):max(xx)+1]\n    return out\n\n\ndef reduce_blocks(a, k):\n    h, w = a.shape\n    if h % k or w % k:\n        return None\n    blocks = a.reshape(h//k, k, w//k, k)\n    out = blocks[:, 0, :, 0]\n    if not np.all(blocks == out[:, None, :, None]):\n        return None\n    return out\n\n\ndef compress(a, axis):\n    if axis == 0:\n        keep = np.r_[True, np.any(a[1:] != a[:-1], axis=1)]\n        return a[keep]\n    return compress(a.T, 0).T\n\n\ndef remove_lines(a):\n    rows = ~np.all(a == a[:, :1], axis=1)\n    cols = ~np.all(a == a[:1, :], axis=0)\n    return a[rows][:, cols]\n\n\ndef half(a, axis, side, gap):\n    n = a.shape[axis]\n    if (n - gap) % 2:\n        return None\n    k = (n-gap)//2\n    if axis == 0:\n        return a[:k] if side == 0 else a[k+gap:]\n    return a[:, :k] if side == 0 else a[:, k+gap:]\n\n\nclass Operation(NamedTuple):\n    name: str\n    fn: object\n\n\ndef operations(train):\n    """Fixed task grammar. Parameters depend only on demonstration inputs/labels."""\n    ops = []\n    def add(name, fn):\n        ops.append(Operation(name, fn))\n    for k in (1,2,3):\n        add(f\'rotate{k*90}\', lambda a,k=k: np.rot90(a,k))\n    add(\'flip_lr\', np.fliplr)\n    add(\'flip_ud\', np.flipud)\n    add(\'transpose\', lambda a:a.T)\n    add(\'anti_transpose\', lambda a:np.rot90(a.T,2))\n    add(\'crop_nonzero\', lambda a:bbox(a,a!=0))\n    add(\'crop_nonbackground\', lambda a:bbox(a,a!=background(a)))\n    add(\'compress_rows\', lambda a:compress(a,0))\n    add(\'compress_columns\', lambda a:compress(a,1))\n    add(\'remove_uniform_lines\', remove_lines)\n    for k in (2,3):\n        add(f\'upscale{k}\',lambda a,k=k:np.repeat(np.repeat(a,k,0),k,1))\n        add(f\'downscale{k}\',lambda a,k=k:reduce_blocks(a,k))\n        add(f\'tile{k}x{k}\',lambda a,k=k:np.tile(a,(k,k)))\n    for axis in (0,1):\n        add(f\'repeat_axis{axis}\',lambda a,axis=axis:np.concatenate((a,a),axis=axis))\n        add(f\'mirror_axis{axis}\',lambda a,axis=axis:np.concatenate((a,np.flip(a,axis)),axis=axis))\n        for side in (0,1):\n            for gap in (0,1):\n                add(f\'half{axis}_{side}_gap{gap}\',lambda a,axis=axis,side=side,gap=gap:half(a,axis,side,gap))\n    colors = sorted({c for t in train for row in t[\'input\'] for c in row})\n    for c in colors:\n        add(f\'crop_color{c}\',lambda a,c=c:bbox(a,a==c))\n        add(f\'keep_color{c}\',lambda a,c=c:np.where(a==c,a,0))\n        add(f\'erase_color{c}\',lambda a,c=c:np.where(a==c,0,a))\n    for bgmode in (\'zero\',\'mode\'):\n        for same in (False,True):\n            for small in (False,True):\n                for crop in (False,True):\n                    add(f\'object_{bgmode}_same{int(same)}_small{int(small)}_crop{int(crop)}\',\n                        lambda a,bgmode=bgmode,same=same,small=small,crop=crop:\n                        select_object(a,0 if bgmode==\'zero\' else background(a),same,small,crop))\n    # Infer a single consistent color map from demonstrations of equal shape.\n    mapping, consistent = {}, True\n    for pair in train:\n        a,b = np.array(pair[\'input\']),np.array(pair[\'output\'])\n        if a.shape != b.shape:\n            consistent = False\n            break\n        for x,y in zip(a.ravel(),b.ravel()):\n            x,y = int(x),int(y)\n            if x in mapping and mapping[x] != y:\n                consistent = False\n                break\n            mapping[x] = y\n    if consistent:\n        lut = np.array([mapping.get(c,c) for c in range(10)],dtype=np.uint8)\n        add(\'demonstration_color_map\',lambda a,lut=lut:lut[a])\n    return ops\n\n\ndef apply(op, values):\n    out = []\n    for s in values:\n        try:\n            a = op.fn(unpack(s))\n        except (ValueError,IndexError):\n            return None\n        if a is None or a.ndim != 2 or min(a.shape)<1 or max(a.shape)>30:\n            return None\n        out.append(pack(a))\n    return tuple(out)\n\n\ndef search(task, mode=\'query\', depth=2, max_states=0, ops_override=None):\n    """Collect goals *before* merging, preserving final-step alternatives in both arms.\n\n    max_states=0 is exhaustive quotient search. A positive cap is approximate.\n    Grammar state is only remaining depth, and BFS stores minimum-depth states.\n    """\n    started = time.perf_counter()\n    train, queries = task[\'train\'], task[\'test\']\n    n = len(train)\n    initial = tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in train+queries)\n    goal = tuple(pack(np.array(p[\'output\'],dtype=np.uint8)) for p in train)\n    ops = operations(train) if ops_override is None else ops_override\n    signature = (lambda v:v[:n]) if mode==\'train\' else (lambda v:v)\n    seen = {signature(initial): (initial, ())}\n    frontier = [(initial,())]\n    solutions, witnesses = {}, []\n    metrics = Counter()\n    def collect(v,path):\n        if v[:n] == goal:\n            predictions = v[n:]\n            if predictions not in solutions:\n                solutions[predictions] = path\n    collect(initial,())\n    layers=[]\n    for d in range(1,depth+1):\n        next_frontier=[]\n        for values,path in frontier:\n            for i,op in enumerate(ops):\n                metrics[\'expansions\'] += 1\n                v = apply(op,values)\n                if v is None:\n                    metrics[\'undefined\'] += 1\n                    continue\n                p = path+(i,)\n                collect(v,p)\n                sig = signature(v)\n                if sig in seen:\n                    metrics[\'merged\'] += 1\n                    old,oldpath=seen[sig]\n                    if old[n:] != v[n:]:\n                        metrics[\'query_divergent_merges\'] += 1\n                        if len(witnesses)<3:\n                            witnesses.append({\'kept\':[ops[j].name for j in oldpath],\n                                              \'discarded\':[ops[j].name for j in p],\n                                              \'at_depth\':d})\n                    continue\n                # Final-depth states need not be stored or deduplicated: all goals\n                # have already been collected. Avoid quadratic memory for evidence.\n                if d < depth:\n                    if max_states and len(next_frontier)>=max_states:\n                        metrics[\'capped\'] += 1\n                        continue\n                    seen[sig]=(v,p)\n                    next_frontier.append((v,p))\n        layers.append({\'depth\':d,\'frontier\':len(frontier),\'next\':len(next_frontier)})\n        frontier=next_frontier\n    # Deterministic shortest-program, then grammar-order tie break. No labels used.\n    candidates=sorted(solutions.items(),key=lambda kv:(len(kv[1]),kv[1],kv[0]))\n    out=[]\n    for pred,path in candidates:\n        out.append({\'predictions\':[unpack(s).tolist() for s in pred],\n                    \'program\':[ops[i].name for i in path]})\n    return {\'candidates\':out,\'metrics\':dict(metrics),\'layers\':layers,\n            \'merge_witnesses\':witnesses,\'num_operations\':len(ops),\n            \'seconds\':time.perf_counter()-started,\'mode\':mode,\'depth\':depth,\n            \'max_states\':max_states}\n\n\ndef solve(task, mode=\'query\', depth=2, max_states=0):\n    """Kaggle-style two attempts per query, deduplicated independently per input."""\n    result=search(task,mode,depth,max_states)\n    answer=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for candidate in result[\'candidates\']:\n            grid=candidate[\'predictions\'][i]\n            if grid not in grids:\n                grids.append(grid)\n            if len(grids)==2:\n                break\n        while len(grids)<2:\n            grids.append(q[\'input\'])\n        answer.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return answer,result\n', 'learner/screened_grammar.py': '"""Demonstration-screened unary constructors, fixed before search. MIT-0.\n\nThe unfiltered object catalog is deliberately not the searched grammar. A new\noperation is admitted iff its direct application to every original input is\ndefined and fits all output grids, either literally or under one consistent\nterminal color map. No query inputs or outputs enter this screening. There is\nno semantic deduplication: distinct admitted operations remain distinct.\n"""\nimport numpy as np\nfrom solver import pack\nfrom solver_extended import fit_map\nfrom object_grammar import ObjectGrammar\n\n\nclass ScreenedGrammar:\n    def __init__(self,train):\n        self.catalog=ObjectGrammar(train)\n        self.catalog.clear_cache()\n        self.operations=list(self.catalog.base.operations)\n        values=tuple(pack(np.asarray(p[\'input\'],dtype=np.uint8)) for p in train)\n        goals=tuple(pack(np.asarray(p[\'output\'],dtype=np.uint8)) for p in train)\n        self.screening_grid_evaluations=0\n        self.proposed_operations=len(self.catalog.operations)-len(self.operations)\n        for op in self.catalog.operations[len(self.operations):]:\n            predictions=[]\n            for value in values:\n                self.screening_grid_evaluations+=1\n                result=self.catalog.apply(op,(value,))\n                if result is None:break\n                predictions.append(result[0])\n            if len(predictions)!=len(values):continue\n            predictions=tuple(predictions)\n            if predictions==goals or fit_map(predictions,goals) is not None:\n                self.operations.append(op)\n        self.admitted_operations=len(self.operations)-len(self.catalog.base.operations)\n\n    def clear_cache(self):self.catalog.clear_cache()\n    def apply(self,op,values):return self.catalog.apply(op,values)\n', 'learner/relational_grammar.py': '"""Compile demonstration-fitted relational trees into pure unary operations.\nMIT-0. The prior relational learner is used unchanged and sees no query inputs.\nAll demonstration-fitting trees are retained in fixed demonstration-LOO order.\n"""\nfrom collections import OrderedDict\nimport json\nimport numpy as np\nfrom solver import Operation,pack,unpack\nfrom screened_grammar import ScreenedGrammar\nimport relational\n\n\nclass RelationalGrammar:\n    def __init__(self,train,features_fn=None,solve_fn=None):\n        self.screened=ScreenedGrammar(train)\n        self.operations=list(self.screened.operations)\n        self.screening_grid_evaluations=self.screened.screening_grid_evaluations\n        self.proposed_operations=self.screened.proposed_operations\n        self.admitted_operations=self.screened.admitted_operations\n        # With an empty query list the unmodified learner constructs and ranks\n        # programs using only demonstration fit and fold-local LOO predictions.\n        learned=(relational.solve if solve_fn is None else solve_fn)({\'train\':train,\'test\':[]},features_fn=features_fn)\n        self.constructor_metrics=learned[\'metrics\']\n        self.fit_counts=learned.get(\'fit_counts\',{})\n        self.programs={};self.feature_cache=OrderedDict();seen=set()\n        for candidate in learned[\'consistent_programs\']:\n            program=candidate[\'program\']\n            encoded=json.dumps(program,sort_keys=True,separators=(\',\',\':\'))\n            if encoded in seen:continue\n            seen.add(encoded)\n            op=Operation(\'relational_tree:\'+encoded,None)\n            self.operations.append(op);self.programs[id(op)]=program\n        self.learned_operations=len(self.programs)\n\n    def clear_cache(self):\n        self.screened.clear_cache();self.feature_cache.clear()\n\n    def primitive(self,program,s):\n        a=relational.transform(unpack(s).astype(np.int16),program[\'transform\'])\n        key=pack(a)\n        if key not in self.feature_cache:\n            X,names,tags=relational.features(a)\n            self.feature_cache[key]=(X,{name:i for i,name in enumerate(names)})\n            if len(self.feature_cache)>128:self.feature_cache.popitem(last=False)\n        X,index=self.feature_cache[key]\n        self.feature_cache.move_to_end(key)\n        out=np.empty(a.size,dtype=np.uint8)\n        def node(tree,indices):\n            if isinstance(tree,int):out[indices]=tree;return\n            name,threshold,left,right=tree\n            mask=X[indices,index[name]]<=threshold\n            if mask.any():node(left,indices[mask])\n            if (~mask).any():node(right,indices[~mask])\n        node(program[\'tree\'],np.arange(a.size))\n        return pack(out.reshape(a.shape))\n\n    def apply(self,op,values):\n        if id(op) not in self.programs:return self.screened.apply(op,values)\n        return tuple(self.primitive(self.programs[id(op)],s) for s in values)\n', 'learner/solver_extended.py': '"""Structural search with a demonstration-fitted terminal color map.\n\nSPDX-License-Identifier: MIT-0\nThis is a separately versioned grammar extension. Structural-prefix equivalence\nstill preserves query outputs; terminal map fitting is a deterministic function\nof the complete demonstration state vector. Unknown colors map to themselves.\n"""\nfrom collections import Counter\nimport time\nimport numpy as np\nfrom solver import operations,pack,unpack,apply\n\n\ndef fit_map(values,goal):\n    if any(v[:2] != g[:2] for v,g in zip(values,goal)):\n        return None\n    lut=np.arange(10,dtype=np.uint8)\n    known={}\n    for v,g in zip(values,goal):\n        a=np.frombuffer(v[2:],dtype=np.uint8)\n        b=np.frombuffer(g[2:],dtype=np.uint8)\n        pairs=np.unique(a.astype(np.uint16)*10+b)\n        for pair in pairs:\n            x,y=divmod(int(pair),10)\n            if x in known and known[x]!=y:return None\n            known[x]=y;lut[x]=y\n    return lut\n\n\ndef search(task,mode=\'query\',depth=3,max_states=0,ops_override=None):\n    start=time.perf_counter();train=task[\'train\'];n=len(train)\n    initial=tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in train+task[\'test\'])\n    goal=tuple(pack(np.array(p[\'output\'],dtype=np.uint8)) for p in train)\n    ops=operations(train) if ops_override is None else ops_override\n    signature=(lambda v:v[:n]) if mode==\'train\' else (lambda v:v)\n    seen={signature(initial):(initial,())};frontier=[(initial,())]\n    solutions={};fit_cache={};metrics=Counter();witnesses=[];layers=[]\n    def collect(v,path):\n        if v[:n]==goal:\n            predictions=v[n:];names=[ops[i].name for i in path]\n            rank=(len(path),path,0)\n        else:\n            if len(path)>=depth:return\n            key=v[:n]\n            if key not in fit_cache:fit_cache[key]=fit_map(key,goal)\n            lut=fit_cache[key]\n            if lut is None:return\n            predictions=tuple(pack(lut[unpack(s)]) for s in v[n:])\n            names=[ops[i].name for i in path]+[\'fit_color_map:\'+\',\'.join(map(str,lut.tolist()))]\n            rank=(len(path)+1,path,1)\n        if predictions not in solutions or rank<solutions[predictions][0]:\n            solutions[predictions]=(rank,names)\n    collect(initial,())\n    for d in range(1,depth+1):\n        next_frontier=[]\n        for values,path in frontier:\n            for i,op in enumerate(ops):\n                metrics[\'expansions\']+=1\n                v=apply(op,values)\n                if v is None:\n                    metrics[\'undefined\']+=1;continue\n                p=path+(i,);collect(v,p);sig=signature(v)\n                if sig in seen:\n                    metrics[\'merged\']+=1;old,oldpath=seen[sig]\n                    if old[n:]!=v[n:]:\n                        metrics[\'query_divergent_merges\']+=1\n                        if len(witnesses)<3:witnesses.append({\'kept\':[ops[j].name for j in oldpath],\'discarded\':[ops[j].name for j in p],\'at_depth\':d})\n                    continue\n                if d<depth:\n                    if max_states and len(next_frontier)>=max_states:\n                        metrics[\'capped\']+=1;continue\n                    seen[sig]=(v,p);next_frontier.append((v,p))\n        layers.append({\'depth\':d,\'frontier\':len(frontier),\'next\':len(next_frontier)})\n        frontier=next_frontier\n    out=[{\'predictions\':[unpack(s).tolist() for s in pred],\'program\':info[1]}\n         for pred,info in sorted(solutions.items(),key=lambda kv:(kv[1][0],kv[0]))]\n    return {\'candidates\':out,\'metrics\':dict(metrics),\'layers\':layers,\'merge_witnesses\':witnesses,\n            \'num_operations\':len(ops),\'seconds\':time.perf_counter()-start,\'mode\':mode,\n            \'depth\':depth,\'max_states\':max_states,\'grammar\':\'structural_with_terminal_color_map\'}\n\n\ndef solve(task,mode=\'query\',depth=3,max_states=0):\n    result=search(task,mode,depth,max_states)\n    answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for c in result[\'candidates\']:\n            grid=c[\'predictions\'][i]\n            if grid not in grids:grids.append(grid)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return answers,result\n', 'learner/lazy_residual_runner.py': '"""Standalone lazy residual runtime; explicit native input, frozen weights."""\nimport argparse,gzip,json,time\nfrom pathlib import Path\nfrom concurrent.futures import ProcessPoolExecutor\nimport numpy as np\nfrom sklearn.tree import DecisionTreeClassifier\nfrom lazy_residual import certify\nfrom inference_guard import blind_view,inference_access\ndef one(item):\n    name,task,baseline,model=item\n    DecisionTreeClassifier(max_depth=1,random_state=0).fit(np.array([[0.],[1.]]),[0,1])\n    start=time.process_time()\n    with inference_access():row=certify(blind_view(task),baseline,model)\n    row[\'selector_cpu_seconds\']=time.process_time()-start;row[\'task_id\']=name\n    return name,row[\'selected_attempts\'],row\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--challenges\',required=True);p.add_argument(\'--baseline\',required=True);p.add_argument(\'--gate\',required=True);p.add_argument(\'--out\',required=True);p.add_argument(\'--workers\',type=int,default=2);a=p.parse_args()\n    tasks=json.loads(Path(a.challenges).read_text());base=json.loads(Path(a.baseline).read_text());model=json.loads(Path(a.gate).read_text());assert set(tasks)==set(base)\n    items=[(n,dict(train=[dict(input=p[\'input\'],output=p[\'output\']) for p in t[\'train\']],test=[dict(input=q[\'input\']) for q in t[\'test\']]),base[n],model) for n,t in sorted(tasks.items())]\n    out=Path(a.out);out.mkdir(exist_ok=False);submission={};cpu=fold_cpu=primitive=skipped=0;start=time.perf_counter()\n    with ProcessPoolExecutor(max_workers=a.workers) as pool,gzip.open(out/\'certificates.jsonl.gz\',\'wt\') as f:\n        for n,ans,row in pool.map(one,items,chunksize=1):\n            submission[n]=ans;cpu+=row[\'selector_cpu_seconds\'];fold_cpu+=row[\'fold_cpu_seconds\'];primitive+=row[\'fold_primitive_calls\']+row[\'full\'][\'work_counts\'][\'primitive_grid_evaluations\'];skipped+=row[\'skipped_folds\']\n            f.write(json.dumps(row,separators=(\',\',\':\'))+\'\\n\');f.flush()\n            if len(submission)%20==0:print(\'lazy residual\',len(submission),len(tasks),flush=True)\n    (out/\'submission.json\').write_text(json.dumps(submission,separators=(\',\',\':\'))+\'\\n\')\n    receipt=dict(complete=True,tasks=len(tasks),stage_cpu_seconds=cpu,fold_cpu_seconds=fold_cpu,primitive_calls=primitive,skipped_folds=skipped,wall_seconds=time.perf_counter()-start,model_name=model[\'name\'],workers=a.workers,baseline_recomputed=False)\n    (out/\'receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\');print(json.dumps(receipt),flush=True)\nif __name__==\'__main__\':main()\n', 'learner/entry.py': '"""Portable object grammar / exact engine adapter. MIT-0."""\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nfrom grid_cache import GridEvaluator\nfrom relational_grammar import RelationalGrammar\nfrom lazy_grammar import LazyGrammar\nfrom cached_grammar import CachedGrammar\nimport constructors  # Load source during setup, before guarded inference.\nfrom packed_2d import PackedGrammar\nfrom staged_solver import search as staged\nfrom solver_deferred import search as deferred\n\n\ndef attempts(task, candidates):\n    output = []\n    for qi, query in enumerate(task[\'test\']):\n        distinct = []\n        for candidate in candidates:\n            grid = candidate[\'predictions\'][qi]\n            if grid not in distinct: distinct.append(grid)\n            if len(distinct) == 2: break\n        while len(distinct) < 2: distinct.append(query[\'input\'])\n        output.append(dict(zip((\'attempt_1\', \'attempt_2\'), distinct)))\n    return output\n\n\ndef solve(task, engine=\'staged\', grammar=\'objects\', depth=2, backend=\'cached\'):\n    cpu, wall = time.process_time(), time.perf_counter()\n    blind = {\'train\': [{\'input\': p[\'input\'], \'output\': p[\'output\']} for p in task[\'train\']],\n             \'test\': [{\'input\': p[\'input\']} for p in task[\'test\']]}\n    factory = ({\'lazy\':LazyGrammar,\'cached\':CachedGrammar,\'original\':RelationalGrammar}[backend] if grammar == \'objects\' else PackedGrammar)(blind[\'train\'])\n    factory.clear_cache()\n    constructor_cpu = time.process_time()-cpu\n    evaluator = GridEvaluator(factory.operations, factory.apply, True, 200000)\n    result = {\'staged\': staged, \'deferred\': deferred}[engine](\n        blind, depth, factory.operations, evaluator, True)\n    return dict(candidates=result[\'candidates\'], attempts=attempts(blind, result[\'candidates\']),\n                metrics=result[\'metrics\'], stages=result.get(\'stages\'),\n                work_counts=evaluator.metrics(), num_operations=len(factory.operations),\n                screening_grid_evaluations=getattr(factory,\'screening_grid_evaluations\',0),\n                proposed_operations=getattr(factory,\'proposed_operations\',0),\n                admitted_operations=getattr(factory,\'admitted_operations\',0),\n                learned_operations=getattr(factory,\'learned_operations\',0),\n                constructor_metrics=getattr(factory,\'constructor_metrics\',{}),\n                grammar_sha256=hashlib.sha256(json.dumps([op.name for op in factory.operations],separators=(\',\',\':\')).encode()).hexdigest(),\n                feature_counts=getattr(factory,\'feature_counts\',{}),\n                fit_counts=getattr(factory,\'fit_counts\',{}),\n                constructor_cpu_seconds=constructor_cpu,\n                cpu_seconds=time.process_time()-cpu, wall_seconds=time.perf_counter()-wall)\n\n\ndef run(challenges, engine=\'staged\', grammar=\'objects\', depth=2):\n    submission, totals = {}, Counter()\n    with gzip.open(\'tasks.jsonl.gz\', \'wt\') as stream:\n        for task_id, task in challenges.items():\n            row = solve(task, engine, grammar, depth)\n            submission[task_id] = row[\'attempts\']\n            row[\'task_id\'] = task_id\n            stream.write(json.dumps(row, separators=(\',\', \':\'))+\'\\n\'); stream.flush()\n            totals[\'tasks\'] += 1\n            totals[\'primitive_calls\'] += row[\'work_counts\'][\'primitive_grid_evaluations\']\n            totals[\'cpu_seconds\'] += row[\'cpu_seconds\']\n    Path(\'submission.json\').write_text(json.dumps(submission)+\'\\n\')\n    receipt = dict(complete=True, totals=dict(totals), engine=engine, grammar=grammar,\n                   depth=depth, verified=False)\n    Path(\'receipt.json\').write_text(json.dumps(receipt, indent=2)+\'\\n\')\n    return receipt\n', 'learner/relational.py': '"""Demonstration-fitted relational cell rules; no query labels or task IDs.\nSPDX-License-Identifier: MIT-0\n"""\nfrom __future__ import annotations\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport time\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.tree import DecisionTreeClassifier\n\nVERSION = \'relational-rules-v3\'\nDEPTHS = (2, 4, 6, 10)\nGROUPS = (\'local\', \'objects\', \'rays\', \'all\')\n\n\ndef clean_task(task):\n    return {\'train\': [{\'input\': p[\'input\'], \'output\': p[\'output\']} for p in task[\'train\']],\n            \'test\': [{\'input\': p[\'input\']} for p in task[\'test\']]}\n\n\ndef mode(a):\n    return int(np.bincount(a.ravel(), minlength=10).argmax())\n\n\ndef transform(a, name):\n    if name == \'identity\': return a\n    bg = 0 if name == \'crop_zero\' else mode(a)\n    y, x = np.nonzero(a != bg)\n    if len(y) == 0: return a\n    return a[y.min():y.max()+1, x.min():x.max()+1]\n\n\ndef features(a):\n    """Fixed input-only features. Each feature is an integer per grid cell."""\n    a = np.asarray(a, dtype=np.int16)\n    h, w = a.shape\n    yy, xx = np.indices((h, w))\n    cols, names, tags = [], [], []\n    bg = mode(a)\n    def add(name, v, tag):\n        if np.isscalar(v): v = np.full_like(a, v)\n        cols.append(np.array(v, dtype=np.int16, copy=True).ravel()); names.append(name); tags.append(tag)\n    add(\'color\', a, \'base\')\n    add(\'nonzero\', a != 0, \'base\'); add(\'nonbackground\', a != bg, \'base\')\n    add(\'background\', bg, \'base\')\n    # Absolute geometry is separate: simpler feature subsets can generalize away\n    # from accidental coordinate correlations.\n    for name, v in [(\'row\',yy),(\'column\',xx),(\'bottom\',h-1-yy),(\'right\',w-1-xx),\n                    (\'row_parity\',yy%2),(\'col_parity\',xx%2),\n                    (\'border\',np.minimum.reduce([yy,xx,h-1-yy,w-1-xx])),\n                    (\'diagonal\',yy-xx),(\'antidiagonal\',yy+xx),\n                    (\'h\',h),(\'w\',w)]: add(name,v,\'position\')\n    pad = np.pad(a,2,constant_values=10)\n    for dy in range(-2,3):\n        for dx in range(-2,3):\n            if dy == dx == 0: continue\n            add(f\'neighbor_{dy}_{dx}\',pad[2+dy:2+dy+h,2+dx:2+dx+w], \'local\')\n    add(\'mirror_ud\',a[::-1], \'local\'); add(\'mirror_lr\',a[:,::-1], \'local\')\n    add(\'mirror_both\',a[::-1,::-1], \'local\')\n    if h == w:\n        add(\'transpose\',a.T, \'local\'); add(\'antitranspose\',a[::-1,::-1].T,\'local\')\n    else:\n        add(\'transpose\',10,\'local\'); add(\'antitranspose\',10,\'local\')\n    for c in range(10):\n        mask = a == c\n        rows = mask.sum(1)[:,None] + np.zeros_like(a)\n        columns = mask.sum(0)[None,:] + np.zeros_like(a)\n        add(f\'color{c}_count\',mask.sum(),\'color\')\n        add(f\'color{c}_rowcount\',rows,\'rays\'); add(f\'color{c}_colcount\',columns,\'rays\')\n        y,x=np.nonzero(mask)\n        if len(y):\n            y0,y1,x0,x1=y.min(),y.max(),x.min(),x.max()\n            add(f\'color{c}_inbbox\',(yy>=y0)&(yy<=y1)&(xx>=x0)&(xx<=x1),\'color\')\n            add(f\'color{c}_bboxdy\',np.maximum(y0-yy,0)+np.maximum(yy-y1,0),\'color\')\n            add(f\'color{c}_bboxdx\',np.maximum(x0-xx,0)+np.maximum(xx-x1,0),\'color\')\n            dist = ndi.distance_transform_cdt(~mask,metric=\'taxicab\')\n            add(f\'color{c}_distance\',dist,\'color\')\n        else:\n            for suffix in (\'inbbox\',\'bboxdy\',\'bboxdx\',\'distance\'): add(f\'color{c}_{suffix}\',-1,\'color\')\n        # Counts on each side support intervals, parity, rays, and crossings.\n        for axis, label in [(0,\'vertical\'),(1,\'horizontal\')]:\n            forward=np.cumsum(mask,axis=axis)-mask\n            backward=np.flip(np.cumsum(np.flip(mask,axis=axis),axis=axis),axis=axis)-mask\n            add(f\'color{c}_{label}_before\',forward,\'rays\')\n            add(f\'color{c}_{label}_after\',backward,\'rays\')\n    # Component geometry for both monochrome regions and foreground objects.\n    for conn in (1,2):\n        structure=ndi.generate_binary_structure(2,conn)\n        maps={name:np.zeros_like(a) for name in (\n            \'area\',\'height\',\'width\',\'top\',\'left\',\'bottom\',\'right\',\'row\',\'col\',\n            \'edge\',\'holes\',\'colors\',\'color\',\'fill\',\'perimeter\',\'rankarea\',\n            \'nobjects\',\'rankrow\',\'rankcol\',\'sym_ud\',\'sym_lr\')}\n        groups=[]\n        for c in range(10):\n            labels,num=ndi.label(a==c,structure)\n            for label,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; sublabels=labels[sl]\n                mask=sublabels==label\n                groups.append((sy,sx,mask,c))\n        def component_maps(groups, prefix):\n            areas=[int(m.sum()) for _,_,m,_ in groups]\n            rows=sorted(set(sy.start for sy,_,_,_ in groups))\n            xs=sorted(set(sx.start for _,sx,_,_ in groups))\n            for sy,sx,mask,c in groups:\n                ah,aw=mask.shape; area=int(mask.sum())\n                gy,gx=np.indices(mask.shape); filled=ndi.binary_fill_holes(mask)\n                holes=int((filled&~mask).sum())\n                erosion=ndi.binary_erosion(mask)\n                properties={\'area\':area,\'height\':ah,\'width\':aw,\n                    \'top\':sy.start,\'left\':sx.start,\'bottom\':h-sy.stop,\'right\':w-sx.stop,\n                    \'row\':gy,\'col\':gx,\'edge\':int(sy.start==0 or sx.start==0 or sy.stop==h or sx.stop==w),\n                    \'holes\':holes,\'colors\':len(np.unique(a[sy,sx][mask])),\n                    \'color\':c,\'fill\':area==ah*aw,\'perimeter\':int((mask&~erosion).sum()),\n                    \'rankarea\':sum(z<area for z in areas),\'nobjects\':len(groups),\n                    \'rankrow\':rows.index(sy.start),\'rankcol\':xs.index(sx.start),\n                    \'sym_ud\':int(np.array_equal(mask,mask[::-1])),\n                    \'sym_lr\':int(np.array_equal(mask,mask[:,::-1]))}\n                for k,v in properties.items():\n                    view=maps[k][sy,sx]\n                    view[mask]=v[mask] if isinstance(v,np.ndarray) else v\n            for name,v in maps.items(): add(prefix+name,v,\'objects\')\n        component_maps(groups,f\'same{conn}_\')\n        for bgval,bgname in [(0,\'zero\'),(bg,\'mode\')]:\n            for v in maps.values(): v.fill(0)\n            labels,num=ndi.label(a!=bgval,structure)\n            groups=[]\n            for i,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None: continue\n                sy,sx=sl; mask=labels[sl]==i\n                vals=a[sl][mask]\n                c=int(np.bincount(vals,minlength=10).argmax())\n                groups.append((sy,sx,mask,c))\n            component_maps(groups,f\'fg_{bgname}{conn}_\')\n    # Nearest non-background and nearest differing cell in eight directions.\n    for dy,dx in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:\n        raycolor=np.full_like(a,10); raydistance=np.full_like(a,31)\n        diffcolor=np.full_like(a,10); diffdistance=np.full_like(a,31)\n        for y in range(h):\n            for x in range(w):\n                for dist in range(1,31):\n                    ny,nx=y+dy*dist,x+dx*dist\n                    if not (0<=ny<h and 0<=nx<w): break\n                    if raycolor[y,x]==10 and a[ny,nx]!=bg:\n                        raycolor[y,x]=a[ny,nx];raydistance[y,x]=dist\n                    if diffcolor[y,x]==10 and a[ny,nx]!=a[y,x]:\n                        diffcolor[y,x]=a[ny,nx];diffdistance[y,x]=dist\n                    if raycolor[y,x]!=10 and diffcolor[y,x]!=10: break\n        for name,v in [(\'color\',raycolor),(\'distance\',raydistance),(\'diffcolor\',diffcolor),(\'diffdistance\',diffdistance)]:\n            add(f\'ray{dy}_{dx}_{name}\',v,\'rays\')\n    return np.stack(cols,axis=1).astype(np.float32),names,tags\n\n\ndef feature_indices(tags, group):\n    allowed={\'local\':{\'base\',\'local\'},\'objects\':{\'base\',\'objects\',\'color\'},\n             \'rays\':{\'base\',\'rays\',\'color\'},\'all\':set(tags)}[group]\n    return np.array([i for i,t in enumerate(tags) if t in allowed])\n\n\ndef tree_spec(tree,names,indices):\n    t=tree.tree_\n    def node(i):\n        if t.children_left[i] < 0:\n            return int(tree.classes_[np.argmax(t.value[i][0])])\n        return [names[indices[t.feature[i]]],float(t.threshold[i]),node(t.children_left[i]),node(t.children_right[i])]\n    return node(0)\n\n\ndef solve(task, execution=\'deferred\', features_fn=None):\n    startcpu,startwall=time.process_time(),time.perf_counter()\n    task=clean_task(task)\n    stats=Counter();candidates=[];all_proposals=[]\n    cache={}\n    def feat(a):\n        key=(a.shape,a.tobytes())\n        if key not in cache:\n            stats[\'feature_grid_calls\']+=1\n            stats[\'feature_cells\']+=a.size\n            cache[key]=(features if features_fn is None else features_fn)(a)\n        return cache[key]\n    train=[(np.array(p[\'input\'],dtype=np.int16),np.array(p[\'output\'],dtype=np.int16)) for p in task[\'train\']]\n    query=[np.array(p[\'input\'],dtype=np.int16) for p in task[\'test\']]\n    for transform_name in (\'identity\',\'crop_zero\',\'crop_mode\'):\n        transformed=[transform(a,transform_name) for a,b in train]\n        if any(a.shape!=b.shape for a,(_,b) in zip(transformed,train)):continue\n        # Identical transform behavior may differ on a query: preserve all names.\n        ftrain=[feat(a) for a in transformed]\n        names,tags=ftrain[0][1:]\n        y=np.concatenate([b.ravel() for a,b in train])\n        boundaries=np.cumsum([0]+[a.size for a in transformed])\n        for group in GROUPS:\n            ix=feature_indices(tags,group)\n            X=np.concatenate([f[0][:,ix] for f in ftrain])\n            for depth in DEPTHS:\n                model=DecisionTreeClassifier(max_depth=depth,random_state=0,criterion=\'entropy\')\n                model.fit(X,y); stats[\'models_fit\']+=1\n                predtrain=model.predict(X);stats[\'model_training_cells\']+=len(y)\n                fit=bool(np.array_equal(predtrain,y))\n                preds=None\n                # Eager explicitly materializes every proposal. Deferred delays\n                # query features and predictions until the demonstration fit.\n                if execution==\'eager\' or fit:\n                    preds=[]\n                    for q in query:\n                        tq=transform(q,transform_name)\n                        fq=feat(tq)[0][:,ix]\n                        pq=model.predict(fq).astype(np.int16).reshape(tq.shape).tolist()\n                        stats[\'query_model_grid_calls\']+=1;stats[\'query_model_cells\']+=tq.size\n                        preds.append(pq)\n                proposal={\'transform\':transform_name,\'group\':group,\'depth\':depth,\'fits\':fit,\n                          \'nodes\':int(model.tree_.node_count)}\n                all_proposals.append(proposal)\n                if not fit:continue\n                loo_exact=loo_cells=loo_total=0\n                if len(train)>1:\n                    for i in range(len(train)):\n                        keep=np.ones(len(y),dtype=bool);keep[boundaries[i]:boundaries[i+1]]=False\n                        cv=DecisionTreeClassifier(max_depth=depth,random_state=0,criterion=\'entropy\')\n                        cv.fit(X[keep],y[keep]);stats[\'models_fit\']+=1\n                        got=cv.predict(X[~keep]);truth=y[~keep]\n                        loo_exact+=int(np.array_equal(got,truth));loo_cells+=int((got==truth).sum());loo_total+=len(truth)\n                rank=[-loo_exact,-(loo_cells/max(1,loo_total)),int(model.tree_.node_count),\n                      (\'identity\',\'crop_zero\',\'crop_mode\').index(transform_name),GROUPS.index(group),depth]\n                candidates.append({\'predictions\':preds,\'program\':{\'transform\':transform_name,\'tree\':tree_spec(model,names,ix)},\n                    \'rank\':rank,\'loo_exact\':loo_exact,\'loo_cells\':loo_cells,\'loo_total\':loo_total,\'proposal\':proposal})\n    candidates.sort(key=lambda c:(c[\'rank\'],json.dumps(c[\'predictions\'],separators=(\',\',\':\'))))\n    distinct=[];seen=set()\n    for c in candidates:\n        key=json.dumps(c[\'predictions\'],separators=(\',\',\':\'))\n        if key not in seen: seen.add(key);distinct.append(c)\n    answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for c in distinct:\n            g=c[\'predictions\'][i]\n            if g not in grids:grids.append(g)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return {\'attempts\':answers,\'candidates\':distinct,\'consistent_programs\':candidates,\n            \'proposals\':all_proposals,\'metrics\':dict(stats),\'cpu_seconds\':time.process_time()-startcpu,\n            \'wall_seconds\':time.perf_counter()-startwall,\'execution\':execution}\n\n\ndef replay(program, grid):\n    a=transform(np.asarray(grid,dtype=np.int16),program[\'transform\'])\n    X,names,tags=features(a);index={s:i for i,s in enumerate(names)}\n    def node(tree, selected):\n        if isinstance(tree,int):return np.full(len(selected),tree,dtype=np.int16)\n        name,threshold,left,right=tree\n        mask=X[selected,index[name]]<=threshold\n        out=np.empty(len(selected),dtype=np.int16)\n        out[mask]=node(left,selected[mask]);out[~mask]=node(right,selected[~mask])\n        return out\n    return node(program[\'tree\'],np.arange(a.size)).reshape(a.shape).tolist()\n', 'learner/anytime.py': '"""Baseline-preserving anytime depth-three adapter. MIT-0."""\nimport time\nimport hashlib\nimport json\nfrom adaptive import BudgetEvaluator, fill_slots, has_vacancy\nfrom cached_grammar import CachedGrammar\nfrom staged_solver import search as staged\nfrom entry import attempts\nfrom anytime_search import search\n\nEXTRA_BUDGET = 50000\nMAX_DEPTH = 3\n\n\ndef solve(task, extra_budget=EXTRA_BUDGET, max_depth=MAX_DEPTH):\n    cpu, wall = time.process_time(), time.perf_counter()\n    blind = {\'train\': [{\'input\':p[\'input\'], \'output\':p[\'output\']} for p in task[\'train\']],\n             \'test\': [{\'input\':p[\'input\']} for p in task[\'test\']]}\n    factory = CachedGrammar(blind[\'train\']); factory.clear_cache()\n    constructor_cpu = time.process_time()-cpu\n    evaluator = BudgetEvaluator(factory.operations, factory.apply)\n    base = staged(blind, 2, factory.operations, evaluator, True)\n    base_answers = attempts(blind, base[\'candidates\'])\n    baseline = dict(candidates=base[\'candidates\'], attempts=base_answers,\n        metrics=base[\'metrics\'], stages=base[\'stages\'], work_counts=evaluator.metrics(),\n        constructor_metrics=factory.constructor_metrics, fit_counts=factory.fit_counts,\n        grammar_sha256=hashlib.sha256(json.dumps([op.name for op in factory.operations],separators=(\',\',\':\')).encode()).hexdigest(),\n        constructor_cpu_seconds=constructor_cpu, cpu_seconds=time.process_time()-cpu)\n    before = evaluator.counts[\'primitive_grid_evaluations\']\n    evaluator.ceiling = before + extra_budget\n    extension = None\n    selected = base_answers\n    witnesses = [None]*len(base_answers)\n    if extra_budget and has_vacancy(selected):\n        extension = search(blind, max_depth, factory.operations, evaluator)\n        selected, witnesses = fill_slots(selected, extension[\'candidates\'])\n    extra = evaluator.counts[\'primitive_grid_evaluations\']-before\n    assert 0 <= extra <= extra_budget\n    return dict(baseline=baseline, extension=extension, attempts=selected, witnesses=witnesses,\n        extra_budget=extra_budget, max_depth=max_depth, extra_primitive_calls=extra,\n        work_counts=evaluator.metrics(), cpu_seconds=time.process_time()-cpu,\n        wall_seconds=time.perf_counter()-wall)\n\n\ndef run(challenges):\n    import gzip\n    from collections import Counter\n    from pathlib import Path\n    submission, totals = {}, Counter()\n    with gzip.open(\'tasks.jsonl.gz\', \'wt\') as stream:\n        for task_id, task in challenges.items():\n            row = solve(task); row[\'task_id\'] = task_id\n            submission[task_id] = row[\'attempts\']\n            stream.write(json.dumps(row,separators=(\',\',\':\'))+\'\\n\');stream.flush()\n            totals[\'tasks\'] += 1; totals[\'cpu_seconds\'] += row[\'cpu_seconds\']\n            totals[\'primitive_grid_evaluations\'] += row[\'work_counts\'][\'primitive_grid_evaluations\']\n    Path(\'submission.json\').write_text(json.dumps(submission)+\'\\n\')\n    receipt = dict(complete=True, totals=dict(totals), verified=False,\n                   extra_budget=EXTRA_BUDGET, max_depth=MAX_DEPTH)\n    Path(\'receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\')\n    return receipt\n', 'learner/packed_2d.py': '"""Exact packed-byte evaluator for the frozen solver.operations factory. MIT-0.\n\nContract: packed immutable grids, dimensions 1..30, palette 0..9. Dispatch is\nbound to the identities created by this grammar, not caller-supplied names.\n"""\nfrom functools import lru_cache\nfrom solver import operations\n\n\ndef _grid(h, w, data):\n    if not (1 <= h <= 30 and 1 <= w <= 30):\n        return None\n    return bytes((h, w)) + data\n\n\ndef _transpose(s):\n    h, w, b = s[0], s[1], s[2:]\n    return bytes((w, h)) + b\'\'.join(b[x::w] for x in range(w))\n\n\ndef _crop(s, positions):\n    if not positions:\n        return None\n    h, w, b = s[0], s[1], s[2:]\n    y0, y1 = positions[0] // w, positions[-1] // w\n    x0 = min(p % w for p in positions)\n    x1 = max(p % w for p in positions)\n    return _grid(y1-y0+1, x1-x0+1,\n                 b\'\'.join(b[y*w+x0:y*w+x1+1] for y in range(y0, y1+1)))\n\n\ndef _background(b):\n    return max(range(10), key=b.count)  # First maximum: lowest color wins ties.\n\n\n@lru_cache(maxsize=128)\ndef _components(s, bg, same):\n    h, w, b = s[0], s[1], s[2:]\n    remaining = {i for i, c in enumerate(b) if c != bg}\n    result = []\n    while remaining:\n        start = min(remaining)\n        remaining.remove(start)\n        stack, cells = [start], [start]\n        while stack:\n            p = stack.pop()\n            y, x = divmod(p, w)\n            neighbors = []\n            if y: neighbors.append(p-w)\n            if y+1 < h: neighbors.append(p+w)\n            if x: neighbors.append(p-1)\n            if x+1 < w: neighbors.append(p+1)\n            for q in neighbors:\n                if q in remaining and (not same or b[q] == b[start]):\n                    remaining.remove(q)\n                    cells.append(q)\n                    stack.append(q)\n        result.append(tuple(sorted(cells)))\n    return tuple(sorted(result, key=lambda c: (-len(c), c)))\n\n\ndef _object(s, bgmode, same, small, crop):\n    h, w, b = s[0], s[1], s[2:]\n    bg = 0 if bgmode == \'zero\' else _background(b)\n    cs = _components(s, bg, same)\n    if not cs:\n        return None\n    cells = min(cs, key=lambda c: (len(c), c)) if small else cs[0]\n    data = bytearray([bg]) * (h*w)\n    for p in cells:\n        data[p] = b[p]\n    value = bytes((h, w)) + bytes(data)\n    return _crop(value, cells) if crop else value\n\n\ndef _transform(name, s):\n    h, w, b = s[0], s[1], s[2:]\n    if name == \'rotate90\':\n        return _grid(w, h, b\'\'.join(b[x::w] for x in range(w-1, -1, -1)))\n    if name == \'rotate180\':\n        return s[:2] + b[::-1]\n    if name == \'rotate270\':\n        return _grid(w, h, b\'\'.join(b[x::w][::-1] for x in range(w)))\n    if name == \'flip_lr\':\n        return s[:2] + b\'\'.join(b[y*w:(y+1)*w][::-1] for y in range(h))\n    if name == \'flip_ud\':\n        return s[:2] + b\'\'.join(b[y*w:(y+1)*w] for y in range(h-1, -1, -1))\n    if name == \'transpose\':\n        return _transpose(s)\n    if name == \'anti_transpose\':\n        return _grid(w, h, b\'\'.join(b[x::w][::-1] for x in range(w-1, -1, -1)))\n    if name in (\'crop_nonzero\', \'crop_nonbackground\'):\n        bg = 0 if name == \'crop_nonzero\' else _background(b)\n        return _crop(s, [i for i, c in enumerate(b) if c != bg])\n    if name == \'compress_rows\':\n        rows = [b[y*w:(y+1)*w] for y in range(h)]\n        kept = [row for i, row in enumerate(rows) if not i or row != rows[i-1]]\n        return _grid(len(kept), w, b\'\'.join(kept))\n    if name == \'compress_columns\':\n        return _transpose(_transform(\'compress_rows\', _transpose(s)))\n    if name == \'remove_uniform_lines\':\n        rows = [y for y in range(h) if b[y*w:(y+1)*w] != bytes((b[y*w],))*w]\n        cols = [x for x in range(w) if b[x::w] != bytes((b[x],))*h]\n        return _grid(len(rows), len(cols), bytes(b[y*w+x] for y in rows for x in cols))\n    if name.startswith(\'upscale\'):\n        k = int(name[-1])\n        if h*k > 30 or w*k > 30:\n            return None\n        return _grid(h*k, w*k,\n                     b\'\'.join(b\'\'.join(bytes((c,))*k for c in b[y*w:(y+1)*w])*k for y in range(h)))\n    if name.startswith(\'downscale\'):\n        k = int(name[-1])\n        if h % k or w % k:\n            return None\n        rows = []\n        for y in range(0, h, k):\n            row = b[y*w:(y+1)*w:k]\n            expanded = b\'\'.join(bytes((c,))*k for c in row)\n            if any(b[(y+j)*w:(y+j+1)*w] != expanded for j in range(k)):\n                return None\n            rows.append(row)\n        return _grid(h//k, w//k, b\'\'.join(rows))\n    if name.startswith(\'tile\'):\n        k = int(name[4])\n        if h*k > 30 or w*k > 30:\n            return None\n        return _grid(h*k, w*k, b\'\'.join(b[y*w:(y+1)*w]*k for y in range(h))*k)\n    if name.startswith((\'repeat_axis\', \'mirror_axis\')):\n        axis, mirror = int(name[-1]), name.startswith(\'mirror\')\n        if (h if axis == 0 else w)*2 > 30:\n            return None\n        rows = [b[y*w:(y+1)*w] for y in range(h)]\n        if axis == 0:\n            return _grid(h*2, w, b + b\'\'.join(reversed(rows) if mirror else rows))\n        return _grid(h, w*2, b\'\'.join(row+(row[::-1] if mirror else row) for row in rows))\n    if name.startswith(\'half\'):\n        axis, side, gap = int(name[4]), int(name[6]), int(name[-1])\n        n = h if axis == 0 else w\n        if (n-gap) % 2:\n            return None\n        k = (n-gap)//2\n        if k < 1:\n            return None\n        start = k+gap if side else 0\n        if axis == 0:\n            return _grid(k, w, b[start*w:(start+k)*w])\n        return _grid(h, k, b\'\'.join(b[y*w+start:y*w+start+k] for y in range(h)))\n    if name.startswith(\'crop_color\'):\n        color = int(name[10:])\n        return _crop(s, [i for i, c in enumerate(b) if c == color])\n    raise ValueError(\'Unsupported frozen operation: \' + name)\n\n\nclass PackedGrammar:\n    def __init__(self, train):\n        self.operations = operations(train)\n        self._functions = {}\n        for op in self.operations:\n            name = op.name\n            if name.startswith(\'object_\'):\n                _, bg, same, small, crop = name.split(\'_\')\n                params = (bg, bool(int(same[-1])), bool(int(small[-1])), bool(int(crop[-1])))\n                fn = lambda s, p=params: _object(s, *p)\n            elif name.startswith((\'keep_color\', \'erase_color\')):\n                keep = name.startswith(\'keep\')\n                color = int(name[10:] if keep else name[11:])\n                table = bytes((c if c == color else 0) if keep else (0 if c == color else c)\n                              for c in range(256))\n                fn = lambda s, t=table: s[:2] + s[2:].translate(t)\n            elif name == \'demonstration_color_map\':\n                # The frozen factory stores the learned ten-entry array here.\n                table = op.fn.__defaults__[0].tobytes() + bytes(range(10, 256))\n                fn = lambda s, t=table: s[:2] + s[2:].translate(t)\n            else:\n                fn = lambda s, n=name: _transform(n, s)\n            self._functions[id(op)] = fn\n\n    def apply(self, op, values):\n        fn = self._functions[id(op)]\n        out = []\n        for value in values:\n            result = fn(value)\n            if result is None:\n                return None\n            out.append(result)\n        return tuple(out)\n\n    @staticmethod\n    def clear_cache():\n        _components.cache_clear()\n', 'learner/learned_runner.py': '"""Offline standalone inference process; separate namespace from native. MIT-0."""\nimport argparse,gzip,json,time\nfrom pathlib import Path\nfrom concurrent.futures import ProcessPoolExecutor\nimport numpy as np\nfrom sklearn.tree import DecisionTreeClassifier\nfrom selector import certify,merge\nfrom inference_guard import blind_view,inference_access\n\ndef one(item):\n    name,task,baseline,gate=item\n    DecisionTreeClassifier(max_depth=1,random_state=0).fit(np.array([[0.],[1.]]),[0,1])\n    with inference_access():\n        row=certify(blind_view(task))\n        answer=merge(baseline,row,gate)\n    row[\'task_id\']=name;row[\'selected_attempts\']=answer\n    return name,answer,row\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--challenges\',required=True);p.add_argument(\'--baseline\',required=True)\n    p.add_argument(\'--gate\',required=True);p.add_argument(\'--out\',required=True);p.add_argument(\'--workers\',type=int,default=2);a=p.parse_args()\n    tasks=json.loads(Path(a.challenges).read_text());base=json.loads(Path(a.baseline).read_text());gate=json.loads(Path(a.gate).read_text())\n    assert set(tasks)==set(base)\n    # Query labels/metadata are erased before worker serialization.\n    items=[(n,{\'train\':[dict(input=p[\'input\'],output=p[\'output\']) for p in t[\'train\']],\n               \'test\':[{\'input\':q[\'input\']} for q in t[\'test\']]},base[n],gate) for n,t in sorted(tasks.items())]\n    out=Path(a.out);out.mkdir(exist_ok=True);submission={};cpu=fold_cpu=primitive=0;start=time.perf_counter()\n    with ProcessPoolExecutor(max_workers=a.workers) as pool,gzip.open(out/\'certificates.jsonl.gz\',\'wt\') as f:\n        for n,ans,row in pool.map(one,items,chunksize=1):\n            submission[n]=ans;cpu+=row[\'certificate_cpu_seconds\'];fold_cpu+=row[\'fold_cpu_seconds\'];primitive+=row[\'fold_primitive_calls\']+row[\'full\'][\'work_counts\'][\'primitive_grid_evaluations\']\n            f.write(json.dumps(row,separators=(\',\',\':\'))+\'\\n\');f.flush()\n            if len(submission)%10==0:print(\'learned selector\',len(submission),len(tasks),flush=True)\n    (out/\'submission.json\').write_text(json.dumps(submission,separators=(\',\',\':\'))+\'\\n\')\n    receipt=dict(complete=True,tasks=len(tasks),stage_cpu_seconds=cpu,fold_cpu_seconds=fold_cpu,primitive_calls=primitive,\n                 wall_seconds=time.perf_counter()-start,gate=gate,workers=a.workers,baseline_recomputed=False)\n    (out/\'receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\');print(json.dumps(receipt),flush=True)\nif __name__==\'__main__\':main()\n', 'learner/lazy_grammar.py': '"""Exact lazy evaluation of learned relational trees. MIT-0."""\nimport numpy as np\nfrom solver import pack,unpack\nimport relational\nfrom relational_grammar import RelationalGrammar\nfrom fast_features import FeatureGrid,features\n\n\nclass LazyGrammar(RelationalGrammar):\n    def __init__(self,train):\n        super().__init__(train,features_fn=features)\n        self.feature_counts={\'feature_grids\':0,\'requested_columns\':0,\'computed_columns\':0}\n\n    def primitive(self,program,s):\n        a=relational.transform(unpack(s).astype(np.int16),program[\'transform\'])\n        key=pack(a)\n        if key not in self.feature_cache:\n            self.feature_cache[key]=FeatureGrid(a)\n            self.feature_counts[\'feature_grids\']+=1\n            if len(self.feature_cache)>128:self.feature_cache.popitem(last=False)\n        features=self.feature_cache[key];self.feature_cache.move_to_end(key)\n        out=np.empty(a.size,dtype=np.uint8)\n        def node(tree,indices):\n            if isinstance(tree,int):out[indices]=tree;return\n            name,threshold,left,right=tree\n            self.feature_counts[\'requested_columns\']+=1\n            self.feature_counts[\'computed_columns\']+=int(name not in features.cache)\n            mask=features.column(name)[indices]<=threshold\n            if mask.any():node(left,indices[mask])\n            if (~mask).any():node(right,indices[~mask])\n        node(program[\'tree\'],np.arange(a.size))\n        return pack(out.reshape(a.shape))\n', 'learner/cached_grammar.py': '"""Exact lazy evaluation of learned relational trees. MIT-0."""\nimport numpy as np\nfrom solver import pack,unpack\nimport relational\nfrom relational_grammar import RelationalGrammar\nfrom component_features import FeatureGrid,features\nfrom cached_learner import solve\n\n\nclass CachedGrammar(RelationalGrammar):\n    def __init__(self,train):\n        super().__init__(train,features_fn=features,solve_fn=solve)\n        self.feature_counts={\'feature_grids\':0,\'requested_columns\':0,\'computed_columns\':0}\n\n    def primitive(self,program,s):\n        a=relational.transform(unpack(s).astype(np.int16),program[\'transform\'])\n        key=pack(a)\n        if key not in self.feature_cache:\n            self.feature_cache[key]=FeatureGrid(a)\n            self.feature_counts[\'feature_grids\']+=1\n            if len(self.feature_cache)>128:self.feature_cache.popitem(last=False)\n        features=self.feature_cache[key];self.feature_cache.move_to_end(key)\n        out=np.empty(a.size,dtype=np.uint8)\n        def node(tree,indices):\n            if isinstance(tree,int):out[indices]=tree;return\n            name,threshold,left,right=tree\n            self.feature_counts[\'requested_columns\']+=1\n            self.feature_counts[\'computed_columns\']+=int(name not in features.cache)\n            mask=features.column(name)[indices]<=threshold\n            if mask.any():node(left,indices[mask])\n            if (~mask).any():node(right,indices[~mask])\n        node(program[\'tree\'],np.arange(a.size))\n        return pack(out.reshape(a.shape))\n', 'learner/selector.py': '"""Demonstration-resampled learned/native attempt arbitration. MIT-0."""\nimport time\nfrom anytime import solve as learned_solve\n\n\ndef candidates(row):\n    return row[\'baseline\'][\'candidates\'] + (row.get(\'extension\') or {}).get(\'candidates\', [])\n\n\ndef proposals(row, nq):\n    result = [[] for _ in range(nq)]\n    for c in candidates(row):\n        for qi in range(nq):\n            if len(result[qi]) < 2 and all(c[\'predictions\'][qi] != x[\'grid\'] for x in result[qi]):\n                result[qi].append(dict(grid=c[\'predictions\'][qi],program=c[\'program\']))\n    return result\n\n\ndef certify(task, full=None):\n    start=time.process_time()\n    blind={\'train\':[{\'input\':p[\'input\'],\'output\':p[\'output\']} for p in task[\'train\']],\n           \'test\':[{\'input\':q[\'input\']} for q in task[\'test\']]}\n    full_fresh=full is None\n    if full_fresh: full=learned_solve(blind)\n    n=len(blind[\'train\']); nq=len(blind[\'test\']); choices=proposals(full,nq)\n    folds=[]\n    if n >= 2 and any(choices):\n        for omitted, pair in enumerate(blind[\'train\']):\n            fold={\'train\':blind[\'train\'][:omitted]+blind[\'train\'][omitted+1:],\n                  \'test\':[{\'input\':pair[\'input\']}]+blind[\'test\']}\n            row=learned_solve(fold)\n            # Canonical candidates are distinct as whole prediction tuples.\n            selected=[]\n            for c in candidates(row):\n                if all(c[\'predictions\']!=s[\'predictions\'] for s in selected):selected.append(c)\n                if len(selected)==2:break\n            folds.append(dict(omitted=omitted,row=row,selected=selected,\n                              correct=[c[\'predictions\'][0]==pair[\'output\'] for c in selected]))\n    chosen=[]\n    for qi, options in enumerate(choices):\n        for rank,option in enumerate(options):\n            joint=agreement=exact=0\n            for fold in folds:\n                cs=fold[\'selected\']; flags=fold[\'correct\']\n                joint+=any(ok and c[\'predictions\'][qi+1]==option[\'grid\'] for c,ok in zip(cs,flags))\n                agreement+=any(c[\'predictions\'][qi+1]==option[\'grid\'] for c in cs)\n                exact+=any(flags)\n            option.update(joint=joint,agreement=agreement,fold_exact=exact,demonstrations=n,rank=rank)\n        options.sort(key=lambda c:(-c[\'joint\'],-c[\'agreement\'],c[\'rank\']))\n        chosen.append(options[0] if options else None)\n    rows=[f[\'row\'] for f in folds]\n    return dict(chosen=chosen,choices=choices,folds=folds,full=full,full_fresh=full_fresh,\n                certificate_cpu_seconds=time.process_time()-start,\n                fold_cpu_seconds=sum(r[\'cpu_seconds\'] for r in rows),\n                fold_primitive_calls=sum(r[\'work_counts\'][\'primitive_grid_evaluations\'] for r in rows),\n                fold_constructor_cpu_seconds=sum(r[\'baseline\'][\'constructor_cpu_seconds\'] for r in rows))\n\n\ndef accept(proposal, gate):\n    return bool(proposal and proposal[\'demonstrations\']>=gate[\'min_demonstrations\']\n                and proposal[\'joint\']/proposal[\'demonstrations\']>=gate[\'min_joint_fraction\'])\n\n\ndef merge(baseline,certificate,gate):\n    answers=[]\n    for old,proposal in zip(baseline,certificate[\'chosen\']):\n        new=dict(old)\n        if accept(proposal,gate) and proposal[\'grid\']!=old[\'attempt_1\']:\n            new[\'attempt_2\']=proposal[\'grid\']\n        answers.append(new)\n    return answers\n', 'learner/staged_solver.py': '"""Exact, successive demonstration refinement of a provenance DAG.\nSPDX-License-Identifier: MIT-0\n\nThe grammar is constructed from all demonstrations before search. Only evaluation\nis staged: first demonstrate potential, then refine the productive graph with\neach remaining demonstration, then materialize query values. Every incoming\nedge is retained, including paths whose earlier query evaluation would fail.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nimport time\nimport numpy as np\nfrom solver import pack, unpack, operations, apply\nfrom solver_extended import fit_map\n\n\ndef search(task, depth=2, ops_override=None, apply_fn=apply, terminal_map=True):\n    started = time.perf_counter()\n    train = tuple(pack(np.array(p[\'input\'], dtype=np.uint8)) for p in task[\'train\'])\n    goal = tuple(pack(np.array(p[\'output\'], dtype=np.uint8)) for p in task[\'train\'])\n    queries = tuple(pack(np.array(p[\'input\'], dtype=np.uint8)) for p in task[\'test\'])\n    if not train or depth < 0:\n        raise ValueError(\'At least one demonstration and nonnegative depth required\')\n    ops = operations(task[\'train\']) if ops_override is None else ops_override\n    metrics = Counter()\n    stages = []\n\n    def prune(layers, targets):\n        """Retain precisely ancestors of possible goals on the known examples."""\n        fit_cache = {}\n        terminals = []\n        needed = [set() for _ in layers]\n        for d, layer in enumerate(layers):\n            for values in layer:\n                if values == targets:\n                    lut, extra = None, 0\n                elif terminal_map and d < depth:\n                    if values not in fit_cache:\n                        fit_cache[values] = fit_map(values, targets)\n                    lut, extra = fit_cache[values], 1\n                    if lut is None:\n                        continue\n                else:\n                    continue\n                terminals.append((d, values, lut, extra))\n                needed[d].add(values)\n        for d in range(depth, 0, -1):\n            for values in needed[d]:\n                for parent, _ in layers[d][values]:\n                    needed[d - 1].add(parent)\n        metrics[\'nodes_before_pruning\'] += sum(map(len, layers))\n        metrics[\'edges_before_pruning\'] += sum(len(e) for layer in layers for e in layer.values())\n        kept = [{v: edges for v, edges in layer.items() if v in needed[d]}\n                for d, layer in enumerate(layers)]\n        stages.append({\'demonstrations\': len(targets),\n                       \'nodes_before\': sum(map(len, layers)),\n                       \'nodes_after\': sum(map(len, kept)),\n                       \'edges_after\': sum(len(e) for layer in kept for e in layer.values()),\n                       \'terminals\': len(terminals)})\n        return kept, terminals\n\n    # The coarse DAG records all first-demonstration-defined syntax paths.\n    layers = [{train[:1]: []}]\n    for d in range(1, depth + 1):\n        current = {}\n        for parent in layers[-1]:\n            for i, op in enumerate(ops):\n                metrics[\'pivot_expansions\'] += 1\n                values = apply_fn(op, parent)\n                if values is None:\n                    metrics[\'demonstration_undefined\'] += 1\n                    continue\n                if d == depth and values != goal[:1]:\n                    continue\n                current.setdefault(values, []).append((parent, i))\n        layers.append(current)\n    layers, terminals = prune(layers, goal[:1])\n\n    # Refine only coarse paths that can reach a coarse goal. Distinct additional\n    # demonstration values split a node; all predecessor edges survive the split.\n    for n in range(1, len(train)):\n        refined = [{train[:n + 1]: []} if layers[0] else {}]\n        by_coarse = {train[:n]: [train[:n + 1]]} if layers[0] else {}\n        for d in range(1, depth + 1):\n            current = {}\n            next_by_coarse = {}\n            for coarse_values, edges in layers[d].items():\n                for coarse_parent, i in edges:\n                    for parent in by_coarse.get(coarse_parent, ()):\n                        metrics[\'refinement_expansions\'] += 1\n                        extra = apply_fn(ops[i], parent[n:])\n                        if extra is None:\n                            metrics[\'demonstration_undefined\'] += 1\n                            continue\n                        values = coarse_values + extra\n                        if d == depth and values != goal[:n + 1]:\n                            continue\n                        if values not in current:\n                            current[values] = []\n                            next_by_coarse.setdefault(coarse_values, []).append(values)\n                        current[values].append((parent, i))\n            refined.append(current)\n            by_coarse = next_by_coarse\n        layers, terminals = prune(refined, goal[:n + 1])\n\n    @lru_cache(maxsize=None)\n    def materialize(d, values):\n        metrics[\'materialized_nodes\'] += 1\n        if d == 0:\n            return {queries: ()}\n        outputs = {}\n        for parent, i in layers[d][values]:\n            for query_values, path in materialize(d - 1, parent).items():\n                metrics[\'query_expansions\'] += 1\n                pred = apply_fn(ops[i], query_values)\n                if pred is None:\n                    metrics[\'query_undefined\'] += 1\n                    continue\n                new_path = path + (i,)\n                if pred not in outputs or new_path < outputs[pred]:\n                    outputs[pred] = new_path\n        metrics[\'materialized_query_tuples\'] += len(outputs)\n        return outputs\n\n    solutions = {}\n    for d, values, lut, extra in terminals:\n        for pred, path in materialize(d, values).items():\n            if lut is not None:\n                pred = tuple(pack(lut[unpack(s)]) for s in pred)\n                metrics[\'terminal_query_maps\'] += len(pred)\n            rank = (d + extra, path, extra)\n            if pred not in solutions or rank < solutions[pred][0]:\n                names = [ops[i].name for i in path]\n                if extra:\n                    names += [\'fit_color_map:\' + \',\'.join(map(str, lut.tolist()))]\n                solutions[pred] = rank, names\n    candidates = [{\'predictions\': [unpack(s).tolist() for s in pred], \'program\': info[1]}\n                  for pred, info in sorted(solutions.items(), key=lambda kv: (kv[1][0], kv[0]))]\n    return {\'candidates\': candidates, \'metrics\': dict(metrics), \'stages\': stages,\n            \'seconds\': time.perf_counter() - started, \'depth\': depth, \'max_states\': 0,\n            \'mode\': \'staged\', \'num_operations\': len(ops),\n            \'grammar\': \'successive_demonstration_refinement\'}\n', 'learner/residual.py': '"""Training-calibrated second-slot arbitration. Pure inference, MIT-0."""\nimport math\n\nFEATURES=[\'native_second\',\'learned\',\'base\',\'extra\',\'log_rank\',\'min_depth\',\'log_programs\',\n    \'tree\',\'learned_map\',\'relational\',\'geometric\',\'demonstrations\',\'log_pool\',\n    \'two_joint\',\'two_agree\',\'two_exact\',\'all_joint\',\'all_agree\',\'all_exact\',\n    \'two_joint_count\',\'all_joint_count\',\'no_two_joint\',\'all_joint_perfect\',\n    \'same_shape_input\',\'same_shape_all_targets\',\'ratio_shape_consistent\',\n    \'same_color_set_input\',\'colors_seen_in_targets\',\'input_copy\',\n    \'height\',\'width\',\'log_area\',\'color_count\',\'uniform\',\'native_distinct\',\n    \'two_joint_margin\',\'all_joint_margin\']\n\ndef key(grid):return tuple(tuple(row) for row in grid)\ndef candidates(row):return row[\'baseline\'][\'candidates\']+(row.get(\'extension\') or {}).get(\'candidates\',[])\ndef colors(grid):return set(x for row in grid for x in row)\n\ndef features(task, certificate, native):\n    """Uses only demonstrations, unlabeled inputs, and label-blind certificates."""\n    full=certificate[\'full\'];base=full[\'baseline\'][\'candidates\'];extra=(full.get(\'extension\') or {}).get(\'candidates\',[])\n    train=task[\'train\'];n=len(train);folds=certificate[\'folds\'];answers=[]\n    target_colors=set().union(*(colors(p[\'output\']) for p in train))\n    for qi,q in enumerate(task[\'test\']):\n        first=key(native[qi][\'attempt_1\']);second=key(native[qi][\'attempt_2\'])\n        pool={second:dict(grid=native[qi][\'attempt_2\'],programs=[],ranks=[],base=False,extra=False,native_second=True)}\n        for rank,c in enumerate(base+extra):\n            k=key(c[\'predictions\'][qi])\n            if k==first:continue\n            e=pool.setdefault(k,dict(grid=c[\'predictions\'][qi],programs=[],ranks=[],base=False,extra=False,native_second=False))\n            e[\'programs\'].append(c[\'program\']);e[\'ranks\'].append(rank)\n            e[\'base\']|=rank<len(base);e[\'extra\']|=rank>=len(base)\n        if len(pool)>1:pool.pop(first,None)\n        entries=[]\n        for k,e in pool.items():\n            two_joint=two_agree=two_exact=all_joint=all_agree=all_exact=0\n            for fold in folds:\n                held=train[fold[\'omitted\']][\'output\']; cs=candidates(fold[\'row\']);tw=fold[\'selected\']\n                two_joint+=any(c[\'predictions\'][0]==held and key(c[\'predictions\'][qi+1])==k for c in tw)\n                two_agree+=any(key(c[\'predictions\'][qi+1])==k for c in tw)\n                two_exact+=any(c[\'predictions\'][0]==held for c in tw)\n                all_joint+=any(c[\'predictions\'][0]==held and key(c[\'predictions\'][qi+1])==k for c in cs)\n                all_agree+=any(key(c[\'predictions\'][qi+1])==k for c in cs)\n                all_exact+=any(c[\'predictions\'][0]==held for c in cs)\n            g=e[\'grid\'];h,w=len(g),len(g[0]);ih,iw=len(q[\'input\']),len(q[\'input\'][0]);gc=colors(g)\n            ops=[op for p in e[\'programs\'] for op in p]\n            same_targets=all((h,w)==(len(p[\'output\']),len(p[\'output\'][0])) for p in train)\n            ratios=all(h*len(p[\'input\'])==ih*len(p[\'output\']) and w*len(p[\'input\'][0])==iw*len(p[\'output\'][0]) for p in train)\n            x=[e[\'native_second\'],bool(e[\'programs\']),e[\'base\'],e[\'extra\'],math.log1p(min(e[\'ranks\']) if e[\'ranks\'] else len(base+extra)+1),\n               min(map(len,e[\'programs\'])) if e[\'programs\'] else 4,math.log1p(len(e[\'programs\'])),\n               any(\'tree\' in op for op in ops),any(\'learned\' in op or \'fit_color\' in op for op in ops),\n               any(op.startswith(\'rel\') for op in ops),any(op.startswith((\'rotate\',\'flip\',\'transpose\',\'crop\',\'scale\')) for op in ops),\n               n,math.log1p(len(pool)),two_joint/n,two_agree/n,two_exact/n,all_joint/n,all_agree/n,all_exact/n,\n               two_joint,all_joint,two_joint==0,all_joint==n,(h,w)==(ih,iw),same_targets,ratios,\n               gc==colors(q[\'input\']),gc<=target_colors,k==key(q[\'input\']),h/30,w/30,math.log1p(h*w),len(gc)/10,len(gc)==1,\n               first!=second,0,0]\n            assert len(x)==len(FEATURES)\n            entries.append(dict(grid=g,x=list(map(float,x)),native_second=e[\'native_second\'],\n                order=min(e[\'ranks\']) if e[\'ranks\'] else -1,programs=e[\'programs\'],two_joint=two_joint,all_joint=all_joint))\n        for e in entries:\n            others=[z for z in entries if z is not e]\n            e[\'x\'][-2]=(e[\'two_joint\']-max((z[\'two_joint\'] for z in others),default=0))/n\n            e[\'x\'][-1]=(e[\'all_joint\']-max((z[\'all_joint\'] for z in others),default=0))/n\n        answers.append(entries)\n    return answers\n\ndef confidence(x,model):\n    z=model[\'intercept\']+sum(c*(v-m)/s for v,m,s,c in zip(x,model[\'mean\'],model[\'scale\'],model[\'coefficients\']))\n    return z  # Monotone log-odds; avoids needless sigmoid overflow.\n\ndef rank(entries,model):\n    return sorted(entries,key=lambda e:(-confidence(e[\'x\'],model),not e[\'native_second\'],e[\'order\'],key(e[\'grid\'])))\n\ndef predict_from_features(native,certificate,entries,model):\n    if model[\'kind\']==\'native\':return [dict(a) for a in native]\n    if model[\'kind\']==\'prior\':\n        from selector import merge\n        return merge(native,certificate,model[\'gate\'])\n    return [dict(attempt_1=a[\'attempt_1\'],attempt_2=rank(es,model)[0][\'grid\']) for a,es in zip(native,entries)]\n\ndef predict(task,certificate,native,model):\n    return predict_from_features(native,certificate,features(task,certificate,native),model)\n', 'learner/lazy_residual.py': '"""Exact answer stopping for a frozen linear residual ranker. MIT-0.\n\nBounds include all future fold votes, not just votes for current winners.\nNo model, grammar, candidate or tie-breaking changes are made.\n"""\nimport math\nimport time\nfrom selector import candidates, learned_solve\nfrom residual import features, predict_from_features, confidence\n\n\ndef bounds(entries, n, remaining, model):\n    """Conservative per-candidate logit intervals for arbitrary future folds.\n\nEvery vote can gain 0..remaining. Joint votes cannot exceed agreement votes.\nJoint counts also determine two Boolean features and competitor margins.\n    Exact-held-demo terms (15,18) are identical for every candidate. Cancel\n    their exact real contributions, retaining their maximum magnitudes in\n    the floating-point error budget for each original complete score.\n"""\n    result=[]\n    for e in entries:\n        x=e[\'x\']; lo=x.copy(); hi=x.copy()\n        for i in (13,14,15,16,17,18):hi[i]=min(n,round(x[i]*n)+remaining)/n\n        hi[19]=x[19]+remaining;hi[20]=x[20]+remaining\n        lo[21]=float(x[19]+remaining==0);hi[21]=float(x[19]==0)\n        lo[22]=float(x[20]==n);hi[22]=float(x[20]+remaining==n)\n        others=[z for z in entries if z is not e]\n        tj=max((z[\'two_joint\'] for z in others),default=0)\n        aj=max((z[\'all_joint\'] for z in others),default=0)\n        lo[35]=(x[19]-tj-(remaining if others else 0))/n\n        hi[35]=(x[19]+remaining-tj)/n\n        lo[36]=(x[20]-aj-(remaining if others else 0))/n\n        hi[36]=(x[20]+remaining-aj)/n\n        lows=[];highs=[];maxpad=0.0\n        for dt in range(remaining+1):\n            for da in range(dt,remaining+1):\n                a=lo.copy();b=hi.copy();t=int(x[19])+dt;u=int(x[20])+da\n                a[13]=b[13]=t/n;a[16]=b[16]=u/n\n                a[19]=b[19]=t;a[20]=b[20]=u\n                a[21]=b[21]=float(t==0);a[22]=b[22]=float(u==n)\n                a[14]=(round(x[14]*n)+dt)/n\n                a[17]=(round(x[17]*n)+da)/n\n                a[35]=(t-tj-(remaining if others else 0))/n;b[35]=(t-tj)/n\n                a[36]=(u-aj-(remaining if others else 0))/n;b[36]=(u-aj)/n\n                lower=[];upper=[]\n                for av,bv,c,m,s in zip(a,b,model[\'coefficients\'],model[\'mean\'],model[\'scale\']):\n                    assert av<=bv and s>0 and all(map(math.isfinite,(av,bv,c,m,s)))\n                    v=c*(av-m)/s;w=c*(bv-m)/s\n                    lower.append(min(v,w));upper.append(max(v,w))\n                magnitude=abs(model[\'intercept\'])+sum(max(abs(v),abs(w)) for v,w in zip(lower,upper))\n                assert math.isfinite(magnitude) and magnitude<1e300\n                pad=512*math.ulp(max(1.0,4*magnitude));maxpad=max(maxpad,pad)\n                lows.append(model[\'intercept\']+sum(v for i,v in enumerate(lower) if i not in (15,18))-pad)\n                highs.append(model[\'intercept\']+sum(v for i,v in enumerate(upper) if i not in (15,18))+pad)\n        result.append(dict(lower=min(lows),upper=max(highs),feature_lower=lo,feature_upper=hi,padding=maxpad))\n    return result\n\n\ndef decision(entries,n,remaining,model):\n    if len(entries)==1:return 0,[]\n    if remaining==0:\n        winner=min(range(len(entries)),key=lambda i:(-confidence(entries[i][\'x\'],model),\n                   not entries[i][\'native_second\'],entries[i][\'order\'],\n                   tuple(map(tuple,entries[i][\'grid\']))))\n        return winner,[]\n    intervals=bounds(entries,n,remaining,model)\n    for i,b in enumerate(intervals):\n        if all(i==j or b[\'lower\']>other[\'upper\'] for j,other in enumerate(intervals)):\n            return i,intervals\n    return None,intervals\n\n\ndef certify(task,baseline,model,full=None,fold_provider=None):\n    start=time.process_time()\n    blind={\'train\':[dict(input=p[\'input\'],output=p[\'output\']) for p in task[\'train\']],\n           \'test\':[dict(input=q[\'input\']) for q in task[\'test\']]}\n    if full is None:full=learned_solve(blind)\n    n=len(blind[\'train\']);nq=len(blind[\'test\']);folds=[];trace=[]\n    count=n if n>=2 and candidates(full) else 0\n    for completed in range(count+1):\n        entries=features(blind,dict(full=full,folds=folds),baseline)\n        decisions=[decision(es,n,count-completed,model) for es in entries]\n        trace.append(dict(completed=completed,remaining=count-completed,\n                          winners=[w for w,_ in decisions],intervals=[b for _,b in decisions]))\n        if all(w is not None for w,_ in decisions):break\n        assert completed<count\n        pair=blind[\'train\'][completed]\n        if fold_provider is None:\n            ft=dict(train=blind[\'train\'][:completed]+blind[\'train\'][completed+1:],\n                    test=[dict(input=pair[\'input\'])]+blind[\'test\'])\n            row=learned_solve(ft)\n        else:row=fold_provider(completed)\n        selected=[]\n        for c in candidates(row):\n            if all(c[\'predictions\']!=s[\'predictions\'] for s in selected):selected.append(c)\n            if len(selected)==2:break\n        folds.append(dict(omitted=completed,row=row,selected=selected,\n                          correct=[c[\'predictions\'][0]==pair[\'output\'] for c in selected]))\n    answers=[dict(attempt_1=old[\'attempt_1\'],attempt_2=es[w][\'grid\'])\n             for old,es,(w,_) in zip(baseline,entries,decisions)]\n    return dict(full=full,folds=folds,entries=entries,bound_trace=trace,\n                selected_attempts=answers,completed_folds=len(folds),\n                original_fold_count=count,skipped_folds=count-len(folds),\n                certificate_cpu_seconds=time.process_time()-start,\n                fold_primitive_calls=sum(f[\'row\'][\'work_counts\'][\'primitive_grid_evaluations\'] for f in folds),\n                fold_cpu_seconds=sum(f[\'row\'][\'cpu_seconds\'] for f in folds))\n', 'learner/cached_learner.py': '"""Task-local exact fit reuse; fixed seed and all fit inputs retained. MIT-0."""\nfrom relational import *\n\ndef solve(task, execution=\'deferred\', features_fn=None):\n    startcpu,startwall=time.process_time(),time.perf_counter()\n    task=clean_task(task)\n    stats=Counter();candidates=[];all_proposals=[]\n    cache={}\n    fitted_models={}; fit_counts=Counter()\n    def fitted(X,y,key):\n        fit_counts[\'logical_fit_requests\']+=1\n        if key not in fitted_models:\n            model=DecisionTreeClassifier(max_depth=key[2],random_state=0,criterion=\'entropy\')\n            model.fit(X,y)\n            fitted_models[key]=model\n            fit_counts[\'actual_model_fits\']+=1\n        else:\n            fit_counts[\'reused_model_fits\']+=1\n        return fitted_models[key]\n    def feat(a):\n        key=(a.shape,a.tobytes())\n        if key not in cache:\n            stats[\'feature_grid_calls\']+=1\n            stats[\'feature_cells\']+=a.size\n            cache[key]=(features if features_fn is None else features_fn)(a)\n        return cache[key]\n    train=[(np.array(p[\'input\'],dtype=np.int16),np.array(p[\'output\'],dtype=np.int16)) for p in task[\'train\']]\n    query=[np.array(p[\'input\'],dtype=np.int16) for p in task[\'test\']]\n    for transform_name in (\'identity\',\'crop_zero\',\'crop_mode\'):\n        transformed=[transform(a,transform_name) for a,b in train]\n        if any(a.shape!=b.shape for a,(_,b) in zip(transformed,train)):continue\n        # Identical transform behavior may differ on a query: preserve all names.\n        signature=tuple((a.shape,a.tobytes()) for a in transformed)\n        ftrain=[feat(a) for a in transformed]\n        names,tags=ftrain[0][1:]\n        y=np.concatenate([b.ravel() for a,b in train])\n        boundaries=np.cumsum([0]+[a.size for a in transformed])\n        for group in GROUPS:\n            ix=feature_indices(tags,group)\n            X=np.concatenate([f[0][:,ix] for f in ftrain])\n            for depth in DEPTHS:\n                model=fitted(X,y,(signature,group,depth,-1)); stats[\'models_fit\']+=1\n                predtrain=model.predict(X);stats[\'model_training_cells\']+=len(y)\n                fit=bool(np.array_equal(predtrain,y))\n                preds=None\n                # Eager explicitly materializes every proposal. Deferred delays\n                # query features and predictions until the demonstration fit.\n                if execution==\'eager\' or fit:\n                    preds=[]\n                    for q in query:\n                        tq=transform(q,transform_name)\n                        fq=feat(tq)[0][:,ix]\n                        pq=model.predict(fq).astype(np.int16).reshape(tq.shape).tolist()\n                        stats[\'query_model_grid_calls\']+=1;stats[\'query_model_cells\']+=tq.size\n                        preds.append(pq)\n                proposal={\'transform\':transform_name,\'group\':group,\'depth\':depth,\'fits\':fit,\n                          \'nodes\':int(model.tree_.node_count)}\n                all_proposals.append(proposal)\n                if not fit:continue\n                loo_exact=loo_cells=loo_total=0\n                if len(train)>1:\n                    for i in range(len(train)):\n                        keep=np.ones(len(y),dtype=bool);keep[boundaries[i]:boundaries[i+1]]=False\n                        cv=fitted(X[keep],y[keep],(signature,group,depth,i));stats[\'models_fit\']+=1\n                        got=cv.predict(X[~keep]);truth=y[~keep]\n                        loo_exact+=int(np.array_equal(got,truth));loo_cells+=int((got==truth).sum());loo_total+=len(truth)\n                rank=[-loo_exact,-(loo_cells/max(1,loo_total)),int(model.tree_.node_count),\n                      (\'identity\',\'crop_zero\',\'crop_mode\').index(transform_name),GROUPS.index(group),depth]\n                candidates.append({\'predictions\':preds,\'program\':{\'transform\':transform_name,\'tree\':tree_spec(model,names,ix)},\n                    \'rank\':rank,\'loo_exact\':loo_exact,\'loo_cells\':loo_cells,\'loo_total\':loo_total,\'proposal\':proposal})\n    candidates.sort(key=lambda c:(c[\'rank\'],json.dumps(c[\'predictions\'],separators=(\',\',\':\'))))\n    distinct=[];seen=set()\n    for c in candidates:\n        key=json.dumps(c[\'predictions\'],separators=(\',\',\':\'))\n        if key not in seen: seen.add(key);distinct.append(c)\n    answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for c in distinct:\n            g=c[\'predictions\'][i]\n            if g not in grids:grids.append(g)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return {\'attempts\':answers,\'candidates\':distinct,\'consistent_programs\':candidates,\n            \'proposals\':all_proposals,\'metrics\':dict(stats),\'cpu_seconds\':time.process_time()-startcpu,\n            \'wall_seconds\':time.perf_counter()-startwall,\'execution\':execution,\'fit_counts\':dict(fit_counts)}\n', 'learner/fast_features.py': '"""Exact demand-driven implementation of the frozen relational features. MIT-0.\n\nInput-only; names, int16 conversion and column order match relational.features.\nAn instance belongs to one immutable grid. No task identity or output is used.\n"""\nimport numpy as np\nfrom scipy import ndimage as ndi\n\nPROPERTIES = (\'area\',\'height\',\'width\',\'top\',\'left\',\'bottom\',\'right\',\'row\',\'col\',\n              \'edge\',\'holes\',\'colors\',\'color\',\'fill\',\'perimeter\',\'rankarea\',\n              \'nobjects\',\'rankrow\',\'rankcol\',\'sym_ud\',\'sym_lr\')\nDIRECTIONS = ((-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1))\n\n\ndef schema():\n    names, tags = [], []\n    def add(name, tag): names.append(name); tags.append(tag)\n    for n in (\'color\',\'nonzero\',\'nonbackground\',\'background\'): add(n,\'base\')\n    for n in (\'row\',\'column\',\'bottom\',\'right\',\'row_parity\',\'col_parity\',\'border\',\n              \'diagonal\',\'antidiagonal\',\'h\',\'w\'): add(n,\'position\')\n    for dy in range(-2,3):\n        for dx in range(-2,3):\n            if dy or dx: add(f\'neighbor_{dy}_{dx}\',\'local\')\n    for n in (\'mirror_ud\',\'mirror_lr\',\'mirror_both\',\'transpose\',\'antitranspose\'): add(n,\'local\')\n    for c in range(10):\n        for n,t in ((\'count\',\'color\'),(\'rowcount\',\'rays\'),(\'colcount\',\'rays\'),\n                    (\'inbbox\',\'color\'),(\'bboxdy\',\'color\'),(\'bboxdx\',\'color\'),(\'distance\',\'color\'),\n                    (\'vertical_before\',\'rays\'),(\'vertical_after\',\'rays\'),\n                    (\'horizontal_before\',\'rays\'),(\'horizontal_after\',\'rays\')): add(f\'color{c}_{n}\',t)\n    for conn in (1,2):\n        for prefix in (f\'same{conn}_\',f\'fg_zero{conn}_\',f\'fg_mode{conn}_\'):\n            for name in PROPERTIES: add(prefix+name,\'objects\')\n    for dy,dx in DIRECTIONS:\n        for n in (\'color\',\'distance\',\'diffcolor\',\'diffdistance\'): add(f\'ray{dy}_{dx}_{n}\',\'rays\')\n    return names,tags\n\n\nNAMES,TAGS=schema()\n\n\nclass FeatureGrid:\n    def __init__(self,a):\n        self.a=np.asarray(a,dtype=np.int16)\n        self.h,self.w=self.a.shape\n        self.bg=int(np.bincount(self.a.ravel(),minlength=10).argmax())\n        self.yy,self.xx=np.indices(self.a.shape)\n        self.cache={};self.groups={};self.masks={}\n\n    def column(self,name):\n        if name not in self.cache:\n            value=self.compute(name)\n            # Original add() broadcasts scalars, copies to int16, then flattens.\n            if np.isscalar(value): value=np.full_like(self.a,value)\n            self.cache[name]=np.array(value,dtype=np.int16,copy=True).ravel()\n        return self.cache[name]\n\n    def compute(self,name):\n        a=self.a;h,w=self.h,self.w;y,x=self.yy,self.xx\n        if name==\'color\': return a\n        if name==\'nonzero\': return a!=0\n        if name==\'nonbackground\': return a!=self.bg\n        if name==\'background\': return self.bg\n        if name==\'row\': return y\n        if name==\'column\': return x\n        if name==\'bottom\': return h-1-y\n        if name==\'right\': return w-1-x\n        if name==\'row_parity\': return y%2\n        if name==\'col_parity\': return x%2\n        if name==\'border\': return np.minimum.reduce([y,x,h-1-y,w-1-x])\n        if name==\'diagonal\': return y-x\n        if name==\'antidiagonal\': return y+x\n        if name==\'h\': return h\n        if name==\'w\': return w\n        if name.startswith(\'neighbor_\'):\n            _,dy,dx=name.split(\'_\');dy,dx=int(dy),int(dx)\n            if \'pad\' not in self.groups:self.groups[\'pad\']=np.pad(a,2,constant_values=10)\n            return self.groups[\'pad\'][2+dy:2+dy+h,2+dx:2+dx+w]\n        if name==\'mirror_ud\':return a[::-1]\n        if name==\'mirror_lr\':return a[:,::-1]\n        if name==\'mirror_both\':return a[::-1,::-1]\n        if name==\'transpose\':return a.T if h==w else 10\n        if name==\'antitranspose\':return a[::-1,::-1].T if h==w else 10\n        if name.startswith(\'color\'):\n            head,suffix=name.split(\'_\',1);c=int(head[5:])\n            if c not in self.masks:self.masks[c]=a==c\n            mask=self.masks[c]\n            if suffix==\'count\':return mask.sum()\n            if suffix==\'rowcount\':return np.broadcast_to(mask.sum(1)[:,None],a.shape)\n            if suffix==\'colcount\':return np.broadcast_to(mask.sum(0)[None,:],a.shape)\n            if suffix in (\'inbbox\',\'bboxdy\',\'bboxdx\',\'distance\'):\n                yy,xx=np.nonzero(mask)\n                if not len(yy):return -1\n                if suffix==\'distance\':return ndi.distance_transform_cdt(~mask,metric=\'taxicab\')\n                y0,y1,x0,x1=yy.min(),yy.max(),xx.min(),xx.max()\n                if suffix==\'inbbox\':return (y>=y0)&(y<=y1)&(x>=x0)&(x<=x1)\n                if suffix==\'bboxdy\':return np.maximum(y0-y,0)+np.maximum(y-y1,0)\n                return np.maximum(x0-x,0)+np.maximum(x-x1,0)\n            axis,label=(0,\'vertical\') if suffix.startswith(\'vertical\') else (1,\'horizontal\')\n            if suffix==label+\'_before\':return np.cumsum(mask,axis=axis)-mask\n            if suffix==label+\'_after\':return np.flip(np.cumsum(np.flip(mask,axis=axis),axis=axis),axis=axis)-mask\n        if name.startswith((\'same\',\'fg_\')):\n            prefix,prop=name.rsplit(\'_\',1)\n            # sym_ud and sym_lr are two-token properties.\n            if prop in (\'ud\',\'lr\'):prefix,prop=name.rsplit(\'_\',2)[0],\'sym_\'+prop\n            conn=int(prefix[-1]);kind=prefix[:-1]\n            bg=None if kind==\'same\' else (0 if kind==\'fg_zero\' else self.bg)\n            key=(\'components\',conn,bg)\n            if key not in self.groups:self.groups[key]=self.component_maps(conn,bg)\n            return self.groups[key][prop]\n        if name.startswith(\'ray\'):\n            dy,dx,suffix=name[3:].split(\'_\',2);dy,dx=int(dy),int(dx)\n            key=(\'ray\',dy,dx)\n            if key not in self.groups:self.groups[key]=self.rays(dy,dx)\n            return self.groups[key][suffix]\n        raise KeyError(name)\n\n    def component_maps(self,conn,bg):\n        a=self.a;h,w=self.h,self.w\n        structure=ndi.generate_binary_structure(2,conn)\n        groups=[]\n        for c in (range(10) if bg is None else (None,)):\n            labels,_=ndi.label(a==c if bg is None else a!=bg,structure)\n            for label,sl in enumerate(ndi.find_objects(labels),1):\n                if sl is None:continue\n                sy,sx=sl;mask=labels[sl]==label\n                color=c if bg is None else int(np.bincount(a[sl][mask],minlength=10).argmax())\n                groups.append((sy,sx,mask,color))\n        maps={name:np.zeros_like(a) for name in PROPERTIES}\n        areas=[int(m.sum()) for _,_,m,_ in groups]\n        rows=sorted(set(sy.start for sy,_,_,_ in groups))\n        cols=sorted(set(sx.start for _,sx,_,_ in groups))\n        for sy,sx,mask,c in groups:\n            ah,aw=mask.shape;area=int(mask.sum());gy,gx=np.indices(mask.shape)\n            properties={\'area\':area,\'height\':ah,\'width\':aw,\'top\':sy.start,\'left\':sx.start,\n                \'bottom\':h-sy.stop,\'right\':w-sx.stop,\'row\':gy,\'col\':gx,\n                \'edge\':int(sy.start==0 or sx.start==0 or sy.stop==h or sx.stop==w),\n                \'holes\':int((ndi.binary_fill_holes(mask)&~mask).sum()),\n                \'colors\':len(np.unique(a[sy,sx][mask])),\'color\':c,\'fill\':area==ah*aw,\n                \'perimeter\':int((mask&~ndi.binary_erosion(mask)).sum()),\n                \'rankarea\':sum(z<area for z in areas),\'nobjects\':len(groups),\n                \'rankrow\':rows.index(sy.start),\'rankcol\':cols.index(sx.start),\n                \'sym_ud\':int(np.array_equal(mask,mask[::-1])),\n                \'sym_lr\':int(np.array_equal(mask,mask[:,::-1]))}\n            for name,value in properties.items():\n                view=maps[name][sy,sx];view[mask]=value[mask] if isinstance(value,np.ndarray) else value\n        return maps\n\n    def rays(self,dy,dx):\n        a=self.a;h,w=a.shape\n        rc=np.full_like(a,10);rd=np.full_like(a,31)\n        dc=np.full_like(a,10);dd=np.full_like(a,31)\n        # Whole-grid shifted comparisons preserve the original nearest-first\n        # scan (including its distance-30 cutoff), without Python cell loops.\n        for d in range(1,31):\n            sy,sx=dy*d,dx*d\n            if abs(sy)>=h or abs(sx)>=w:break\n            dst=(slice(max(0,-sy),min(h,h-sy)),slice(max(0,-sx),min(w,w-sx)))\n            src=(slice(max(0,sy),min(h,h+sy)),slice(max(0,sx),min(w,w+sx)))\n            v=a[src]\n            choose=(rc[dst]==10)&(v!=self.bg)\n            rc[dst][choose]=v[choose];rd[dst][choose]=d\n            choose=(dc[dst]==10)&(v!=a[dst])\n            dc[dst][choose]=v[choose];dd[dst][choose]=d\n        return dict(color=rc,distance=rd,diffcolor=dc,diffdistance=dd)\n\n\ndef features(a):\n    grid=FeatureGrid(a)\n    return np.stack([grid.column(name) for name in NAMES],axis=1).astype(np.float32),list(NAMES),list(TAGS)\n', 'learner/grid_cache.py': '"""Task-local, bounded memoization of pure unary grid transformations.\nSPDX-License-Identifier: MIT-0\n\nThe cache stores packed immutable results, including undefined results. It never\nmerges search nodes, chooses programs, or reads labels. Operations are keyed by\nidentity, not name: same-name functions may implement different transformations.\n"""\nfrom collections import Counter\n\n\nclass GridEvaluator:\n    def __init__(self, operations, apply_fn, cache=True, max_entries=200000):\n        self.apply_fn = apply_fn\n        self.operation_ids = {id(op): i for i, op in enumerate(operations)}\n        self.enabled = cache\n        self.max_entries = max_entries\n        self.cache = {}\n        self.counts = Counter()\n\n    def __call__(self, op, values):\n        output = []\n        index = self.operation_ids[id(op)]\n        for packed in values:\n            self.counts[\'logical_grid_requests\'] += 1\n            key = (index, packed)\n            if self.enabled and key in self.cache:\n                self.counts[\'cache_hits\'] += 1\n                value = self.cache[key]\n            else:\n                self.counts[\'primitive_grid_evaluations\'] += 1\n                self.counts[\'primitive_input_cells\'] += packed[0] * packed[1]\n                transformed = self.apply_fn(op, (packed,))\n                value = None if transformed is None else transformed[0]\n                if self.enabled and len(self.cache) < self.max_entries:\n                    self.cache[key] = value\n                elif self.enabled:\n                    self.counts[\'uncached_at_capacity\'] += 1\n            if value is None:\n                return None\n            output.append(value)\n        return tuple(output)\n\n    def metrics(self):\n        return dict(self.counts, cache_entries=len(self.cache),\n                    cache_payload_bytes=sum(len(k[1]) + (0 if v is None else len(v))\n                                            for k, v in self.cache.items()))\n', 'learner/solver_deferred.py': '"""Layered demonstration DAG with demand-driven query evaluation.\nSPDX-License-Identifier: MIT-0\nPure fixed unary grammars only. No beam, syntax-dependent semantics or cost.\nEach training-equivalent node retains all predecessor edges; no representative\nprefix is discarded. Query vectors are materialized only for goal ancestors.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nimport time\nimport numpy as np\nfrom solver import pack,unpack,operations,apply\nfrom solver_extended import fit_map\n\n\ndef search(task,depth=2,ops_override=None,apply_fn=apply,terminal_map=True):\n    started=time.perf_counter()\n    train=tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in task[\'train\'])\n    queries=tuple(pack(np.array(p[\'input\'],dtype=np.uint8)) for p in task[\'test\'])\n    goal=tuple(pack(np.array(p[\'output\'],dtype=np.uint8)) for p in task[\'train\'])\n    ops=operations(task[\'train\']) if ops_override is None else ops_override\n    layers=[{train:[]}];metrics=Counter();terminals=[];fit_cache={}\n\n    def terminal(d,values):\n        if values==goal:return None,0\n        if not terminal_map or d>=depth:return False,0\n        if values not in fit_cache:fit_cache[values]=fit_map(values,goal)\n        lut=fit_cache[values]\n        return (lut,1) if lut is not None else (False,0)\n\n    for d in range(depth+1):\n        if d:\n            current={}\n            for parent in layers[-1]:\n                for i,op in enumerate(ops):\n                    metrics[\'demonstration_expansions\']+=1\n                    values=apply_fn(op,parent)\n                    if values is None:\n                        metrics[\'demonstration_undefined\']+=1;continue\n                    # Final layer requires only training-fitting terminal nodes.\n                    if d==depth and values!=goal:continue\n                    current.setdefault(values,[]).append((parent,i))\n            layers.append(current)\n        for values in layers[d]:\n            lut,extra=terminal(d,values)\n            if lut is not False:terminals.append((d,values,lut,extra))\n        metrics[\'training_nodes\']+=len(layers[d])\n        metrics[\'provenance_edges\']+=sum(map(len,layers[d].values()))\n\n    @lru_cache(maxsize=None)\n    def materialize(d,values):\n        metrics[\'materialized_nodes\']+=1\n        if d==0:return {queries:()}\n        outputs={}\n        for parent,i in layers[d][values]:\n            for q,path in materialize(d-1,parent).items():\n                metrics[\'query_expansions\']+=1\n                pred=apply_fn(ops[i],q)\n                if pred is None:\n                    metrics[\'query_undefined\']+=1;continue\n                p=path+(i,)\n                if pred not in outputs or p<outputs[pred]:outputs[pred]=p\n        metrics[\'materialized_query_tuples\']+=len(outputs)\n        return outputs\n\n    solutions={}\n    for d,values,lut,extra in terminals:\n        for pred,path in materialize(d,values).items():\n            if lut is not None:\n                pred=tuple(pack(lut[unpack(s)]) for s in pred)\n                metrics[\'terminal_query_maps\']+=len(pred)\n            rank=(d+extra,path,extra)\n            if pred not in solutions or rank<solutions[pred][0]:\n                names=[ops[i].name for i in path]\n                if extra:names+=[\'fit_color_map:\'+\',\'.join(map(str,lut.tolist()))]\n                solutions[pred]=(rank,names)\n    candidates=[{\'predictions\':[unpack(s).tolist() for s in pred],\'program\':info[1]}\n                for pred,info in sorted(solutions.items(),key=lambda kv:(kv[1][0],kv[0]))]\n    return {\'candidates\':candidates,\'metrics\':dict(metrics),\'seconds\':time.perf_counter()-started,\n            \'depth\':depth,\'mode\':\'deferred\',\'max_states\':0,\'num_operations\':len(ops),\n            \'grammar\':\'layered_demonstration_dag_terminal_map\' if terminal_map else \'layered_demonstration_dag\',\n            \'layers\':[{\'depth\':d,\'nodes\':len(nodes)} for d,nodes in enumerate(layers)]}\n\n\ndef solve(task,depth=2,ops_override=None,apply_fn=apply,terminal_map=True):\n    result=search(task,depth,ops_override,apply_fn,terminal_map);answers=[]\n    for i,q in enumerate(task[\'test\']):\n        grids=[]\n        for candidate in result[\'candidates\']:\n            grid=candidate[\'predictions\'][i]\n            if grid not in grids:grids.append(grid)\n            if len(grids)==2:break\n        while len(grids)<2:grids.append(q[\'input\'])\n        answers.append({\'attempt_1\':grids[0],\'attempt_2\':grids[1]})\n    return answers,result\n', 'learner/anytime_search.py': '"""Stream sound goals from an increasingly refined provenance DAG. MIT-0.\n\nUnbounded enumeration has the same ranked semantics as staged_solver. Under\ninterruption only already validated candidates survive; search completeness\nand final ranking over unseen programs are explicitly not claimed.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nimport time\nimport numpy as np\nfrom solver import pack, unpack\nfrom solver_extended import fit_map\nfrom adaptive import BudgetExpired\n\n\ndef search(task, depth, ops, evaluator, terminal_map=True):\n    start = time.process_time()\n    train = tuple(pack(np.asarray(p[\'input\'], dtype=np.uint8)) for p in task[\'train\'])\n    goal = tuple(pack(np.asarray(p[\'output\'], dtype=np.uint8)) for p in task[\'train\'])\n    queries = tuple(pack(np.asarray(p[\'input\'], dtype=np.uint8)) for p in task[\'test\'])\n    if not train or depth < 0:\n        raise ValueError(\'Expected demonstrations and nonnegative depth\')\n    layers = [{train[:1]: []}]\n    refined = [{train: []}] + [{} for _ in range(depth)]\n    metrics = Counter()\n    solutions = {}\n    fit_cache = {}\n\n    def terminal(d, values, targets):\n        if values == targets:\n            return None, 0\n        if not terminal_map or d >= depth:\n            return False, 0\n        key = (values, targets)\n        if key not in fit_cache:\n            fit_cache[key] = fit_map(values, targets)\n        lut = fit_cache[key]\n        return (lut, 1) if lut is not None else (False, 0)\n\n    @lru_cache(maxsize=None)\n    def refine(d, coarse):\n        # Called only for a node in an already complete coarse layer. Every\n        # incoming syntax edge is retained; this memo never sees a growing node.\n        metrics[\'refined_nodes\'] += 1\n        if d == 0:\n            return (train,)\n        full_values = {}\n        for parent_coarse, i in layers[d][coarse]:\n            for parent in refine(d-1, parent_coarse):\n                metrics[\'refinement_expansions\'] += 1\n                extra = evaluator(ops[i], parent[1:])\n                if extra is None:\n                    continue\n                values = coarse + extra\n                full_values.setdefault(values, []).append((parent, i))\n        for values, edges in full_values.items():\n            assert values not in refined[d]\n            refined[d][values] = edges\n        return tuple(full_values)\n\n    @lru_cache(maxsize=None)\n    def materialize(d, values):\n        metrics[\'materialized_nodes\'] += 1\n        if d == 0:\n            return {queries: ()}\n        outputs = {}\n        for parent, i in refined[d][values]:\n            for query_values, path in materialize(d-1, parent).items():\n                metrics[\'query_expansions\'] += 1\n                pred = evaluator(ops[i], query_values)\n                if pred is None:\n                    continue\n                path = path + (i,)\n                if pred not in outputs or path < outputs[pred]:\n                    outputs[pred] = path\n        return outputs\n\n    def emit(d, pred, path, lut, extra):\n        if lut is not None:\n            pred = tuple(pack(lut[unpack(s)]) for s in pred)\n            metrics[\'terminal_query_maps\'] += len(pred)\n        rank = (d+extra, path, extra)\n        if pred not in solutions or rank < solutions[pred][0]:\n            names = [ops[i].name for i in path]\n            if extra:\n                names += [\'fit_color_map:\' + \',\'.join(map(str, lut.tolist()))]\n            solutions[pred] = (rank, names)\n            metrics[\'candidate_updates\'] += 1\n\n    # Refine a terminal edge immediately, rather than waiting for the complete\n    # deepest layer. Prefix DAGs are stable; the current layer is never memoized.\n    def inspect(d, parent_coarse, i, coarse_value):\n        lut, extra = terminal(d, coarse_value, goal[:1])\n        if lut is False:\n            return\n        metrics[\'coarse_terminal_edges\'] += 1\n        for parent in refine(d-1, parent_coarse):\n            more = evaluator(ops[i], parent[1:])\n            metrics[\'terminal_refinement_expansions\'] += 1\n            if more is None:\n                continue\n            full = coarse_value + more\n            lut, extra = terminal(d, full, goal)\n            if lut is False:\n                continue\n            metrics[\'full_terminal_edges\'] += 1\n            for q, path in materialize(d-1, parent).items():\n                metrics[\'terminal_query_expansions\'] += 1\n                pred = evaluator(ops[i], q)\n                if pred is not None:\n                    emit(d, pred, path+(i,), lut, extra)\n\n    complete = True\n    try:\n        lut, extra = terminal(0, train, goal)\n        if lut is not False:\n            emit(0, queries, (), lut, extra)\n        for d in range(1, depth+1):\n            current = {}\n            for parent in layers[-1]:\n                for i, op in enumerate(ops):\n                    metrics[\'pivot_expansions\'] += 1\n                    value = evaluator(op, parent)\n                    if value is None:\n                        continue\n                    if d < depth:\n                        current.setdefault(value, []).append((parent, i))\n                    inspect(d, parent, i, value)\n            layers.append(current)\n            metrics[\'completed_layers\'] += 1\n    except BudgetExpired:\n        complete = False\n    candidates = [dict(predictions=[unpack(s).tolist() for s in pred], program=info[1])\n                  for pred, info in sorted(solutions.items(), key=lambda kv: (kv[1][0], kv[0]))]\n    return dict(candidates=candidates, complete=complete, budget_expired=not complete,\n                depth=depth, metrics=dict(metrics), cpu_seconds=time.process_time()-start)\n', 'learner/singleton_residual_runner.py': '"""Standalone singleton residual runtime; explicit native input, frozen weights."""\nimport argparse,gzip,json,time\nfrom pathlib import Path\nfrom concurrent.futures import ProcessPoolExecutor\nimport numpy as np\nfrom sklearn.tree import DecisionTreeClassifier\nfrom singleton_residual import certify\nfrom inference_guard import blind_view,inference_access\ndef one(item):\n    name,task,baseline,model=item\n    DecisionTreeClassifier(max_depth=1,random_state=0).fit(np.array([[0.],[1.]]),[0,1])\n    start=time.process_time()\n    with inference_access():row=certify(blind_view(task),baseline,model)\n    row[\'selector_cpu_seconds\']=time.process_time()-start;row[\'task_id\']=name\n    return name,row[\'selected_attempts\'],row\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--challenges\',required=True);p.add_argument(\'--baseline\',required=True);p.add_argument(\'--gate\',required=True);p.add_argument(\'--out\',required=True);p.add_argument(\'--workers\',type=int,default=2);a=p.parse_args()\n    tasks=json.loads(Path(a.challenges).read_text());base=json.loads(Path(a.baseline).read_text());model=json.loads(Path(a.gate).read_text());assert set(tasks)==set(base)\n    items=[(n,dict(train=[dict(input=p[\'input\'],output=p[\'output\']) for p in t[\'train\']],test=[dict(input=q[\'input\']) for q in t[\'test\']]),base[n],model) for n,t in sorted(tasks.items())]\n    out=Path(a.out);out.mkdir(exist_ok=False);submission={};cpu=fold_cpu=primitive=skipped=0;start=time.perf_counter()\n    with ProcessPoolExecutor(max_workers=a.workers) as pool,gzip.open(out/\'certificates.jsonl.gz\',\'wt\') as f:\n        for n,ans,row in pool.map(one,items,chunksize=1):\n            submission[n]=ans;cpu+=row[\'selector_cpu_seconds\'];fold_cpu+=row[\'fold_cpu_seconds\'];primitive+=row[\'fold_primitive_calls\']+row[\'full\'][\'work_counts\'][\'primitive_grid_evaluations\'];skipped+=row[\'skipped_folds\']\n            f.write(json.dumps(row,separators=(\',\',\':\'))+\'\\n\');f.flush()\n            if len(submission)%20==0:print(\'singleton residual\',len(submission),len(tasks),flush=True)\n    (out/\'submission.json\').write_text(json.dumps(submission,separators=(\',\',\':\'))+\'\\n\')\n    receipt=dict(complete=True,tasks=len(tasks),stage_cpu_seconds=cpu,fold_cpu_seconds=fold_cpu,primitive_calls=primitive,skipped_folds=skipped,wall_seconds=time.perf_counter()-start,model_name=model[\'name\'],workers=a.workers,baseline_recomputed=False)\n    (out/\'receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\');print(json.dumps(receipt),flush=True)\nif __name__==\'__main__\':main()\n', 'learner/object_grammar.py': '"""Finite object/relational grammar for exact staged search. MIT-0.\n\nOnly demonstration pairs construct the grammar. Runtime operations are pure\nfunctions of one packed grid. No identifiers, query labels, stored answers,\nexternal files, libraries beyond NumPy, or nondeterministic ties.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nfrom solver import Operation\nfrom packed_2d import PackedGrammar as BaseGrammar\n\n\ndef background(s, role):\n    return 0 if role == \'zero\' else max(range(10), key=s[2:].count)\n\n\ndef neighbors(p, h, w, diagonal=False):\n    y, x = divmod(p, w)\n    for dy, dx in ((-1, 0), (0, -1), (0, 1), (1, 0)):\n        yy, xx = y + dy, x + dx\n        if 0 <= yy < h and 0 <= xx < w:\n            yield yy*w + xx\n    if diagonal:\n        for dy, dx in ((-1, -1), (-1, 1), (1, -1), (1, 1)):\n            yy, xx = y + dy, x + dx\n            if 0 <= yy < h and 0 <= xx < w:\n                yield yy*w + xx\n\n\ndef normalized_shape(points, w):\n    coords = [divmod(p, w) for p in points]\n    # D4-invariant binary object shape; color and location do not enter.\n    variants = []\n    for swap in (False, True):\n        for sy, sx in ((1, 1), (1, -1), (-1, 1), (-1, -1)):\n            q = [(sy*(x if swap else y), sx*(y if swap else x)) for y, x in coords]\n            y0, x0 = min(y for y, x in q), min(x for y, x in q)\n            variants.append(tuple(sorted((y-y0, x-x0) for y, x in q)))\n    return min(variants)\n\n\n@lru_cache(maxsize=4096)\ndef objects(s, role, kind):\n    h, w, b = s[0], s[1], s[2:]\n    bg = background(s, role)\n    remaining = {p for p, c in enumerate(b) if c != bg}\n    groups = []\n    if kind == \'colors\':\n        groups = [tuple(p for p in sorted(remaining) if b[p] == c)\n                  for c in sorted(set(b)-{bg})]\n    else:\n        while remaining:\n            start = min(remaining)\n            remaining.remove(start)\n            queue, group = [start], [start]\n            while queue:\n                for p in neighbors(queue.pop(), h, w, kind == \'mono8\'):\n                    if p in remaining and (kind == \'fg4\' or b[p] == b[start]):\n                        remaining.remove(p)\n                        queue.append(p)\n                        group.append(p)\n            groups.append(tuple(sorted(group)))\n    result = []\n    for cells in groups:\n        ys, xs = zip(*(divmod(p, w) for p in cells))\n        y0, y1, x0, x1 = min(ys), max(ys), min(xs), max(xs)\n        box = (y1-y0+1)*(x1-x0+1)\n        # Internal empty cells not 4-connected to the bounding-box exterior.\n        empty = {y*w+x for y in range(y0, y1+1) for x in range(x0, x1+1)} - set(cells)\n        border = {p for p in empty if p//w in (y0, y1) or p%w in (x0, x1)}\n        queue = list(border)\n        while queue:\n            for p in neighbors(queue.pop(), h, w):\n                if p in empty and p not in border:\n                    border.add(p)\n                    queue.append(p)\n        shape = normalized_shape(cells, w)\n        result.append((cells, y0, y1, x0, x1, box, len(empty-border),\n                       len({b[p] for p in cells}), shape))\n    return tuple(result)\n\n\n@lru_cache(maxsize=16384)\ndef select(s, role, kind, criterion):\n    objs = objects(s, role, kind)\n    if not objs:\n        return ()\n    frequencies = Counter(o[8] for o in objs)\n    if criterion == \'all\':\n        return objs\n    if criterion == \'rare_shape\':\n        values = [frequencies[o[8]] for o in objs]\n        target = min(values)\n    elif criterion == \'common_shape\':\n        values = [frequencies[o[8]] for o in objs]\n        target = max(values)\n    else:\n        key, direction = criterion.rsplit(\'_\', 1)\n        f = {\'area\': lambda o: len(o[0]), \'box\': lambda o: o[5],\n             \'holes\': lambda o: o[6], \'colors\': lambda o: o[7],\n             \'row\': lambda o: o[1], \'col\': lambda o: o[3],\n             \'density\': lambda o: len(o[0])/o[5]}[key]\n        values = [f(o) for o in objs]\n        target = (min if direction == \'min\' else max)(values)\n    return tuple(o for o, value in zip(objs, values) if value == target)\n\n\ndef object_transform(s, role, kind, criterion, action, color=None):\n    chosen = select(s, role, kind, criterion)\n    if not chosen:\n        return None\n    h, w, b = s[0], s[1], s[2:]\n    bg = background(s, role)\n    out = bytearray([bg])*(h*w) if action in (\'crop\', \'keep\') else bytearray(b)\n    cells = sorted({p for o in chosen for p in o[0]})\n    for p in cells:\n        out[p] = bg if action == \'erase\' else (color if action == \'paint\' else b[p])\n    if action == \'crop\':\n        y0, y1 = cells[0]//w, cells[-1]//w\n        x0, x1 = min(p%w for p in cells), max(p%w for p in cells)\n        return bytes((y1-y0+1, x1-x0+1)) + b\'\'.join(\n            out[y*w+x0:y*w+x1+1] for y in range(y0, y1+1))\n    if action in (\'boxes\', \'outlines\'):\n        for o in chosen:\n            own = b[o[0][0]] if color is None else color\n            for y in range(o[1], o[2]+1):\n                for x in range(o[3], o[4]+1):\n                    if action == \'boxes\' or y in (o[1], o[2]) or x in (o[3], o[4]):\n                        if out[y*w+x] == bg:\n                            out[y*w+x] = own\n    return s[:2] + bytes(out)\n\n\ndef connect(s, role, direction, mode):\n    h, w, b = s[0], s[1], s[2:]\n    bg = background(s, role)\n    out, claims = bytearray(b), {}\n    # Same-colored endpoints extend only through background. Conflicting colors\n    # make an operation undefined, avoiding an arbitrary color-order overwrite.\n    for color in sorted(set(b)-{bg}):\n        points = [divmod(p, w) for p, c in enumerate(b) if c == color]\n        groups = {}\n        for y, x in points:\n            key = y if direction == \'row\' else x if direction == \'col\' else y-x if direction == \'diag\' else y+x\n            groups.setdefault(key, []).append((y, x))\n        for group in groups.values():\n            if len(group) < 2:\n                continue\n            group.sort()\n            if direction == \'row\': group.sort(key=lambda p: p[1])\n            y0, x0 = group[0]; y1, x1 = group[-1]\n            dy, dx = (y1 > y0)-(y1 < y0), (x1 > x0)-(x1 < x0)\n            n = max(abs(y1-y0), abs(x1-x0))\n            for k in range(n+1):\n                p = (y0+k*dy)*w+x0+k*dx\n                if b[p] != bg and b[p] != color:\n                    if mode == \'clear\': return None\n                    continue\n                if p in claims and claims[p] != color: return None\n                claims[p] = color\n    for p, c in claims.items(): out[p] = c\n    return s[:2]+bytes(out)\n\n\ndef holes(s, role, color):\n    h, w, b = s[0], s[1], s[2:]\n    bg = background(s, role)\n    empty = {p for p, c in enumerate(b) if c == bg}\n    reached = {p for p in empty if p//w in (0, h-1) or p%w in (0, w-1)}\n    queue = list(reached)\n    while queue:\n        for p in neighbors(queue.pop(), h, w):\n            if p in empty and p not in reached:\n                reached.add(p); queue.append(p)\n    out = bytearray(b)\n    for p in empty-reached: out[p] = color\n    return s[:2]+bytes(out)\n\n\ndef symmetry(s, role, axis):\n    h, w, b = s[0], s[1], s[2:]\n    if axis in (\'transpose\', \'anti\') and h != w: return None\n    bg, out = background(s, role), bytearray(b)\n    for p, c in enumerate(b):\n        y, x = divmod(p, w)\n        yy, xx = {\'lr\': (y, w-1-x), \'ud\': (h-1-y, x),\n                  \'rot\': (h-1-y, w-1-x), \'transpose\': (x, y),\n                  \'anti\': (w-1-x, h-1-y)}[axis]\n        q = yy*w+xx\n        if c != bg:\n            if b[q] != bg and b[q] != c: return None\n            out[q] = c\n    return s[:2]+bytes(out)\n\n\nclass ObjectGrammar:\n    def __init__(self, train):\n        self.base = BaseGrammar(train)\n        self.operations = list(self.base.operations)\n        self.functions = {}\n        def add(name, fn):\n            op = Operation(\'rel_\'+name, None)\n            self.operations.append(op); self.functions[id(op)] = fn\n        # Color literals are derived only from demonstration output palettes.\n        colors = sorted({c for pair in train for row in pair[\'output\'] for c in row})\n        selectors = (\'area_min\', \'area_max\', \'box_min\', \'box_max\', \'holes_max\',\n                     \'colors_max\', \'rare_shape\', \'common_shape\', \'density_min\',\n                     \'row_min\', \'row_max\', \'col_min\', \'col_max\')\n        paint_selectors = (\'area_min\', \'area_max\', \'holes_max\', \'rare_shape\', \'common_shape\')\n        for role in (\'zero\', \'mode\'):\n            for kind in (\'fg4\', \'mono4\', \'mono8\'):\n                for criterion in selectors:\n                    for action in (\'crop\', \'keep\', \'erase\'):\n                        add(f\'{role}_{kind}_{criterion}_{action}\',\n                            lambda s, r=role, k=kind, c=criterion, a=action:\n                            object_transform(s, r, k, c, a))\n                for criterion in paint_selectors:\n                    for color in colors:\n                        add(f\'{role}_{kind}_{criterion}_paint{color}\',\n                            lambda s, r=role, k=kind, c=criterion, v=color:\n                            object_transform(s, r, k, c, \'paint\', v))\n            for kind in (\'mono4\', \'colors\'):\n                for action in (\'boxes\', \'outlines\'):\n                    add(f\'{role}_{kind}_{action}\', lambda s, r=role, k=kind, a=action:\n                        object_transform(s, r, k, \'all\', a))\n            for direction in (\'row\', \'col\', \'diag\', \'anti\'):\n                for mode in (\'clear\', \'skip\'):\n                    add(f\'{role}_connect_{direction}_{mode}\',\n                        lambda s, r=role, d=direction, m=mode: connect(s, r, d, m))\n            for color in colors:\n                add(f\'{role}_holes{color}\', lambda s, r=role, c=color: holes(s, r, c))\n            for axis in (\'lr\', \'ud\', \'rot\', \'transpose\', \'anti\'):\n                add(f\'{role}_symmetry_{axis}\', lambda s, r=role, a=axis: symmetry(s, r, a))\n        # Imported after this module is initialized to share the pure object\n        # representation without constructor state in primitive evaluation.\n        from constructors import construct, apply_rule, name\n        self.rules = construct(train)\n        for spec in self.rules:\n            add(name(spec), lambda s, spec=spec: apply_rule(s, spec))\n\n    def clear_cache(self):\n        self.base.clear_cache(); objects.cache_clear(); select.cache_clear()\n\n    def apply(self, op, values):\n        fn = self.functions.get(id(op))\n        if fn is None: return self.base.apply(op, values)\n        out = []\n        for s in values:\n            value = fn(s)\n            if value is None: return None\n            out.append(value)\n        return tuple(out)\n', 'learner/constructors.py': '"""Demonstration-induced finite object-color programs. MIT-0.\n\nEach rule maps one input-derived object property to a constant color or to\nidentity. Unseen property values preserve the input object\'s colors. Learned\nrules are retained only after exact replay of every demonstration. The search\nthen treats each immutable rule as one deterministic unary operation.\n"""\nfrom collections import Counter\nimport json\nimport numpy as np\nfrom solver import pack\nfrom object_grammar import objects, background\n\nFEATURES = (\'area\', \'area_rank\', \'box\', \'box_rank\', \'holes\', \'colors\',\n            \'shape_frequency\', \'row_rank\', \'col_rank\', \'density\',\n            \'border_count\', \'aspect\', \'area_parity\', \'symmetric_lr\', \'symmetric_ud\')\n\n\ndef values(s, objs, feature):\n    h,w=s[0],s[1]\n    counts=Counter(o[8] for o in objs)\n    sizes=sorted({len(o[0]) for o in objs}); boxes=sorted({o[5] for o in objs})\n    rows=sorted({o[1] for o in objs}); columns=sorted({o[3] for o in objs})\n    result=[]\n    for o in objs:\n        cells,y0,y1,x0,x1,box,holes,ncolors,shape=o\n        if feature==\'area\': value=len(cells)\n        elif feature==\'area_rank\': value=sizes.index(len(cells))\n        elif feature==\'box\': value=box\n        elif feature==\'box_rank\': value=boxes.index(box)\n        elif feature==\'holes\': value=holes\n        elif feature==\'colors\': value=ncolors\n        elif feature==\'shape_frequency\': value=counts[shape]\n        elif feature==\'row_rank\': value=rows.index(y0)\n        elif feature==\'col_rank\': value=columns.index(x0)\n        elif feature==\'density\':\n            from math import gcd\n            g=gcd(len(cells),box); value=f\'{len(cells)//g}/{box//g}\'\n        elif feature==\'border_count\': value=sum(p//w in (0,h-1) or p%w in (0,w-1) for p in cells)\n        elif feature==\'aspect\': value=(y1-y0>x1-x0)-(y1-y0<x1-x0)\n        elif feature==\'area_parity\': value=len(cells)%2\n        elif feature==\'symmetric_lr\': value=int({(p//w,x0+x1-p%w) for p in cells}=={divmod(p,w) for p in cells})\n        elif feature==\'symmetric_ud\': value=int({(y0+y1-p//w,p%w) for p in cells}=={divmod(p,w) for p in cells})\n        else: raise ValueError(feature)\n        result.append(str(value))\n    return result\n\n\ndef apply_rule(s, spec):\n    role,kind,feature,mapping=spec\n    objs=objects(s,role,kind); out=bytearray(s[2:])\n    for o,value in zip(objs,values(s,objs,feature)):\n        color=mapping.get(value,-1)\n        if color>=0:\n            for p in o[0]: out[p]=color\n    return s[:2]+bytes(out)\n\n\ndef construct(train):\n    pairs=[(pack(np.asarray(p[\'input\'],dtype=np.uint8)),pack(np.asarray(p[\'output\'],dtype=np.uint8))) for p in train]\n    if any(a[:2]!=b[:2] for a,b in pairs): return []\n    rules=[]\n    for role in (\'zero\',\'mode\'):\n        for kind in (\'fg4\',\'mono4\',\'mono8\',\'colors\'):\n            examples=[]; valid=True\n            for a,b in pairs:\n                objs=objects(a,role,kind); labels=[]\n                bg=background(a,role)\n                if any(x==bg and y!=x for x,y in zip(a[2:],b[2:])):\n                    valid=False; break\n                for o in objs:\n                    # Prefer identity whenever observed colors are unchanged;\n                    # otherwise require a uniform target over the entire object.\n                    if all(a[2+p]==b[2+p] for p in o[0]): labels.append(-1)\n                    else:\n                        colors={b[2+p] for p in o[0]}\n                        if len(colors)!=1:valid=False;break\n                        labels.append(next(iter(colors)))\n                if not valid:break\n                examples.append((a,objs,labels))\n            if not valid:continue\n            for feature in FEATURES:\n                allowed={}\n                for a,objs,labels in examples:\n                    for value,label,o in zip(values(a,objs,feature),labels,objs):\n                        # An unchanged monochrome object permits both identity\n                        # and a constant equal to its input color.\n                        options={label}\n                        if label==-1 and len({a[2+p] for p in o[0]})==1:\n                            options.add(a[2+o[0][0]])\n                        allowed[value]=allowed.get(value,options)&options\n                if not allowed or any(not v for v in allowed.values()):continue\n                mapping={k:min(v) for k,v in sorted(allowed.items())}\n                if all(v==-1 for v in mapping.values()):continue\n                spec=[role,kind,feature,mapping]\n                if all(apply_rule(a,spec)==b for a,b in pairs):\n                    rules.append(spec)\n    return rules\n\n\ndef name(spec): return \'learned_object_color:\'+json.dumps(spec,sort_keys=True,separators=(\',\',\':\'))\n', 'learner/gate.json': '{\n  "kind": "logistic",\n  "name": "logistic-C10",\n  "C": 10,\n  "intercept": -5.1173588149289735,\n  "coefficients": [\n    1.1489197108131286,\n    -0.12284874967356074,\n    -0.12284874967356074,\n    0.0,\n    0.4978576919350526,\n    -0.9234623858640394,\n    -0.13959866669776358,\n    0.4404905223491359,\n    -1.0575555451600875,\n    -0.11285641572105298,\n    -0.20866949552446637,\n    0.2743500945143578,\n    0.1591979687937727,\n    0.4413933131602151,\n    -0.9794321967780608,\n    0.418884702730957,\n    0.4492684405570742,\n    -0.7313885022665884,\n    0.9210698658841522,\n    -0.18794188018547672,\n    0.37971751338422405,\n    -0.5871094832192223,\n    -0.27164340032454765,\n    -0.8770605716773927,\n    -0.18111119820881946,\n    0.7376294636525951,\n    0.23025324111897147,\n    -0.42357278082124594,\n    -0.40431654528298133,\n    -0.2641689195149226,\n    -0.7847227032405617,\n    0.8838126610720303,\n    -0.4235309590363814,\n    -0.5613645222807104,\n    0.09646921954685543,\n    0.08057177052882476,\n    1.1057301041545267\n  ],\n  "mean": [\n    0.22803738317757008,\n    0.7794392523364486,\n    0.7794392523364486,\n    0.0,\n    1.4089107743963094,\n    2.020560747663551,\n    0.5423196010881606,\n    0.6785046728971963,\n    0.11588785046728972,\n    0.7757009345794392,\n    0.028037383177570093,\n    3.0785046728971963,\n    2.166061119072715,\n    0.016105919003115265,\n    0.04785046728971962,\n    0.2100934579439253,\n    0.0832710280373832,\n    0.19152647975077886,\n    0.21800623052959506,\n    0.0616822429906542,\n    0.2691588785046729,\n    0.9682242990654205,\n    0.03364485981308411,\n    0.8934579439252337,\n    0.5046728971962616,\n    0.9065420560747663,\n    0.1925233644859813,\n    0.8579439252336448,\n    0.028037383177570093,\n    0.3823052959501576,\n    0.4012461059190047,\n    4.740751558026528,\n    0.3456074766355152,\n    0.029906542056074768,\n    0.994392523364486,\n    -0.16694704049844236,\n    -0.11037383177570088\n  ],\n  "scale": [\n    0.41956684217308526,\n    0.4146247752530547,\n    0.4146247752530547,\n    1.0,\n    1.051260515268927,\n    1.1412190823734378,\n    0.2905605851214497,\n    0.4670504060097438,\n    0.3200903881427256,\n    0.4171198804564753,\n    0.16507964236126818,\n    0.8567519665697575,\n    0.975485297002751,\n    0.10328808775421719,\n    0.16470197279153206,\n    0.37640548879168595,\n    0.23079166714991736,\n    0.3156391857999036,\n    0.38238304527696093,\n    0.4031754470284084,\n    0.7700480886735386,\n    0.1754024109432245,\n    0.18031329185959136,\n    0.3085301384988617,\n    0.4999781635549617,\n    0.2910731121943431,\n    0.39428177565413475,\n    0.34910764298183383,\n    0.16507964236126818,\n    0.15751998750027793,\n    0.17082812368896252,\n    0.8701177688500804,\n    0.1607643981726075,\n    0.1703295065404785,\n    0.07467283871191766,\n    0.3785341244647363,\n    0.2803181426175287\n  ],\n  "features": [\n    "native_second",\n    "learned",\n    "base",\n    "extra",\n    "log_rank",\n    "min_depth",\n    "log_programs",\n    "tree",\n    "learned_map",\n    "relational",\n    "geometric",\n    "demonstrations",\n    "log_pool",\n    "two_joint",\n    "two_agree",\n    "two_exact",\n    "all_joint",\n    "all_agree",\n    "all_exact",\n    "two_joint_count",\n    "all_joint_count",\n    "no_two_joint",\n    "all_joint_perfect",\n    "same_shape_input",\n    "same_shape_all_targets",\n    "ratio_shape_consistent",\n    "same_color_set_input",\n    "colors_seen_in_targets",\n    "input_copy",\n    "height",\n    "width",\n    "log_area",\n    "color_count",\n    "uniform",\n    "native_distinct",\n    "two_joint_margin",\n    "all_joint_margin"\n  ],\n  "iterations": [\n    49\n  ]\n}\n'}
for name,source in learner_sources.items():
    target=bundle/name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source)


In [ ]:
# One uniform native profile per notebook run. Default uses color-conservative.
NATIVE_PROFILE = 'color-conservative'
paths=list(Path('/kaggle/input').glob('**/arc-agi_test_challenges.json'))
assert len(paths)==1,f'Expected one challenge file, got {paths}'
challenges=json.loads(paths[0].read_text())
challenges={name:{'train':[{'input':p['input'],'output':p['output']} for p in task['train']],
                  'test':[{'input':q['input']} for q in task['test']]} for name,task in challenges.items()}
Path('blind-challenges.json').write_text(json.dumps(challenges,separators=(',',':'))+'\n')


In [ ]:
native_predict = color_predict if NATIVE_PROFILE == 'color-conservative' else diversity_predict
assert NATIVE_PROFILE in ('color-conservative','diversity')
native_submission,native_receipt=native_predict(challenges,bundle/'native',profile=NATIVE_PROFILE,timeout=1500,workers=2)
Path('native-submission.json').write_text(json.dumps(native_submission,separators=(',',':'))+'\n')
Path('native-receipt.json').write_text(json.dumps(native_receipt,separators=(',',':'))+'\n')


In [ ]:
subprocess.run([sys.executable,str((bundle/'learner'/'singleton_residual_runner.py').resolve()),
    '--challenges',str(Path('blind-challenges.json').resolve()),'--baseline',str(Path('native-submission.json').resolve()),
    '--gate',str((bundle/'learner'/'gate.json').resolve()),'--out',str(Path('learned-output').resolve()),'--workers','2'],check=True)
Path('submission.json').write_bytes(Path('learned-output/submission.json').read_bytes())
print(json.loads(Path('learned-output/receipt.json').read_text()))
